# ARC-AGI-2: Qwen 3.8 Flash-Next on 4 x L4 (Public LB 83.47)

# Competition Switch and Timer

In [ ]:
import os, re

m = re.search(r"-(\d+)-", os.environ.get("KAGGLE_CONTAINER_NAME", ""))
print("SESSION_ID:", m.group(1) if m else "not found", flush=True)

In [ ]:
import glob, json, os, subprocess, threading, time
from datetime import datetime

TIME_BUDGET_MIN = 11 * 60 + 55
VALIDATION_MODE = True
VALIDATION_TIME_MIN = 80  # E36 gEAN calibration: 80 min from kernel start
VALIDATION_TASKS = 120  # E36: first 120 ARC-AGI-2 TRAINING tasks (calibration, not eval)
CHECK_ATTEMPTS = True

IS_RERUN = os.getenv("KAGGLE_IS_COMPETITION_RERUN") is not None
INPUT_ROOT = "/kaggle/input"
SUBMISSION = "/kaggle/working/submission.json"
WORK = "/tmp/arc_run"

def _kernel_start():
    btime = int(next(l for l in open("/proc/stat") if l.startswith("btime")).split()[1])
    ticks = int(open("/proc/self/stat").read().rsplit(")", 1)[1].split()[19])
    return btime + ticks / os.sysconf("SC_CLK_TCK")

T0 = _kernel_start()
VALIDATING = VALIDATION_MODE and not IS_RERUN
if VALIDATION_MODE and IS_RERUN:
    print("VALIDATION_MODE = True is IGNORED: this is the competition rerun, the hidden test tasks run (set it back to False to avoid confusion)")
_BUDGET_MIN = VALIDATION_TIME_MIN if VALIDATING else TIME_BUDGET_MIN
DEADLINE = T0 + _BUDGET_MIN * 60
_fmt = lambda t: datetime.fromtimestamp(t).strftime("%H:%M:%S")
assert T0 <= time.time() + 5, f"kernel start time {T0} is in the future: cannot set the timer"

_test_name = "arc-agi_test_challenges.json"
_found = sorted(os.path.join(d, _test_name) for d, _, fs in os.walk(INPUT_ROOT) if _test_name in fs)
if len(_found) != 1:
    raise RuntimeError(f"need exactly one {_test_name} under {INPUT_ROOT} (attach the ARC Prize competition data), found {len(_found)}:\n  " + "\n  ".join(_found or ["(none)"]))
TEST_FILE = _found[0]
if VALIDATING:
    TEST_FILE = os.path.join(os.path.dirname(_found[0]), "arc-agi_training_challenges.json")  # E36 calibration on training tasks
    for _f in (TEST_FILE, TEST_FILE.replace("_challenges.json", "_solutions.json")):
        if not os.path.exists(_f):
            raise RuntimeError(f"VALIDATION_MODE: {_f} is missing (the competition data must hold the evaluation tasks and their solutions)")
TEST_TASKS = json.load(open(TEST_FILE))
if VALIDATING and VALIDATION_TASKS > 0:
    TEST_TASKS = dict(list(TEST_TASKS.items())[:VALIDATION_TASKS])
print(f"competition rerun: {IS_RERUN} | " + ("VALIDATION MODE: evaluation" if VALIDATING else "test") + f" tasks: {len(TEST_TASKS)} from {TEST_FILE}")
print(f"kernel started {_fmt(T0)} | now {_fmt(time.time())} | DEADLINE {_fmt(DEADLINE)} ({_BUDGET_MIN // 60} h {_BUDGET_MIN % 60} min after the kernel start)")

_SUB_LOCK = threading.Lock()
_LAST_GOOD = {}
_LAST_REC = {}

def _grid_ok(g):
    return (isinstance(g, list) and 0 < len(g) <= 30 and all(isinstance(r, list) and 0 < len(r) <= 30 and len(r) == len(g[0]) for r in g)
            and all(isinstance(v, int) and not isinstance(v, bool) and 0 <= v <= 9 for r in g for v in r))

_TP = lambda g: [list(r) for r in zip(*g)]
_FL = lambda g: [list(r)[::-1] for r in g]
_BACK = {"": None, "P": None, "T": _TP, "F": _FL, "2": None, "3": None}
_ROUND_ORDER = {"": 0, "2": 1, "P": 1, "T": 2, "3": 2, "F": 3}

def _split_id(rid):
    head, _, var = rid.rpartition("_")
    return (head, var) if head and var in ("P", "T", "F", "2", "3") else (rid, "")

def _back(g, var):
    f = _BACK[var]
    return f(g) if (f and isinstance(g, list) and g and all(isinstance(r, list) for r in g)) else g

def _records():
    for f in glob.glob(f"{WORK}/out_*/*/*/check_attempts.json") + glob.glob(f"{WORK}/parked/*/*/*/check_attempts.json"):
        try:
            _LAST_REC[f] = (json.load(open(os.path.join(os.path.dirname(os.path.dirname(f)), "metadata.json")))["puzzle_id"],
                            json.load(open(f)).get("records") or [])
        except Exception:
            pass
    recs = {}
    for rid, rs in _LAST_REC.values():
        tid, var = _split_id(rid)
        recs.setdefault(tid, []).extend(dict(r, tests=[_back(g, var) for g in (r.get("tests") or [])]) for r in rs)
    return {tid: sorted(rs, key=lambda r: (r["ok"] / max(1, r["n"]), r["pct"]), reverse=True) for tid, rs in recs.items()}

def write_submission(quiet=False):
    with _SUB_LOCK:
        for f in sorted(glob.glob(f"{WORK}/sub_*/submission.json")):
            try:
                _LAST_GOOD[f] = json.load(open(f))
            except Exception:
                pass
        finals = {}
        for content in _LAST_GOOD.values():
            for rid, entries in content.items():
                tid, var = _split_id(rid)
                if isinstance(entries, list):
                    finals.setdefault(tid, []).append((_ROUND_ORDER[var], [_back(e.get("attempt_1"), var) if isinstance(e, dict) else None for e in entries]))
        recs = _records() if CHECK_ATTEMPTS else {}
        out, answered, bad, from_ck = {}, 0, 0, [0, 0]
        for tid, task in TEST_TASKS.items():
            rows, has = [], False
            fs = [a for _, a in sorted(finals.get(tid, []), key=lambda x: x[0])]
            for i in range(len(task["test"])):
                cands = []
                for a in fs:
                    g = a[i] if i < len(a) else None
                    if g is not None and not _grid_ok(g):
                        bad += 1
                    elif g is not None:
                        cands.append((g, "final"))
                cands += [(r["tests"][i], "check") for r in recs.get(tid, []) if i < len(r.get("tests") or []) and _grid_ok(r["tests"][i])]
                a1 = cands[0] if cands else (None, None)
                a2 = next((c for c in cands if c[0] != a1[0]), (None, None))
                from_ck[0] += a1[1] == "check"
                from_ck[1] += a2[1] == "check"
                row = {"attempt_1": a1[0], "attempt_2": a2[0]}
                for k in ("attempt_1", "attempt_2"):
                    has |= row[k] is not None
                    row[k] = row[k] if row[k] is not None else [[0]]
                rows.append(row)
            out[tid] = rows
            answered += has
        tmp = SUBMISSION + ".tmp"
        with open(tmp, "w") as fh:
            json.dump(out, fh)
        os.replace(tmp, SUBMISSION)
    if not quiet:
        print(f"submission.json: {len(out)} tasks, {answered} with an answer" + (f", {bad} malformed answers left out" if bad else "")
              + (f" | from check() records: attempt_1 {from_ck[0]}, attempt_2 {from_ck[1]} test inputs" if CHECK_ATTEMPTS else ""), flush=True)
    return answered, bad

def _skip_later_cells():
    ip = get_ipython()
    if _skip_cell not in ip.input_transformers_cleanup:
        ip.input_transformers_cleanup.append(_skip_cell)

def _skip_cell(lines):
    return ["pass\n"]

STOP_EVERYTHING = ("arcagi2.evaluation.evaluate", "stage_manager.py", "vllm.entrypoints", "VLLM::", "/kaggle/working/prefetch.py")

def _deadline_watch():
    while time.time() < DEADLINE:
        time.sleep(15)
    DEADLINE_HIT.set()
    print(f"\n[{_fmt(time.time())}] DEADLINE reached: writing submission.json, stopping the harness and vLLM, skipping the later cells", flush=True)
    write_submission()
    _skip_later_cells()
    for _ in range(240):
        for pattern in STOP_EVERYTHING:
            subprocess.run(["pkill", "-9", "-f", pattern], capture_output=True)
        if _ == 1:
            write_submission()
        time.sleep(15)

DEADLINE_HIT = threading.Event()
write_submission()
if IS_RERUN or VALIDATING:
    if VALIDATING:
        print(f"NOT the competition rerun, VALIDATION MODE: {len(TEST_TASKS)} evaluation tasks, score at the end of step 7")
    if not any(t.name == "deadline-watch" and t.is_alive() for t in threading.enumerate()):
        threading.Thread(target=_deadline_watch, name="deadline-watch", daemon=True).start()
    print(f"the pipeline runs: steps 0-7; everything stops at {_fmt(DEADLINE)} at the latest")
else:
    _skip_later_cells()
    print("NOT the competition rerun: placeholder submission.json written, ALL LATER CELLS ARE SKIPPED (this is what 'Save Version' should do).\n"
          "To validate here: set VALIDATION_MODE = True, restart the kernel and run all (set it back to False before you submit).")

In [ ]:
# E35 B0: save every text trace (full thinking, no images) to /kaggle/working every 10 min and after the deadline.
import tarfile, threading, time as _t, os as _os, pathlib as _pl
def save_traces():
    src, dst = _pl.Path(WORK), "/kaggle/working/traces.tar.gz"
    if not src.exists(): return
    tmp = dst + ".part"
    with tarfile.open(tmp, "w:gz", compresslevel=6) as t:
        for f in src.rglob("*"):
            if f.is_file() and "images" not in f.parts and f.suffix in (".md", ".json", ".jsonl", ".log", ".txt", ".py"):
                t.add(f, arcname=str(f.relative_to(src)))
    _os.replace(tmp, dst)
    print(f"[TRACES] saved {_os.path.getsize(dst)/2**20:.1f} MiB", flush=True)
def _trace_loop():
    while not DEADLINE_HIT.is_set():
        DEADLINE_HIT.wait(600)
        try: save_traces()
        except Exception as e: print("[TRACES] save failed:", e, flush=True)
    _t.sleep(60)
    try: save_traces()
    except Exception as e: print("[TRACES] final save failed:", e, flush=True)
threading.Thread(target=_trace_loop, name="trace-save", daemon=True).start()
print("[TRACES] saver on: every 10 min + after the deadline")


# Environment and Model

In [ ]:
import os
from pathlib import Path

INPUT_ROOT = "/kaggle/input"
manifests, model_dirs = [], []
for d, _, files in os.walk(INPUT_ROOT):
    fs = set(files)
    if "UTILITY_MANIFEST.json" in fs:
        manifests.append(d)
    if "config.json" in fs and any(f.endswith(".safetensors") for f in fs):
        model_dirs.append(d)

def _one(what, found, hint):
    if len(found) != 1:
        raise RuntimeError(f"step 0: {what}: found {len(found)}, need exactly 1:\n  " + "\n  ".join(sorted(found) or ["(none)"]) + f"\n{hint}")
    return found[0]

_inside = lambda d, top: d == top or d.startswith(top.rstrip("/") + "/")
UTILITY_PATH = _one("the vLLM utility (folder with UTILITY_MANIFEST.json)", manifests,
                    "Attach the utility notebook's output (one version only).")
MODEL_PATH = _one("the model (folder with config.json and .safetensors files)", [d for d in model_dirs if not _inside(d, UTILITY_PATH)],
                  "Attach exactly one model version.")
assert "TEST_FILE" in globals(), "run the START cell first"
COMP_DIR = os.path.dirname(TEST_FILE)
print("utility    :", UTILITY_PATH)
print("model      :", MODEL_PATH)
print("test tasks :", TEST_FILE, f"({len(TEST_TASKS)} tasks)")

PREFETCH_READERS = 24
PREFETCH_LOG = "/kaggle/working/prefetch.log"
NGRAM_SHM = "/dev/shm/flashnext_ngram.bin"
_PREFETCH_PY = r"""
import json, os, re, struct, sys, time, glob
from concurrent.futures import ThreadPoolExecutor
# session 16 (startup test 6): argv = root, readers, log, [n-gram table file in /dev/shm]. With the 4th argument, the n-gram rows of the
# ple-*.safetensors files are written ONCE into that file (rows in global order, layout in <file>.json; read from the network and dropped
# from the page cache right away, so the 51 GB are not held twice); every other file is read into the page cache as before.
root, n, log = sys.argv[1], int(sys.argv[2]), sys.argv[3]
shm = sys.argv[4] if len(sys.argv) > 4 and sys.argv[4] else None
# in the order vLLM loads them (its natural sort, weight_utils._natural_sort_key), each file by many readers at once: when RAM gets
# tight while vLLM pins the n-gram table, Linux drops the OLDEST cached data first, i.e. what vLLM has already loaded (session 11:
# the earlier order "first pieces of every file first" let it drop the starts of files vLLM still needed -> network re-reads)
nat = lambda p: [int(s) if s.isdigit() else s for s in re.split(r"(\d+)", os.path.basename(p))]
files = sorted(glob.glob(os.path.join(root, "*.safetensors")), key=nat)
piece = 256 << 20
ESIZE = {"F8_E4M3": 1, "F8_E5M2": 1, "U8": 1, "I8": 1, "BF16": 2, "F16": 2, "F32": 4}
TORCH = {"F8_E4M3": "float8_e4m3fn", "F8_E5M2": "float8_e5m2", "U8": "uint8", "I8": "int8", "BF16": "bfloat16", "F16": "float16", "F32": "float32"}

def header(f):
    with open(f, "rb") as fh:
        k = struct.unpack("<Q", fh.read(8))[0]
        return json.loads(fh.read(k)), 8 + k

table_jobs, shm_meta = [], None
if shm:
    shards, ple_files = {}, [f for f in files if os.path.basename(f).startswith("ple-")]
    for f in ple_files:
        h, data0 = header(f)
        for name, v in h.items():
            m = re.search(r"ngram_embedding\.shard_(\d+)\.weight$", name)
            if m:
                if int(m.group(1)) in shards:
                    raise SystemExit(f"n-gram shm: shard {m.group(1)} appears twice")
                shards[int(m.group(1))] = (f, data0 + v["data_offsets"][0], v["data_offsets"][1] - v["data_offsets"][0], v["dtype"], v["shape"])
    if not shards:
        raise SystemExit("n-gram shm: no ngram_embedding.shard_N.weight tensors in the ple-*.safetensors files")
    idx = sorted(shards)
    if idx != list(range(len(idx))):
        raise SystemExit(f"n-gram shm: shard numbers are not 0..{len(idx) - 1}")
    dtypes = {shards[i][3] for i in idx}; dims = {tuple(shards[i][4][1:]) for i in idx}
    if len(dtypes) != 1 or len(dims) != 1 or len(next(iter(dims))) != 1:
        raise SystemExit(f"n-gram shm: mixed shard dtypes/shapes {dtypes} {dims}")
    dt = dtypes.pop(); dim = next(iter(dims))[0]; row = dim * ESIZE[dt]
    shard_rows = shards[0][4][0]
    if any(shards[i][4][0] != shard_rows for i in idx[:-1]) or shards[idx[-1]][4][0] > shard_rows:
        raise SystemExit("n-gram shm: the shards do not all have the same number of rows (only the last may be shorter)")
    total_rows = sum(shards[i][4][0] for i in idx)
    size = (total_rows + 65536) * row                     # spare rows (zeros, never touched) for vLLM's per-GPU padding
    shm_meta = {"source": root, "torch_dtype": TORCH[dt], "row_bytes": row, "rows": total_rows, "shard_rows": shard_rows,
                "shards": len(idx), "file_bytes": size, "complete": False}
    old = json.load(open(shm + ".json")) if os.path.exists(shm + ".json") else None
    if old and old.get("complete") and {k: old.get(k) for k in shm_meta if k != "complete"} == {k: v for k, v in shm_meta.items() if k != "complete"} \
            and os.path.exists(shm) and os.path.getsize(shm) == size:
        shm_meta = None                                    # already built in this session
    else:
        free = os.statvfs(os.path.dirname(shm)); free = free.f_bavail * free.f_frsize
        if free < total_rows * row:
            raise SystemExit(f"n-gram shm: {os.path.dirname(shm)} has {free / 1e9:.1f} GB free, the table needs {total_rows * row / 1e9:.1f} GB")
        for p in (shm, shm + ".json"):
            if os.path.exists(p):
                os.remove(p)
        with open(shm, "wb") as fh:
            fh.truncate(size)
        for i in idx:
            f, off, nbytes = shards[i][:3]
            dst = i * shard_rows * row
            for o in range(0, nbytes, 64 << 20):
                table_jobs.append((f, off + o, dst + o, min(64 << 20, nbytes - o)))
    files = [f for f in files if not os.path.basename(f).startswith("ple-")]   # vLLM reads only the tiny non-table tensors there itself

jobs = [("cache", f, o, None, min(piece, os.path.getsize(f) - o)) for f in files for o in range(0, os.path.getsize(f), piece)] + \
       [("table", f, o, d, s) for f, o, d, s in table_jobs]
total, done, t0 = sum(j[4] for j in jobs), [0], time.time()
shm_fd = os.open(shm, os.O_RDWR) if table_jobs else None

def run(j):
    kind, f, off, dst, size = j
    fd = os.open(f, os.O_RDONLY)
    try:
        start, left = off, size
        while left > 0:
            b = os.pread(fd, min(left, 16 << 20), off)
            if not b:
                raise IOError(f"short read {f} at {off}")
            if kind == "table":
                w = os.pwrite(shm_fd, b, dst + (off - start))
                if w != len(b):
                    raise IOError(f"short write to {shm}")
            off += len(b); left -= len(b); done[0] += len(b)
        if kind == "table":
            os.posix_fadvise(fd, start, size, os.POSIX_FADV_DONTNEED)   # the rows are in /dev/shm now: do not keep the network copy
    finally:
        os.close(fd)

with open(log, "w") as L:
    L.write(f"prefetch: {len(files)} files to the page cache"
            + (f" + n-gram table {sum(j[3] for j in table_jobs) / 2**30:.1f} GiB to {shm}" if table_jobs else
               (f" | n-gram table already in {shm}" if shm else "")) + f", {total / 2**30:.1f} GiB, {n} readers\n"); L.flush()
    with ThreadPoolExecutor(n) as ex:
        futs = [ex.submit(run, j) for j in jobs]
        while not all(f.done() for f in futs):
            time.sleep(10)
            L.write(f"{time.time() - t0:6.0f} s  {done[0] / 2**30:6.1f} / {total / 2**30:.1f} GiB  {done[0] / 1e6 / (time.time() - t0):6.0f} MB/s\n"); L.flush()
        for f in futs:
            f.result()
    if shm_meta is not None:
        os.fsync(shm_fd) if shm_fd is not None else None
        shm_meta["complete"] = True
        json.dump(shm_meta, open(shm + ".json", "w"))
        L.write(f"NGRAM SHM DONE: {shm_meta['rows']:,} rows x {shm_meta['row_bytes']} bytes = {shm_meta['rows'] * shm_meta['row_bytes'] / 2**30:.1f} GiB in {shm}\n")
    L.write(f"PREFETCH DONE: {done[0] / 2**30:.1f} GiB in {time.time() - t0:.0f} s ({done[0] / 1e6 / (time.time() - t0):.0f} MB/s)\n")
"""
PREFETCH_AFTER_STEP2 = False
if PREFETCH_READERS and PREFETCH_AFTER_STEP2:
    print(f"model prefetch: starts in step 2, after the utility tars ({PREFETCH_READERS} readers, log {PREFETCH_LOG})")
elif PREFETCH_READERS:
    _old = globals().get("PREFETCH_PROC")
    if _old is not None and _old.poll() is None:
        print("model prefetch: already running (started by an earlier run of this cell), log", PREFETCH_LOG)
    elif os.path.exists(PREFETCH_LOG) and "PREFETCH DONE" in open(PREFETCH_LOG).read():
        print("model prefetch: already done in this session:", open(PREFETCH_LOG).read().strip().splitlines()[-1])
    else:
        import subprocess, sys
        Path("/kaggle/working/prefetch.py").write_text(_PREFETCH_PY)
        PREFETCH_PROC = subprocess.Popen([sys.executable, "/kaggle/working/prefetch.py", MODEL_PATH, str(PREFETCH_READERS), PREFETCH_LOG, NGRAM_SHM or ""],
                                         stdout=subprocess.DEVNULL, stderr=open(PREFETCH_LOG + ".err", "w"), start_new_session=True)
        print(f"model prefetch: reading {MODEL_PATH} into RAM in the background with {PREFETCH_READERS} readers (log {PREFETCH_LOG}); step 6 waits for it")

In [ ]:
import json, os, platform, re, shutil, subprocess, sys
from pathlib import Path

assert all(n in globals() for n in ("UTILITY_PATH", "MODEL_PATH", "COMP_DIR")), "run step 0 first: it finds the utility, the model and the competition data"
print("utility:", UTILITY_PATH, "| model:", MODEL_PATH, "| competition data:", COMP_DIR)
MODEL_KIND   = "flashnext"

KINDS = {
    "flashnext": dict(vllm="0.31.0", tp=4, replicas=1, kv_dtype="fp8",  gpu_util=0.92, expert_parallel=True,  min_gpus=4),
    "gsq27":     dict(vllm="0.27.1", tp=2, replicas=2, kv_dtype="fp8",  gpu_util=0.95, expert_parallel=False, min_gpus=2),
}
assert MODEL_KIND in KINDS, f"MODEL_KIND must be one of {list(KINDS)}"
KIND = KINDS[MODEL_KIND]
assert "PUT-THE" not in UTILITY_PATH + MODEL_PATH, "type the two paths at the top of this cell first"

print("python", sys.version.split()[0], "|", platform.platform())
assert sys.version_info[:2] == (3, 12), "The utility is built for Python 3.12. Use that environment (Session options)."
ALLOW_OTHER_GPU = False
gpus = subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total,driver_version,compute_cap", "--format=csv,noheader"],
                      capture_output=True, text=True).stdout.strip().splitlines()
print("gpus:"); print("\n".join("  " + g for g in gpus))
N_GPUS = len(gpus)
caps = [float(g.split(",")[-1]) for g in gpus]
if min(caps) < 8.9 and not ALLOW_OTHER_GPU:
    raise RuntimeError(f"These GPUs have compute capability {min(caps)}, but this notebook needs 8.9 (L4 or newer). In the Session options choose the accelerator 'GPU L4 x4'.")
if N_GPUS < KIND["min_gpus"]:
    raise RuntimeError(f"{MODEL_KIND} needs at least {KIND['min_gpus']} GPUs, this machine has {N_GPUS}.")
gpu_gb = float(gpus[0].split(",")[2].replace("MiB", "").strip()) / 1024
smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
m = re.search(r"CUDA Version:\s*([\d.]+)", smi)
print("driver supports CUDA:", m.group(1) if m else "unknown")
if m and float(m.group(1)) < 13.0:
    print("WARNING: torch 2.13 in the utility is built for CUDA 13; this driver supports an older CUDA.")

TP = KIND["tp"] if N_GPUS >= KIND["tp"] else N_GPUS
REPLICAS = KIND["replicas"] if N_GPUS >= TP * KIND["replicas"] else 1
KV_DTYPE = KIND["kv_dtype"]
GPU_UTIL = KIND["gpu_util"]
EXPERT_PARALLEL = KIND["expert_parallel"] and TP > 1
print(f"\nkind {MODEL_KIND}: vLLM {KIND['vllm']} | {REPLICAS} copy x TP {TP} | KV cache {KV_DTYPE} | expert parallel {EXPERT_PARALLEL}")

mem_gb = lambda key: int(next(l for l in open("/proc/meminfo") if l.startswith(key)).split()[1]) / 1048576 * 1.073741824
shm = shutil.disk_usage("/dev/shm")
print(f"RAM {mem_gb('MemTotal'):.0f} GB (available {mem_gb('MemAvailable'):.0f}) | CPU cores {os.cpu_count()} | /dev/shm {shm.total / 1e9:.1f} GB | free disk /tmp {shutil.disk_usage('/tmp').free / 1e9:.0f} GB")
if MODEL_KIND == "flashnext":
    print("Flash-Next keeps its 51 GB n-gram table in pinned host memory (split over the GPUs' processes): this needs about 70 GB of free RAM.",
          "OK" if mem_gb("MemAvailable") >= 70 else "<- WARNING: not enough free RAM")
if shm.total < 2e9:
    print("WARNING: /dev/shm is small; vLLM with several GPUs passes data through shared memory. If the server hangs at startup, tell me.")

In [ ]:
import hashlib, shutil, subprocess, sys, tarfile, time
from pathlib import Path

UTIL = Path(UTILITY_PATH)
if not (UTIL / "UTILITY_MANIFEST.json").is_file():
    found = subprocess.run("find /kaggle -maxdepth 6 -name UTILITY_MANIFEST.json 2>/dev/null", shell=True, capture_output=True, text=True).stdout
    raise RuntimeError(f"UTILITY_MANIFEST.json is not in {UTIL}. Manifests found on this machine:\n{found or '(none)'}\nSet UTILITY_PATH in step 1 to the folder that holds it.")
MANIFEST = json.loads((UTIL / "UTILITY_MANIFEST.json").read_text())
WHEELHOUSE = UTIL
print("utility:", UTIL, "\n", {k: MANIFEST.get(k) for k in ("created", "vllm", "torch", "humming", "layout", "pack", "pivotools_commit")})
assert MANIFEST.get("vllm") == KIND["vllm"], (f"This utility holds vLLM {MANIFEST.get('vllm')!r}, but MODEL_KIND {MODEL_KIND!r} needs {KIND['vllm']}. "
                                              "Point UTILITY_PATH at the right utility, or change MODEL_KIND.")
VSITE, ASITE = Path("/tmp/site-vllm"), Path("/tmp/site-arc")

def _warm(path, readers=16, piece=64 << 20):
    from concurrent.futures import ThreadPoolExecutor
    size, t0 = path.stat().st_size, time.time()
    def _read(off):
        fd = os.open(path, os.O_RDONLY)
        try:
            left = min(piece, size - off)
            while left > 0:
                b = os.pread(fd, min(left, 16 << 20), off)
                if not b:
                    raise IOError(f"short read {path} at {off}")
                off += len(b); left -= len(b)
        finally:
            os.close(fd)
    with ThreadPoolExecutor(readers) as ex:
        list(ex.map(_read, range(0, size, piece)))
    print(f"  {path.name}: {size / 1e9:.1f} GB read into RAM in {time.time() - t0:.0f} s ({size / 1e6 / max(time.time() - t0, 1e-3):.0f} MB/s, {readers} readers)", flush=True)

def materialize(name, dest):
    if (dest / ".ready").exists():
        print(f"{name}: already in {dest}"); return
    shutil.rmtree(dest, ignore_errors=True)
    t0 = time.time()
    tar = UTIL / f"{name}.tar"
    if tar.exists():
        dest.mkdir(parents=True)
        _warm(tar)
        _t1 = time.time()
        _r = subprocess.run(["tar", "-xf", str(tar), "-C", str(dest)], capture_output=True, text=True)
        if _r.returncode:
            raise RuntimeError(f"tar failed for {tar} (exit {_r.returncode}): {_r.stderr[-2000:]}")
        print(f"  {tar.name}: unpacked from RAM in {time.time() - _t1:.0f} s", flush=True)
    else:
        shutil.copytree(UTIL / name, dest, symlinks=True)
    (dest / ".ready").write_text("ok")
    print(f"{name}: ready in {dest} after {time.time() - t0:.0f}s")

def offline_install(site, find_links, lock_file, label):
    if (site / ".install_ok").exists():
        print(f"{label}: already installed in {site}"); return
    shutil.rmtree(site, ignore_errors=True); site.mkdir(parents=True)
    t0 = time.time()
    r = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-index", "--find-links", str(find_links), "--requirement", str(lock_file),
                        "--target", str(site), "--upgrade", "--ignore-installed", "--only-binary", ":all:", "--no-compile", "--disable-pip-version-check",
                        "--no-warn-conflicts"], capture_output=True, text=True)
    print(f"{label}: pip exit {r.returncode} after {time.time() - t0:.0f}s")
    if r.returncode:
        print((r.stderr or r.stdout)[-3000:]); raise RuntimeError(f"{label} install failed")
    (site / ".install_ok").write_text("ok")

if MANIFEST.get("layout") == "nopip-sites":
    materialize("site-vllm", VSITE); materialize("site-arc", ASITE)
    for d in [VSITE / "bin", ASITE / "bin", *VSITE.glob("nvidia/*/bin"), *VSITE.glob("nvidia/*/nvvm/bin"), VSITE / "triton" / "backends" / "nvidia" / "bin"]:
        if d.is_dir():
            for f in d.iterdir():
                if f.is_file():
                    f.chmod(f.stat().st_mode | 0o755)
else:
    print("older utility layout: offline install from its wheels (pip with --no-index)")
    offline_install(VSITE, UTIL, UTIL / "requirements.lock", "vLLM environment")
    offline_install(ASITE, UTIL / "arc", UTIL / "arc" / "requirements.arc.lock", "harness environment")

if globals().get("PREFETCH_AFTER_STEP2") and PREFETCH_READERS:
    _old = globals().get("PREFETCH_PROC")
    if _old is not None and _old.poll() is None:
        print("model prefetch: already running, log", PREFETCH_LOG)
    elif os.path.exists(PREFETCH_LOG) and "PREFETCH DONE" in open(PREFETCH_LOG).read():
        print("model prefetch: already done in this session")
    else:
        Path("/kaggle/working/prefetch.py").write_text(_PREFETCH_PY)
        PREFETCH_PROC = subprocess.Popen([sys.executable, "/kaggle/working/prefetch.py", MODEL_PATH, str(PREFETCH_READERS), PREFETCH_LOG, NGRAM_SHM or ""],
                                         stdout=subprocess.DEVNULL, stderr=open(PREFETCH_LOG + ".err", "w"), start_new_session=True)
        PREFETCH_T0 = time.time()
        print(f"model prefetch: started now, {time.time() - T0:.0f} s after the kernel start ({PREFETCH_READERS} readers, log {PREFETCH_LOG})", flush=True)

homes = [p.parent.parent for p in VSITE.glob("nvidia/*/bin/nvcc")]
assert homes, "nvcc not found in the vLLM packages of the utility"
CUDA_HOME = homes[0]
lib = CUDA_HOME / "lib"
if lib.is_dir() and not (CUDA_HOME / "lib64").exists():
    (CUDA_HOME / "lib64").symlink_to(lib, target_is_directory=True)
for d in (CUDA_HOME / "lib", CUDA_HOME / "lib64"):
    if d.is_dir():
        for f in d.glob("lib*.so.*"):
            base = f.name.split(".so.")[0] + ".so"
            if not (d / base).exists():
                try: (d / base).symlink_to(f.name)
                except OSError: pass
print("CUDA_HOME =", CUDA_HOME)

DRIVER_STUB = Path("/tmp/cuda-driver-stub")

def vllm_env(extra=None):
    env = dict(os.environ)
    if DRIVER_STUB.exists():
        env["LIBRARY_PATH"] = os.pathsep.join([str(DRIVER_STUB)] + ([env["LIBRARY_PATH"]] if env.get("LIBRARY_PATH") else []))
    env["PYTHONPATH"] = os.pathsep.join([str(VSITE / "nvidia_cutlass_dsl" / "dsl_packages"), str(VSITE)])
    env["CUDA_HOME"] = env["CUDA_PATH"] = str(CUDA_HOME)
    env["PATH"] = os.pathsep.join([str(CUDA_HOME / "bin"), str(VSITE / "bin"), env.get("PATH", "")])
    env["LD_LIBRARY_PATH"] = os.pathsep.join([str(CUDA_HOME / "lib"), str(VSITE / "torch" / "lib")] + ([env["LD_LIBRARY_PATH"]] if env.get("LD_LIBRARY_PATH") else []))
    env.update({"HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1", "VLLM_NO_USAGE_STATS": "1", "DO_NOT_TRACK": "1",
                "TOKENIZERS_PARALLELISM": "false", "VLLM_ENGINE_READY_TIMEOUT_S": "3600"})
    if extra: env.update(extra)
    return env

def find_libcuda():
    code = "import torch; torch.cuda.init(); print('\\n'.join(l.split()[-1] for l in open('/proc/self/maps') if 'libcuda.so' in l))"
    r = subprocess.run([sys.executable, "-c", code], env=vllm_env(), capture_output=True, text=True)
    paths = [p for p in r.stdout.split() if "libcuda.so" in p]
    if paths:
        return os.path.realpath(paths[0])
    out = subprocess.run("find /usr /lib /lib64 /opt /run -name 'libcuda.so*' 2>/dev/null | head -5", shell=True, capture_output=True, text=True).stdout.split()
    return os.path.realpath(out[0]) if out else None

DRIVER_STUB.mkdir(exist_ok=True)
import glob as _glob
_libs = sorted({os.path.realpath(p) for d in ("/usr/local/nvidia/lib64", "/usr/lib/x86_64-linux-gnu", "/usr/lib64", "/usr/lib")
                for p in _glob.glob(f"{d}/libcuda.so*") if os.path.isfile(p)})
real = _libs[0] if len(_libs) == 1 else find_libcuda()
print("driver library lookup:", "direct (one file in the driver folders)" if len(_libs) == 1 else f"CUDA probe ({len(_libs)} candidates found directly)")
if real is None:
    print("WARNING: could not find the NVIDIA driver library (libcuda). Kernels that compile at first use may fail to link.")
else:
    link = DRIVER_STUB / "libcuda.so"
    if link.is_symlink() or link.exists():
        link.unlink()
    link.symlink_to(real)
    Path("/tmp/t_link.c").write_text("int main(){return 0;}")
    r = subprocess.run(["gcc", "/tmp/t_link.c", "-lcuda", "-o", "/tmp/t_link"], env=vllm_env(), capture_output=True, text=True)
    print(f"driver library {real} -> {link} | linking with -lcuda:", "OK" if r.returncode == 0 else r.stderr[-300:])

chk = "import torch, vllm; print('torch', torch.__version__, '| cuda', torch.version.cuda, '| vllm', vllm.__version__)\nimport humming; print('humming ok')"
STEP2_CHECK = (chk, vllm_env())
print("step 2 import check (torch, vllm, humming): runs in the background when the server starts (step 6)")

In [ ]:
import ast, os, re, subprocess, sys
from pathlib import Path

MODEL_DIR = Path(MODEL_PATH)
MTP_PRESENT = False
need = ["config.json", "model.safetensors.index.json", "tokenizer.json"] + (["patch_vllm_qwen35_embedding.py"] if MODEL_KIND == "gsq27" else [])
missing = [n for n in need if not (MODEL_DIR / n).exists()]
if not MODEL_DIR.exists() or missing:
    found = subprocess.run("find /kaggle/input -maxdepth 7 -name config.json 2>/dev/null | head -20", shell=True, capture_output=True, text=True).stdout
    raise RuntimeError(f"model folder problem: {MODEL_DIR} (missing: {missing}). config.json files found under /kaggle/input:\n{found or '(none)'}\nSet MODEL_PATH in step 1.")
files = sorted(MODEL_DIR.glob("*.safetensors"))
cfg = json.loads((MODEL_DIR / "config.json").read_text())
qc = cfg.get("quantization_config") or (cfg.get("text_config") or {}).get("quantization_config") or {}
print(f"model: {MODEL_DIR}\n  {len(files)} safetensors files, {sum(p.stat().st_size for p in files) / 1e9:.1f} GB | architectures {cfg.get('architectures')} | quant_method {qc.get('quant_method')}")
if MODEL_KIND == "flashnext":
    assert any("Qwen4Exp" in a for a in (cfg.get("architectures") or [])), "this config is not a Flash-Next (Qwen4Exp) model: check MODEL_PATH or MODEL_KIND"
    print("  n-gram table dtype in the config:", (cfg.get("text_config") or cfg).get("ple_embedding_dtype"))

# vLLM Patches: Parallel Layout and KV-Cache Capacity

In [ ]:
DCP_SIZE = 2
COMPRESSED_BLOCK_FACTOR = 8
GPU_UTIL_OVERRIDE = 0.945
SPARSE_INDEXER_MAX_LOGITS_MB = 128
DCP_COMM_BACKEND = "a2a"
MAX_CUDAGRAPH_CAPTURE_SIZE = 512
CUDAGRAPH_CAPTURE_SIZES = [1, 2, 4, 8, 16, 32, 48, 64, 72, 128, 256, 512]
assert max(CUDAGRAPH_CAPTURE_SIZES) == MAX_CUDAGRAPH_CAPTURE_SIZE, (CUDAGRAPH_CAPTURE_SIZES, MAX_CUDAGRAPH_CAPTURE_SIZE)
MAX_BATCHED_TOKENS = 1024
MM_MAX_PIXELS = 2097152
DISABLE_VIDEO = True
EXPANDABLE_SEGMENTS = True
FP8_DENSE_OVERLAY = False
FP8_OVERLAY_PATH = "/kaggle/input/datasets/darrenamadeusmartin/qwen38-flash-next-fp8-dense"
FP8_HC_LMHEAD = True
FP8_HC_OVERLAY_PATH = "/kaggle/input/datasets/darrenamadeusmartin/qwen38-flash-next-fp8-dense-hc"
MAMBA_SSM_FP16_SR = True
MAMBA_SSM_BF16 = False
PROFILE = False
CUSTOM_ALL_REDUCE = False
MEM_TRIMS = True
PREFIX_MATCH_UNIT = None
PREFIX_RETENTION = None
PREFIX_REUSE = True
QAR = None
QAR_CODEC = "int8"
QAR_BLOCK = 32
QAR_MIN_MB = 1.5
SINK_WINDOW = None
SINK_JUMP = ("auto", 49152, 65536)
TRIM_FREE = True
QSA_SKIP_GAP = True
KV_OFFLOAD_GIB = 30
OFFLOAD_EVICT = True
DIAG_ENCODER_MISS = True
DIAG_WEIGHTS = True

import json, os, py_compile, shutil, subprocess, sys
from pathlib import Path

VROOT = Path(os.environ.get("QSA_DCP_VLLM_ROOT", "/tmp/site-vllm/vllm"))
BAK = ".qsa_dcp.bak"
assert (VROOT / "models/qwen4_exp/nvidia/qsa.py").exists(), f"{VROOT} has no Qwen4Exp QSA: run steps 1-3 first (step 2 unpacks vLLM)"

QSA_DCP_PY = '# SPDX-License-Identifier: Apache-2.0\n"""Decode-context-parallel (DCP) kernels for Qwen4Exp QSA sparse attention.\n\nInstalled by patch_vllm_qsa_dcp.py as vllm/models/qwen4_exp/nvidia/ops/qsa_dcp.py.\n\nWith DCP of size W the ranks that share a KV head each store a share of every request\'s tokens: logical position t is\nowned by rank (t // I) % W at local position (t // (I*W)) * I + t % I (vLLM\'s interleave convention,\nI = cp_kv_cache_interleave_size). The indexer still selects logical positions over the whole sequence (its caches are\nnot sharded); each rank keeps the selected tokens it owns, attends with the all-gathered query heads, and returns a\nnormalized FP32 partial plus its log2 LSE. vLLM\'s dcp_combine (cp_lse_ag_out_rs / dcp_a2a_lse_reduce, with\nis_lse_base_on_e=False) merges the partials and hands each rank its own heads; the caller applies the output gate.\n\nVerified on one L4 by simulating W=2 (dev/patches/qsa_dcp/README.md).\n"""\n\nimport torch\n\nfrom vllm.models.qwen4_exp.nvidia.ops import qsa as _qsa\nfrom vllm.triton_utils import tl, triton\n\n\n@triton.jit\ndef _dcp_filter_indices_kernel(\n    src_ptr,\n    dst_ptr,\n    stride_src,\n    stride_dst,\n    rank,\n    TOPK: tl.constexpr,\n    BLOCK: tl.constexpr,\n    W: tl.constexpr,\n    I: tl.constexpr,\n):\n    row = tl.program_id(0)\n    cols = tl.arange(0, BLOCK)\n    count = tl.load(src_ptr + row * stride_src + TOPK)\n    tok = tl.load(src_ptr + row * stride_src + cols, mask=cols < TOPK, other=-1)\n    live = (cols < TOPK) & (cols < count) & (tok >= 0)\n    keep = live & (((tok // I) % W) == rank)\n    keep_i = keep.to(tl.int32)\n    kept_before = tl.cumsum(keep_i, axis=0) - keep_i\n    n_keep = tl.sum(keep_i, axis=0)\n    local = (tok // (I * W)) * I + tok % I\n    # Kept entries are compacted to the front; every other column gets -1. The positions are disjoint and cover all\n    # TOPK columns (the attention kernel trusts -1 for columns past the count inside the last tile).\n    pos = tl.where(keep, kept_before, n_keep + (cols - kept_before))\n    val = tl.where(keep, local, -1)\n    tl.store(dst_ptr + row * stride_dst + pos, val, mask=cols < TOPK)\n    tl.store(dst_ptr + row * stride_dst + TOPK, n_keep)\n\n\ndef dcp_filter_indices(\n    packed: torch.Tensor,\n    rank: int,\n    world: int,\n    interleave: int = 1,\n    out: torch.Tensor | None = None,\n) -> torch.Tensor:\n    """Packed [rows, TOPK + 1] logical selection -> this rank\'s local selection (same layout, new count column)."""\n    rows, width = packed.shape\n    topk = width - 1\n    if out is None:\n        out = torch.empty_like(packed)\n    if rows:\n        _dcp_filter_indices_kernel[(rows,)](\n            packed,\n            out,\n            packed.stride(0),\n            out.stride(0),\n            rank,\n            TOPK=topk,\n            BLOCK=triton.next_power_of_2(topk),\n            W=world,\n            I=interleave,\n            num_warps=4,\n        )\n    return out\n\n\n@triton.jit(do_not_specialize=["num_rows"])\ndef _dcp_local_merge_kernel(\n    partial_output_ptr,\n    partial_lse_ptr,\n    out_ptr,\n    lse_ptr,\n    num_rows,\n    HEAD_DIM: tl.constexpr,\n    NUM_QUERY_HEADS: tl.constexpr,\n    NUM_SPLITS: tl.constexpr,\n    BLOCK_SPLITS: tl.constexpr,\n):\n    # Same math as _qsa_merge_splitk_kernel without the gate: FP32 merged partial and its LSE (log2 units).\n    row = tl.program_id(0)\n    head = tl.program_id(1)\n    s = tl.arange(0, BLOCK_SPLITS)\n    d = tl.arange(0, HEAD_DIM)\n    smask = s < NUM_SPLITS\n    lse = tl.load(\n        partial_lse_ptr + (s * num_rows + row) * NUM_QUERY_HEADS + head,\n        mask=smask,\n        other=-float("inf"),\n    )\n    lse_max = tl.max(lse, axis=0)\n    has = lse_max > -float("inf")\n    w = tl.math.exp2(tl.where(smask & has, lse - lse_max, -float("inf")))\n    den = tl.sum(w, axis=0)\n    srows = s.to(tl.int64) * num_rows + row\n    po = tl.load(\n        partial_output_ptr + (srows[:, None] * NUM_QUERY_HEADS + head) * HEAD_DIM + d[None, :],\n        mask=smask[:, None],\n        other=0.0,\n    )\n    merged = tl.where(den > 0, tl.sum(po * w[:, None], axis=0) / den, 0.0)\n    tl.store(out_ptr + (row * NUM_QUERY_HEADS + head) * HEAD_DIM + d, merged)\n    tl.store(\n        lse_ptr + row * NUM_QUERY_HEADS + head,\n        tl.where(den > 0, lse_max + tl.math.log2(tl.maximum(den, 1e-30)), -float("inf")),\n    )\n\n\ndef qsa_dcp_local_partial(\n    q: torch.Tensor,\n    k_cache: torch.Tensor,\n    v_cache: torch.Tensor,\n    local_indices: torch.Tensor,\n    block_table: torch.Tensor,\n    token_to_req: torch.Tensor,\n    k_scale: float,\n    v_scale: float,\n    num_splits: int,\n    block_n: int,\n    num_warps: int,\n    out: torch.Tensor | None = None,\n    lse: torch.Tensor | None = None,\n) -> tuple[torch.Tensor, torch.Tensor]:\n    """This rank\'s attention over its own tokens: FP32 normalized output [rows, H, D] and log2 LSE [rows, H]."""\n    rows, num_heads, head_dim = q.shape\n    is_fp8 = k_cache.dtype == torch.float8_e4m3fn\n    softmax_scale = (head_dim**-0.5) * (float(k_scale) if is_fp8 else 1.0)\n    output_scale = float(v_scale) if is_fp8 else 1.0\n    topk = local_indices.shape[1] - 1\n    num_tiles = triton.cdiv(topk, block_n)\n    # At least 2 splits so the split-K kernel writes normalized partials + LSE instead of the gated output.\n    splits = max(2, min(num_splits, num_tiles))\n    partial_output = torch.empty(\n        (splits, rows, num_heads, head_dim), dtype=torch.float32, device=q.device\n    )\n    partial_lse = torch.empty((splits, rows, num_heads), dtype=torch.float32, device=q.device)\n    if out is None:\n        out = torch.empty((rows, num_heads, head_dim), dtype=torch.float32, device=q.device)\n    if lse is None:\n        lse = torch.empty((rows, num_heads), dtype=torch.float32, device=q.device)\n    if not rows:\n        return out, lse\n    group_size = num_heads // k_cache.shape[2]\n    _qsa._qsa_sparse_paged_gqa_splitk_kernel[(rows, k_cache.shape[2], splits)](\n        q,\n        k_cache,\n        v_cache,\n        local_indices,\n        block_table,\n        token_to_req,\n        partial_output,\n        partial_lse,\n        q,  # output: unused when NUM_SPLITS > 1\n        softmax_scale,\n        output_scale,\n        q,  # output gate: unused when NUM_SPLITS > 1\n        q.stride(0),\n        q.stride(1),\n        k_cache.stride(0),\n        k_cache.stride(1),\n        k_cache.stride(2),\n        v_cache.stride(0),\n        v_cache.stride(1),\n        v_cache.stride(2),\n        local_indices.stride(0),\n        block_table.stride(0),\n        q.stride(0),\n        q.stride(1),\n        q.stride(0),\n        q.stride(1),\n        rows,\n        k_cache.shape[0],\n        block_table.shape[0],\n        TOPK=topk,\n        PAGE_SIZE=k_cache.shape[1],\n        PAGE_TABLE_WIDTH=block_table.shape[1],\n        GROUP_SIZE=group_size,\n        HEAD_DIM=head_dim,\n        NUM_QUERY_HEADS=num_heads,\n        NUM_SPLITS=splits,\n        NUM_TILES=num_tiles,\n        BLOCK_M=triton.next_power_of_2(group_size),\n        BLOCK_N=block_n,\n        IS_FP8=is_fp8,\n        num_warps=num_warps,\n        num_stages=2,\n    )\n    _dcp_local_merge_kernel[(rows, num_heads)](\n        partial_output,\n        partial_lse,\n        out,\n        lse,\n        rows,\n        HEAD_DIM=head_dim,\n        NUM_QUERY_HEADS=num_heads,\n        NUM_SPLITS=splits,\n        BLOCK_SPLITS=triton.next_power_of_2(splits),\n        num_warps=2,\n        num_stages=1,\n    )\n    return out, lse\n\n\ndef qsa_dcp_combine(\n    all_out: torch.Tensor,\n    all_lse: torch.Tensor,\n    output_gate: torch.Tensor,\n    out: torch.Tensor,\n) -> torch.Tensor:\n    """Single-process reference combine: all_out [W, rows, H, D] FP32, all_lse [W, rows, H] -> gated output.\n\n    In vLLM the ranks call dcp_combine instead; this is used by the one-GPU tests.\n    """\n    _, rows, num_heads, head_dim = all_out.shape\n    gate = output_gate.view(rows, num_heads, head_dim)\n    _qsa._qsa_merge_splitk_kernel[(rows, num_heads)](\n        all_out,\n        all_lse,\n        out,\n        gate,\n        out.stride(0),\n        out.stride(1),\n        gate.stride(0),\n        gate.stride(1),\n        rows,\n        HEAD_DIM=head_dim,\n        NUM_QUERY_HEADS=num_heads,\n        NUM_SPLITS=all_out.shape[0],\n        BLOCK_SPLITS=triton.next_power_of_2(all_out.shape[0]),\n        num_warps=2,\n        num_stages=1,\n    )\n    return out\n'
QSA_EDITS = [('impl supports_dcp', '    supports_dcp: bool = False\n    supports_pcp: bool = False\n', '    supports_dcp: bool = True\n    supports_pcp: bool = False\n'), ('impl DCP refusal', '        if self.dcp_world_size != 1:\n            raise NotImplementedError(\n                "Qwen4Exp QSA does not support decode context parallelism"\n            )\n', '        # Decode context parallel (patch_vllm_qsa_dcp): each DCP rank stores its share of the tokens.\n        self._qsa_dcp_interleave = 1\n        if self.dcp_world_size > 1:\n            from vllm.config import get_current_vllm_config\n\n            self._qsa_dcp_interleave = int(\n                get_current_vllm_config().parallel_config.cp_kv_cache_interleave_size\n            )\n'), ('forward_qsa DCP branch', '        from .ops.qsa import qsa_sparse_paged_attention\n\n        qsa_sparse_paged_attention(\n', '        if self.dcp_world_size > 1:\n            return self._forward_qsa_dcp(\n                query,\n                key_cache,\n                value_cache,\n                logical_indices,\n                attn_metadata,\n                output,\n                token_to_req,\n                output_gate,\n                num_tokens,\n                k_scale,\n                v_scale,\n            )\n\n        from .ops.qsa import qsa_sparse_paged_attention\n\n        qsa_sparse_paged_attention(\n'), ('_forward_qsa_dcp method', '        return output\n\n\nclass Qwen4ExpQSAAttention(Qwen3NextAttention, AttentionLayerBase):\n', "        return output\n\n    # Split-K settings for one DCP rank's partial (12 query heads over its tokens), measured on L4 (sm_89).\n    _QSA_DCP_SPLITS = 4\n    _QSA_DCP_BLOCK_N = 32\n    _QSA_DCP_WARPS = 8\n\n    def _forward_qsa_dcp(\n        self,\n        query: torch.Tensor,\n        key_cache: torch.Tensor,\n        value_cache: torch.Tensor,\n        logical_indices: torch.Tensor,\n        attn_metadata: FlashAttentionMetadata,\n        output: torch.Tensor,\n        token_to_req: torch.Tensor,\n        output_gate: torch.Tensor,\n        num_tokens: int,\n        k_scale: float | None,\n        v_scale: float | None,\n    ) -> torch.Tensor:\n        from vllm.distributed.parallel_state import get_dcp_group\n\n        from .ops.qsa_dcp import dcp_filter_indices, qsa_dcp_local_partial\n\n        group = get_dcp_group()\n        # The DCP group is the pair of TP ranks that share this KV head: gather both ranks' query heads.\n        query_all = group.all_gather(query[:num_tokens].contiguous(), dim=1)\n        local_indices = dcp_filter_indices(\n            logical_indices, self.dcp_rank, self.dcp_world_size, self._qsa_dcp_interleave\n        )\n        partial, lse = qsa_dcp_local_partial(\n            query_all,\n            key_cache,\n            value_cache,\n            local_indices,\n            attn_metadata.block_table,\n            token_to_req,\n            1.0 if k_scale is None else k_scale,\n            1.0 if v_scale is None else v_scale,\n            self._QSA_DCP_SPLITS,\n            self._QSA_DCP_BLOCK_N,\n            self._QSA_DCP_WARPS,\n        )\n        # LSE-weighted merge across the group; returns this rank's own heads [num_tokens, H_local, D].\n        merged = self.dcp_combine(partial, lse, group, is_lse_base_on_e=False)\n        num_heads = query.shape[1]\n        gate = output_gate[:num_tokens].reshape(num_tokens, num_heads, -1)\n        # Same rounding as the unpatched kernel: BF16 attention output, gate applied in FP32.\n        output[:num_tokens] = (\n            merged.to(torch.bfloat16).float() * torch.sigmoid(gate.float())\n        ).to(output.dtype)\n        return output\n\n\nclass Qwen4ExpQSAAttention(Qwen3NextAttention, AttentionLayerBase):\n"), ('layer supports_dcp', 'class Qwen4ExpQSAAttention(Qwen3NextAttention, AttentionLayerBase):\n    """Merged Qwen full-attention owner with a QSA index side branch."""\n\n    supports_dcp = False\n', 'class Qwen4ExpQSAAttention(Qwen3NextAttention, AttentionLayerBase):\n    """Merged Qwen full-attention owner with a QSA index side branch."""\n\n    supports_dcp = True\n'), ('layer context-parallel refusal', '        if (\n            parallel_config.prefill_context_parallel_size > 1\n            or parallel_config.decode_context_parallel_size > 1\n        ):\n            raise NotImplementedError(\n                "Qwen4Exp QSA does not support context parallelism"\n            )\n', '        if parallel_config.prefill_context_parallel_size > 1:\n            raise NotImplementedError(\n                "Qwen4Exp QSA does not support prefill context parallelism"\n            )\n')]
CACHE_EDITS = [('ring buffer not DCP-sharded', '        return CircularBufferSpec(\n            block_size=capacity,\n            num_kv_heads=1,\n            head_size=self.head_size,\n            head_size_v=0,\n            dtype=self.dtype,\n        )\n', '        return CircularBufferSpec(\n            block_size=capacity,\n            num_kv_heads=1,\n            head_size=self.head_size,\n            head_size_v=0,\n            dtype=self.dtype,\n            dcp_sharded=False,\n        )\n'), ('compressed keys not DCP-sharded, wider blocks under DCP', '    def get_kv_cache_spec(self, vllm_config: VllmConfig) -> KVCacheSpec:\n        del vllm_config\n        return MLAAttentionSpec(\n            block_size=self.cache_config.block_size,\n            num_kv_heads=1,\n            head_size=self.head_size,\n            dtype=self.dtype,\n            tokens_per_state=self.compress_ratio,\n        )\n', '    def get_kv_cache_spec(self, vllm_config: VllmConfig) -> KVCacheSpec:\n        # patch_vllm_qsa_dcp: under DCP the main QSA KV is sharded but these compressed keys must stay whole on every\n        # rank (the indexer selects over the whole sequence), so vLLM puts them in their own KV-cache group. In the\n        # packed (block-outermost) layout every block is as wide as the widest group, so with the default block size\n        # this group used 21 full-width blocks per 32K request for ~1/8 of their bytes (measured on 4 x L4: 13.85x\n        # instead of 16.73x). A block of FACTOR x block_size tokens fills the block width (8 -> 3 blocks per request).\n        block_size = self.cache_config.block_size\n        if vllm_config.parallel_config.decode_context_parallel_size > 1:\n            import os\n\n            block_size *= int(os.environ.get("FLASHNEXT_QSA_COMPRESSED_BLOCK_FACTOR", "8"))\n        return MLAAttentionSpec(\n            block_size=block_size,\n            num_kv_heads=1,\n            head_size=self.head_size,\n            dtype=self.dtype,\n            tokens_per_state=self.compress_ratio,\n            dcp_sharded=False,\n        )\n')]
ARGS_EDITS = [('EngineArgs DCP default from the environment', '    decode_context_parallel_size: int = ParallelConfig.decode_context_parallel_size\n', '    decode_context_parallel_size: int = int(os.environ.get("FLASHNEXT_DCP_SIZE", ParallelConfig.decode_context_parallel_size))  # qsa_dcp cell\n'), ('CLI --decode-context-parallel-size default from the environment', '            **parallel_kwargs["decode_context_parallel_size"],\n', '            **{**parallel_kwargs["decode_context_parallel_size"], "default": int(os.environ.get("FLASHNEXT_DCP_SIZE", "1"))},  # qsa_dcp cell\n'), ('CLI --dcp-comm-backend default from the environment', '            **parallel_kwargs["dcp_comm_backend"],\n', '            **{**parallel_kwargs["dcp_comm_backend"], **({"default": os.environ["FLASHNEXT_DCP_COMM"]} if os.environ.get("FLASHNEXT_DCP_COMM") else {})},  # qsa_dcp cell\n'), ("environment overrides applied after the server's command line is parsed", '        engine_args = cls(\n            **{attr: getattr(args, attr) for attr in attrs if hasattr(args, attr)}\n        )\n        return engine_args\n', '        engine_args = cls(\n            **{attr: getattr(args, attr) for attr in attrs if hasattr(args, attr)}\n        )\n        # qsa_dcp cell: overrides that must win over flags the notebook\'s step 6 passes explicitly\n        if os.environ.get("FLASHNEXT_MAX_BATCHED"):\n            engine_args.max_num_batched_tokens = int(os.environ["FLASHNEXT_MAX_BATCHED"])\n        if os.environ.get("FLASHNEXT_MM_PROCESSOR_KWARGS"):\n            engine_args.mm_processor_kwargs = {\n                **(engine_args.mm_processor_kwargs or {}),\n                **json.loads(os.environ["FLASHNEXT_MM_PROCESSOR_KWARGS"]),\n            }\n        if os.environ.get("FLASHNEXT_MAMBA_SSM_DTYPE"):\n            engine_args.mamba_ssm_cache_dtype = os.environ["FLASHNEXT_MAMBA_SSM_DTYPE"]\n        if os.environ.get("FLASHNEXT_LIMIT_MM"):\n            engine_args.limit_mm_per_prompt = {\n                **(engine_args.limit_mm_per_prompt or {}),\n                **json.loads(os.environ["FLASHNEXT_LIMIT_MM"]),\n            }\n        if os.environ.get("FLASHNEXT_PROFILER_DIR"):\n            engine_args.profiler_config = ProfilerConfig(\n                profiler="torch", torch_profiler_dir=os.environ["FLASHNEXT_PROFILER_DIR"]\n            )\n        if os.environ.get("FLASHNEXT_CUSTOM_ALL_REDUCE") == "1":\n            engine_args.disable_custom_all_reduce = False\n        if os.environ.get("FLASHNEXT_KV_OFFLOAD_GIB"):\n            engine_args.kv_offloading_size = float(os.environ["FLASHNEXT_KV_OFFLOAD_GIB"])\n            # vLLM refuses a KV connector with PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True unless the cumem allocator\n            # holds the weights + KV cache (in its pool expandable_segments is off, so the KV cache never moves).\n            engine_args.enable_cumem_allocator = True\n        return engine_args\n'), ('CLI --max-cudagraph-capture-size default from the environment', '            **compilation_kwargs["max_cudagraph_capture_size"],\n', '            **{**compilation_kwargs["max_cudagraph_capture_size"], **({"default": int(os.environ["FLASHNEXT_MAX_CUDAGRAPH"])} if os.environ.get("FLASHNEXT_MAX_CUDAGRAPH") else {})},  # qsa_dcp cell\n'), ('CLI --prefix-cache-retention-interval default from the environment', '            "--prefix-cache-retention-interval",\n            **cache_kwargs["prefix_cache_retention_interval"],\n', '            "--prefix-cache-retention-interval",\n            **{**cache_kwargs["prefix_cache_retention_interval"], **({"default": (None if os.environ["FLASHNEXT_PREFIX_RETENTION"] == "dense" else int(os.environ["FLASHNEXT_PREFIX_RETENTION"]))} if os.environ.get("FLASHNEXT_PREFIX_RETENTION") else {})},  # qsa_dcp cell\n'), ('CLI --prefix-match-unit default from the environment', '            "--prefix-match-unit", **cache_kwargs["prefix_match_unit"]\n', '            "--prefix-match-unit", **{**cache_kwargs["prefix_match_unit"], **({"default": int(os.environ["FLASHNEXT_PREFIX_MATCH_UNIT"])} if os.environ.get("FLASHNEXT_PREFIX_MATCH_UNIT") else {})},  # qsa_dcp cell\n'), ('CLI --profiler-config default from the environment', '        vllm_group.add_argument("--profiler-config", **vllm_kwargs["profiler_config"])\n', '        vllm_group.add_argument("--profiler-config", **{**vllm_kwargs["profiler_config"], **({"default": ProfilerConfig(profiler="torch", torch_profiler_dir=os.environ["FLASHNEXT_PROFILER_DIR"])} if os.environ.get("FLASHNEXT_PROFILER_DIR") else {})})  # qsa_dcp cell\n')]
GDN_EDITS = [('FP16 state SR helper', 'from .op import exp, log\n', "from .op import exp, log\n\n\n@triton.jit\ndef _qsa_sr_hash(x):  # PCG-style integer hash (FP16 state stochastic rounding, patch_vllm_qsa_dcp)\n    s = x * 747796405 + 2891336453\n    w = ((s >> ((s >> 28) + 4)) ^ s) * 277803737\n    return (w >> 22) ^ w\n\n\n@triton.jit\ndef _qsa_sr_fp16(b_h, elem, b_v):\n    # Unbiased FP32 -> FP16 rounding of the recurrent state: add 13 random bits below FP16's mantissa, clear them.\n    bits = b_h.to(tl.uint32, bitcast=True)\n    key = elem.to(tl.uint32) ^ ((bits >> 13) * 0x9E3779B9) ^ (b_v.to(tl.uint32, bitcast=True)[:, None] * 0x85EBCA6B)\n    bits = (bits + (_qsa_sr_hash(key) & 0x1FFF)) & 0xFFFFE000\n    return bits.to(tl.float32, bitcast=True).to(tl.float16)\n"), ('FP16 state SR: fused_recurrent fwd, in-place state', '\n                p_ht = p_ht + i_hv * V * K + o_v[:, None] * K + o_k[None, :]\n                tl.store(p_ht, b_h.to(p_ht.dtype.element_ty), mask=mask_h)\n', '\n                p_ht = p_ht + i_hv * V * K + o_v[:, None] * K + o_k[None, :]\n                if p_ht.dtype.element_ty == tl.float16:\n                    tl.store(p_ht, _qsa_sr_fp16(b_h, i_hv * V * K + o_v[:, None] * K + o_k[None, :], b_v), mask=mask_h)\n                else:\n                    tl.store(p_ht, b_h.to(p_ht.dtype.element_ty), mask=mask_h)\n'), ('FP16 state SR: fused_recurrent fwd, per-token states', '\n            p_ht = p_ht + i_hv * V * K + o_v[:, None] * K + o_k[None, :]\n            tl.store(p_ht, b_h.to(p_ht.dtype.element_ty), mask=mask_h)\n', '\n            p_ht = p_ht + i_hv * V * K + o_v[:, None] * K + o_k[None, :]\n            if p_ht.dtype.element_ty == tl.float16:\n                tl.store(p_ht, _qsa_sr_fp16(b_h, i_hv * V * K + o_v[:, None] * K + o_k[None, :], b_v), mask=mask_h)\n            else:\n                tl.store(p_ht, b_h.to(p_ht.dtype.element_ty), mask=mask_h)\n'), ('FP16 state SR: packed decode', '\n    p_ht = p_ht + i_hv * V * K + o_v[:, None] * K + o_k[None, :]\n    tl.store(p_ht, b_h.to(p_ht.dtype.element_ty), mask=mask_h)\n', '\n    p_ht = p_ht + i_hv * V * K + o_v[:, None] * K + o_k[None, :]\n    if p_ht.dtype.element_ty == tl.float16:\n        tl.store(p_ht, _qsa_sr_fp16(b_h, i_hv * V * K + o_v[:, None] * K + o_k[None, :], b_v), mask=mask_h)\n    else:\n        tl.store(p_ht, b_h.to(p_ht.dtype.element_ty), mask=mask_h)\n')]
HC_EDITS = [('FP8 hyper-connection: quant config helper', 'from .ops.cute_dsl.hc_down_silu import MAX_FUSED_M, hc_down_silu\n', 'from .ops.cute_dsl.hc_down_silu import MAX_FUSED_M, hc_down_silu\n\n\ndef _flashnext_fp8_hc_quant_config():\n    # FP8 hyper-connection projections (Kaggle cell FP8_HC_LMHEAD): use the checkpoint\'s quant config.\n    import os\n\n    if os.environ.get("FLASHNEXT_FP8_HC") != "1":\n        return None\n    from vllm.config import get_current_vllm_config\n\n    return get_current_vllm_config().quant_config\n'), ('FP8 hyper-connection: input_mix_weight_down_block_inject', '                quant_config=None,\n                prefix=maybe_prefix(prefix, "input_mix_weight_down_block_inject"),\n', '                quant_config=_flashnext_fp8_hc_quant_config(),\n                prefix=maybe_prefix(prefix, "input_mix_weight_down_block_inject"),\n'), ('FP8 hyper-connection: input_mix_weight_down', '                quant_config=None,\n                prefix=maybe_prefix(prefix, "input_mix_weight_down"),\n', '                quant_config=_flashnext_fp8_hc_quant_config(),\n                prefix=maybe_prefix(prefix, "input_mix_weight_down"),\n'), ('FP8 hyper-connection: input_mix_weight_up', '            quant_config=None,\n            prefix=maybe_prefix(prefix, "input_mix_weight_up"),\n', '            quant_config=_flashnext_fp8_hc_quant_config(),\n            prefix=maybe_prefix(prefix, "input_mix_weight_up"),\n')]
LMHEAD_EDITS = [('FP8 lm_head', '        self.lm_head = ParallelLMHead(\n            config.vocab_size,\n            config.hidden_size,\n            prefix=maybe_prefix(prefix, "lm_head"),\n        )\n', '        import os\n\n        self.lm_head = ParallelLMHead(\n            config.vocab_size,\n            config.hidden_size,\n            # FP8 lm_head (Kaggle cell FP8_HC_LMHEAD): use the checkpoint\'s quant config.\n            quant_config=(\n                vllm_config.quant_config\n                if os.environ.get("FLASHNEXT_FP8_HC") == "1"\n                else None\n            ),\n            prefix=maybe_prefix(prefix, "lm_head"),\n        )\n')]
DIAG_WEIGHTS_V2_EDITS = [('log GPU memory per model part after loading (runner V2)', '        logger.info(\n            "Model loading took %s GiB memory and %.6f seconds",\n            format_gib(m.consumed_memory),\n            time_after_load - time_before_load,\n        )\n', '        logger.info(\n            "Model loading took %s GiB memory and %.6f seconds",\n            format_gib(m.consumed_memory),\n            time_after_load - time_before_load,\n        )\n        if __import__("os").environ.get("FLASHNEXT_DIAG_WEIGHTS") == "1":  # qsa_dcp cell: per-module GPU memory (log only)\n            import collections as _c, re as _re, torch as _t\n            _mdl = getattr(self, "model", None)\n            _g, _seen, _tot = _c.defaultdict(int), set(), 0\n            def _add(_name, _x):\n                global_ok = isinstance(_x, _t.Tensor) and _x.is_cuda\n                if not global_ok:\n                    return 0\n                _s = _x.untyped_storage()\n                if _s.data_ptr() in _seen:\n                    return 0\n                _seen.add(_s.data_ptr())\n                _g[_re.sub(r"\\.\\d+(?=\\.|$)", ".N", _name)] += _s.nbytes()\n                return _s.nbytes()\n            if _mdl is not None:\n                for _n, _p in list(_mdl.named_parameters(remove_duplicate=False)) + list(_mdl.named_buffers(remove_duplicate=False)):\n                    _tot += _add(_n, _p)\n                for _mn, _mod in _mdl.named_modules():\n                    for _an, _av in list(vars(_mod).items()):\n                        if _an not in ("_parameters", "_buffers"):\n                            _tot += _add(f"{_mn}.<attr>{_an}", _av)\n                _top = sorted(_g.items(), key=lambda kv: -kv[1])[:30]\n                logger.info("WEIGHTS DIAG: counted %.3f GiB in %d storages | torch allocated %.3f GiB, reserved %.3f GiB",\n                            _tot / 2**30, len(_seen), _t.cuda.memory_allocated() / 2**30, _t.cuda.memory_reserved() / 2**30)\n                for _k, _v in _top:\n                    logger.info("WEIGHTS DIAG: %9.1f MiB  %s", _v / 2**20, _k)\n')]
DIAG_WEIGHTS_V1_EDITS = [('log GPU memory per model part after loading (runner V1)', '        logger.info_once(\n            "Model loading took %s GiB memory and %.6f seconds",\n            format_gib(self.model_memory_usage),\n            time_after_load - time_before_load,\n        )\n', '        logger.info_once(\n            "Model loading took %s GiB memory and %.6f seconds",\n            format_gib(self.model_memory_usage),\n            time_after_load - time_before_load,\n        )\n        if __import__("os").environ.get("FLASHNEXT_DIAG_WEIGHTS") == "1":  # qsa_dcp cell: per-module GPU memory (log only)\n            import collections as _c, re as _re, torch as _t\n            _mdl = getattr(self, "model", None)\n            _g, _seen, _tot = _c.defaultdict(int), set(), 0\n            def _add(_name, _x):\n                global_ok = isinstance(_x, _t.Tensor) and _x.is_cuda\n                if not global_ok:\n                    return 0\n                _s = _x.untyped_storage()\n                if _s.data_ptr() in _seen:\n                    return 0\n                _seen.add(_s.data_ptr())\n                _g[_re.sub(r"\\.\\d+(?=\\.|$)", ".N", _name)] += _s.nbytes()\n                return _s.nbytes()\n            if _mdl is not None:\n                for _n, _p in list(_mdl.named_parameters(remove_duplicate=False)) + list(_mdl.named_buffers(remove_duplicate=False)):\n                    _tot += _add(_n, _p)\n                for _mn, _mod in _mdl.named_modules():\n                    for _an, _av in list(vars(_mod).items()):\n                        if _an not in ("_parameters", "_buffers"):\n                            _tot += _add(f"{_mn}.<attr>{_an}", _av)\n                _top = sorted(_g.items(), key=lambda kv: -kv[1])[:30]\n                logger.info("WEIGHTS DIAG: counted %.3f GiB in %d storages | torch allocated %.3f GiB, reserved %.3f GiB",\n                            _tot / 2**30, len(_seen), _t.cuda.memory_allocated() / 2**30, _t.cuda.memory_reserved() / 2**30)\n                for _k, _v in _top:\n                    logger.info("WEIGHTS DIAG: %9.1f MiB  %s", _v / 2**20, _k)\n')]
QSA_TRIM_EDITS = [('memory trims: shared top-k buffer registry', 'from .indexer_qsa import QSAIndexer\n', 'from .indexer_qsa import QSAIndexer\n\n_FLASHNEXT_SHARED_TOPK: dict = {}  # memory trims (Kaggle cell MEM_TRIMS)\n'), ('memory trims: rotary cache sized to the serving window', '        self.rotary_emb = get_rope(\n            head_size=self.head_dim,\n            max_position=config.max_position_embeddings,\n            rope_parameters=config.rope_parameters,\n        )\n', '        import os as _os\n\n        self.rotary_emb = get_rope(\n            head_size=self.head_dim,\n            # memory trims (Kaggle cell MEM_TRIMS): positions never exceed the serving window\n            max_position=(\n                min(config.max_position_embeddings, vllm_config.model_config.max_model_len)\n                if _os.environ.get("FLASHNEXT_MEM_TRIMS") == "1"\n                else config.max_position_embeddings\n            ),\n            rope_parameters=config.rope_parameters,\n        )\n'), ('memory trims: one top-k buffer for all QSA layers', '        self.register_buffer(\n            "topk_indices_buffer",\n            torch.empty(\n                max_tokens,\n                self.indexer.packed_output_width,\n                dtype=torch.int32,\n            ),\n            persistent=False,\n        )\n', '        _shape = (max_tokens, self.indexer.packed_output_width)\n        if _os.environ.get("FLASHNEXT_MEM_TRIMS") == "1":\n            # memory trims: each QSA layer writes and reads its selection before the next layer runs -> one shared buffer\n            _buf = _FLASHNEXT_SHARED_TOPK.get(_shape)\n            if _buf is None:\n                _buf = _FLASHNEXT_SHARED_TOPK[_shape] = torch.empty(*_shape, dtype=torch.int32)\n        else:\n            _buf = torch.empty(*_shape, dtype=torch.int32)\n        self.register_buffer("topk_indices_buffer", _buf, persistent=False)\n')]
HUMMING_TRIM_EDITS = [('memory trims: Humming padding 64 for the hyper-connection layers', '        pad_n_to_multiple=256,\n        pad_k_to_multiple=128,\n        has_bias=layer.has_bias,\n', '        pad_n_to_multiple=64 if _flashnext_small_pad(layer) else 256,\n        pad_k_to_multiple=64 if _flashnext_small_pad(layer) else 128,\n        has_bias=layer.has_bias,\n'), ('memory trims: small-padding selector', 'def prepare_humming_linear_layer_config(\n', 'def _flashnext_small_pad(layer) -> bool:\n    # memory trims (Kaggle cell MEM_TRIMS): the FP8 hyper-connection mixers (336 x 10,240 and 10,240 x 320) only\n    import os\n\n    return os.environ.get("FLASHNEXT_MEM_TRIMS") == "1" and "hyper_connection" in getattr(layer, "prefix", "")\n\n\ndef prepare_humming_linear_layer_config(\n')]
PREFIX_REUSE_EDITS = [('prefix reuse: partial cache entry at the latest generated-token boundary', '        hash_block_size = self.block_pool.hash_block_size\n        boundary_tokens = request.num_prompt_tokens // hash_block_size * hash_block_size\n        if boundary_tokens == 0 or boundary_tokens > num_tokens:\n            return\n        if boundary_tokens % self.block_size == 0:\n            return\n', '        hash_block_size = self.block_pool.hash_block_size\n        boundary_tokens = request.num_prompt_tokens // hash_block_size * hash_block_size\n        import os\n        if os.environ.get("FLASHNEXT_DECODE_PARTIAL") == "1":\n            # flashnext (Kaggle cell PREFIX_REUSE): also the latest hash boundary of the GENERATED tokens, so a follow-up call can\n            # resume inside the previous reply (the Mamba groups keep those states under dense retention; without this entry a\n            # large attention block caps the hit at the prompt end). Limited to boundaries that already have block hashes.\n            latest = min(num_tokens, len(request.block_hashes) * hash_block_size) // hash_block_size * hash_block_size\n            boundary_tokens = max(boundary_tokens, latest)\n        if boundary_tokens == 0 or boundary_tokens > num_tokens:\n            return\n        if boundary_tokens % self.block_size == 0:\n            return\n')]
QAR_EDITS = [('compressed all-reduce: algorithm + dispatch helper', 'class CudaCommunicator(DeviceCommunicatorBase):\n', '# ---- flashnext compressed all-reduce (dev/patches/qsa_dcp/qar.py) ----\n# flashnext compressed all-reduce for LARGE (prefill) messages. One source, used by the vLLM patch (qar_edits.py inserts this\n# text into vllm/distributed/device_communicators/cuda_communicator.py) and by tools/qar_bench_cell.py.\n# An all-reduce = reduce-scatter + all-gather. Kaggle\'s 4 x L4 move NCCL data through host memory at ~4.9 GB/s, and a 5-10 MB\n# prefill message is pure bytes (3.4 ms at 10.5 MB vs 85 us latency), so sending fewer bytes is the only lever left:\n#   mode "a2a": each rank quantizes its partial result to 8 bits (codec below), exchanges the shards\n#               (grouped send/recv), sums the received shards + its OWN EXACT shard in FP32, quantizes the sum, all-gathers it\n#               -> ~56 % of the BF16 bytes (int8, block 32).\n#   mode "rs":  BF16 reduce-scatter (exact NCCL sum, as today), then 8-bit all-gather of the summed shard -> ~78 % of the bytes,\n#               one quantization only.\n# Every rank dequantizes the SAME gathered bytes (its own shard included), so all ranks keep identical hidden states.\n# Messages below the size threshold (all decode steps) and anything inside CUDA-graph capture keep the normal BF16 path.\n# Packed layout ("segment"): per rank, `shard` 1-byte codes then shard / block FP32 scales; W segments of pk bytes back to back.\n# flashnext_qar uses 3 fused Triton kernels (pack, dequant+sum+repack, unpack); flashnext_qar_ref is the same algorithm in plain\n# torch ops (the reference the kernels are tested against bit-exactly, and the CPU multi-process test). Session 8 dry run on one\n# L4: the torch-op version costs ~430-700 us per call in launch overhead, too much for a ~1.7 ms (5.2 MB) all-reduce.\nimport os\n\nimport torch\n\n# codec: "int8" (default: round half away from zero of x / scale, scale = block absmax / 127) or "fp8" (e4m3, scale = block\n# absmax / 448); one FP32 scale per FLASHNEXT_QAR_BLOCK values (default 32). Mini-model test (docs/10): fp8 / 128 added 2-4x\n# vLLM\'s own run-to-run prompt-logprob noise, int8 / 32 ~1.5-2x.\n_FN_QAR_CODEC = os.environ.get("FLASHNEXT_QAR_CODEC", "int8")\n_FN_QAR_BLOCK = int(os.environ.get("FLASHNEXT_QAR_BLOCK", "32"))\n_FN_QAR_FP8 = torch.float8_e4m3fn\n_FN_QAR_MAX = 127.0 if _FN_QAR_CODEC == "int8" else 448.0\n_FN_QAR_INV = 1.0 / _FN_QAR_MAX                                         # scale = absmax * (1 / max), same rounding in torch and Triton\nif _FN_QAR_CODEC not in ("int8", "fp8") or 128 % _FN_QAR_BLOCK:\n    raise ValueError(f"FLASHNEXT_QAR_CODEC {_FN_QAR_CODEC!r} / FLASHNEXT_QAR_BLOCK {_FN_QAR_BLOCK}: use int8|fp8 and a divisor of 128")\n\n\n# ---------------------------------------------------------------- reference (plain torch ops)\ndef _fn_qar_pack(v, out=None):\n    """FP32/BF16 vector (length multiple of 128) -> one segment: uint8 [n + 4 * n / block] (codes, then FP32 scales)."""\n    n = v.numel()\n    blocks = v.float().view(-1, _FN_QAR_BLOCK)\n    scale = blocks.abs().amax(dim=1, keepdim=True).clamp_(min=1e-30).mul_(_FN_QAR_INV)\n    if out is None:\n        out = torch.empty(n + 4 * (n // _FN_QAR_BLOCK), dtype=torch.uint8, device=v.device)\n    q = (blocks / scale).clamp_(-_FN_QAR_MAX, _FN_QAR_MAX)\n    if _FN_QAR_CODEC == "int8":\n        out[:n].view(torch.int8).view(-1, _FN_QAR_BLOCK).copy_(q.sign() * (q.abs() + 0.5).floor())\n    else:\n        out[:n].view(_FN_QAR_FP8).view(-1, _FN_QAR_BLOCK).copy_(q)\n    out[n:].view(torch.float32).copy_(scale.view(-1))\n    return out\n\n\ndef _fn_qar_unpack(buf, n):\n    """one segment uint8 [n + 4 * n / block] -> FP32 [n]."""\n    vals = buf[:n].view(torch.int8 if _FN_QAR_CODEC == "int8" else _FN_QAR_FP8).view(-1, _FN_QAR_BLOCK).float()\n    return (vals * buf[n:].view(torch.float32).view(-1, 1)).view(-1)\n\n\ndef _fn_qar_layout(n, world):\n    shard = -(-n // (world * 128)) * 128                                # elements per rank, multiple of 128\n    return shard, shard + 4 * (shard // _FN_QAR_BLOCK)                  # packed bytes per segment (multiple of 4)\n\n\ndef flashnext_qar_ref(pc, world, rank, x, mode):\n    """Reference implementation (torch ops). pc: group_start/group_end/send/recv/all_gather(out, inp)/reduce_scatter(out, inp)."""\n    n = x.numel()\n    shard, pk = _fn_qar_layout(n, world)\n    flat = x.reshape(-1)\n    if shard * world != n:\n        flat = torch.nn.functional.pad(flat, (0, shard * world - n))\n    if mode == "rs":\n        mine = torch.empty(shard, dtype=x.dtype, device=x.device)\n        pc.reduce_scatter(mine, flat)\n        total = mine.float()\n    elif mode == "a2a":\n        send = torch.empty(world * pk, dtype=torch.uint8, device=x.device)\n        for p in range(world):\n            _fn_qar_pack(flat[p * shard:(p + 1) * shard], send[p * pk:(p + 1) * pk])\n        recv = torch.empty_like(send)\n        pc.group_start()\n        for p in range(world):\n            if p != rank:\n                pc.send(send[p * pk:(p + 1) * pk], p)\n                pc.recv(recv[p * pk:(p + 1) * pk], p)\n        pc.group_end()\n        total = flat[rank * shard:(rank + 1) * shard].float()\n        for p in range(world):\n            if p != rank:\n                total = total + _fn_qar_unpack(recv[p * pk:(p + 1) * pk], shard)\n    else:\n        raise ValueError(f"FLASHNEXT_QAR mode {mode!r}: use \'a2a\' or \'rs\'")\n    gathered = torch.empty(world * pk, dtype=torch.uint8, device=x.device)\n    pc.all_gather(gathered, _fn_qar_pack(total))\n    out = torch.cat([_fn_qar_unpack(gathered[p * pk:(p + 1) * pk], shard) for p in range(world)])\n    return out[:n].to(x.dtype).view_as(x)\n\n\n# ---------------------------------------------------------------- fused Triton kernels (same arithmetic, bit-exact)\n_FN_QAR_ROWS = 64                                                       # blocks per Triton program\n\ntry:\n    import triton\n    import triton.language as tl\nexcept ImportError:                                                      # CPU-only test environments\n    triton = None\n\nif triton is not None:\n    @triton.jit\n    def _fn_qar_quant(x, MAXV: tl.constexpr, INV: tl.constexpr, IS_INT8: tl.constexpr):\n        """x [ROWS, BLOCK] fp32 -> (codes, scale [ROWS])"""\n        scale = tl.maximum(tl.max(tl.abs(x), axis=1), 1e-30) * INV\n        y = tl.math.div_rn(x, scale[:, None])                           # IEEE division = torch\'s tensor / tensor\n        y = tl.minimum(tl.maximum(y, -MAXV), MAXV)\n        if IS_INT8:\n            r = tl.where(y >= 0, tl.floor(y + 0.5), -tl.floor(-y + 0.5))\n            return r.to(tl.int8), scale\n        else:\n            return y.to(tl.float8e4nv), scale\n\n    @triton.jit\n    def _fn_qar_pack_kernel(x_ptr, n, codes_ptr, scales_ptr, nb_seg, pk, n_blocks,\n                            BLOCK: tl.constexpr, ROWS: tl.constexpr, MAXV: tl.constexpr, INV: tl.constexpr, IS_INT8: tl.constexpr):\n        """x (n values, any float dtype) -> packed segments: block b goes to segment b // nb_seg."""\n        b = tl.program_id(0) * ROWS + tl.arange(0, ROWS)\n        c = tl.arange(0, BLOCK)\n        bm = b < n_blocks\n        off = b[:, None] * BLOCK + c[None, :]\n        x = tl.load(x_ptr + off, mask=bm[:, None] & (off < n), other=0.0).to(tl.float32)\n        q, s = _fn_qar_quant(x, MAXV, INV, IS_INT8)\n        seg = b // nb_seg\n        loc = b - seg * nb_seg\n        tl.store(codes_ptr + (seg * pk + loc * BLOCK)[:, None] + c[None, :], q, mask=bm[:, None])\n        tl.store(scales_ptr + (seg * pk + nb_seg * BLOCK) // 4 + loc, s, mask=bm)\n\n    @triton.jit\n    def _fn_qar_sum_repack_kernel(own_ptr, rcodes_ptr, rscales_ptr, ocodes_ptr, oscales_ptr, nb_seg, pk, rank,\n                                  WORLD: tl.constexpr, BLOCK: tl.constexpr, ROWS: tl.constexpr, MAXV: tl.constexpr,\n                                  INV: tl.constexpr, IS_INT8: tl.constexpr):\n        """a2a: own exact shard + dequantized peer segments (rank order) -> one packed segment of the sum."""\n        b = tl.program_id(0) * ROWS + tl.arange(0, ROWS)\n        c = tl.arange(0, BLOCK)\n        bm = b < nb_seg\n        off = b[:, None] * BLOCK + c[None, :]\n        total = tl.load(own_ptr + off, mask=bm[:, None], other=0.0).to(tl.float32)\n        for p in tl.static_range(WORLD):\n            if p != rank:\n                v = tl.load(rcodes_ptr + p * pk + off, mask=bm[:, None]).to(tl.float32)\n                s = tl.load(rscales_ptr + (p * pk + nb_seg * BLOCK) // 4 + b, mask=bm, other=0.0)\n                total = total + v * s[:, None]\n        q, s2 = _fn_qar_quant(total, MAXV, INV, IS_INT8)\n        tl.store(ocodes_ptr + off, q, mask=bm[:, None])\n        tl.store(oscales_ptr + nb_seg * BLOCK // 4 + b, s2, mask=bm)\n\n    @triton.jit\n    def _fn_qar_unpack_kernel(codes_ptr, scales_ptr, out_ptr, n, nb_seg, pk, n_blocks,\n                              BLOCK: tl.constexpr, ROWS: tl.constexpr):\n        """packed segments -> out (n values, out dtype)."""\n        b = tl.program_id(0) * ROWS + tl.arange(0, ROWS)\n        c = tl.arange(0, BLOCK)\n        bm = b < n_blocks\n        seg = b // nb_seg\n        loc = b - seg * nb_seg\n        v = tl.load(codes_ptr + (seg * pk + loc * BLOCK)[:, None] + c[None, :], mask=bm[:, None]).to(tl.float32)\n        s = tl.load(scales_ptr + (seg * pk + nb_seg * BLOCK) // 4 + loc, mask=bm, other=0.0)\n        off = b[:, None] * BLOCK + c[None, :]\n        tl.store(out_ptr + off, (v * s[:, None]).to(out_ptr.dtype.element_ty), mask=bm[:, None] & (off < n))\n\n\ndef _fn_qar_views(buf):\n    codes = buf.view(torch.int8) if _FN_QAR_CODEC == "int8" else buf.view(_FN_QAR_FP8)\n    return codes, buf.view(torch.float32)\n\n\ndef flashnext_qar(pc, world, rank, x, mode):\n    """Compressed all-reduce of x over `world` ranks with fused kernels (3 launches + 2 NCCL calls); same result bits as\n    flashnext_qar_ref. pc: vLLM PyNcclCommunicator-style object."""\n    n = x.numel()\n    shard, pk = _fn_qar_layout(n, world)\n    nb_seg = shard // _FN_QAR_BLOCK\n    flat = x.reshape(-1)\n    grid_seg = (triton.cdiv(nb_seg, _FN_QAR_ROWS),)\n    kw = dict(BLOCK=_FN_QAR_BLOCK, ROWS=_FN_QAR_ROWS, MAXV=_FN_QAR_MAX, INV=_FN_QAR_INV, IS_INT8=_FN_QAR_CODEC == "int8")\n    mine = torch.empty(pk, dtype=torch.uint8, device=x.device)\n    mc, ms = _fn_qar_views(mine)\n    if mode == "rs":\n        if shard * world != n:\n            flat = torch.nn.functional.pad(flat, (0, shard * world - n))\n        red = torch.empty(shard, dtype=x.dtype, device=x.device)\n        pc.reduce_scatter(red, flat)\n        _fn_qar_pack_kernel[grid_seg](red, shard, mc, ms, nb_seg, pk, nb_seg, **kw)\n    elif mode == "a2a":\n        send = torch.empty(world * pk, dtype=torch.uint8, device=x.device)\n        sc, ss = _fn_qar_views(send)\n        _fn_qar_pack_kernel[(triton.cdiv(world * nb_seg, _FN_QAR_ROWS),)](flat, n, sc, ss, nb_seg, pk, world * nb_seg, **kw)\n        recv = torch.empty_like(send)\n        pc.group_start()\n        for p in range(world):\n            if p != rank:\n                pc.send(send[p * pk:(p + 1) * pk], p)\n                pc.recv(recv[p * pk:(p + 1) * pk], p)\n        pc.group_end()\n        own = flat[rank * shard:(rank + 1) * shard]\n        if own.numel() < shard:                                          # last rank\'s shard runs past n: pad it\n            own = torch.nn.functional.pad(own, (0, shard - own.numel()))\n        rc, rs_ = _fn_qar_views(recv)\n        _fn_qar_sum_repack_kernel[grid_seg](own, rc, rs_, mc, ms, nb_seg, pk, rank, WORLD=world, **kw)\n    else:\n        raise ValueError(f"FLASHNEXT_QAR mode {mode!r}: use \'a2a\' or \'rs\'")\n    gathered = torch.empty(world * pk, dtype=torch.uint8, device=x.device)\n    pc.all_gather(gathered, mine)\n    out = torch.empty(n, dtype=x.dtype, device=x.device)\n    gc, gs = _fn_qar_views(gathered)\n    _fn_qar_unpack_kernel[(triton.cdiv(world * nb_seg, _FN_QAR_ROWS),)](gc, gs, out, n, nb_seg, pk, world * nb_seg,\n                                                                         BLOCK=_FN_QAR_BLOCK, ROWS=_FN_QAR_ROWS)\n    return out.view_as(x)\n\n_FN_QAR_MODE = os.environ.get("FLASHNEXT_QAR") or None\n_FN_QAR_MIN_BYTES = float(os.environ.get("FLASHNEXT_QAR_MIN_MB", "1.5")) * 2**20\n_FN_QAR_CALLS = [0]\n\n\ndef _fn_qar_maybe(comm, x):\n    """flashnext: compressed all-reduce for large TP messages, or None (= vLLM\'s normal path)."""\n    if (_FN_QAR_MODE is None or comm.world_size == 1 or not str(comm.unique_name).startswith("tp:")\n            or x.dtype not in (torch.bfloat16, torch.float16) or not x.is_cuda\n            or x.numel() * x.element_size() < _FN_QAR_MIN_BYTES or torch.cuda.is_current_stream_capturing()):\n        return None\n    pc = comm.pynccl_comm\n    if pc is None or pc.disabled:\n        raise RuntimeError("FLASHNEXT_QAR is set but the TP group has no PyNccl communicator (no silent fallback)")\n    _FN_QAR_CALLS[0] += 1\n    if _FN_QAR_CALLS[0] == 1:\n        logger.info("flashnext QAR active: mode %s, codec %s, block %d, min %.1f MB; first message %s %.2f MB",\n                    _FN_QAR_MODE, _FN_QAR_CODEC, _FN_QAR_BLOCK, _FN_QAR_MIN_BYTES / 2**20, tuple(x.shape),\n                    x.numel() * x.element_size() / 2**20)\n    return flashnext_qar(pc, comm.world_size, comm.rank_in_group, x.contiguous(), _FN_QAR_MODE)\n\n\nclass CudaCommunicator(DeviceCommunicatorBase):\n'), ('compressed all-reduce: first choice in CudaCommunicator.all_reduce', '    def all_reduce(self, input_):\n        fi_ar_comm = self.fi_ar_comm\n', '    def all_reduce(self, input_):\n        _fn_out = _fn_qar_maybe(self, input_)\n        if _fn_out is not None:\n            return _fn_out\n        fi_ar_comm = self.fi_ar_comm\n')]
SW_EDITS = [('sink+window: candidate mask helper', '\n\ndef _topk(', '\n\n# ---- flashnext: sink + window candidate mask (dev/patches/qsa_dcp/sink_window_edits.py) ----\nimport os as _fn_os\n\n_FN_SW_SINK_AUTO = _fn_os.environ.get("FLASHNEXT_QSA_SINK") == "auto"\n_FN_SW_SINK = 0 if _FN_SW_SINK_AUTO else int(_fn_os.environ.get("FLASHNEXT_QSA_SINK", "0") or 0)\n_FN_SW_SINK_TOK = None          # "auto": per-token sink lengths, a buffer filled by Qwen4ExpModelState.prepare_inputs\n_FN_SW_WINDOW = int(_fn_os.environ.get("FLASHNEXT_QSA_WINDOW", "0") or 0)\n_FN_SW_CHECK = _fn_os.environ.get("FLASHNEXT_QSA_SW_CHECK") == "1"\n# "jump" mode (owner\'s variant): like today\'s trim, but without re-reading. Up to LIMIT tokens nothing changes; when attention\n# would see LIMIT tokens, it keeps the sink + the most recent tokens so that it sees KEEP tokens; that part grows back to LIMIT,\n# then the next cut (every LIMIT - KEEP tokens). The DeltaNet layers keep the whole conversation.\n_FN_SW_KEEP = int(_fn_os.environ.get("FLASHNEXT_QSA_JUMP_KEEP", "0") or 0)\n_FN_SW_LIMIT = int(_fn_os.environ.get("FLASHNEXT_QSA_JUMP_LIMIT", "0") or 0)\n_FN_SW_CALLS = [0]\n# skip (session 9): the scoring kernels do not even compute the cut-away groups (their scores are set to -1 anyway), so the\n# scoring cost stays at sink + recent part instead of growing with the whole conversation. Same selection, bit for bit.\n_FN_SW_SKIP = _fn_os.environ.get("FLASHNEXT_QSA_SKIP_GAP") == "1"\n\n\ndef _fn_sw_gap(visible_blocks, token_topk, compress_ratio, row_start=0):\n    """(sink, first recent group) per row in compressed groups (int64), or None when sink+window is inactive."""\n    if _FN_SW_WINDOW <= 0 and _FN_SW_LIMIT <= 0:\n        return None\n    rows = visible_blocks.shape[0]\n    v = visible_blocks.to(torch.int64)\n    if _FN_SW_SINK_AUTO:\n        if _FN_SW_SINK_TOK is None:\n            raise RuntimeError("FLASHNEXT_QSA_SINK=auto but the model state did not publish per-token sinks")\n        sink_c = _FN_SW_SINK_TOK[row_start:row_start + rows] // compress_ratio\n    else:\n        sink_c = torch.full_like(v, _FN_SW_SINK // compress_ratio)\n    if _FN_SW_LIMIT > 0:                                                    # jump mode\n        if _FN_SW_LIMIT <= _FN_SW_KEEP:\n            raise RuntimeError(f"QSA jump mode: need LIMIT > KEEP (keep {_FN_SW_KEEP}, limit {_FN_SW_LIMIT})")\n        lim_c, keep_c = _FN_SW_LIMIT // compress_ratio, _FN_SW_KEEP // compress_ratio\n        period = lim_c - keep_c\n        n_cuts = torch.div(v - lim_c, period, rounding_mode="floor") + 1        # cuts so far (<= 0: none yet)\n        start = lim_c + (n_cuts - 1) * period - (keep_c - sink_c)              # first recent group after the last cut\n        hi = torch.where(v > lim_c, start, sink_c)\n    else:                                                                   # sliding mode\n        if _FN_SW_WINDOW < token_topk + compress_ratio:\n            raise RuntimeError(f"FLASHNEXT_QSA_WINDOW {_FN_SW_WINDOW} must be >= the QSA budget {token_topk} + {compress_ratio}")\n        hi = v - _FN_SW_WINDOW // compress_ratio\n    # at least token_topk + compress_ratio recent tokens stay eligible (also with a long sink); never cut into the sink\n    hi = torch.maximum(torch.minimum(hi, v - (token_topk // compress_ratio + 1)), sink_c)\n    return sink_c, hi\n\n\ndef _fn_gap_kwargs(gap):\n    """Extra scoring-kernel arguments: skip the tiles that lie inside every row\'s cut-away range."""\n    if gap is None or not _FN_SW_SKIP:\n        return {}\n    return dict(gap_lo_ptr=gap[0].to(torch.int32).contiguous(), gap_hi_ptr=gap[1].to(torch.int32).contiguous(), SKIP_GAP=True)\n\n\ndef _fn_compact_warmup_kwargs():\n    if not _FN_SW_SKIP or (_FN_SW_WINDOW <= 0 and _FN_SW_LIMIT <= 0):\n        return {}\n    return dict(gap_lo_ptr=TritonWarmupTensor(torch.int32), gap_shift_ptr=TritonWarmupTensor(torch.int32), COMPACT=True)\n\n\ndef _fn_sink_window_mask_note():\n    _FN_SW_CALLS[0] += 1\n    if _FN_SW_CALLS[0] == 1:\n        print("flashnext QSA sink+window active, decode scoring COMPACT (the cut-away part is not scored)", flush=True)\n\n\ndef _fn_sink_window_mask(logits, visible_blocks, token_topk, compress_ratio, row_start=0, gap=None):\n    """Set the scores of the groups between the sink and the recent part to -1 (rows = queries, columns = compressed groups).\n    Returns (sink, first recent group) per row for the optional check, or None when inactive."""\n    if gap is None:\n        gap = _fn_sw_gap(visible_blocks, token_topk, compress_ratio, row_start)\n    if gap is None:\n        return None\n    sink_c, hi = gap\n    cols = torch.arange(logits.shape[1], device=logits.device)\n    cut = (cols[None, :] >= sink_c[:, None]) & (cols[None, :] < hi[:, None])\n    _FN_SW_CALLS[0] += 1\n    if _FN_SW_CALLS[0] == 1:                                                # no GPU sync per call: the mask is applied always\n        sink_txt = "auto (first prompt of each request)" if _FN_SW_SINK_AUTO else str(_FN_SW_SINK)\n        if _FN_SW_SKIP:\n            print("flashnext QSA scoring skips the cut-away part (FLASHNEXT_QSA_SKIP_GAP=1)", flush=True)\n        if _FN_SW_LIMIT > 0:\n            print(f"flashnext QSA sink+window active (jump): sink {sink_txt}, attention cut back to {_FN_SW_KEEP} tokens "\n                  f"each time it reaches {_FN_SW_LIMIT}", flush=True)\n        else:\n            print(f"flashnext QSA sink+window active: sink {sink_txt} + window {_FN_SW_WINDOW} tokens", flush=True)\n    logits.masked_fill_(cut, -1.0)\n    return sink_c, hi\n\n\ndef _fn_sink_window_check(sel, block_indices, visible_blocks):\n    if sel is None or not _FN_SW_CHECK or torch.cuda.is_current_stream_capturing():   # a check needs a sync: not inside graph capture\n        return\n    sink_c, hi = sel\n    k = visible_blocks.to(torch.int64).clamp(max=block_indices.shape[1])\n    valid = torch.arange(block_indices.shape[1], device=block_indices.device)[None, :] < k[:, None]\n    b = block_indices.to(torch.int64)\n    bad = valid & (b >= sink_c[:, None]) & (b < hi[:, None])\n    if bool(bad.any()):\n        r = int(bad.any(dim=1).nonzero()[0])\n        raise RuntimeError(f"QSA sink+window CHECK: row {r} selected a masked group (sink groups {int(sink_c[r])}, recent from "\n                           f"{int(hi[r])}, selected {b[r][bad[r]][:8].tolist()})")\n\n\ndef _topk('), ('sink+window: decode selection', '    _topk(\n        logits,\n        visible_blocks,\n        token_topk,\n        compress_ratio,\n        block_indices,\n        torch.empty((_TOPK_WORKSPACE_BYTES,), dtype=torch.uint8, device=q.device),\n    )\n', '    _fn_sel = _fn_sink_window_mask(logits, visible_blocks, token_topk, compress_ratio, 0)\n    _topk(\n        logits,\n        visible_blocks,\n        token_topk,\n        compress_ratio,\n        block_indices,\n        torch.empty((_TOPK_WORKSPACE_BYTES,), dtype=torch.uint8, device=q.device),\n    )\n    _fn_sink_window_check(_fn_sel, block_indices, visible_blocks)\n'), ('sink+window: prefill selection takes its first batch row', '    block_indices: torch.Tensor,\n    max_seq_len: int,\n) -> None:\n', '    block_indices: torch.Tensor,\n    max_seq_len: int,\n    _fn_row_start: int = 0,\n) -> None:\n'), ('sink+window: prefill selection', '        _topk(\n            logits,\n            visible_blocks[query_slice],\n            token_topk,\n            compress_ratio,\n            block_indices[query_slice],\n            topk_workspace,\n        )\n', '        _fn_sel = _fn_sink_window_mask(logits, visible_blocks[query_slice], token_topk, compress_ratio,\n                                       _fn_row_start + query_start)\n        _topk(\n            logits,\n            visible_blocks[query_slice],\n            token_topk,\n            compress_ratio,\n            block_indices[query_slice],\n            topk_workspace,\n        )\n        _fn_sink_window_check(_fn_sel, block_indices[query_slice], visible_blocks[query_slice])\n'), ('skip: decode scoring kernel arguments', '    STAGES: tl.constexpr,\n) -> None:\n    NUM_COLUMNS: tl.constexpr = PAGE_TABLE_WIDTH * PAGE_SIZE\n', '    STAGES: tl.constexpr,\n    gap_lo_ptr=None,\n    gap_shift_ptr=None,\n    COMPACT: tl.constexpr = False,\n) -> None:\n    NUM_COLUMNS: tl.constexpr = PAGE_TABLE_WIDTH * PAGE_SIZE\n'), ('skip: decode compact column -> group', '    column_offsets = tl.arange(0, BLOCK_N)\n    for tile in tl.range(tile_start, tile_end, num_stages=STAGES):\n        columns = tile * BLOCK_N + column_offsets\n        live = columns < max_visible\n        logical_page = tl.minimum(columns // PAGE_SIZE, PAGE_TABLE_WIDTH - 1)\n        page_offset = columns % PAGE_SIZE\n', '    column_offsets = tl.arange(0, BLOCK_N)\n    if COMPACT:\n        # flashnext compact scoring (DECODE_QUERY_LEN == 1): column j = group j below the sink, group j + shift above it\n        c_lo = tl.load(gap_lo_ptr + request)\n        c_shift = tl.load(gap_shift_ptr + request)\n    for tile in tl.range(tile_start, tile_end, num_stages=STAGES):\n        columns = tile * BLOCK_N + column_offsets\n        live = columns < max_visible\n        if COMPACT:\n            groups = columns + tl.where(columns >= c_lo, c_shift, 0)\n        else:\n            groups = columns\n        logical_page = tl.minimum(groups // PAGE_SIZE, PAGE_TABLE_WIDTH - 1)\n        page_offset = groups % PAGE_SIZE\n'), ('skip: prefill scoring kernel arguments', '    STAGES: tl.constexpr,\n) -> None:\n    num_columns = page_table_width * PAGE_SIZE\n', '    STAGES: tl.constexpr,\n    gap_lo_ptr=None,\n    gap_hi_ptr=None,\n    SKIP_GAP: tl.constexpr = False,\n) -> None:\n    num_columns = page_table_width * PAGE_SIZE\n'), ('skip: prefill scoring loop jumps over the cut-away tiles', '    for tile in tl.range(k_tile_start, k_tile_end, num_stages=STAGES):\n        columns = tile * BLOCK_N + column_offsets\n', '    if SKIP_GAP:\n        # flashnext: jump over the tiles that lie inside the cut-away range of EVERY row of this program (their scores\n        # would be overwritten with -1 by the sink+window mask anyway)\n        g_lo = tl.load(gap_lo_ptr + absolute_rows, mask=valid_rows, other=0)\n        g_hi = tl.load(gap_hi_ptr + absolute_rows, mask=valid_rows, other=0)\n        skip_lo = tl.max(tl.where(valid_rows, g_lo, 0), axis=0)\n        skip_hi = tl.min(tl.where(valid_rows, g_hi, 0x7FFFFFFF), axis=0)\n        first_end = tl.minimum(k_tile_end, tl.maximum(k_tile_start, tl.cdiv(skip_lo, BLOCK_N)))\n        second_start = tl.minimum(k_tile_end, tl.maximum(first_end, skip_hi // BLOCK_N))\n        n_first = first_end - k_tile_start\n        jump = second_start - first_end\n        n_tiles = n_first + (k_tile_end - second_start)\n    else:\n        n_tiles = k_tile_end - k_tile_start\n    for i in tl.range(0, n_tiles, num_stages=STAGES):\n        if SKIP_GAP:\n            tile = k_tile_start + i + tl.where(i >= n_first, jump, 0)\n        else:\n            tile = k_tile_start + i\n        columns = tile * BLOCK_N + column_offsets\n'), ('skip: decode warmup also compiles the compact kernel', '            num_warps=1 if k_cache.dtype == torch.float8_e4m3fn else 2,\n            grid=(', '            num_warps=1 if k_cache.dtype == torch.float8_e4m3fn else 2,\n            **(_fn_compact_warmup_kwargs() if decode_query_len == 1 else {}),\n            grid=('), ('skip: decode computes the cut-away range before scoring', '    columns = page_table.shape[1] * k_cache.shape[1]\n    logits = torch.empty((q.shape[0], columns), dtype=torch.float32, device=q.device)\n', '    columns = page_table.shape[1] * k_cache.shape[1]\n    logits = torch.empty((q.shape[0], columns), dtype=torch.float32, device=q.device)\n    _fn_gap = _fn_sw_gap(visible_blocks, token_topk, compress_ratio, 0)\n    _fn_compact = _fn_gap is not None and _FN_SW_SKIP and decode_query_len == 1\n    _fn_kw, _fn_vis = {}, visible_blocks\n    if _fn_compact:\n        _fn_lo = _fn_gap[0].to(torch.int32).contiguous()\n        _fn_shift = (_fn_gap[1] - _fn_gap[0]).to(torch.int32).contiguous()\n        _fn_vis = (visible_blocks - _fn_shift).to(visible_blocks.dtype)\n        _fn_kw = dict(gap_lo_ptr=_fn_lo, gap_shift_ptr=_fn_shift, COMPACT=True)\n'), ('skip: decode scoring sees the compact row length', '        page_table,\n        visible_blocks,\n        logits,\n', '        page_table,\n        _fn_vis,\n        logits,\n'), ('skip: decode scoring compact arguments; mask only without compaction', '        num_warps=1 if k_cache.dtype == torch.float8_e4m3fn else 2,\n    )\n    _fn_sel = _fn_sink_window_mask(logits, visible_blocks, token_topk, compress_ratio, 0)\n    _topk(\n        logits,\n        visible_blocks,\n', '        num_warps=1 if k_cache.dtype == torch.float8_e4m3fn else 2,\n        **_fn_kw,\n    )\n    if _fn_compact:\n        _fn_sel = _fn_gap\n        _fn_sink_window_mask_note()\n    else:\n        _fn_sel = _fn_sink_window_mask(logits, visible_blocks, token_topk, compress_ratio, 0, gap=_fn_gap)\n    _topk(\n        logits,\n        _fn_vis,\n'), ('skip: decode selected columns back to group numbers', '        torch.empty((_TOPK_WORKSPACE_BYTES,), dtype=torch.uint8, device=q.device),\n    )\n    _fn_sink_window_check(_fn_sel, block_indices, visible_blocks)\n', '        torch.empty((_TOPK_WORKSPACE_BYTES,), dtype=torch.uint8, device=q.device),\n    )\n    if _fn_compact:\n        block_indices.add_(torch.where(block_indices >= _fn_lo[:, None], _fn_shift[:, None], 0).to(block_indices.dtype))\n    _fn_sink_window_check(_fn_sel, block_indices, visible_blocks)\n'), ('skip: prefill logits take the range', '    query_offset: int,\n    num_queries: int,\n) -> torch.Tensor:\n', '    query_offset: int,\n    num_queries: int,\n    _fn_gap=None,\n) -> torch.Tensor:\n'), ('skip: prefill scoring gets the range', '        STAGES=STAGES,\n        num_warps=num_warps,\n    )\n    return logits\n', '        STAGES=STAGES,\n        num_warps=num_warps,\n        **_fn_gap_kwargs(_fn_gap),\n    )\n    return logits\n'), ('skip: prefill computes the range once for all its rows', '    for query_start in range(0, rows, rows_per_chunk):\n', '    _fn_gap = _fn_sw_gap(visible_blocks, token_topk, compress_ratio, _fn_row_start)\n    for query_start in range(0, rows, rows_per_chunk):\n'), ('skip: prefill chunks pass the range on, the mask reuses it', '            num_queries=query_end - query_start,\n        )\n        _fn_sel = _fn_sink_window_mask(logits, visible_blocks[query_slice], token_topk, compress_ratio,\n                                       _fn_row_start + query_start)\n', '            num_queries=query_end - query_start,\n            _fn_gap=_fn_gap,\n        )\n        _fn_sel = _fn_sink_window_mask(logits, visible_blocks[query_slice], token_topk, compress_ratio,\n                                       _fn_row_start + query_start,\n                                       gap=None if _fn_gap is None else (_fn_gap[0][query_slice], _fn_gap[1][query_slice]))\n')]
SW_INDEXER_EDITS = [('sink+window: prefill rows start after the decode rows', '                compressed_metadata.max_seq_len,\n            )\n', '                compressed_metadata.max_seq_len,\n                _fn_row_start=num_decode_tokens,\n            )\n')]
SW_STATE_EDITS = [('sink+window: per-request sink buffers', '        super().__init__(vllm_config, model, encoder_cache, device)\n        config = self.model_config.hf_text_config\n', '        super().__init__(vllm_config, model, encoder_cache, device)\n        import os as _fn_os\n\n        self._fn_sink_auto = _fn_os.environ.get("FLASHNEXT_QSA_SINK") == "auto"\n        if self._fn_sink_auto:\n            # flashnext sink+window prototype: per request, everything before the first "<|im_start|>assistant"\n            from vllm.models.qwen4_exp.nvidia.ops import qsa_indexer as _fn_qi\n            from vllm.tokenizers.registry import cached_tokenizer_from_config\n\n            _tok = cached_tokenizer_from_config(vllm_config.model_config)\n            self._fn_im_start = int(_tok.convert_tokens_to_ids("<|im_start|>"))\n            self._fn_assistant = [int(t) for t in _tok.encode("assistant", add_special_tokens=False)]\n            self._fn_sink_req = torch.zeros(self.max_num_reqs, dtype=torch.int64, device=self.device)\n            self._fn_sink_tok = torch.zeros(self.max_num_tokens + 1024, dtype=torch.int64, device=self.device)\n            self._fn_sink_logged = 0\n            _fn_qi._FN_SW_SINK_TOK = self._fn_sink_tok\n        config = self.model_config.hf_text_config\n'), ('sink+window: per-request sink from the prompt + per-token copy each step', '    def prepare_inputs(\n        self,\n        input_batch: InputBatch,\n        req_states: RequestState,\n    ) -> dict[str, Any]:\n        model_inputs = super().prepare_inputs(input_batch, req_states)\n', '    def add_request(self, req_index: int, new_req_data) -> None:\n        super().add_request(req_index, new_req_data)\n        if getattr(self, "_fn_sink_auto", False):\n            ids = new_req_data.prompt_token_ids or []\n            marker, n, sink, i = self._fn_assistant, len(self._fn_assistant), len(ids), -1\n            while True:\n                try:\n                    i = ids.index(self._fn_im_start, i + 1)\n                except ValueError:\n                    break\n                if ids[i + 1:i + 1 + n] == marker:\n                    sink = i\n                    break\n            self._fn_sink_req[req_index].fill_(sink)\n            if self._fn_sink_logged < 10 and not new_req_data.req_id.startswith("_warmup"):\n                self._fn_sink_logged += 1\n                print(f"flashnext QSA sink (auto): request {new_req_data.req_id[:24]} prompt {len(ids)} tokens -> sink {sink}",\n                      flush=True)\n\n    def prepare_inputs(\n        self,\n        input_batch: InputBatch,\n        req_states: RequestState,\n    ) -> dict[str, Any]:\n        model_inputs = super().prepare_inputs(input_batch, req_states)\n        if getattr(self, "_fn_sink_auto", False):\n            _n = input_batch.num_reqs\n            _qsl = input_batch.query_start_loc[: _n + 1].long()\n            _sinks = self._fn_sink_req[input_batch.idx_mapping[:_n].long()]\n            self._fn_sink_tok[: input_batch.num_tokens].copy_(\n                torch.repeat_interleave(_sinks, _qsl[1:] - _qsl[:-1], output_size=input_batch.num_tokens)\n            )\n')]
TF_SINGLE_EDITS = [('trim free: gap manager for the QSA groups', '\n\ndef get_manager_for_kv_cache_spec(', '\n\n# ---- flashnext: trim without re-reading, memory freeing (dev/patches/qsa_dcp/trim_free_edits.py) ----\nimport bisect as _fn_bisect\nimport os as _fn_os\n\n_FN_TRIM = {\n    "on": _fn_os.environ.get("FLASHNEXT_TRIM_FREE") == "1",\n    "check": _fn_os.environ.get("FLASHNEXT_TRIM_CHECK") == "1",\n    "keep": int(_fn_os.environ.get("FLASHNEXT_QSA_JUMP_KEEP", "0") or 0),\n    "limit": int(_fn_os.environ.get("FLASHNEXT_QSA_JUMP_LIMIT", "0") or 0),\n    "configured": False, "topk": 0, "ratio": 0, "chunk": 0, "im_start": -1, "assistant": [],\n    "sinks": {}, "cur_sink": None, "groups": [], "cuts": 0, "freed": 0, "hits": 0,\n    "gap_group_ids": set(), "usage_fn": None,\n}\n\n\ndef fn_trim_configure(vllm_config) -> None:\n    """Called once by the scheduler. Raises when the switches do not fit together (no silent fallback)."""\n    t = _FN_TRIM\n    if not t["on"]:\n        return\n    if _fn_os.environ.get("FLASHNEXT_QSA_SINK") != "auto" or t["limit"] <= t["keep"] or t["keep"] <= 0:\n        raise RuntimeError("FLASHNEXT_TRIM_FREE=1 needs FLASHNEXT_QSA_SINK=auto and FLASHNEXT_QSA_JUMP_LIMIT > "\n                           f"FLASHNEXT_QSA_JUMP_KEEP > 0 (got sink {_fn_os.environ.get(\'FLASHNEXT_QSA_SINK\')!r}, "\n                           f"keep {t[\'keep\']}, limit {t[\'limit\']})")\n    if not t["groups"]:\n        raise RuntimeError("FLASHNEXT_TRIM_FREE=1 but no KV-cache group got the trim manager")\n    cfg = vllm_config.model_config.hf_text_config\n    t["topk"] = int(cfg.indexer_budget)\n    t["ratio"] = int(cfg.indexer_compress_ratio)\n    t["chunk"] = int(vllm_config.scheduler_config.max_num_batched_tokens)\n    from vllm.tokenizers.registry import cached_tokenizer_from_config\n\n    tok = cached_tokenizer_from_config(vllm_config.model_config)\n    t["im_start"] = int(tok.convert_tokens_to_ids("<|im_start|>"))\n    t["assistant"] = [int(x) for x in tok.encode("assistant", add_special_tokens=False)]\n    t["configured"] = True\n    print(f"flashnext trim (memory freed): attention cut back to {t[\'keep\']} tokens each time it reaches {t[\'limit\']}, "\n          f"sink = the first prompt of each request; QSA groups with the gap manager: {t[\'groups\']} "\n          f"(budget {t[\'topk\']}, compress ratio {t[\'ratio\']}, chunk {t[\'chunk\']})", flush=True)\n\n\ndef fn_trim_sink_of(prompt_token_ids) -> int:\n    """Index of the first "<|im_start|>assistant" in a prompt (the whole prompt if there is none)."""\n    t = _FN_TRIM\n    ids = list(prompt_token_ids or [])\n    marker, n, s, i = t["assistant"], len(t["assistant"]), len(ids), -1\n    while True:\n        try:\n            i = ids.index(t["im_start"], i + 1)\n        except ValueError:\n            break\n        if ids[i + 1:i + 1 + n] == marker:\n            return i\n    return s\n\n\ndef fn_trim_sink(request) -> int | None:\n    """Sink of a request (cached per request id until its blocks are freed)."""\n    t = _FN_TRIM\n    if not t["configured"]:\n        return None\n    s = t["sinks"].get(request.request_id)\n    if s is None:\n        s = t["sinks"][request.request_id] = fn_trim_sink_of(request.prompt_token_ids)\n    return s\n\n\ndef fn_trim_gap_tokens(sink: int, pos: int) -> tuple[int, int]:\n    """Token range [lo, hi) that no query at a position >= pos can select (the model mask in qsa_indexer.py, with the\n    visible group count of position pos rounded down: the range only grows with the position, so this is safe)."""\n    t = _FN_TRIM\n    r = t["ratio"]\n    v = pos // r\n    sink_c = sink // r\n    lim_c, keep_c = t["limit"] // r, t["keep"] // r\n    period = lim_c - keep_c\n    if v > lim_c:\n        n_cuts = (v - lim_c) // period + 1\n        hi = lim_c + (n_cuts - 1) * period - (keep_c - sink_c)\n    else:\n        hi = sink_c\n    hi = max(min(hi, v - (t["topk"] // r + 1)), sink_c)\n    return sink_c * r, hi * r\n\n\nclass FnTrimGapManager(FullAttentionManager):\n    """QSA KV group with the middle freed (see trim_free_edits.py)."""\n\n    fn_trim_gap: ClassVar[bool] = True\n\n    def __init__(self, kv_cache_spec, **kwargs) -> None:\n        super().__init__(kv_cache_spec, **kwargs)\n        _FN_TRIM["groups"].append(f"{type(kv_cache_spec).__name__} group {self.kv_cache_group_id} "\n                                  f"(blocks of {self.block_size} tokens)")\n        _FN_TRIM["gap_group_ids"].add(self.kv_cache_group_id)\n\n    def _fn_gap_blocks(self, sink: int, pos: int) -> tuple[int, int]:\n        lo, hi = fn_trim_gap_tokens(sink, pos)\n        return cdiv(lo, self.block_size), hi // self.block_size\n\n    def remove_skipped_blocks(self, request_id, processed_computed_tokens, num_prompt_tokens=None) -> None:\n        sink = _FN_TRIM["sinks"].get(request_id)\n        if sink is None or not _FN_TRIM["configured"]:\n            return\n        # free only what is cut away one cache-hit boundary BEFORE the last one (session 11 fix). A follow-up call resumes at\n        # a cached boundary <= the processed count; freeing the gap of the processed count itself made every such hit\n        # invalid whenever a cut fell after the last boundary (find_longest_cache_hit then returned 0 and the follow-up\n        # recomputed the WHOLE conversation: RTX PRO 6000, 50,725 of 50,725 tokens). Freeing later is always safe.\n        a = getattr(self, "cache_hit_alignment_tokens", 0) or self.block_pool.hash_block_size\n        lo, hi = self._fn_gap_blocks(sink, max(0, processed_computed_tokens // a * a - a))\n        blocks = self.req_to_blocks.get(request_id)\n        if blocks is None:\n            return\n        hi = min(hi, len(blocks))\n        if hi > lo and blocks[hi - 1] is not self._null_block:\n            before = sum(b is not self._null_block for b in blocks)\n            self._remove_blocks_in_range(request_id, lo, hi)\n            freed = before - sum(b is not self._null_block for b in blocks)\n            t = _FN_TRIM\n            t["cuts"] += 1\n            t["freed"] += freed * self.block_size\n            if t["cuts"] <= 12 or t["cuts"] % 500 == 0:\n                print(f"flashnext trim: request {request_id[:24]} (sink {sink}) at token {processed_computed_tokens}: freed {freed} blocks "\n                      f"of group {self.kv_cache_group_id} (tokens {lo * self.block_size}-{hi * self.block_size}); "\n                      f"it now holds {len(blocks) - blocks.count(self._null_block)} of {len(blocks)} blocks "\n                      f"[cuts so far {t[\'cuts\']}, tokens freed {t[\'freed\']}]", flush=True)\n        if _FN_TRIM["check"]:\n            n = min(processed_computed_tokens // self.block_size, len(blocks))\n            bad = [i for i in range(n) if blocks[i] is self._null_block and not (lo <= i < hi)]\n            if bad:\n                raise RuntimeError(f"flashnext trim CHECK: request {request_id} group {self.kv_cache_group_id} at "\n                                   f"{processed_computed_tokens}: blocks {bad[:8]} are null outside the gap [{lo}, {hi})")\n\n    def get_num_blocks_to_allocate(self, request_id, num_tokens, new_computed_blocks, total_computed_tokens,\n                                   num_local_computed_tokens, num_tokens_main_model, apply_admission_cap=False,\n                                   prefill_end=0) -> int:\n        sink = _FN_TRIM["sinks"].get(request_id)\n        if not apply_admission_cap or sink is None or not _FN_TRIM["configured"]:\n            return super().get_num_blocks_to_allocate(\n                request_id, num_tokens, new_computed_blocks, total_computed_tokens, num_local_computed_tokens,\n                num_tokens_main_model, apply_admission_cap=apply_admission_cap, prefill_end=prefill_end)\n        # admission: a trimmed request holds at most the sink + LIMIT + one chunk (vLLM would reserve the whole prompt)\n        old = self.max_admission_blocks_per_request\n        self.max_admission_blocks_per_request = (cdiv(sink, self.block_size) + 2\n                                                 + cdiv(_FN_TRIM["limit"] + _FN_TRIM["chunk"], self.block_size))\n        try:\n            return super().get_num_blocks_to_allocate(\n                request_id, num_tokens, new_computed_blocks, total_computed_tokens, num_local_computed_tokens,\n                num_tokens_main_model, apply_admission_cap=True, prefill_end=prefill_end)\n        finally:\n            self.max_admission_blocks_per_request = old\n\n    def add_local_computed_blocks(self, request_id, new_computed_blocks, num_local_computed_tokens,\n                                  num_external_computed_tokens) -> None:\n        super().add_local_computed_blocks(request_id, new_computed_blocks, num_local_computed_tokens,\n                                          num_external_computed_tokens)\n        sink = _FN_TRIM["sinks"].get(request_id)\n        if _FN_TRIM["check"] and sink is not None:\n            blocks = self.req_to_blocks[request_id]\n            lo, hi = self._fn_gap_blocks(sink, num_local_computed_tokens)\n            bad = [i for i, b in enumerate(blocks) if b is self._null_block and not (lo <= i < hi)]\n            if bad:\n                raise RuntimeError(f"flashnext trim CHECK: hit of request {request_id} group {self.kv_cache_group_id} "\n                                   f"({num_local_computed_tokens} tokens): null blocks {bad[:8]} outside the gap [{lo}, {hi})")\n\n    def allocate_external_computed_blocks(self, request_id, num_local_computed_tokens, num_external_computed_tokens) -> None:\n        """Blocks for tokens restored from RAM (KV offloading): null blocks in the cut-away middle, real ones elsewhere."""\n        sink = _FN_TRIM["sinks"].get(request_id)\n        if sink is None or not _FN_TRIM["configured"] or num_external_computed_tokens <= 0:\n            return super().allocate_external_computed_blocks(request_id, num_local_computed_tokens,\n                                                             num_external_computed_tokens)\n        total = num_local_computed_tokens + num_external_computed_tokens\n        req_blocks = self.req_to_blocks[request_id]\n        lo, hi = self._fn_gap_blocks(sink, total)\n        positions = range(len(req_blocks), cdiv(total, self.block_size))\n        new_blocks = iter(self.block_pool.get_new_blocks(sum(not (lo <= i < hi) for i in positions)))\n        added = [self._null_block if lo <= i < hi else next(new_blocks) for i in positions]\n        req_blocks.extend(added)\n        if self._record_new_block_ids:\n            self.new_block_ids.extend(b.block_id for b in added if b is not self._null_block)\n        if _FN_TRIM["check"]:\n            bad = [i for i, b in enumerate(req_blocks) if b is self._null_block and not (lo <= i < hi)]\n            if bad:\n                raise RuntimeError(f"flashnext trim CHECK: RAM restore of request {request_id} group {self.kv_cache_group_id} "\n                                   f"({total} tokens): null blocks {bad[:8]} outside the gap [{lo}, {hi})")\n\n    def pop_blocks_for_free(self, request_id: str) -> list[KVCacheBlock]:\n        _FN_TRIM["sinks"].pop(request_id, None)\n        return super().pop_blocks_for_free(request_id)\n\n    def get_num_common_prefix_blocks(self, running_request_id: str) -> int:\n        return 0\n\n    @classmethod\n    def find_longest_cache_hit(cls, block_hashes, max_length, kv_cache_group_ids, block_pool, kv_cache_spec,\n                               drop_eagle_block, alignment_tokens, dcp_world_size=1, pcp_world_size=1):\n        sink = _FN_TRIM["cur_sink"]\n        if sink is None or drop_eagle_block or not _FN_TRIM["configured"]:\n            # contiguous hits only (always valid: every block is real)\n            return super().find_longest_cache_hit(\n                block_hashes, max_length, kv_cache_group_ids, block_pool, kv_cache_spec, drop_eagle_block,\n                alignment_tokens, dcp_world_size, pcp_world_size)\n        block_size = kv_cache_spec.block_size\n        if dcp_world_size > 1:\n            block_size *= dcp_world_size\n        block_hashes = resolve_block_hashes(\n            block_hashes, block_pool.hash_block_size, block_size,\n            supports_fine_grained_hash_lookup=cls.supports_fine_grained_hash_lookup, alignment_tokens=alignment_tokens)\n        fine = alignment_tokens < block_size and block_size % alignment_tokens == 0\n        full = BlockHashListWithBlockSize(block_hashes, alignment_tokens, block_size) if fine else block_hashes\n        n_full = min(max_length // block_size, len(full))\n        present = [block_pool.get_cached_block(full[i], kv_cache_group_ids) or None for i in range(n_full)]\n        missing = [i for i, p in enumerate(present) if p is None]\n        null = block_pool.null_block\n\n        def gap(pos):\n            lo, hi = fn_trim_gap_tokens(sink, pos)\n            return cdiv(lo, block_size), hi // block_size\n\n        def ok(pos, k):                       # every block before index k that the model may read must be cached\n            j = _fn_bisect.bisect_left(missing, k)\n            if j == 0:\n                return True\n            lo, hi = gap(pos)\n            return missing[0] >= lo and missing[j - 1] < hi\n\n        def build(pos, k, tail):\n            lo, hi = gap(pos)\n            out = tuple([] for _ in kv_cache_group_ids)\n            for i in range(k):\n                if lo <= i < hi:\n                    for o in out:\n                        o.append(null)\n                else:\n                    for o, c in zip(out, present[i]):\n                        o.append(c)\n            if tail:\n                for o, c in zip(out, tail):\n                    o.append(c)\n            if missing and missing[0] < k:\n                _FN_TRIM["hits"] += 1\n            return out, pos\n\n        max_aligned = max_length - max_length % alignment_tokens\n        for k in range(n_full, -1, -1):\n            if fine and (k == n_full or present[k] is None):\n                # a cached entry of block k may end PAST max_length: KV is append-only, so the same block also serves the\n                # shorter hit (vLLM\'s coordinator does this by truncation; copy-on-write protects the shared partial block)\n                scale = block_size // alignment_tokens\n                first = k * scale\n                last = min(first + scale, len(block_hashes))\n                for fine_idx in range(last - 1, first - 1, -1):\n                    tail = block_pool.get_cached_block(block_hashes[fine_idx], kv_cache_group_ids)\n                    if not tail:\n                        continue\n                    pos = min((fine_idx + 1) * alignment_tokens, max_aligned)\n                    if pos > k * block_size and ok(pos, k):\n                        return build(pos, k, tail)\n                    break                     # a shorter tail in this block needs at least the same blocks\n            pos = k * block_size\n            if pos % alignment_tokens == 0 and ok(pos, k):\n                return build(pos, k, None)\n        return tuple([] for _ in kv_cache_group_ids), 0\n\n\ndef get_manager_for_kv_cache_spec('), ('trim free: QSA groups use the gap manager', '    manager_class = KVCacheSpecRegistry.get_manager_class(kv_cache_spec, role)\n', '    manager_class = KVCacheSpecRegistry.get_manager_class(kv_cache_spec, role)\n    if _FN_TRIM["on"] and manager_class is FullAttentionManager:\n        manager_class = FnTrimGapManager\n')]
TF_COORD_EDITS = [('trim free: a hit with a freed middle is not downward-closed, ask the gap manager again when the length shrinks', 'if isinstance(spec, FullAttentionSpec) and cached_blocks is not None:', 'if isinstance(spec, FullAttentionSpec) and cached_blocks is not None and not getattr(manager_cls, "fn_trim_gap", False):')]
TF_KVM_EDITS = [("trim free: the hit search knows the request's sink", '        max_cache_hit_length = request.num_tokens - 1\n        computed_blocks, num_new_computed_tokens, num_uncached = (\n            self.coordinator.find_longest_cache_hit(\n                request.block_hashes, max_cache_hit_length\n            )\n        )\n', '        max_cache_hit_length = request.num_tokens - 1\n        from vllm.v1.core.single_type_kv_cache_manager import _FN_TRIM as _fn_t, fn_trim_sink as _fn_sink\n\n        _fn_t["cur_sink"] = _fn_sink(request)\n        try:\n            computed_blocks, num_new_computed_tokens, num_uncached = (\n                self.coordinator.find_longest_cache_hit(\n                    request.block_hashes, max_cache_hit_length\n                )\n            )\n        finally:\n            _fn_t["cur_sink"] = None\n'), ('trim free: register the sink before freeing', '        self.coordinator.remove_skipped_blocks(\n            request.request_id,\n            max(0, total_computed_tokens - request.num_in_flight_tokens),\n', '        from vllm.v1.core.single_type_kv_cache_manager import fn_trim_sink as _fn_sink\n\n        _fn_sink(request)\n        self.coordinator.remove_skipped_blocks(\n            request.request_id,\n            max(0, total_computed_tokens - request.num_in_flight_tokens),\n')]
TF_SCHED_EDITS = [('trim free: configure the rule once', '        # Bind after construction so connectors can access the cache manager.\n        if self.connector is not None:\n            self.connector.bind_kv_cache_manager(self.kv_cache_manager)\n', '        # Bind after construction so connectors can access the cache manager.\n        if self.connector is not None:\n            self.connector.bind_kv_cache_manager(self.kv_cache_manager)\n        from vllm.v1.core.single_type_kv_cache_manager import _FN_TRIM as _fn_t, fn_trim_configure as _fn_trim_configure\n\n        _fn_trim_configure(vllm_config)\n        _fn_t["usage_fn"] = lambda: self.kv_cache_manager.usage  # for the lazy KV offloading (offload_edits.py)\n')]
OFF_EDITS = [('offload: helpers', 'from vllm.v1.request import Request, RequestStatus\n', 'from vllm.v1.request import Request, RequestStatus\n\n# ---- flashnext: gap-aware + lazy offloading (dev/patches/qsa_dcp/offload_edits.py) ----\nimport math as _fn_math\nimport os as _fn_os\n\n_FN_OFF_LAZY = _fn_os.environ.get("FLASHNEXT_OFFLOAD_LAZY") == "1"\n_FN_OFF_PRESSURE = float(_fn_os.environ.get("FLASHNEXT_OFFLOAD_PRESSURE", "0.8") or 0.8)\n\n\ndef _fn_trim_state():\n    from vllm.v1.core import single_type_kv_cache_manager as _S\n\n    return _S._FN_TRIM\n'), ('offload: sinks, state spacing, gap range, gap-aware lookup', '        self._events_tracker = OffloadingEventsTracker(spec.kv_events_config)\n', '        self._events_tracker = OffloadingEventsTracker(spec.kv_events_config)\n        # flashnext: per-request sink (for the cut-away range) and the DeltaNet checkpoint spacing a restore can use\n        self._fn_sinks: dict[str, int] = {}\n        _attn_chunks = [c.tokens_per_chunk for c in self.config.kv_group_configs if not c.requires_cow_source]\n        self._fn_state_every = _fn_math.lcm(*_attn_chunks) if (_FN_OFF_LAZY and _attn_chunks) else 0\n        if _FN_OFF_LAZY or _fn_trim_state()["on"]:\n            if self.config.blocks_per_chunk != 1:\n                raise RuntimeError("flashnext offloading edits need blocks_per_chunk == 1")\n            print(f"flashnext offload: middle-aware {_fn_trim_state()[\'on\']} | lazy {_FN_OFF_LAZY} (running requests copied "\n                  f"only above {_FN_OFF_PRESSURE:.0%} GPU KV use, finished calls always) | DeltaNet states every "\n                  f"{self._fn_state_every or \'chunk\'} tokens", flush=True)\n\n    def _fn_gap_chunks(self, req, group_config, end_tokens):\n        """Chunk range [lo, hi) of the cut-away middle for a QSA group at end_tokens, or None (not a trimmed group)."""\n        t = _fn_trim_state()\n        if not t["configured"] or group_config.group_idx not in t["gap_group_ids"]:\n            return None\n        sink = self._fn_sinks.get(req.request_id)\n        if sink is None:\n            from vllm.v1.core.single_type_kv_cache_manager import fn_trim_sink_of\n\n            sink = self._fn_sinks[req.request_id] = fn_trim_sink_of(req.prompt_token_ids or [])\n        from vllm.v1.core.single_type_kv_cache_manager import fn_trim_gap_tokens\n\n        lo, hi = fn_trim_gap_tokens(sink, end_tokens)\n        tpc = group_config.tokens_per_chunk\n        return cdiv(lo, tpc), hi // tpc\n\n    def _fn_gap_lookup(self, keys, req_status, group_config, start_chunk_idx):\n        """Like _maximal_prefix_lookup, but chunks inside the cut-away middle may be missing."""\n        present = []\n        for key in keys:\n            result = self.manager.lookup(key, req_status.req_context)\n            if result in (LookupResult.RETRY, LookupResult.HIT_PENDING):\n                return None\n            present.append(result is LookupResult.HIT)\n        tpc = group_config.tokens_per_chunk\n        for h in range(len(present), 0, -1):\n            lo, hi = self._fn_gap_chunks(req_status.req, group_config, (start_chunk_idx + h) * tpc)\n            if all(present[i] or lo <= start_chunk_idx + i < hi for i in range(h)):\n                return h\n        return 0\n\n    def _fn_low_pressure(self) -> bool:\n        usage_fn = _fn_trim_state().get("usage_fn")\n        return usage_fn is not None and usage_fn() < _FN_OFF_PRESSURE\n\n    def _fn_gap_ok_end(self, req_status, end: int) -> int:\n        """Largest token end <= end at which every chunk a RAM restore must load for the QSA groups is in RAM. The GPU side\n        puts null blocks only in the cut-away middle of the FINAL hit length, and that middle shrinks with the length."""\n        base = req_status.num_locally_computed_tokens\n        ok = end\n        for group_config, group_state in zip(self.config.kv_group_configs, req_status.group_states):\n            gap = self._fn_gap_chunks(req_status.req, group_config, end)\n            if gap is None:\n                continue\n            tpc = group_config.tokens_per_chunk\n            for i in range(base // tpc, min(cdiv(end, tpc), len(group_state.offload_keys))):\n                if gap[0] <= i < gap[1]:\n                    continue\n                if self.manager.lookup(group_state.offload_keys[i], req_status.req_context) is not LookupResult.HIT:\n                    ok = min(ok, i * tpc)\n                    break\n        return ok\n\n    def _fn_lookup(self, req_status, max_num_new_tokens):\n        """vLLM\'s lookup, then re-checked for the QSA groups at the final hit length. vLLM shrinks the hit to the shortest\n        group and re-checks only groups whose shorter hit can become invalid; a QSA group\'s hit can (its cut-away middle\n        is narrower at a shorter length), so shrink until the final length is valid (Kaggle run: \'Chunk ... not found\')."""\n        n = self._lookup(req_status, max_num_new_tokens)\n        if not n or not _fn_trim_state()["configured"]:\n            return n\n        base = req_status.num_locally_computed_tokens\n        for _ in range(256):\n            ok_end = self._fn_gap_ok_end(req_status, base + n)\n            if ok_end >= base + n:\n                return n\n            print(f"flashnext offload: RAM hit of request {req_status.req.request_id} shortened {base + n} -> {ok_end} tokens "\n                  f"(a chunk to load was in the cut-away middle at the longer length and is not in RAM)", flush=True)\n            if ok_end <= base:\n                return 0\n            cap = ok_end - base if max_num_new_tokens is None else min(ok_end - base, max_num_new_tokens)\n            n = self._lookup(req_status, cap)\n            if not n:\n                return n\n        raise RuntimeError(f"flashnext offload: RAM hit of request {req_status.req.request_id} did not converge")\n'), ('offload: QSA groups look up across the cut-away middle', '                if sliding_window_size_in_chunks is None:\n                    num_hit_chunks = self._maximal_prefix_lookup(\n', '                if sliding_window_size_in_chunks is None and self._fn_gap_chunks(\n                    req_status.req, group_config, 0\n                ) is not None:\n                    num_hit_chunks = self._fn_gap_lookup(\n                        offload_keys, req_status, group_config, start_chunk_idx\n                    )\n                elif sliding_window_size_in_chunks is None:\n                    num_hit_chunks = self._maximal_prefix_lookup(\n'), ('offload: QSA groups restore only the sink + recent part', '            num_chunks = cdiv(num_cached_tokens, tokens_per_chunk)\n            if num_pending_gpu_blocks:\n', '            if num_pending_gpu_blocks and self._fn_gap_chunks(request, group_config, 0) is not None:\n                # flashnext: the GPU manager put null blocks in the cut-away middle: load only the real blocks\n                assert partial_tail_boundary is None\n                _fn_idx = [i for i in range(load_start_gpu_block_idx, num_gpu_blocks) if not group_blocks[i].is_null]\n                keys_to_load.extend(offload_keys[i] for i in _fn_idx)\n                dst_block_ids.extend(group_blocks[i].block_id for i in _fn_idx)\n                group_sizes.append(len(_fn_idx))\n                block_indices.append(load_start_gpu_block_idx)\n                if req_status.offloading_context.policy == OffloadPolicy.CHUNK_LEVEL:\n                    group_state.next_stored_chunk_idx = cdiv(num_cached_tokens, tokens_per_chunk)\n                continue\n            num_chunks = cdiv(num_cached_tokens, tokens_per_chunk)\n            if num_pending_gpu_blocks:\n'), ('offload: re-check the QSA groups at the final RAM hit length', '            num_hit_tokens = self._lookup(req_status, max_num_new_tokens)\n', '            num_hit_tokens = self._fn_lookup(req_status, max_num_new_tokens)\n'), ('offload: lazy - running requests only under memory pressure, finished calls always', '            req = req_status.req\n\n            if req.status is RequestStatus.FINISHED_ABORTED:\n', '            req = req_status.req\n            if _FN_OFF_LAZY and not req.is_finished() and self._fn_low_pressure():\n                continue  # flashnext lazy: nothing advances, a later step (pressure or end of call) catches up\n\n            if req.status is RequestStatus.FINISHED_ABORTED:\n'), ('offload: cut-away range of each QSA group', '                offload_keys = group_state.offload_keys[start_chunk_idx:num_chunks]\n', '                _fn_gap = self._fn_gap_chunks(req, group_config, num_tokens_after_batch)\n                offload_keys = group_state.offload_keys[start_chunk_idx:num_chunks]\n'), ('offload: never store the cut-away middle', '                    abs_chunk_idx = start_chunk_idx + key_idx\n', '                    abs_chunk_idx = start_chunk_idx + key_idx\n                    if _fn_gap is not None and _fn_gap[0] <= abs_chunk_idx < _fn_gap[1]:\n                        continue\n'), ('offload: DeltaNet checkpoints only where a restore can land', '                    or boundary % group_config.tokens_per_chunk != 0\n', '                    or boundary % group_config.tokens_per_chunk != 0\n                    or (self._fn_state_every and boundary % self._fn_state_every != 0)\n'), ('offload: forget the sink of a finished request', '            req_status.finished_signaled = True\n', '            req_status.finished_signaled = True\n            self._fn_sinks.pop(req_id, None)\n')]
EV_POOL_EDITS = [('flashnext evict offload: eviction hook', '        self._emit_block_removed_events(evicted_hashes)\n        return True\n', '        self._emit_block_removed_events(evicted_hashes)\n        _fn_cb = getattr(self, "_fn_on_evict", None)\n        if _fn_cb is not None:\n            _fn_cb(block, evicted_hashes)\n        return True\n')]
EV_SINGLE_EDITS = [('flashnext evict offload: trim state keys', '    "gap_group_ids": set(), "usage_fn": None,\n}', '    "gap_group_ids": set(), "usage_fn": None,\n    # evict offload (evict_offload_edits.py): drop the cache entry of cut-away blocks; RAM lookups start at token cur_off\n    "drop_gap": _fn_os.environ.get("FLASHNEXT_OFFLOAD_EVICT") == "1", "gap_dropped": 0, "cur_off": 0,\n}'), ('flashnext evict offload: cut-away blocks are reused first and never copied', '        if hi > lo and blocks[hi - 1] is not self._null_block:\n            before = sum(b is not self._null_block for b in blocks)\n            self._remove_blocks_in_range(request_id, lo, hi)\n', '        if hi > lo and blocks[hi - 1] is not self._null_block:\n            before = sum(b is not self._null_block for b in blocks)\n            if _FN_TRIM["drop_gap"]:\n                # the model never reads these again: without a cache entry they go to the FRONT of the free queue (reused\n                # first) instead of the back (kept longer than the parked conversations), and are never copied to RAM\n                pool = self.block_pool\n                pool._fn_dropping = True\n                try:\n                    for i in range(hi - 1, lo - 1, -1):\n                        b = blocks[i]\n                        if b is self._null_block:\n                            break\n                        if b.ref_cnt == 1 and b.block_hash is not None and pool._maybe_evict_cached_block(b):\n                            _FN_TRIM["gap_dropped"] += 1\n                finally:\n                    pool._fn_dropping = False\n            self._remove_blocks_in_range(request_id, lo, hi)\n'), ('flashnext evict offload: DeltaNet states leaving a decoding request are held or dropped', '                blocks = self.req_to_blocks[request_id]\n                if blocks[last_state_block_idx] != self._null_block:\n                    self.block_pool.free_blocks([blocks[last_state_block_idx]])\n                    blocks[last_state_block_idx] = self._null_block\n', '                blocks = self.req_to_blocks[request_id]\n                if blocks[last_state_block_idx] != self._null_block:\n                    if not self._fn_retire(request_id, last_state_block_idx, blocks[last_state_block_idx]):\n                        self.block_pool.free_blocks([blocks[last_state_block_idx]])\n                    blocks[last_state_block_idx] = self._null_block\n'), ('flashnext evict offload: DeltaNet states retired after a prefill chunk are held or dropped', '        for i in range(last_block - 1, first_block - 1, -1):\n            if blocks[i].is_null:\n                continue\n            freed.append(blocks[i])\n            blocks[i] = self._null_block\n', '        for i in range(last_block - 1, first_block - 1, -1):\n            if blocks[i].is_null:\n                continue\n            if not self._fn_retire(request_id, i, blocks[i]):\n                freed.append(blocks[i])\n            blocks[i] = self._null_block\n'), ('flashnext evict offload: held DeltaNet states go back to the request when it is freed', '        return super().pop_blocks_for_free(request_id)\n\n    def get_num_skipped_tokens(self, num_computed_tokens: int) -> int:\n', '        _fn_h = self.__dict__.get("_fn_held", {}).pop(request_id, None)\n        if _fn_h:\n            _blocks = self.req_to_blocks.get(request_id)\n            for _i, _b, _a in _fn_h:\n                if _blocks is not None and _i < len(_blocks) and _blocks[_i].is_null:\n                    _blocks[_i] = _b\n                else:\n                    print(f"flashnext evict offload: held DeltaNet state {_b.block_id} of request {request_id} (index "\n                          f"{_i}) has no free slot; freed on its own", flush=True)\n                    self.block_pool.free_blocks([_b])\n        return super().pop_blocks_for_free(request_id)\n\n    def _fn_retire(self, request_id, idx, block) -> bool:\n        """flashnext evict offload: a DeltaNet state leaves a running request (vLLM frees it with its cache entry, so it\n        becomes the OLDEST data of the task, evicted before the task\'s attention data, and dense retention floods the\n        eviction queue with them). Hold the 2 newest states at segment boundaries (a RAM restore ends there) and the newest\n        other state (a follow-up may resume there); drop the cache entry of every older one so it is reused first."""\n        from vllm.v1.core.kv_cache_coordinator import _FN_SEG\n\n        if not _FN_SEG["on"] or block.block_hash is None or block.ref_cnt != 1:\n            return False\n        held = self.__dict__.setdefault("_fn_held", {}).setdefault(request_id, [])\n        aligned = ((idx + 1) * self.block_size) % self.scheduler_block_size == 0\n        held.append((idx, block, aligned))\n        same = sorted((h for h in held if h[2] == aligned), key=lambda h: h[0])\n        pool = self.block_pool\n        for h in same[: -(2 if aligned else 1)]:\n            held.remove(h)\n            pool._fn_dropping = True\n            try:\n                pool._maybe_evict_cached_block(h[1])\n            finally:\n                pool._fn_dropping = False\n            pool.free_blocks([h[1]])\n            _FN_SEG["states_dropped"] += 1\n        return True\n\n    def get_num_skipped_tokens(self, num_computed_tokens: int) -> int:\n'), ('flashnext evict offload: the gap rule knows where a RAM lookup starts', '        def gap(pos):\n            lo, hi = fn_trim_gap_tokens(sink, pos)\n            return cdiv(lo, block_size), hi // block_size\n', '        off = _FN_TRIM["cur_off"]              # RAM lookup (evict_offload_edits.py): block_hashes start at token off\n        if off % block_size:\n            raise RuntimeError(f"flashnext evict offload: RAM lookup starts at token {off}, not a multiple of the "\n                               f"{block_size}-token block of a QSA group")\n        off_b = off // block_size\n\n        def gap(pos):\n            lo, hi = fn_trim_gap_tokens(sink, pos + off)\n            return max(cdiv(lo, block_size) - off_b, 0), max(hi // block_size - off_b, 0)\n')]
EV_COORD_EDITS = [('flashnext evict offload: segment helpers', 'from vllm.v1.request import Request\n', 'from vllm.v1.request import Request\n\n# ---- flashnext: segment-ordered freeing for the evict offload (dev/patches/qsa_dcp/evict_offload_edits.py) ----\nimport os as _fn_os\n\n_FN_SEG = {\n    "on": _fn_os.environ.get("FLASHNEXT_OFFLOAD_EVICT") == "1",\n    "of": {},          # GPU block id -> segment id (blocks of finished calls that are worth copying)\n    "segs": {},        # segment id -> list of KVCacheBlock\n    "left": {},        # segment id -> number of its blocks still recorded\n    "old_states": set(),  # GPU block ids of DeltaNet states not on a segment boundary (GPU-only, never copied to RAM)\n    "next": 0,\n    "states_dropped": 0,\n    "state_groups": set(),  # KV-cache group ids of the DeltaNet (Mamba) groups  # older DeltaNet states of running requests whose cache entry was dropped (reused first)\n}\n\n\ndef _fn_seg_forget(block_id):\n    sid = _FN_SEG["of"].pop(block_id, None)\n    if sid is not None:\n        n = _FN_SEG["left"][sid] - 1\n        if n:\n            _FN_SEG["left"][sid] = n\n        else:\n            _FN_SEG["left"].pop(sid)\n            _FN_SEG["segs"].pop(sid, None)\n    _FN_SEG["old_states"].discard(block_id)\n\n\ndef _fn_seg_order(managers, request_id):\n    """Pop a request\'s blocks in allocation order such that reversed() frees them newest SEGMENT first (every group of a\n    segment together); records the segments so the offload copies a segment as a whole."""\n    from vllm.v1.kv_cache_interface import MambaSpec\n\n    entries = []\n    for m in managers:\n        is_state = isinstance(m.kv_cache_spec, MambaSpec)\n        if is_state:\n            _FN_SEG["state_groups"].add(m.kv_cache_group_id)\n        seg = m.scheduler_block_size\n        for i, b in enumerate(m.pop_blocks_for_free(request_id)):\n            entries.append(((i * m.block_size) // seg, m.kv_cache_group_id, i, b,\n                            is_state and ((i + 1) * m.block_size) % seg != 0))\n    entries.sort(key=lambda e: e[:3])\n    # a RAM restore ends at the end of the newest complete segment: copy only that DeltaNet state of each group (every\n    # state costs a whole block; one per segment would add ~80 % to a conversation\'s RAM on Kaggle)\n    newest = {}\n    for s, g, i, b, old_state in entries:\n        if not old_state and not b.is_null and b.block_hash is not None and g in _FN_SEG["state_groups"]:\n            newest[g] = max(newest.get(g, -1), i)\n    entries = [(s, g, i, b, old_state or (g in _FN_SEG["state_groups"] and i != newest.get(g)))\n               for s, g, i, b, old_state in entries]\n    cur, sid = None, None\n    for s, _g, _i, b, old_state in entries:\n        if b.is_null or b.block_hash is None:\n            continue\n        _fn_seg_forget(b.block_id)\n        if old_state:\n            _FN_SEG["old_states"].add(b.block_id)\n            continue\n        if s != cur:\n            cur, sid = s, _FN_SEG["next"]\n            _FN_SEG["next"] += 1\n            _FN_SEG["segs"][sid], _FN_SEG["left"][sid] = [], 0\n        _FN_SEG["segs"][sid].append(b)\n        _FN_SEG["left"][sid] += 1\n        _FN_SEG["of"][b.block_id] = sid\n    return [e[3] for e in entries]\n'), ('flashnext evict offload: free a finished call segment by segment', '        for manager in self.single_type_managers:\n            manager.free(request_id)\n', '        if _FN_SEG["on"]:\n            self.block_pool.free_blocks(reversed(_fn_seg_order(self.single_type_managers, request_id)))\n            return\n        for manager in self.single_type_managers:\n            manager.free(request_id)\n'), ('flashnext evict offload: deferred frees in segment order too', '        blocks: list[KVCacheBlock] = []\n        for manager in self.single_type_managers:\n            blocks.extend(manager.pop_blocks_for_free(request_id))\n        return blocks\n', '        if _FN_SEG["on"]:\n            return _fn_seg_order(self.single_type_managers, request_id)\n        blocks: list[KVCacheBlock] = []\n        for manager in self.single_type_managers:\n            blocks.extend(manager.pop_blocks_for_free(request_id))\n        return blocks\n')]
EV_SIMPLE_EDITS = [('flashnext evict offload: settings, counters, start-up line', '        else:\n            self._target_free = 0\n        self._store_event_to_blocks: dict[int, TransferMeta] = {}\n', '        else:\n            self._target_free = 0\n        # flashnext evict offload (dev/patches/qsa_dcp/evict_offload_edits.py)\n        import os as _fn_os\n        import time as _fn_time\n\n        self._fn_on = lazy_offload and _fn_os.environ.get("FLASHNEXT_OFFLOAD_EVICT") == "1"\n        if self._fn_on:\n            # vLLM\'s estimate (one prefill chunk per group, x2) is smaller than what one step can take when several\n            # requests are admitted at once (Colab E1: 650 blocks lost in the first 10 min with 20): look further ahead\n            self._target_free = int(self._target_free * float(_fn_os.environ.get("FLASHNEXT_OFFLOAD_AHEAD", "4")))\n        self._fn_state_groups = {i for i, g in enumerate(kv_cache_config.kv_cache_groups)\n                                 if isinstance(g.kv_cache_spec, MambaSpec)}\n        self._fn_n = dict.fromkeys(("copied", "restored_blocks", "restored_tokens", "restores", "safe", "lost_kv",\n                                    "lost_state", "old_state", "ram_evicted"), 0)\n        self._fn_tot = dict(self._fn_n)\n        self._fn_ram_held = 0\n        self._fn_t = _fn_time.time()\n        if self._fn_on:\n            _g = ", ".join(f"{i}:{type(g.kv_cache_spec).__name__}/{self.group_block_sizes[i]}"\n                           f"{\'\' if i in self.prefix_cacheable_group_ids else \'(not cached)\'}"\n                           for i, g in enumerate(kv_cache_config.kv_cache_groups))\n            print(f"flashnext evict offload: RAM {self.num_cpu_blocks} blocks per rank; copies only blocks within "\n                  f"{self._target_free} blocks of the GPU eviction point, in {self.block_size}-token segments; DeltaNet "\n                  f"groups {sorted(self._fn_state_groups)} copy only segment-end states | groups {_g}", flush=True)\n        self._store_event_to_blocks: dict[int, TransferMeta] = {}\n'), ('flashnext evict offload: eviction counters on both pools', '        Called by Scheduler after kv_cache_manager is ready."""\n        self._gpu_block_pool = gpu_block_pool\n', '        Called by Scheduler after kv_cache_manager is ready."""\n        self._gpu_block_pool = gpu_block_pool\n        if self._fn_on:\n            gpu_block_pool._fn_on_evict = self._fn_gpu_evicted\n            self.cpu_block_pool._fn_on_evict = self._fn_ram_evicted\n\n    def _fn_gpu_evicted(self, block, hashes) -> None:\n        from vllm.v1.core.kv_cache_coordinator import _FN_SEG, _fn_seg_forget\n\n        if getattr(self._gpu_block_pool, "_fn_dropping", False):\n            return  # a cut-away block (counted by the trim)\n        old = block.block_id in _FN_SEG["old_states"]\n        recorded = block.block_id in _FN_SEG["of"]\n        _fn_seg_forget(block.block_id)\n        n = self._fn_n\n        if self.cpu_block_pool.cached_block_hash_to_block.get_one_block(hashes[0]) is not None:\n            n["safe"] += 1\n        elif old or (get_group_id(hashes[0]) in self._fn_state_groups and not recorded):\n            n["old_state"] += 1   # a DeltaNet state kept for GPU hits only (not on a segment boundary)\n        elif get_group_id(hashes[0]) in self._fn_state_groups:\n            n["lost_state"] += 1\n        else:\n            n["lost_kv"] += 1\n\n    def _fn_ram_evicted(self, block, hashes) -> None:\n        self._fn_n["ram_evicted"] += 1\n        self._fn_ram_held -= 1\n\n    def _fn_pick(self, node, gpu_ids, picked, num_cpu_free) -> None:\n        """Copy the segment of a block near the eviction point (every group of it), skipping what is not needed."""\n        from vllm.v1.core.kv_cache_coordinator import _FN_SEG\n\n        if node.block_hash is None or node.is_null or node.block_id in picked or node.block_id in _FN_SEG["old_states"]:\n            return\n        sid = _FN_SEG["of"].get(node.block_id)\n        if sid is None:\n            if get_group_id(node.block_hash) in self._fn_state_groups:\n                return  # a DeltaNet state retired while its request ran: never needed by a follow-up call\n            cands = [node]\n        else:\n            cands = [b for b in _FN_SEG["segs"].get(sid, (node,)) if _FN_SEG["of"].get(b.block_id) == sid]\n        cpu_map = self.cpu_block_pool.cached_block_hash_to_block\n        for b in cands:\n            if len(gpu_ids) >= num_cpu_free:\n                return\n            if (b.block_id in picked or b.ref_cnt != 0 or b.is_null or b.block_hash is None\n                    or cpu_map.get_one_block(b.block_hash) is not None):\n                continue\n            gpu_ids.append(b.block_id)\n            picked.add(b.block_id)\n\n    def _fn_report(self) -> None:\n        import time as _fn_time\n\n        now = _fn_time.time()\n        if now - self._fn_t < 60:\n            return\n        self._fn_t = now\n        n, t = self._fn_n, self._fn_tot\n        for k, v in n.items():\n            t[k] += v\n        from vllm.v1.core.kv_cache_coordinator import _FN_SEG\n        from vllm.v1.core.single_type_kv_cache_manager import _FN_TRIM\n\n        print(f"flashnext evict offload (last 60 s | total): copied to RAM {n[\'copied\']} | {t[\'copied\']} blocks; restores "\n              f"{n[\'restores\']} | {t[\'restores\']} ({n[\'restored_tokens\']} | {t[\'restored_tokens\']} tokens); GPU evictions: "\n              f"already in RAM {n[\'safe\']} | {t[\'safe\']}, LOST (recomputed if needed) KV {n[\'lost_kv\']} | {t[\'lost_kv\']} "\n              f"DeltaNet {n[\'lost_state\']} | {t[\'lost_state\']}, GPU-only DeltaNet states {n[\'old_state\']} | "\n              f"{t[\'old_state\']}; older DeltaNet states dropped {_FN_SEG[\'states_dropped\']}; RAM holds {self._fn_ram_held}/{self.num_cpu_blocks} blocks "\n              f"({100 * self._fn_ram_held / max(self.num_cpu_blocks, 1):.0f} %), RAM evictions {n[\'ram_evicted\']} | "\n              f"{t[\'ram_evicted\']}; cut-away blocks dropped {_FN_TRIM[\'gap_dropped\']}; GPU free queue "\n              f"{self._gpu_block_pool.get_num_free_blocks()} blocks", flush=True)\n        for k in n:\n            n[k] = 0\n        # census of the parked part of the GPU cache (free queue, eviction order): what would be lost next\n        c = dict.fromkeys(("empty", "conversation KV", "segment-end DeltaNet state", "GPU-only DeltaNet state (call ended)",\n                           "DeltaNet state not held (retired while running)", "other cached KV"), 0)\n        in_ram = 0\n        cpu_map = self.cpu_block_pool.cached_block_hash_to_block\n        for b in self._gpu_block_pool.free_block_queue.iter_blocks_after(None):\n            h = b.block_hash\n            if h is None or b.is_null:\n                c["empty"] += 1\n                continue\n            in_ram += cpu_map.get_one_block(h) is not None\n            st = get_group_id(h) in self._fn_state_groups\n            if b.block_id in _FN_SEG["old_states"]:\n                c["GPU-only DeltaNet state (call ended)"] += 1\n            elif b.block_id in _FN_SEG["of"]:\n                c["segment-end DeltaNet state" if st else "conversation KV"] += 1\n            else:\n                c["DeltaNet state not held (retired while running)" if st else "other cached KV"] += 1\n        tot = self._gpu_block_pool.num_gpu_blocks - 1\n        print("flashnext evict offload: GPU blocks " + f"{tot}: in use by running requests "\n              f"{tot - sum(c.values())}, parked: " + ", ".join(f"{k} {v}" for k, v in c.items())\n              + f" (of the parked cached blocks {in_ram} are also in RAM)", flush=True)\n'), ('flashnext evict offload: the RAM lookup knows the trim (sink, start token)', '        cpu_hit_blocks, hit_length, _ = self.cpu_coordinator.find_longest_cache_hit(\n            remaining_hashes, max_hit_len\n        )\n', '        from vllm.v1.core import single_type_kv_cache_manager as _fn_S\n\n        _fn_t = _fn_S._FN_TRIM if self._fn_on and _fn_S._FN_TRIM["configured"] else None\n        if _fn_t is not None:\n            _fn_t["cur_sink"], _fn_t["cur_off"] = _fn_S.fn_trim_sink(request), num_computed_tokens\n        try:\n            cpu_hit_blocks, hit_length, _ = self.cpu_coordinator.find_longest_cache_hit(\n                remaining_hashes, max_hit_len\n            )\n        finally:\n            if _fn_t is not None:\n                _fn_t["cur_sink"], _fn_t["cur_off"] = None, 0\n'), ('flashnext evict offload: a restore skips the cut-away middle (null on the GPU)', '            for i, cpu_blk in enumerate(cpu_blocks_g):\n                # Skip null blocks (e.g. sliding window or mamba padding).\n                if cpu_blk.is_null:\n                    continue\n                gpu_block_ids.append(group_gpu_ids[gpu_ext_start + i])\n', '            for i, cpu_blk in enumerate(cpu_blocks_g):\n                if self._fn_on and group_gpu_ids[gpu_ext_start + i] == 0:\n                    # the GPU side put the null block here (cut-away middle): nothing to load\n                    continue\n                if self._fn_on and cpu_blk.is_null and group_gpu_ids[gpu_ext_start + i] != 0:\n                    from vllm.v1.core.single_type_kv_cache_manager import _FN_TRIM\n\n                    if g in _FN_TRIM["gap_group_ids"]:\n                        raise RuntimeError(\n                            f"flashnext evict offload: request {req_id} group {g} block {gpu_ext_start + i}: the GPU "\n                            f"expects data but the RAM hit has the cut-away middle there ({num_computed_tokens} + "\n                            f"{num_external_tokens} tokens)")\n                # Skip null blocks (e.g. sliding window or mamba padding).\n                if cpu_blk.is_null:\n                    continue\n                gpu_block_ids.append(group_gpu_ids[gpu_ext_start + i])\n'), ('flashnext evict offload: restore counters', '        # Touch CPU blocks to prevent eviction during async load.\n        self.cpu_block_pool.touch(cpu_blocks_to_touch)\n', '        if self._fn_on:\n            self._fn_n["restores"] += 1\n            self._fn_n["restored_blocks"] += len(cpu_block_ids)\n            self._fn_n["restored_tokens"] += num_external_tokens\n        # Touch CPU blocks to prevent eviction during async load.\n        self.cpu_block_pool.touch(cpu_blocks_to_touch)\n'), ('flashnext evict offload: report every 60 s', '        # --- Stores ---\n        store_event = -1\n', '        if self._fn_on:\n            self._fn_report()\n        # --- Stores ---\n        store_event = -1\n'), ('flashnext evict offload: walk from the eviction point every step', '        # Validate cursor: stale if block was removed from free queue.\n        if self._cursor is not None and self._cursor.ref_cnt > 0:\n            self._cursor = None\n', '        # Validate cursor: stale if block was removed from free queue.\n        if self._cursor is not None and self._cursor.ref_cnt > 0:\n            self._cursor = None\n        if self._fn_on:\n            self._cursor = None  # flashnext: only the blocks next in line for eviction, never a sweep of the whole queue\n        _fn_picked: set[int] = set()\n'), ('flashnext evict offload: copy whole segments, skip what is not needed', '            if (\n                bhash is not None\n                and not node.is_null\n                and cpu_pool.cached_block_hash_to_block.get_one_block(bhash) is None\n            ):\n                gpu_ids.append(node.block_id)\n', '            if self._fn_on:\n                self._fn_pick(node, gpu_ids, _fn_picked, num_cpu_free)\n            elif (\n                bhash is not None\n                and not node.is_null\n                and cpu_pool.cached_block_hash_to_block.get_one_block(bhash) is None\n            ):\n                gpu_ids.append(node.block_id)\n'), ('flashnext evict offload: copy counters', "        # Free CPU and GPU blocks' ref counts to turn them into prefix cache\n        self.cpu_block_pool.free_blocks(cpu_blocks)\n", '        if self._fn_on:\n            self._fn_n["copied"] += len(cpu_blocks)\n            self._fn_ram_held += len(cpu_blocks)\n        # Free CPU and GPU blocks\' ref counts to turn them into prefix cache\n        self.cpu_block_pool.free_blocks(cpu_blocks)\n')]
EV_CONFIG_EDITS = [('flashnext evict offload: --kv-offloading-size uses the lazy SimpleCPUOffloadConnector', '            if envs.VLLM_USE_SIMPLE_KV_OFFLOAD:\n                config_connector = "SimpleCPUOffloadConnector"\n            else:\n                config_connector = "OffloadingConnector"\n            self.kv_transfer_config.kv_connector = config_connector\n            self.kv_transfer_config.kv_connector_extra_config.update(\n                {"cpu_bytes_to_use": kv_offloading_size * (1 << 30)}\n            )\n', '            _fn_evict = os.environ.get("FLASHNEXT_OFFLOAD_EVICT") == "1"  # flashnext evict_offload_edits.py\n            if envs.VLLM_USE_SIMPLE_KV_OFFLOAD or _fn_evict:\n                config_connector = "SimpleCPUOffloadConnector"\n            else:\n                config_connector = "OffloadingConnector"\n            self.kv_transfer_config.kv_connector = config_connector\n            self.kv_transfer_config.kv_connector_extra_config.update(\n                {"cpu_bytes_to_use": kv_offloading_size * (1 << 30)}\n            )\n            if _fn_evict:\n                self.kv_transfer_config.kv_connector_extra_config["lazy_offload"] = True\n')]
_EV = bool(KV_OFFLOAD_GIB and OFFLOAD_EVICT)
DIAG_CACHE_EDITS = [('encoder cache records stores and frees', 'class EncoderCache:\n    def __init__(self):\n        # req_id -> MM features\n        self.mm_features: dict[str, list[MultiModalFeatureSpec]] = {}\n        # MM hash -> encoder outputs\n        self.encoder_outputs: dict[str, torch.Tensor] = {}\n', 'class _QsaDcpTrackedDict(dict):  # qsa_dcp cell diagnostics: history of every store / free per mm hash\n    def __init__(self):\n        super().__init__()\n        self.events = {}\n        self.n = 0\n\n    def _note(self, key, what):\n        self.n += 1\n        self.events[key] = (self.events.get(key, "") + f" {what}@{self.n}")[-300:]\n\n    def __setitem__(self, key, value):\n        self._note(key, "set")\n        super().__setitem__(key, value)\n\n    def update(self, *args, **kwargs):\n        for key, value in dict(*args, **kwargs).items():\n            self[key] = value\n\n    def pop(self, key, *default):\n        if key in self:\n            self._note(key, "pop")\n        return super().pop(key, *default)\n\n\nclass EncoderCache:\n    def __init__(self):\n        # req_id -> MM features\n        self.mm_features: dict[str, list[MultiModalFeatureSpec]] = {}\n        # MM hash -> encoder outputs\n        self.encoder_outputs: dict[str, torch.Tensor] = _QsaDcpTrackedDict()\n')]
DIAG_RUNNER_EDITS = [('log the situation at an encoder cache miss', '                    raise RuntimeError(f"Encoder cache miss for {mm_hash}.")\n', '                    _ev = getattr(self.encoder_cache.encoder_outputs, "events", {})\n                    logger.error(\n                        "ENCODER MISS DIAG: req=%s window=[%d,%d) item=%d of %d offset=%d len=%d "\n                        "data_is_none=%s cache_entries=%d history=%r all_items=%r",\n                        req_id, cur_query_start, cur_query_end, idx, len(mm_features), start_pos,\n                        num_encoder_tokens, getattr(mm_feature, "data", None) is None,\n                        len(self.encoder_cache.encoder_outputs), _ev.get(mm_hash, "never stored"),\n                        [(f.identifier[:10], f.mm_position.offset, f.mm_position.length,\n                          f.identifier in self.encoder_cache.encoder_outputs,\n                          getattr(f, "data", None) is None) for f in mm_features],\n                    )\n                    raise RuntimeError(f"Encoder cache miss for {mm_hash}.")\n')]
TARGETS = [(VROOT / "models/qwen4_exp/nvidia/qsa.py", QSA_EDITS + (QSA_TRIM_EDITS if MEM_TRIMS else [])),
           (VROOT / "models/qwen4_exp/common/qsa_cache.py", CACHE_EDITS),
           (VROOT / "engine/arg_utils.py", ARGS_EDITS),
           (VROOT / "v1/worker/gpu/mm/encoder_cache.py", DIAG_CACHE_EDITS if DIAG_ENCODER_MISS else []),
           (VROOT / "v1/worker/gpu/mm/encoder_runner.py", DIAG_RUNNER_EDITS if DIAG_ENCODER_MISS else []),
           (VROOT / "third_party/flash_linear_attention/ops/fused_recurrent.py", GDN_EDITS if MAMBA_SSM_FP16_SR else []),
           (VROOT / "models/qwen4_exp/nvidia/hyperconnection.py", HC_EDITS if FP8_HC_LMHEAD else []),
           (VROOT / "models/qwen4_exp/nvidia/model.py", LMHEAD_EDITS if FP8_HC_LMHEAD else []),
           (VROOT / "v1/worker/gpu/model_runner.py", DIAG_WEIGHTS_V2_EDITS if DIAG_WEIGHTS else []),
           (VROOT / "v1/worker/gpu_model_runner.py", DIAG_WEIGHTS_V1_EDITS if DIAG_WEIGHTS else []),
           (VROOT / "model_executor/layers/quantization/utils/humming/linear.py", HUMMING_TRIM_EDITS if MEM_TRIMS else []),
           (VROOT / "v1/core/single_type_kv_cache_manager.py", (PREFIX_REUSE_EDITS if PREFIX_REUSE else []) + (TF_SINGLE_EDITS if TRIM_FREE else [])
            + (EV_SINGLE_EDITS if _EV else [])),
           (VROOT / "distributed/device_communicators/cuda_communicator.py", QAR_EDITS),
           (VROOT / "models/qwen4_exp/nvidia/ops/qsa_indexer.py", SW_EDITS),
           (VROOT / "models/qwen4_exp/nvidia/indexer_qsa.py", SW_INDEXER_EDITS),
           (VROOT / "models/qwen4_exp/nvidia/model_state.py", SW_STATE_EDITS),
           (VROOT / "v1/core/kv_cache_coordinator.py", (TF_COORD_EDITS if TRIM_FREE else []) + (EV_COORD_EDITS if _EV else [])),
           (VROOT / "v1/core/kv_cache_manager.py", TF_KVM_EDITS if TRIM_FREE else []),
           (VROOT / "v1/core/sched/scheduler.py", TF_SCHED_EDITS if TRIM_FREE else []),
           (VROOT / "distributed/kv_transfer/kv_connector/v1/offloading/scheduler.py", OFF_EDITS if KV_OFFLOAD_GIB else []),
           (VROOT / "v1/core/block_pool.py", EV_POOL_EDITS if _EV else []),
           (VROOT / "v1/simple_kv_offload/manager.py", EV_SIMPLE_EDITS if _EV else []),
           (VROOT / "config/vllm.py", EV_CONFIG_EDITS if _EV else [])]
OPS = VROOT / "models/qwen4_exp/nvidia/ops/qsa_dcp.py"

def _replace_once(text, old, new, name):
    if new in text:
        print(f"  [already patched] {name}"); return text
    n = text.count(old)
    if n != 1:
        raise RuntimeError(f"QSA DCP patch: anchor '{name}' found {n} times (expected 1). Is this vLLM 0.31.0? Nothing was changed in this file.")
    print(f"  [patch] {name}"); return text.replace(old, new, 1)

if DCP_SIZE == 1:
    for path, _ in TARGETS:
        b = Path(str(path) + BAK)
        if b.exists():
            shutil.copy2(b, path); print(f"  [restored] {path}")
    if OPS.exists():
        OPS.unlink(); print(f"  [removed] {OPS}")
    os.environ.pop("FLASHNEXT_DCP_SIZE", None)
    os.environ.pop("FLASHNEXT_QSA_COMPRESSED_BLOCK_FACTOR", None)
    os.environ.pop("FLASHNEXT_DCP_COMM", None)
    os.environ.pop("FLASHNEXT_MAX_CUDAGRAPH", None)
    os.environ.pop("VLLM_SPARSE_INDEXER_MAX_LOGITS_MB", None)
    for _k in ("FLASHNEXT_MAX_BATCHED", "FLASHNEXT_MM_PROCESSOR_KWARGS", "FLASHNEXT_LIMIT_MM", "PYTORCH_CUDA_ALLOC_CONF",
               "FLASHNEXT_MAMBA_SSM_DTYPE", "FLASHNEXT_PROFILER_DIR", "FLASHNEXT_CUSTOM_ALL_REDUCE", "FLASHNEXT_FP8_HC",
               "FLASHNEXT_DIAG_WEIGHTS", "FLASHNEXT_MEM_TRIMS", "FLASHNEXT_PREFIX_MATCH_UNIT",
               "FLASHNEXT_PREFIX_RETENTION", "FLASHNEXT_DECODE_PARTIAL", "FLASHNEXT_QAR", "FLASHNEXT_QAR_CODEC",
               "FLASHNEXT_QAR_BLOCK", "FLASHNEXT_QAR_MIN_MB", "FLASHNEXT_QSA_SINK", "FLASHNEXT_QSA_WINDOW",
               "FLASHNEXT_QSA_JUMP_KEEP", "FLASHNEXT_QSA_JUMP_LIMIT", "FLASHNEXT_TRIM_FREE", "FLASHNEXT_QSA_SKIP_GAP",
               "FLASHNEXT_KV_OFFLOAD_GIB", "FLASHNEXT_OFFLOAD_LAZY", "FLASHNEXT_OFFLOAD_EVICT"):
        os.environ.pop(_k, None)
    if "MODEL_PATH" in globals():
        MODEL_DIR = Path(MODEL_PATH); print(f"  MODEL_DIR back to the attached model {MODEL_DIR}")
    if "KIND" in globals():
        GPU_UTIL = KIND["gpu_util"]; print(f"  GPU_UTIL back to step 1's value {GPU_UTIL}")
    print("QSA DCP is OFF: vLLM files restored; step 6 starts the server exactly as before.")
else:
    assert DCP_SIZE == 2, "Flash-Next has 2 KV heads at TP 4: the only valid DCP size is 2"
    assert COMPRESSED_BLOCK_FACTOR in (1, 2, 4, 8), "COMPRESSED_BLOCK_FACTOR must be 1, 2, 4 or 8"
    assert DCP_COMM_BACKEND in (None, "ag_rs", "a2a"), "DCP_COMM_BACKEND must be None, 'ag_rs' or 'a2a'"
    assert GPU_UTIL_OVERRIDE is None or 0.5 < GPU_UTIL_OVERRIDE < 0.99
    assert not (MAMBA_SSM_FP16_SR and MAMBA_SSM_BF16), "choose at most one of MAMBA_SSM_FP16_SR / MAMBA_SSM_BF16"
    assert not TRIM_FREE or (SINK_JUMP is not None and SINK_JUMP[0] == "auto"),         "TRIM_FREE needs SINK_JUMP = ('auto', keep, limit) (the freed middle is the one the QSA layers no longer select)"
    assert PREFIX_RETENTION in (None, "dense") or (isinstance(PREFIX_RETENTION, int) and PREFIX_RETENTION >= 0),         "PREFIX_RETENTION must be None, 'dense' or a non-negative int (multiple of the scheduler block size)"
    assert QAR in (None, "a2a", "rs") and QAR_CODEC in ("int8", "fp8") and 128 % QAR_BLOCK == 0 and QAR_MIN_MB >= 1.0,         "QAR must be None, 'a2a' or 'rs'; QAR_CODEC 'int8' or 'fp8'; QAR_BLOCK a divisor of 128; QAR_MIN_MB >= 1.0 (decode stays BF16)"
    assert SINK_WINDOW is None or (len(SINK_WINDOW) == 2 and (SINK_WINDOW[0] == "auto" or SINK_WINDOW[0] >= 0) and SINK_WINDOW[1] >= 2052),         "SINK_WINDOW must be None or (sink, window) with sink = tokens or 'auto' and window >= 2,052 (the QSA budget 2,048 + 4)"
    assert SINK_JUMP is None or (len(SINK_JUMP) == 3 and (SINK_JUMP[0] == "auto" or SINK_JUMP[1] - SINK_JUMP[0] >= 2052)
                                 and SINK_JUMP[2] > SINK_JUMP[1]),         "SINK_JUMP must be None or (sink, keep, limit) with sink = tokens or 'auto', keep - sink >= 2,052, limit > keep"
    assert not (SINK_WINDOW and SINK_JUMP), "choose at most one of SINK_WINDOW / SINK_JUMP"
    assert not _EV or TRIM_FREE, "OFFLOAD_EVICT needs TRIM_FREE (its RAM lookup and restore know the cut-away middle)"
    if _EV:
        assert KV_OFFLOAD_GIB > 0, "KV_OFFLOAD_GIB must be None or a positive number of GiB"
        _ram_avail = int([l.split()[1] for l in open("/proc/meminfo") if l.startswith("MemAvailable:")][0]) / 2**20
        print(f"  KV offloading (copy only before eviction): {KV_OFFLOAD_GIB} GiB asked (pinned host RAM) | RAM available {_ram_avail:.0f} GiB")
        if _ram_avail < KV_OFFLOAD_GIB + 8:
            raise RuntimeError(f"KV_OFFLOAD_GIB = {KV_OFFLOAD_GIB} does not fit: RAM available {_ram_avail:.0f} GiB (needs the size "
                               f"+ 8 GiB). Lower it or set None.")
    elif KV_OFFLOAD_GIB:
        assert KV_OFFLOAD_GIB > 0, "KV_OFFLOAD_GIB must be None or a positive number of GiB"
        _st = os.statvfs("/dev/shm"); _shm_free = _st.f_bavail * _st.f_frsize / 2**30
        _ram_free = os.sysconf("SC_AVPHYS_PAGES") * os.sysconf("SC_PAGE_SIZE") / 2**30
        print(f"  KV offloading: {KV_OFFLOAD_GIB} GiB asked | /dev/shm free {_shm_free:.0f} GiB | RAM free {_ram_free:.0f} GiB")
        if _shm_free < KV_OFFLOAD_GIB * 1.05 or _ram_free < KV_OFFLOAD_GIB + 8:
            raise RuntimeError(f"KV_OFFLOAD_GIB = {KV_OFFLOAD_GIB} does not fit: /dev/shm free {_shm_free:.0f} GiB, RAM free "
                               f"{_ram_free:.0f} GiB (needs the size in /dev/shm and size + 8 GiB of RAM). Lower it or set None.")
    assert not CUSTOM_ALL_REDUCE, ("CUSTOM_ALL_REDUCE: vLLM refuses custom all-reduce on more than two PCIe-only GPUs, and forced "
                                   "on it measured 5x slower than NCCL on this machine (docs/10). Leave it False.")
    for path, edits in TARGETS:
        b = Path(str(path) + BAK)
        orig = b.read_text() if b.exists() else path.read_text()
        new = orig
        for name, old, rep in edits:
            new = _replace_once(new, old, rep, name)
        if not b.exists():
            shutil.copy2(path, b)
        if new != path.read_text():
            path.write_text(new)
        py_compile.compile(str(path), doraise=True)
    OPS.write_text(QSA_DCP_PY)
    py_compile.compile(str(OPS), doraise=True)
    print(f"  [installed] {OPS}")
    os.environ["FLASHNEXT_DCP_SIZE"] = str(DCP_SIZE)
    os.environ["FLASHNEXT_QSA_COMPRESSED_BLOCK_FACTOR"] = str(COMPRESSED_BLOCK_FACTOR)
    for _k, _v in (("FLASHNEXT_DCP_COMM", DCP_COMM_BACKEND), ("FLASHNEXT_MAX_CUDAGRAPH", MAX_CUDAGRAPH_CAPTURE_SIZE),
                   ("VLLM_SPARSE_INDEXER_MAX_LOGITS_MB", SPARSE_INDEXER_MAX_LOGITS_MB),
                   ("FLASHNEXT_MAX_BATCHED", MAX_BATCHED_TOKENS),
                   ("FLASHNEXT_MM_PROCESSOR_KWARGS", None if MM_MAX_PIXELS is None else '{"max_pixels": %d}' % MM_MAX_PIXELS),
                   ("FLASHNEXT_LIMIT_MM", '{"video": 0}' if DISABLE_VIDEO else None),
                   ("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True" if EXPANDABLE_SEGMENTS else None),
                   ("FLASHNEXT_MAMBA_SSM_DTYPE", "float16" if MAMBA_SSM_FP16_SR else ("bfloat16" if MAMBA_SSM_BF16 else None)),
                   ("FLASHNEXT_PROFILER_DIR", "/kaggle/working/prof" if PROFILE else None),
                   ("FLASHNEXT_CUSTOM_ALL_REDUCE", "1" if CUSTOM_ALL_REDUCE else None),
                   ("FLASHNEXT_FP8_HC", "1" if FP8_HC_LMHEAD else None),
                   ("FLASHNEXT_DIAG_WEIGHTS", "1" if DIAG_WEIGHTS else None),
                   ("FLASHNEXT_MEM_TRIMS", "1" if MEM_TRIMS else None),
                   ("FLASHNEXT_PREFIX_MATCH_UNIT", PREFIX_MATCH_UNIT or (800 if PREFIX_REUSE else None)),
                   ("FLASHNEXT_PREFIX_RETENTION", PREFIX_RETENTION if PREFIX_RETENTION is not None else ("dense" if PREFIX_REUSE else None)),
                   ("FLASHNEXT_DECODE_PARTIAL", "1" if PREFIX_REUSE else None),
                   ("FLASHNEXT_QAR", QAR), ("FLASHNEXT_QAR_CODEC", QAR_CODEC if QAR else None),
                   ("FLASHNEXT_QAR_BLOCK", QAR_BLOCK if QAR else None), ("FLASHNEXT_QAR_MIN_MB", QAR_MIN_MB if QAR else None),
                   ("FLASHNEXT_QSA_SINK", SINK_WINDOW[0] if SINK_WINDOW else (SINK_JUMP[0] if SINK_JUMP else None)),
                   ("FLASHNEXT_QSA_WINDOW", SINK_WINDOW[1] if SINK_WINDOW else None),
                   ("FLASHNEXT_QSA_JUMP_KEEP", SINK_JUMP[1] if SINK_JUMP else None),
                   ("FLASHNEXT_QSA_JUMP_LIMIT", SINK_JUMP[2] if SINK_JUMP else None),
                   ("FLASHNEXT_TRIM_FREE", "1" if TRIM_FREE else None),
                   ("FLASHNEXT_QSA_SKIP_GAP", "1" if QSA_SKIP_GAP and (SINK_JUMP or SINK_WINDOW) else None),
                   ("FLASHNEXT_KV_OFFLOAD_GIB", KV_OFFLOAD_GIB),
                   ("FLASHNEXT_OFFLOAD_LAZY", "1" if KV_OFFLOAD_GIB else None),
                   ("FLASHNEXT_OFFLOAD_EVICT", "1" if _EV else None)):
        if _v is None: os.environ.pop(_k, None)
        else: os.environ[_k] = str(_v)
    if GPU_UTIL_OVERRIDE is not None:
        GPU_UTIL = GPU_UTIL_OVERRIDE
    _OVERLAY = ["dense-00000.safetensors", "dense-00001.safetensors", "dense-00002.safetensors", "config.json",
                "model.safetensors.index.json"]
    if FP8_DENSE_OVERLAY:
        _src = Path(MODEL_PATH)
        if FP8_HC_LMHEAD:
            _ov = Path(FP8_HC_OVERLAY_PATH)
        elif FP8_OVERLAY_PATH:
            _ov = Path(FP8_OVERLAY_PATH)
        else:
            _found = subprocess.run("find /kaggle/input -maxdepth 8 -name dense-00000.safetensors 2>/dev/null", shell=True,
                                    capture_output=True, text=True).stdout.split()
            _cands = []
            for _f in _found:
                _d = Path(_f).parent
                try:
                    _qc = json.loads((_d / "config.json").read_text()).get("quantization_config", {})
                    if "group_1" in _qc.get("config_groups", {}):
                        _cands.append(_d)
                except Exception:
                    pass
            if len(_cands) != 1:
                raise RuntimeError(f"FP8 overlay: expected exactly one attached FP8 dataset folder, found {_cands}. "
                                   "Attach the dataset 'qwen38-flash-next-fp8-dense' or set FP8_OVERLAY_PATH.")
            _ov = _cands[0]
        _miss = [f for f in _OVERLAY if not (_ov / f).exists()]
        if _miss:
            raise RuntimeError(f"FP8 overlay folder {_ov} is missing {_miss}")
        _has_hc = "hyper_connection" in json.dumps(
            json.loads((_ov / "config.json").read_text())["quantization_config"]["config_groups"])
        if _has_hc != FP8_HC_LMHEAD:
            raise RuntimeError(f"FP8 overlay {_ov}: its config {'has' if _has_hc else 'has no'} FP8 hyper-connection/lm_head "
                               f"layers but FP8_HC_LMHEAD = {FP8_HC_LMHEAD}. Use the matching dataset / switch.")
        if FP8_HC_LMHEAD:
            import re as _re
            _tg = [t[3:] for g in json.loads((_ov / "config.json").read_text())["quantization_config"]["config_groups"].values()
                   for t in g.get("targets", []) if t.startswith("re:")]
            _bad = [n for n in ("language_model.lm_head",
                                "language_model.model.layers.0.attn_hyper_connection.input_mix_weight_down",
                                "language_model.model.layers.47.mlp_hyper_connection.input_mix_weight_up",
                                "language_model.model.hyper_connection_mixer.input_mix_weight_down")
                    if not any(_re.match(t, n) for t in _tg)]
            if _bad:
                raise RuntimeError(f"FP8 overlay {_ov}: its config.json does not give FP8 to {_bad} (old dataset version?). "
                                   "Attach the latest version of qwen38-flash-next-fp8-dense-hc.")
        _out = Path("/tmp/flashnext_fp8")
        if _out.exists():
            shutil.rmtree(_out)
        _out.mkdir()
        for _f in _src.iterdir():
            os.symlink(_ov / _f.name if _f.name in _OVERLAY else _f, _out / _f.name)
        MODEL_DIR = _out
        print(f"  FP8 dense{' + hyper-connection + lm_head' if FP8_HC_LMHEAD else ''}: MODEL_DIR = {_out} (links to {_src} + 5 files from {_ov})")
    elif "MODEL_PATH" in globals():
        MODEL_DIR = Path(MODEL_PATH)
        if FP8_HC_LMHEAD:
            import re as _re
            _qc = json.loads((MODEL_DIR / "config.json").read_text()).get("quantization_config", {})
            _tg = [t[3:] for g in _qc.get("config_groups", {}).values() for t in g.get("targets", []) if t.startswith("re:")]
            _bad = [n for n in ("language_model.lm_head",
                                "language_model.model.layers.0.attn_hyper_connection.input_mix_weight_down",
                                "language_model.model.layers.47.mlp_hyper_connection.input_mix_weight_up",
                                "language_model.model.hyper_connection_mixer.input_mix_weight_down")
                    if not any(_re.match(t, n) for t in _tg)]
            if _bad:
                raise RuntimeError(f"FP8_HC_LMHEAD = True but the attached model {MODEL_DIR} has no FP8 config for {_bad}: attach "
                                   "VERSION 2 of qwen38-flash-next-q2-gsq-vllm (FP8 merged in) and set MODEL_PATH (step 1) to its "
                                   "folder .../pytorch/default/2, or set FP8_DENSE_OVERLAY = True with the overlay dataset.")
            print(f"  FP8 dense + hyper-connection + lm_head: from the attached model itself ({MODEL_DIR}, no overlay)")
    env = vllm_env() if "vllm_env" in globals() else dict(os.environ, PYTHONPATH=str(VROOT.parent))
    code = ("import vllm.models.qwen4_exp.nvidia.model; "
            "from vllm.utils.argparse_utils import FlexibleArgumentParser; from vllm.engine.arg_utils import AsyncEngineArgs as A; "
            "a = A.add_cli_args(FlexibleArgumentParser()).parse_args(['--model', 'x', '--tensor-parallel-size', '4']); "
            "import vllm.envs as E; "
            "import tempfile; _m = tempfile.mkdtemp(); "
            "e = A.from_cli_args(A.add_cli_args(FlexibleArgumentParser()).parse_args(['--model', _m, '--tensor-parallel-size', '4', '--max-num-batched-tokens', '4096'])); "
            "print('server settings now: dcp', e.decode_context_parallel_size, '| dcp_comm_backend', e.dcp_comm_backend, '| max_cudagraph_capture_size', e.max_cudagraph_capture_size, "
            "'| indexer logits MB', E.VLLM_SPARSE_INDEXER_MAX_LOGITS_MB, '| max_num_batched_tokens (step 6 asks 4096)', e.max_num_batched_tokens, "
            "'| mm_processor_kwargs', e.mm_processor_kwargs, '| limit_mm_per_prompt', e.limit_mm_per_prompt, '| alloc conf', __import__('os').environ.get('PYTORCH_CUDA_ALLOC_CONF'), "
            "'| mamba_ssm_cache_dtype', e.mamba_ssm_cache_dtype, '| profiler (endpoints)', getattr(A.add_cli_args(FlexibleArgumentParser()).parse_args(['--model', _m]).profiler_config, 'profiler', None), '| custom all-reduce forced on', __import__('os').environ.get('FLASHNEXT_CUSTOM_ALL_REDUCE') == '1', '| prefix_match_unit', e.prefix_match_unit, '| prefix_cache_retention_interval', e.prefix_cache_retention_interval); "
            "import vllm.models.qwen4_exp.nvidia.qsa as q, vllm.models.qwen4_exp.nvidia.ops.qsa_dcp as d; "
            "import vllm.engine.arg_utils as p, inspect; "
            "assert q.Qwen4ExpQSAFlashAttentionImpl.supports_dcp and hasattr(q.Qwen4ExpQSAFlashAttentionImpl, '_forward_qsa_dcp'); "
            "assert 'FLASHNEXT_DCP_SIZE' in inspect.getsource(p); "
            "import vllm.third_party.flash_linear_attention.ops.fused_recurrent as fr; "
            "print('GDN FP16 stochastic rounding installed:', hasattr(fr, '_qsa_sr_fp16')); "
            "import vllm.models.qwen4_exp.nvidia.hyperconnection as hc; "
            "print('FP8 hyper-connection/lm_head patch installed:', hasattr(hc, '_flashnext_fp8_hc_quant_config')); "
            "import vllm.model_executor.layers.quantization.utils.humming.linear as hl; "
            "print('memory trims installed:', hasattr(q, '_FLASHNEXT_SHARED_TOPK') and hasattr(hl, '_flashnext_small_pad'), "
            "'| active:', __import__('os').environ.get('FLASHNEXT_MEM_TRIMS') == '1'); "
            "import vllm.v1.core.single_type_kv_cache_manager as sk; "
            "print('prefix reuse patch installed:', 'FLASHNEXT_DECODE_PARTIAL' in inspect.getsource(sk), "
            "'| active:', __import__('os').environ.get('FLASHNEXT_DECODE_PARTIAL') == '1'); "
            "import vllm.distributed.device_communicators.cuda_communicator as cc; "
            "print('compressed all-reduce installed:', hasattr(cc, '_fn_qar_maybe'), '| mode:', cc._FN_QAR_MODE, '| codec:', cc._FN_QAR_CODEC, "
            "'| block:', cc._FN_QAR_BLOCK, '| min MB:', cc._FN_QAR_MIN_BYTES / 2**20); "
            "import vllm.models.qwen4_exp.nvidia.ops.qsa_indexer as qi, vllm.models.qwen4_exp.nvidia.model_state as ms; "
            "print('sink+window prototype installed:', hasattr(qi, '_fn_sink_window_mask'), '| sink:', qi._FN_SW_SINK, '| window:', qi._FN_SW_WINDOW, "
            "'| jump keep:', qi._FN_SW_KEEP, '| jump limit:', qi._FN_SW_LIMIT, '| sink auto:', qi._FN_SW_SINK_AUTO, "
            "'| per-request sink code:', '_fn_sink_auto' in inspect.getsource(ms), '| scoring skips the cut-away part:', qi._FN_SW_SKIP); "
            "import vllm.distributed.kv_transfer.kv_connector.v1.offloading.scheduler as osch; "
            "print('KV offloading to RAM (GiB):', e.kv_offloading_size, '| middle-aware lazy patch installed:', hasattr(osch, '_fn_trim_state'), "
            "'| lazy:', getattr(osch, '_FN_OFF_LAZY', None), '| cumem allocator:', e.enable_cumem_allocator); "
            "import vllm.v1.simple_kv_offload.manager as skm, vllm.v1.core.kv_cache_coordinator as kc0; "
            "print('copy-only-before-eviction offload installed:', hasattr(skm.SimpleCPUOffloadScheduler, '_fn_pick') and hasattr(kc0, '_fn_seg_order'), "
            "'| active:', __import__('os').environ.get('FLASHNEXT_OFFLOAD_EVICT') == '1'); "
            "import vllm.v1.core.kv_cache_coordinator as kc, vllm.v1.core.kv_cache_manager as km, vllm.v1.core.sched.scheduler as sc; "
            "print('trim free installed:', hasattr(sk, 'FnTrimGapManager') and 'fn_trim_gap' in inspect.getsource(kc) "
            "and 'fn_trim_sink' in inspect.getsource(km) and 'fn_trim_configure' in inspect.getsource(sc), "
            "'| active:', __import__('os').environ.get('FLASHNEXT_TRIM_FREE') == '1'); "
            "print('import check ok')")
    IMPORT_CHECK = (code, env)
    print("import check: runs in the background when the server starts (step 6); its result is checked before the server counts as ready")
    print(f"QSA DCP is ON (FLASHNEXT_DCP_SIZE={DCP_SIZE}, compressed-key block factor {COMPRESSED_BLOCK_FACTOR}). "
          f"Now run step 6: the server will use --decode-context-parallel-size {DCP_SIZE} with TP 4.")
    print(f"Also set: GPU_UTIL = {globals().get('GPU_UTIL')} | dcp comm backend = {DCP_COMM_BACKEND or 'vLLM default (ag_rs)'} | "
          f"max CUDA-graph batch = {MAX_CUDAGRAPH_CAPTURE_SIZE or 'vLLM default'} | prefill chunk = {MAX_BATCHED_TOKENS or 'step 6 value'} | "
          f"image cap = {MM_MAX_PIXELS or 'checkpoint default'} px | video {'off' if DISABLE_VIDEO else 'on'} | expandable_segments {EXPANDABLE_SEGMENTS} | "
          f"FP8 dense {FP8_DENSE_OVERLAY} | FP8 mixers+lm_head {FP8_HC_LMHEAD} | memory trims {MEM_TRIMS} | prefix reuse {PREFIX_REUSE} | "
          f"compressed all-reduce {QAR or 'off'}{(' ' + QAR_CODEC + '/' + str(QAR_BLOCK)) if QAR else ''} | sink+window {SINK_WINDOW or 'off'} | sink jump {SINK_JUMP or 'off'} | middle freed {TRIM_FREE} | scoring skips it {QSA_SKIP_GAP} | RAM offload GiB {KV_OFFLOAD_GIB} (copy only before eviction {_EV}) | DeltaNet state {'FP16 + stochastic rounding' if MAMBA_SSM_FP16_SR else ('BF16 (plain)' if MAMBA_SSM_BF16 else 'checkpoint (FP32)')} | MODEL_DIR {globals().get('MODEL_DIR')}")
    print("After step 6, run this in a cell and send the output:")
    print('!grep -E "GPU KV cache size|kv cache group sizes|kv lcm block sizes|Available KV cache memory|fine-grained prefix|Traceback|Error" /kaggle/working/vllm-server-0.log | tail -20')
    if DIAG_ENCODER_MISS:
        print("If the server dies with 'Encoder cache miss', send:  !grep -m 4 'ENCODER MISS DIAG' /kaggle/working/vllm-server-0.log")
    if PREFIX_MATCH_UNIT:
        print("Prefix-cache check after step 6 (expect no 'Disabling fine-grained' line):  !grep -iE 'fine-grained|prefix_match|hash_block' /kaggle/working/vllm-server-0.log | head -5")
    if TRIM_FREE:
        print("Trim without re-reading (memory freed) is ON: set MAX_LEN = 262144 at the top of step 6 (else the harness trims first and nothing changes). "
              "After step 7:  !grep -aE 'flashnext trim|flashnext QSA sink' /kaggle/working/vllm-server-0.log | head -14")
    if DIAG_WEIGHTS:
        print("Memory per model part (GPU 0):  !grep -E 'Model loading took|WEIGHTS DIAG' /kaggle/working/vllm-server-0.log | grep -E 'TP0|EngineCore|^[^(]' | head -34")

In [ ]:
from pathlib import Path

_f = Path("/tmp/site-vllm/vllm/v1/core/kv_cache_utils.py")
assert _f.exists(), f"{_f} not found: run the earlier steps (step 2 unpacks vLLM to /tmp/site-vllm)"
_s = _f.read_text()
_ANCHOR = "    vllm_config.cache_config.kv_cache_max_concurrency = max_concurrency\n"
_MARK = "# flashnext KV layout diag (session 17)"
_ADD = _ANCHOR + '''    try:  ''' + _MARK + '''
        for _gi, _g in enumerate(kv_cache_config.kv_cache_groups):
            _sp = _g.kv_cache_spec
            _real = getattr(_sp, "real_page_size_bytes", None)
            logger.info(
                "FLASHNEXT KVDIAG group %d | %s | layers %d | block %d tokens | page %d B | real %s B | pages per task at max_len %d | "
                "mamba_cache_mode %s | prefill ckpt blocks %s | spec blocks %s | tp_replicated %s | host %s | shapes %s | dtypes %s",
                _gi, type(_sp).__name__, len(_g.layer_names), _sp.block_size, _sp.page_size_bytes, _real,
                cdiv(_sp.max_memory_usage_bytes(vllm_config), _sp.page_size_bytes), getattr(_sp, "mamba_cache_mode", "-"),
                getattr(_sp, "num_prefill_checkpoint_blocks", "-"), getattr(_sp, "num_speculative_blocks", "-"),
                getattr(_sp, "tp_replicated", "-"), getattr(_g, "host_resident", "-"), getattr(_sp, "shapes", "-"), getattr(_sp, "dtypes", "-"))
        logger.info("FLASHNEXT KVDIAG pool | num_blocks %d | tokens %d | max_len %d | groups %d", kv_cache_config.num_blocks, num_tokens,
                    vllm_config.model_config.max_model_len, len(kv_cache_config.kv_cache_groups))
    except Exception as _e:
        logger.error("FLASHNEXT KVDIAG FAILED: %r", _e)
'''
if _MARK in _s:
    print("KV layout diag: [already] in", _f)
else:
    assert _s.count(_ANCHOR) == 1, f"anchor found {_s.count(_ANCHOR)} times in {_f} (vLLM changed?)"
    _s = _s.replace(_ANCHOR, _ADD)
    print("KV layout diag: [installed] in", _f, "(server log lines 'FLASHNEXT KVDIAG')")
_MARK2 = "# flashnext group-size override (session 17)"
if _MARK2 in _s:
    print("group-size switch: [already] in", _f)
else:
    _A2 = "    def num_groups_for(spec: UniformTypeKVCacheSpecs, balanced: bool) -> int:\n        if balanced and repeats_per_group is not None:\n"
    _B2 = ("    import os as _fn_os  " + _MARK2 + "\n"
           "    _fn_gl = int(_fn_os.environ[\"FLASHNEXT_KV_GROUP_LAYERS\"]) if _fn_os.environ.get(\"FLASHNEXT_KV_GROUP_LAYERS\") else None\n"
           "    if _fn_gl:\n"
           "        repeats_per_group = _fn_gl\n"
           "        logger.info(\"FLASHNEXT group-size override: %d layers per attention group\", _fn_gl)\n\n"
           "    def num_groups_for(spec: UniformTypeKVCacheSpecs, balanced: bool) -> int:\n"
           "        if balanced and repeats_per_group is not None and not (_fn_gl and isinstance(spec.first_spec, (MambaSpec, CircularBufferSpec))):\n")
    assert _s.count(_A2) == 1, f"group-size anchor found {_s.count(_A2)} times in {_f} (vLLM changed?)"
    _s = _s.replace(_A2, _B2)
    print("group-size switch: [installed] in", _f, "(set FLASHNEXT_KV_GROUP_LAYERS before the server starts; unset = unchanged)")
_f.write_text(_s)
compile(_s, str(_f), "exec")
import os
os.environ["FLASHNEXT_KV_GROUP_LAYERS"] = "4"
print("FLASHNEXT_KV_GROUP_LAYERS =", os.environ["FLASHNEXT_KV_GROUP_LAYERS"])

In [ ]:
from pathlib import Path

_root = Path(globals().get("VROOT") or "/tmp/site-vllm/vllm")
_f = _root / "v1/core/single_type_kv_cache_manager.py"
_fp = _root / "v1/core/block_pool.py"
assert _f.exists() and _fp.exists(), f"{_f} / {_fp} not found: run the earlier steps first"
_s = _f.read_text()
_MARK = "# flashnext hold-states switch (session 17)"
_EDITS = [
    ("take back at the end of the call",
     """            for _i, _b, _a in _fn_h:
                if _blocks is not None and _i < len(_blocks) and _blocks[_i].is_null:
                    _blocks[_i] = _b
""",
     """            for _e in _fn_h:
                _i, _b, _a = _e[:3]
                if len(_e) > 3:   """ + _MARK + """: freed at retire time; take it back if its cache entry survived
                    _FN_HOLD["prot"].pop(_b.block_id, None)
                    if _b.is_null or _b.block_hash is None or _b.block_hash != _e[3]:
                        _FN_HOLD["lost"] += 1
                        continue
                    if _b.ref_cnt == 0:
                        self.block_pool.free_block_queue.remove(_b)
                    _b.ref_cnt += 1
                    _FN_HOLD["back"] += 1
                if _blocks is not None and _i < len(_blocks) and _blocks[_i].is_null:
                    _blocks[_i] = _b
"""),
    ("retire: cached instead of pinned",
     """        held.append((idx, block, aligned))
        same = sorted((h for h in held if h[2] == aligned), key=lambda h: h[0])
        pool = self.block_pool
        for h in same[: -(2 if aligned else 1)]:
            held.remove(h)
            pool._fn_dropping = True
""",
     """        _cached = _FN_HOLD["mode"] != "pin"
        held.append((idx, block, aligned, block.block_hash) if _cached else (idx, block, aligned))
        same = sorted((h for h in held if h[2] == aligned), key=lambda h: h[0])
        pool = self.block_pool
        for h in same[: -(2 if aligned else 1)]:
            held.remove(h)
            if len(h) > 3:            # already freed at retire time: drop its cache entry only while it is still ours and unused
                _FN_HOLD["prot"].pop(h[1].block_id, None)
                if h[1].block_hash is not None and h[1].block_hash == h[3] and h[1].ref_cnt == 0:
                    pool._fn_dropping = True
                    try:
                        pool._maybe_evict_cached_block(h[1])
                    finally:
                        pool._fn_dropping = False
                _FN_SEG["states_dropped"] += 1
                continue
            pool._fn_dropping = True
"""),
    ("retire: free the kept state at once in cached mode",
     """            pool.free_blocks([h[1]])
            _FN_SEG["states_dropped"] += 1
        return True
""",
     """            pool.free_blocks([h[1]])
            _FN_SEG["states_dropped"] += 1
        if _cached:
            pool.free_blocks([block])     # a normal cached block: reclaimable when the pool is full, taken back at the end of the call
            _FN_HOLD["freed"] += 1
            if _FN_HOLD["mode"] == "protect":
                _FN_HOLD["prot"][block.block_id] = float("inf")    # block_pool.py: second chance at the head of the free queue
                pool._fn_protect = _FN_HOLD
        return True
"""),
]
_EDITS.append(("retire: pinnew keeps only the newest state pinned",
               """        aligned = ((idx + 1) * self.block_size) % self.scheduler_block_size == 0
        _cached = _FN_HOLD["mode"] != "pin"
""",
               """        aligned = ((idx + 1) * self.block_size) % self.scheduler_block_size == 0
        if _FN_HOLD["mode"] == "pinnew":   """ + _MARK + """: only the newest state stays pinned
            for h in list(held):
                held.remove(h)
                self.block_pool.free_blocks([h[1]])    # the older state: a normal cached block from now on
                _FN_SEG["states_dropped"] += 1
            held.append((idx, block, aligned))
            return True
        _cached = _FN_HOLD["mode"] not in ("pin", "pinnew")
"""))
_POOL_MARK = "# flashnext hold-states second chance (session 17)"
_POOL_EDIT = ("protect: second chance for held states",
              """        ret: list[KVCacheBlock] = self.free_block_queue.popleft_n(num_blocks)

        if self._reuse_watchers:
""",
              """        ret: list[KVCacheBlock] = self.free_block_queue.popleft_n(num_blocks)
        if getattr(self, "_fn_protect", None) is not None and self._fn_protect["prot"]:   """ + _POOL_MARK + """
            ret = _fn_second_chance(self, ret, self._fn_protect)

        if self._reuse_watchers:
""")
_POOL_FN = '''

''' + _POOL_MARK + '''
_FN_SCAN = 256   # free blocks looked at beyond the ones asked for, per call, before a held state is taken anyway


def _fn_second_chance(pool, ret, hold):
    """The blocks just taken from the head of the free queue: held DeltaNet states among them go back to the TAIL (evicted last) and the
    next free blocks replace them; if only held states are found within _FN_SCAN blocks, the oldest held states are taken ("forced").
    hold["prot"]: block id -> time until which it is protected (inf = a held state; park mode: a call's blocks for FLASHNEXT_PARK_S s)."""
    import time as _t
    prot, q, now = hold["prot"], pool.free_block_queue, _t.monotonic()

    def _p(b):
        e = prot.get(b.block_id)
        if e is None:
            return False
        if e < now:
            del prot[b.block_id]
            return False
        return True

    keep, saved = [], []
    for b in ret:
        (saved if _p(b) else keep).append(b)
    if not saved:
        return ret
    scan = _FN_SCAN
    while len(keep) < len(ret) and scan > 0 and q.num_free_blocks > 0:
        b = q.popleft()
        scan -= 1
        (saved if _p(b) else keep).append(b)
    while len(keep) < len(ret):
        b = saved.pop(0)
        prot.pop(b.block_id, None)
        hold["forced"] += 1
        keep.append(b)
    if saved:
        q.append_n(saved)
    return keep
'''
_fc = _root / "v1/core/kv_cache_coordinator.py"
_PARK_MARK = "# flashnext hold-states park (session 17)"
_PARK_EDIT = ("park: a finished call's blocks are protected for a while",
              "    return [e[3] for e in entries]\n",
              "    _fn_park_blocks(managers, [e[3] for e in entries if not e[4]])   " + _PARK_MARK + "\n    return [e[3] for e in entries]\n")
_PARK_FN = '''

''' + _PARK_MARK + '''
def _fn_park_blocks(managers, blocks):
    from vllm.v1.core import single_type_kv_cache_manager as _st

    hold = getattr(_st, "_FN_HOLD", None)
    if not hold or hold["mode"] != "park" or not managers:
        return
    pool = managers[0].block_pool
    if getattr(getattr(pool, "_fn_on_evict", None), "__name__", "") != "_fn_gpu_evicted":
        return                                   # the GPU pool only (the RAM offload's pool has its own coordinator)
    import time as _t

    until, prot, n = _t.monotonic() + hold["park_s"], hold["prot"], 0
    for b in blocks:
        if not b.is_null and b.block_hash is not None:
            prot[b.block_id] = until
            n += 1
    hold["parked"] += n
    pool._fn_protect = hold
    if len(prot) > 4 * pool.num_gpu_blocks:      # entries never met at the head of the free queue again: forget the expired ones
        now = _t.monotonic()
        for k in [k for k, e in prot.items() if e < now]:
            del prot[k]
'''
if _MARK in _s:
    print("hold-states switch: [already] in", _f)
else:
    for label, old, new in _EDITS:
        assert _s.count(old) == 1, f"hold-states edit '{label}': anchor found {_s.count(old)} times in {_f} (patch cell changed?)"
        _s = _s.replace(old, new)
    _s += ("\n\n" + _MARK + "\nimport os as _fh_os\n_FN_HOLD = {\"mode\": _fh_os.environ.get(\"FLASHNEXT_HOLD_STATES\", \"pin\"), "
           "\"freed\": 0, \"back\": 0, \"lost\": 0, \"forced\": 0, \"parked\": 0, \"prot\": {}, "
           "\"park_s\": float(_fh_os.environ.get(\"FLASHNEXT_PARK_S\", \"60\"))}\nassert _FN_HOLD[\"mode\"] in (\"pin\", \"pinnew\", \"cached\", \"protect\", \"park\"), "
           "f\"FLASHNEXT_HOLD_STATES must be pin, pinnew, cached, protect or park, not {_FN_HOLD['mode']!r}\"\n")
    _f.write_text(_s)
    compile(_s, str(_f), "exec")
    print("hold-states switch: [installed] in", _f, "(FLASHNEXT_HOLD_STATES = pin | pinnew | cached | protect | park before the server starts; unset = pin)")
_p = _fp.read_text()
if _POOL_MARK in _p:
    print("hold-states second chance: [already] in", _fp)
else:
    label, old, new = _POOL_EDIT
    assert _p.count(old) == 1, f"hold-states edit '{label}': anchor found {_p.count(old)} times in {_fp} (vLLM changed?)"
    _p = _p.replace(old, new) + _POOL_FN
    _fp.write_text(_p)
    compile(_p, str(_fp), "exec")
    print("hold-states second chance: [installed] in", _fp)
_c = _fc.read_text()
if _PARK_MARK in _c:
    print("hold-states park: [already] in", _fc)
else:
    label, old, new = _PARK_EDIT
    assert _c.count(old) == 1, f"hold-states edit '{label}': anchor found {_c.count(old)} times in {_fc} (patch cell changed?)"
    _c = _c.replace(old, new) + _PARK_FN
    _fc.write_text(_c)
    compile(_c, str(_fc), "exec")
    print("hold-states park: [installed] in", _fc)
import os
os.environ["FLASHNEXT_HOLD_STATES"] = "pinnew"
print("FLASHNEXT_HOLD_STATES =", os.environ["FLASHNEXT_HOLD_STATES"])

In [ ]:
from pathlib import Path

_root = Path(globals().get("VROOT") or "/tmp/site-vllm/vllm")
_fm = _root / "models/qwen4_exp/nvidia/model.py"
_fa = _root / "engine/arg_utils.py"
assert _fm.exists() and _fa.exists(), f"{_fm} / {_fa} not found: run the earlier steps first"
_MARK = "# flashnext batch-sharded sampling (session 17)"
_EDITS = [
    (_fm, "Qwen4ExpForCausalLM.compute_logits_local",
     """    def compute_logits(self, hidden_states: torch.Tensor) -> torch.Tensor | None:
        return self.logits_processor(self.lm_head, hidden_states)
""",
     """    def compute_logits(self, hidden_states: torch.Tensor) -> torch.Tensor | None:
        return self.logits_processor(self.lm_head, hidden_states)

    def compute_logits_local(self, hidden_states: torch.Tensor) -> torch.Tensor:   """ + _MARK + """
        return self.logits_processor(self.lm_head, hidden_states, skip_gather=True)
"""),
    (_fa, "--enable-batch-sharded-sampling from FLASHNEXT_BATCH_SHARDED_SAMPLING",
     """            engine_args.enable_cumem_allocator = True
        return engine_args
""",
     """            engine_args.enable_cumem_allocator = True
        if os.environ.get("FLASHNEXT_BATCH_SHARDED_SAMPLING") == "1":   """ + _MARK + """
            engine_args.enable_batch_sharded_sampling = True
            print("flashnext batch-sharded sampling: ON (--enable-batch-sharded-sampling)", flush=True)
        return engine_args
"""),
]
for _f, _label, _old, _new in _EDITS:
    _s = _f.read_text()
    if _MARK in _s:
        print("batch-sharded sampling: [already]", _label)
        continue
    assert _s.count(_old) == 1, f"batch-sharded sampling edit '{_label}': anchor found {_s.count(_old)} times in {_f} (patch cell changed?)"
    _s = _s.replace(_old, _new)
    _f.write_text(_s)
    compile(_s, str(_f), "exec")
    print("batch-sharded sampling: [installed]", _label)
print("(FLASHNEXT_BATCH_SHARDED_SAMPLING=1 before the server starts; unset = unchanged)")
import os
os.environ["FLASHNEXT_BATCH_SHARDED_SAMPLING"] = "1"
print("FLASHNEXT_BATCH_SHARDED_SAMPLING =", os.environ["FLASHNEXT_BATCH_SHARDED_SAMPLING"])

In [ ]:
from pathlib import Path

_root = Path(globals().get("VROOT") or "/tmp/site-vllm/vllm")
_f = _root / "models/qwen4_exp/nvidia/model.py"
assert _f.exists(), f"{_f} not found: run the earlier steps first"
_s = _f.read_text()
_MARK = "# flashnext token embedding in host RAM (session 17)"
_ANCHOR = """        loaded = loader.load_weights(
            weights,
            mapper=mapper,
        )
        return loaded
"""
_NEW = """        loaded = loader.load_weights(
            weights,
            mapper=mapper,
        )
        _flashnext_embed_to_host(self)   """ + _MARK + """
        return loaded
"""
_FN = '''

''' + _MARK + '''
def _flashnext_embed_to_host(model) -> None:
    import os

    if os.environ.get("FLASHNEXT_EMBED_HOST") != "1":
        return
    from vllm.utils.platform_utils import is_uva_available
    from vllm.utils.torch_utils import get_accelerator_view_from_cpu_tensor

    if not is_uva_available():
        raise RuntimeError("FLASHNEXT_EMBED_HOST=1: this platform has no UVA; unset FLASHNEXT_EMBED_HOST")
    emb = model.embed_tokens
    if getattr(emb, "_flashnext_host_weight", None) is not None:
        return
    w = emb.weight
    if w.device.type != "cuda":
        raise RuntimeError(f"FLASHNEXT_EMBED_HOST=1: embed_tokens.weight is on {w.device}, expected the GPU")
    host = torch.empty(w.shape, dtype=w.dtype, device="cpu", pin_memory=True)
    host.copy_(w.data)
    view = get_accelerator_view_from_cpu_tensor(host)
    if not torch.equal(view[: min(4096, view.shape[0])], w.data[: min(4096, w.shape[0])]):
        raise RuntimeError("FLASHNEXT_EMBED_HOST=1: the host copy of embed_tokens differs from the GPU weights")
    emb._flashnext_host_weight = host            # keeps the pinned memory alive
    emb.weight.data = view
    torch.cuda.empty_cache()
    print(f"flashnext embed host: embed_tokens {tuple(w.shape)} {w.dtype} moved to pinned host RAM "
          f"({host.numel() * host.element_size() / 2**20:.0f} MiB per GPU freed)", flush=True)
'''
if _MARK in _s:
    print("embed host: [already] in", _f)
else:
    assert _s.count(_ANCHOR) == 1, f"embed host anchor found {_s.count(_ANCHOR)} times in {_f} (vLLM changed?)"
    _s = _s.replace(_ANCHOR, _NEW) + _FN
    _f.write_text(_s)
    compile(_s, str(_f), "exec")
    print("embed host: [installed] in", _f, "(FLASHNEXT_EMBED_HOST=1 before the server starts; unset = unchanged)")
import os
os.environ["FLASHNEXT_EMBED_HOST"] = "1"
print("FLASHNEXT_EMBED_HOST =", os.environ["FLASHNEXT_EMBED_HOST"])

In [ ]:
import json, os
V4_EDITS = json.loads("\"# flashnext QSA V4 (session 17): keys FP8 as today, VALUES 4-bit (+ a float32 scale and zero point per token and KV head) in the 12 QSA\\n# attention layers. Run AFTER the patch cell (and the improvement cells) and BEFORE the server starts. Installs ops/qsa_v4.py and edits:\\n#   platforms/interface.py            hybrid block-size alignment: with FLASHNEXT_QSA_V4=1 an attention token costs V4 row bytes\\n#                                     (400 instead of 512 at head size 256) -> attention blocks of 1,024 tokens at the same page\\n#                                     size (409,600 B on Kaggle), the DeltaNet pages stay padded as today\\n#   models/qwen4_exp/common/qsa_cache.py   the indexer's compressed-key blocks: the largest DCP-aligned multiple of the attention block\\n#                                     that fits the page (Kaggle: 6 x 1,024 = 6,144 tokens), padded to the page (vLLM MLA alignment)\\n#   models/qwen4_exp/nvidia/qsa.py    the QSA owner's KV spec gets state_content_bytes = V4 row bytes; the impl writes K/V with\\n#                                     qsa_v4.v4_store and reads them through qsa_v4.v4_views; both kernel calls get v4=True\\n#   models/qwen4_exp/nvidia/ops/qsa.py     the split-K kernel decodes 4-bit V (constexpr V4); wrapper + warmup; tiles <= 64 tokens\\n#                                     with V4 (128-token tiles need 106-128 KB shared memory, RTX/L4 have 99 KB)\\n#   models/qwen4_exp/nvidia/ops/qsa_dcp.py the DCP local partial passes V4\\n# FLASHNEXT_QSA_V4 unset = every edited path behaves exactly as before (the edits only branch on it). Safe to run again (\\\"already\\\").\\n# FLASHNEXT_V4_UNDO=1 reverses everything. An older V4 install found here is reversed and installed fresh.\\nimport os\\nfrom pathlib import Path\\n\\n_root = Path(globals().get(\\\"VROOT\\\") or \\\"/tmp/site-vllm/vllm\\\")\\n_here = Path(globals().get(\\\"V4_DIR\\\") or (Path(__file__).parent if \\\"__file__\\\" in globals() else \\\".\\\"))\\n_MOD_SRC = globals().get(\\\"V4_MODULE_SRC\\\") or (_here / \\\"qsa_v4.py\\\").read_text()\\n_MARK = \\\"# flashnext V4 (session 17)\\\"\\n_NEW = _root / \\\"models/qwen4_exp/nvidia/ops/qsa_v4.py\\\"\\n\\n\\ndef _write(path, text):\\n    tmp = path.with_name(path.name + \\\".fn_tmp\\\")\\n    tmp.write_text(text)\\n    os.replace(tmp, path)\\n\\n\\n_ROW = \\\"((_fn_d + _fn_d // 2 + 8 + 15) // 16 * 16)\\\"\\nEDITS = {\\n    \\\"platforms/interface.py\\\": [\\n        (\\\"attention page per token\\\",\\n         \\\"\\\"\\\"            attn_page_size_1_token = backend_cls.customize_spec(\\n                attn_spec\\n            ).page_size_bytes\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"            attn_page_size_1_token = backend_cls.customize_spec(\\n                attn_spec\\n            ).page_size_bytes\\n            import os as _fn_os   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\": QSA rows = K fp8 + V 4-bit + scale/zero\\n            if _fn_os.environ.get(\\\"FLASHNEXT_QSA_V4\\\") == \\\"1\\\":\\n                _fn_d = model_config.get_head_size()\\n                attn_page_size_1_token = model_config.get_num_kv_heads(parallel_config) * \\\"\\\"\\\" + _ROW + \\\"\\\"\\\"\\n                logger.info(\\\"FLASHNEXT V4: attention bytes per token %d (FP8 would be %d)\\\", attn_page_size_1_token,\\n                            backend_cls.customize_spec(attn_spec).page_size_bytes)\\n\\\"\\\"\\\"),\\n    ],\\n    \\\"models/qwen4_exp/common/qsa_cache.py\\\": [\\n        (\\\"compressed-key blocks fit the V4 page\\\",\\n         \\\"\\\"\\\"        return MLAAttentionSpec(\\n            block_size=block_size,\\n            num_kv_heads=1,\\n            head_size=self.head_size,\\n            dtype=self.dtype,\\n            tokens_per_state=self.compress_ratio,\\n            dcp_sharded=False,\\n        )\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"        import os as _fn_os   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\"\\n        if _fn_os.environ.get(\\\"FLASHNEXT_QSA_V4\\\") == \\\"1\\\":\\n            # the attention page is block x heads x V4 row; take the largest multiple of the attention block (and of the DCP size,\\n            # so compressed blocks start on global attention-block boundaries) whose compressed keys fit it, pad to the page\\n            _fn_mc = vllm_config.model_config\\n            _fn_d = _fn_mc.get_head_size()\\n            _fn_base = self.cache_config.block_size\\n            _fn_page = _fn_base * _fn_mc.get_num_kv_heads(vllm_config.parallel_config) * \\\"\\\"\\\" + _ROW + \\\"\\\"\\\"\\n            _fn_unit = MLAAttentionSpec(block_size=_fn_base, num_kv_heads=1, head_size=self.head_size, dtype=self.dtype,\\n                                        tokens_per_state=self.compress_ratio, dcp_sharded=False).page_size_bytes\\n            _fn_dcp = vllm_config.parallel_config.decode_context_parallel_size\\n            _fn_factor = (_fn_page // _fn_unit) // _fn_dcp * _fn_dcp\\n            if _fn_factor < 1:\\n                raise RuntimeError(f\\\"FLASHNEXT V4: compressed keys do not fit the attention page ({_fn_unit} > {_fn_page})\\\")\\n            return MLAAttentionSpec(\\n                block_size=_fn_base * _fn_factor,\\n                num_kv_heads=1,\\n                head_size=self.head_size,\\n                dtype=self.dtype,\\n                tokens_per_state=self.compress_ratio,\\n                dcp_sharded=False,\\n                alignment=_fn_page,\\n            )\\n        return MLAAttentionSpec(\\n            block_size=block_size,\\n            num_kv_heads=1,\\n            head_size=self.head_size,\\n            dtype=self.dtype,\\n            tokens_per_state=self.compress_ratio,\\n            dcp_sharded=False,\\n        )\\n\\\"\\\"\\\"),\\n        (\\\"compressed-key blocks stay ~6,400 tokens\\\",\\n         \\\"\\\"\\\"            _fn_factor = (_fn_page // _fn_unit) // _fn_dcp * _fn_dcp\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"            _fn_factor = (_fn_page // _fn_unit) // _fn_dcp * _fn_dcp\\n            # keep the prefix-reuse granularity of the FP8 runs (6,400-token compressed blocks): RTX V4 run 1 with page-filling\\n            # 24,576-token blocks recomputed 7-14K prompt tokens/s from minute ~29 (prefix hits fell, decode 2,200 -> 380 tok/s)\\n            _fn_factor = max(_fn_dcp, min(_fn_factor, 6400 // _fn_base // _fn_dcp * _fn_dcp))   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\"\\n\\\"\\\"\\\"),\\n    ],\\n    \\\"models/qwen4_exp/nvidia/qsa.py\\\": [\\n        (\\\"impl flag\\\",\\n         \\\"\\\"\\\"        self.kv_cache_dtype = real_kv_cache_dtype\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"        self.kv_cache_dtype = real_kv_cache_dtype\\n        from .ops.qsa_v4 import v4_enabled as _fn_v4_enabled   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\"\\n        self._fn_v4 = _fn_v4_enabled()\\n        if self._fn_v4 and real_kv_cache_dtype not in (\\\"fp8\\\", \\\"fp8_e4m3\\\"):\\n            raise RuntimeError(f\\\"FLASHNEXT_QSA_V4=1 needs --kv-cache-dtype fp8 (got {real_kv_cache_dtype})\\\")\\n\\\"\\\"\\\"),\\n        (\\\"impl views\\\",\\n         \\\"\\\"\\\"        key_cache, value_cache = kv_cache.transpose(1, 2).split(self.head_size, dim=-1)\\n        k_scale = v_scale = None\\n        if self.kv_cache_dtype in (\\\"fp8\\\", \\\"fp8_e4m3\\\"):\\n            # The cache is allocated as uint8; reinterpret the e4m3 bytes\\n            # (same itemsize, so shape and strides are preserved).\\n            key_cache = key_cache.view(torch.float8_e4m3fn)\\n            value_cache = value_cache.view(torch.float8_e4m3fn)\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"        if self._fn_v4:   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\": K fp8 view + packed 4-bit V view of the same rows\\n            from .ops.qsa_v4 import v4_views\\n\\n            key_cache, value_cache = v4_views(kv_cache, self.head_size)\\n        else:\\n            key_cache, value_cache = kv_cache.transpose(1, 2).split(self.head_size, dim=-1)\\n        k_scale = v_scale = None\\n        if self.kv_cache_dtype in (\\\"fp8\\\", \\\"fp8_e4m3\\\"):\\n            # The cache is allocated as uint8; reinterpret the e4m3 bytes\\n            # (same itemsize, so shape and strides are preserved).\\n            key_cache = key_cache.view(torch.float8_e4m3fn)\\n            if not self._fn_v4:\\n                value_cache = value_cache.view(torch.float8_e4m3fn)\\n\\\"\\\"\\\"),\\n        (\\\"single-GPU call\\\",\\n         \\\"\\\"\\\"            k_scale=k_scale,\\n            v_scale=v_scale,\\n            output_gate=output_gate[:num_tokens],\\n        )\\n        return output\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"            k_scale=k_scale,\\n            v_scale=v_scale,\\n            output_gate=output_gate[:num_tokens],\\n            v4=self._fn_v4,   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\"\\n        )\\n        return output\\n\\\"\\\"\\\"),\\n        (\\\"DCP call + the V4 cache write\\\",\\n         \\\"\\\"\\\"            self._QSA_DCP_SPLITS,\\n            self._QSA_DCP_BLOCK_N,\\n            self._QSA_DCP_WARPS,\\n        )\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"            self._QSA_DCP_SPLITS,\\n            self._QSA_DCP_BLOCK_N,\\n            self._QSA_DCP_WARPS,\\n            v4=self._fn_v4,   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\"\\n        )\\n\\\"\\\"\\\"),\\n        (\\\"the V4 cache write\\\",\\n         \\\"\\\"\\\"    # Split-K settings for one DCP rank's partial (12 query heads over its tokens), measured on L4 (sm_89).\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"    def do_kv_cache_update(self, layer, key, value, kv_cache, slot_mapping):   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\"\\n        if not self._fn_v4:\\n            return super().do_kv_cache_update(layer, key, value, kv_cache, slot_mapping)\\n        from .ops.qsa_v4 import v4_store\\n\\n        v4_store(key, value, kv_cache, slot_mapping, layer._k_scale_float)\\n\\n    # Split-K settings for one DCP rank's partial (12 query heads over its tokens), measured on L4 (sm_89).\\n\\\"\\\"\\\"),\\n        (\\\"owner KV spec\\\",\\n         \\\"\\\"\\\"            dtype=self.kv_cache_torch_dtype,\\n            kv_quant_mode=get_kv_quant_mode(self.kv_cache_dtype),\\n        )\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"            dtype=self.kv_cache_torch_dtype,\\n            kv_quant_mode=get_kv_quant_mode(self.kv_cache_dtype),\\n            state_content_bytes=_fn_v4_spec_bytes(self.head_dim, self.kv_cache_dtype),   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\"\\n        )\\n\\\"\\\"\\\"),\\n        (\\\"owner spec helper\\\",\\n         \\\"\\\"\\\"class Qwen4ExpQSAMetadataBuilder(FlashAttentionMetadataBuilder):\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"def _fn_v4_spec_bytes(head_dim, kv_cache_dtype):   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\": V4 row bytes, or None = dense K/V as before\\n    from .ops.qsa_v4 import v4_enabled, v4_row_bytes\\n\\n    if not v4_enabled():\\n        return None\\n    if kv_cache_dtype not in (\\\"fp8\\\", \\\"fp8_e4m3\\\"):\\n        raise RuntimeError(f\\\"FLASHNEXT_QSA_V4=1 needs --kv-cache-dtype fp8 (got {kv_cache_dtype})\\\")\\n    return v4_row_bytes(head_dim)\\n\\n\\nclass Qwen4ExpQSAMetadataBuilder(FlashAttentionMetadataBuilder):\\n\\\"\\\"\\\"),\\n    ],\\n    \\\"models/qwen4_exp/nvidia/ops/qsa.py\\\": [\\n        (\\\"kernel constexpr\\\",\\n         \\\"\\\"\\\"    BLOCK_N: tl.constexpr,\\n    IS_FP8: tl.constexpr,\\n) -> None:\\n    row = tl.program_id(0)\\n    kv_head = tl.program_id(1)\\n    split_id = tl.program_id(2)\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"    BLOCK_N: tl.constexpr,\\n    IS_FP8: tl.constexpr,\\n    V4: tl.constexpr = False,   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\": v_cache holds 4-bit V codes + per-row float32 scale/zero\\n) -> None:\\n    row = tl.program_id(0)\\n    kv_head = tl.program_id(1)\\n    split_id = tl.program_id(2)\\n\\\"\\\"\\\"),\\n        (\\\"kernel V decode\\\",\\n         \\\"\\\"\\\"        values = tl.load(\\n            v_cache_ptr\\n            + safe_page[:, None] * stride_v_block\\n            + page_offset[:, None] * stride_v_token\\n            + kv_head * stride_v_head\\n            + dim_offsets[None, :],\\n            mask=valid[:, None],\\n            other=0.0,\\n        )\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"        if V4:   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\": byte i = dim 2i (low nibble) | dim 2i+1 (high nibble); then scale, zero (float32)\\n            v_row = v_cache_ptr + safe_page * stride_v_block + page_offset * stride_v_token + kv_head * stride_v_head\\n            packed = tl.load(v_row[:, None] + tl.arange(0, HEAD_DIM // 2)[None, :], mask=valid[:, None], other=0)\\n            v_meta = (v_row + HEAD_DIM // 2).to(tl.pointer_type(tl.float32))\\n            v_scale_row = tl.load(v_meta, mask=valid, other=0.0)\\n            v_zero_row = tl.load(v_meta + 1, mask=valid, other=0.0)\\n            codes = tl.reshape(tl.join(packed & 0xF, packed >> 4), (BLOCK_N, HEAD_DIM))\\n            values = (codes.to(tl.float32) * v_scale_row[:, None] + v_zero_row[:, None]).to(tl.float16)\\n        else:\\n            values = tl.load(\\n                v_cache_ptr\\n                + safe_page[:, None] * stride_v_block\\n                + page_offset[:, None] * stride_v_token\\n                + kv_head * stride_v_head\\n                + dim_offsets[None, :],\\n                mask=valid[:, None],\\n                other=0.0,\\n            )\\n\\\"\\\"\\\"),\\n        (\\\"wrapper argument\\\",\\n         \\\"\\\"\\\"    *,\\n    output_gate: torch.Tensor,\\n) -> torch.Tensor:\\n    \\\\\\\"\\\\\\\"\\\\\\\"Run sparse GQA directly over paged BF16 or FP8-e4m3 K/V caches.\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"    *,\\n    output_gate: torch.Tensor,\\n    v4: bool = False,   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\": v_cache = packed 4-bit V rows (uint8, D/2 wide) from qsa_v4.v4_views\\n) -> torch.Tensor:\\n    \\\\\\\"\\\\\\\"\\\\\\\"Run sparse GQA directly over paged BF16 or FP8-e4m3 K/V caches.\\n\\\"\\\"\\\"),\\n        (\\\"wrapper shape check\\\",\\n         \\\"\\\"\\\"    if q.ndim != 3 or k_cache.ndim != 4 or v_cache.shape != k_cache.shape:\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"    _fn_v4_shape = v4 and v_cache.shape[:3] == k_cache.shape[:3] and 2 * v_cache.shape[3] == k_cache.shape[3]   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\"\\n    if q.ndim != 3 or k_cache.ndim != 4 or (v_cache.shape != k_cache.shape and not _fn_v4_shape):\\n\\\"\\\"\\\"),\\n        (\\\"wrapper dtype check\\\",\\n         \\\"\\\"\\\"    assert k_cache.dtype == v_cache.dtype\\n    is_fp8 = k_cache.dtype == torch.float8_e4m3fn\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"    assert k_cache.dtype == v_cache.dtype or (v4 and k_cache.dtype == torch.float8_e4m3fn and v_cache.dtype == torch.uint8)\\n    assert not v4 or v_cache.dtype == torch.uint8   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\"\\n    is_fp8 = k_cache.dtype == torch.float8_e4m3fn\\n\\\"\\\"\\\"),\\n        (\\\"wrapper output scale\\\",\\n         \\\"\\\"\\\"        softmax_scale = head_dim**-0.5\\n        output_scale = 1.0\\n    assert logical_indices.dtype == block_table.dtype == torch.int32\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"        softmax_scale = head_dim**-0.5\\n        output_scale = 1.0\\n    if v4:\\n        output_scale = 1.0   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\": each row's own scale/zero dequantize V\\n    assert logical_indices.dtype == block_table.dtype == torch.int32\\n\\\"\\\"\\\"),\\n        (\\\"config: V4 argument\\\",\\n         \\\"\\\"\\\"    num_columns: int,\\n    is_fp8: bool = False,\\n) -> tuple[int, int, int, int]:\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"    num_columns: int,\\n    is_fp8: bool = False,\\n    v4: bool = False,   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\": the 4-bit decode needs more shared memory\\n) -> tuple[int, int, int, int]:\\n\\\"\\\"\\\"),\\n        (\\\"config: tile cap\\\",\\n         \\\"\\\"\\\"    num_tiles = triton.cdiv(num_columns, BLOCK_N)\\n    # Never more splits than tiles, never empty.\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"    if v4 and BLOCK_N > 64:\\n        BLOCK_N = 64   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\": 128-token tiles need 106-128 KB shared memory with 4-bit K/V (RTX/L4 limit 99 KB)\\n    num_tiles = triton.cdiv(num_columns, BLOCK_N)\\n    # Never more splits than tiles, never empty.\\n\\\"\\\"\\\"),\\n        (\\\"wrapper config call\\\",\\n         \\\"\\\"\\\"        q.shape[0], k_cache.shape[2], use_prefill_config, selection_width, is_fp8\\n    )\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"        q.shape[0], k_cache.shape[2], use_prefill_config, selection_width, is_fp8, v4=v4   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\"\\n    )\\n\\\"\\\"\\\"),\\n        (\\\"warmup config call\\\",\\n         \\\"\\\"\\\"            num_rows, num_kv_heads, use_prefill_config, selection_width, is_fp8\\n        )\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"            num_rows, num_kv_heads, use_prefill_config, selection_width, is_fp8, v4=_fn_v4   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\"\\n        )\\n\\\"\\\"\\\"),\\n        (\\\"wrapper launch\\\",\\n         \\\"\\\"\\\"        IS_FP8=is_fp8,\\n        num_warps=partial_warps,\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"        IS_FP8=is_fp8,\\n        V4=v4,   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\"\\n        num_warps=partial_warps,\\n\\\"\\\"\\\"),\\n        (\\\"warmup views\\\",\\n         \\\"\\\"\\\"    head_dim = kv_cache.shape[-1] // 2\\n    key_cache, value_cache = kv_cache.transpose(1, 2).split(head_dim, dim=-1)\\n    # An fp8 cache is allocated as uint8 and viewed as e4m3 at attention time.\\n    is_fp8 = kv_cache.dtype == torch.uint8\\n    cache_dtype = torch.float8_e4m3fn if is_fp8 else key_cache.dtype\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"    from .qsa_v4 import v4_enabled, v4_row_bytes, v4_views   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\"\\n\\n    _fn_v4 = v4_enabled()\\n    if _fn_v4:\\n        head_dim = next(d for d in (64, 128, 256, 512) if v4_row_bytes(d) == kv_cache.shape[-1])\\n        key_cache, value_cache = v4_views(kv_cache, head_dim)\\n    else:\\n        head_dim = kv_cache.shape[-1] // 2\\n        key_cache, value_cache = kv_cache.transpose(1, 2).split(head_dim, dim=-1)\\n    # An fp8 cache is allocated as uint8 and viewed as e4m3 at attention time.\\n    is_fp8 = kv_cache.dtype == torch.uint8\\n    cache_dtype = torch.float8_e4m3fn if is_fp8 else key_cache.dtype\\n\\\"\\\"\\\"),\\n        (\\\"warmup V pointer\\\",\\n         \\\"\\\"\\\"    v_cache_ptr = TritonWarmupTensor(\\n        cache_dtype,\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"    v_cache_ptr = TritonWarmupTensor(\\n        torch.uint8 if _fn_v4 else cache_dtype,   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\"\\n\\\"\\\"\\\"),\\n        (\\\"warmup launch\\\",\\n         \\\"\\\"\\\"            IS_FP8=is_fp8,\\n            num_warps=warps,\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"            IS_FP8=is_fp8,\\n            V4=_fn_v4,   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\"\\n            num_warps=warps,\\n\\\"\\\"\\\"),\\n    ],\\n    \\\"models/qwen4_exp/nvidia/ops/qsa_dcp.py\\\": [\\n        (\\\"partial argument\\\",\\n         \\\"\\\"\\\"    lse: torch.Tensor | None = None,\\n) -> tuple[torch.Tensor, torch.Tensor]:\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"    lse: torch.Tensor | None = None,\\n    v4: bool = False,   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\": v_cache = packed 4-bit V rows (qsa_v4.v4_views)\\n) -> tuple[torch.Tensor, torch.Tensor]:\\n\\\"\\\"\\\"),\\n        (\\\"partial output scale\\\",\\n         \\\"\\\"\\\"    output_scale = float(v_scale) if is_fp8 else 1.0\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"    output_scale = float(v_scale) if is_fp8 else 1.0\\n    if v4:\\n        output_scale = 1.0   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\": each row's own scale/zero dequantize V\\n\\\"\\\"\\\"),\\n        (\\\"partial launch\\\",\\n         \\\"\\\"\\\"        IS_FP8=is_fp8,\\n        num_warps=num_warps,\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"        IS_FP8=is_fp8,\\n        V4=v4,   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\"\\n        num_warps=num_warps,\\n\\\"\\\"\\\"),\\n    ],\\n}\\n\\nif os.environ.get(\\\"FLASHNEXT_V4_UNDO\\\") == \\\"1\\\":\\n    for _rel, _edits in EDITS.items():\\n        _p = _root / _rel\\n        _s = _p.read_text()\\n        _n = 0\\n        for _l, _o, _w in reversed(_edits):\\n            if _s.count(_w) == 1:\\n                _s, _n = _s.replace(_w, _o), _n + 1\\n        assert _MARK not in _s, f\\\"V4 undo: a marked line is left in {_p}\\\"\\n        compile(_s, str(_p), \\\"exec\\\")\\n        _write(_p, _s)\\n        print(f\\\"V4: UNDONE in {_rel} ({_n} edits)\\\")\\n    if _NEW.exists():\\n        _NEW.unlink()\\n    print(\\\"V4: removed\\\", _NEW)\\nelse:\\n    _texts = {rel: (_root / rel).read_text() for rel in EDITS}\\n    _marked = [rel for rel, s in _texts.items() if _MARK in s]\\n    _complete = _marked and all(_texts[rel].count(_w) == 1 for rel, _edits in EDITS.items() for _l, _o, _w in _edits)\\n    if _marked and not (_complete and _NEW.exists() and _NEW.read_text() == _MOD_SRC):\\n        # an older V4 install (its edits are a subset of these): reverse what is there, check nothing marked is left, install fresh\\n        for _rel in list(_texts):\\n            for _l, _o, _w in reversed(EDITS[_rel]):\\n                if _texts[_rel].count(_w) == 1:\\n                    _texts[_rel] = _texts[_rel].replace(_w, _o)\\n            assert _MARK not in _texts[_rel], f\\\"V4: cannot undo the older install in {_rel} - start a fresh runtime\\\"\\n        print(\\\"V4: updating an older install\\\")\\n        _marked = []\\n    if _marked:\\n        print(\\\"V4: [already] installed\\\")\\n    else:\\n        _new = {}\\n        for _rel, _edits in EDITS.items():\\n            _s = _texts[_rel]\\n            for _l, _o, _w in _edits:\\n                assert _s.count(_o) == 1, f\\\"V4 edit '{_l}' in {_rel}: anchor found {_s.count(_o)} times\\\"\\n                _s = _s.replace(_o, _w)\\n            compile(_s, _rel, \\\"exec\\\")\\n            _new[_rel] = _s\\n        compile(_MOD_SRC, str(_NEW), \\\"exec\\\")\\n        _write(_NEW, _MOD_SRC)\\n        for _rel, _s in _new.items():\\n            _write(_root / _rel, _s)\\n        print(\\\"V4: [installed] ops/qsa_v4.py +\\\", \\\", \\\".join(f\\\"{rel} ({len(EDITS[rel])})\\\" for rel in EDITS),\\n              \\\"| active only with FLASHNEXT_QSA_V4=1 at the server start\\\")\\n\"")
V4_MODULE = json.loads("\"# flashnext QSA \\\"V4\\\" KV layout (session 17): keys FP8 as today, values 4-bit with a per-(token, head) scale and zero point.\\n# Installed as vllm/models/qwen4_exp/nvidia/ops/qsa_v4.py by v4_edits.py; active only when FLASHNEXT_QSA_V4=1 at the server start.\\n#\\n# One cache row per (token, KV head), V4_ROW_BYTES = round_up(D + D/2 + 8, 16) bytes (D = 256 -> 400; FP8 today: 2*D = 512):\\n#   [0, D)            K, FP8 e4m3, scaled by 1/k_scale exactly as vLLM's reshape_and_cache_flash writes it\\n#   [D, D + D/2)      V, 4-bit codes 0..15, two per byte: byte i = dim 2i (low nibble) | dim 2i+1 (high nibble)\\n#   [D + D/2, +4)     V scale, float32\\n#   [D + D/2 + 4, +4) V zero point (the minimum of the row), float32;   V ~= zero + code * scale\\n#   rest              zero padding (keeps every row 16-byte aligned)\\n# The row lives in vLLM's packed-slot page ([blocks, heads, tokens, V4_ROW_BYTES] uint8, AttentionSpec.state_content_bytes), so every\\n# block-level mechanism (prefix cache, trim, RAM offload, DCP split) moves K, V and the scales together.\\nimport os\\n\\nimport torch\\n\\nfrom vllm.triton_utils import tl, triton\\n\\n\\ndef v4_enabled() -> bool:\\n    return os.environ.get(\\\"FLASHNEXT_QSA_V4\\\") == \\\"1\\\"\\n\\n\\ndef v4_row_bytes(head_dim: int) -> int:\\n    return (head_dim + head_dim // 2 + 8 + 15) // 16 * 16\\n\\n\\n@triton.jit\\ndef _v4_store_kernel(\\n    key_ptr,\\n    value_ptr,\\n    cache_ptr,\\n    slot_ptr,\\n    inv_k_scale,\\n    stride_key_tok,\\n    stride_key_head,\\n    stride_val_tok,\\n    stride_val_head,\\n    stride_cache_block,\\n    stride_cache_head,\\n    stride_cache_token,\\n    num_cache_slots,\\n    PAGE_SIZE: tl.constexpr,\\n    HEAD_DIM: tl.constexpr,\\n):\\n    tok = tl.program_id(0)\\n    head = tl.program_id(1)\\n    slot = tl.load(slot_ptr + tok).to(tl.int64)\\n    if (slot < 0) | (slot >= num_cache_slots):\\n        return\\n    HALF: tl.constexpr = HEAD_DIM // 2\\n    block = slot // PAGE_SIZE\\n    offset = slot % PAGE_SIZE\\n    row = cache_ptr + block * stride_cache_block + head * stride_cache_head + offset * stride_cache_token\\n    dims = tl.arange(0, HEAD_DIM)\\n    # K: FP8 e4m3 with saturation (round to nearest even), as vLLM's fp8 cache write\\n    k = tl.load(key_ptr + tok * stride_key_tok + head * stride_key_head + dims).to(tl.float32) * inv_k_scale\\n    k = tl.minimum(tl.maximum(k, -448.0), 448.0)\\n    tl.store(row + dims, k.to(tl.float8e4nv).to(tl.uint8, bitcast=True))\\n    # V: asymmetric 4-bit over the row (min .. max)\\n    v = tl.load(value_ptr + tok * stride_val_tok + head * stride_val_head + dims).to(tl.float32)\\n    vmin = tl.min(v, axis=0)\\n    vmax = tl.max(v, axis=0)\\n    scale = (vmax - vmin) / 15.0\\n    inv = tl.where(scale > 0, 1.0 / scale, 0.0)\\n    code = tl.minimum(tl.maximum(tl.floor((v - vmin) * inv + 0.5), 0.0), 15.0).to(tl.uint8)\\n    lo, hi = tl.split(tl.reshape(code, (HALF, 2)))\\n    tl.store(row + HEAD_DIM + tl.arange(0, HALF), lo | (hi << 4))\\n    meta = (row + HEAD_DIM + HALF).to(tl.pointer_type(tl.float32))\\n    tl.store(meta, scale)\\n    tl.store(meta + 1, vmin)\\n\\n\\ndef v4_store(key: torch.Tensor, value: torch.Tensor, kv_cache: torch.Tensor, slot_mapping: torch.Tensor, k_scale: float) -> None:\\n    \\\"\\\"\\\"Write this step's K/V rows. key/value [tokens, heads, D] (bf16/fp16); kv_cache [blocks, heads, page, V4_ROW_BYTES] uint8 (the\\n    QSA owner's cache view); slot_mapping [tokens] (negative = not stored on this rank).\\\"\\\"\\\"\\n    n = slot_mapping.shape[0]\\n    if n == 0:\\n        return\\n    head_dim = key.shape[-1]\\n    assert kv_cache.dtype == torch.uint8 and kv_cache.shape[-1] == v4_row_bytes(head_dim) and kv_cache.stride(-1) == 1\\n    assert key.stride(-1) == 1 and value.stride(-1) == 1 and key.shape[1] == kv_cache.shape[1]\\n    _v4_store_kernel[(n, key.shape[1])](\\n        key, value, kv_cache, slot_mapping, 1.0 / float(k_scale),\\n        key.stride(0), key.stride(1), value.stride(0), value.stride(1),\\n        kv_cache.stride(0), kv_cache.stride(1), kv_cache.stride(2),\\n        kv_cache.shape[0] * kv_cache.shape[2],\\n        PAGE_SIZE=kv_cache.shape[2], HEAD_DIM=head_dim, num_warps=2,\\n    )\\n\\n\\ndef v4_views(kv_cache: torch.Tensor, head_dim: int) -> tuple[torch.Tensor, torch.Tensor]:\\n    \\\"\\\"\\\"(key_cache [blocks, page, heads, D] float8_e4m3fn, packed V [blocks, page, heads, D/2] uint8) over the same bytes, in the\\n    [block, token, head, dim] order the QSA kernel indexes.\\\"\\\"\\\"\\n    t = kv_cache.transpose(1, 2)\\n    return t[..., :head_dim].view(torch.float8_e4m3fn), t[..., head_dim:head_dim + head_dim // 2]\\n\\n\\ndef v4_dequant_reference(kv_cache: torch.Tensor, head_dim: int) -> tuple[torch.Tensor, torch.Tensor]:\\n    \\\"\\\"\\\"Test helper: (K, V) as float32 [blocks, page, heads, D], decoded from the rows the way the kernel decodes them.\\\"\\\"\\\"\\n    k8, packed = v4_views(kv_cache, head_dim)\\n    t = kv_cache.transpose(1, 2)\\n    meta = t[..., head_dim + head_dim // 2:head_dim + head_dim // 2 + 8].contiguous().view(torch.float32)\\n    codes = torch.stack([packed & 0xF, packed >> 4], dim=-1).reshape(*packed.shape[:-1], head_dim).float()\\n    v = meta[..., 1:2] + codes * meta[..., 0:1]\\n    return k8.float(), v\\n\"")
exec(compile(V4_EDITS, 'v4_edits', 'exec'), {'__name__': '__main__', 'VROOT': VROOT, 'V4_MODULE_SRC': V4_MODULE})
os.environ['FLASHNEXT_QSA_V4'] = '1'
assert os.environ.get('FLASHNEXT_PREFIX_MATCH_UNIT') == '800', 'expected the patch cell prefix match unit 800'
os.environ['FLASHNEXT_PREFIX_MATCH_UNIT'] = '1024'
assert os.environ.get('FLASHNEXT_MAX_BATCHED') == '1024', 'expected the patch cell prefill chunk 1024'
os.environ['FLASHNEXT_MAX_BATCHED'] = '2048'
K4_EDITS = json.loads("\"# flashnext QSA K4 (session 17): KEYS 4-bit too, on top of V4 (run v4_edits.py first). Installs ops/qsa_k4.py and edits the V4 lines:\\n#   platforms/interface.py / common/qsa_cache.py   with FLASHNEXT_QSA_K4=1 an attention token costs K4 row bytes (272 at head size 256)\\n#   nvidia/qsa.py                     the owner spec gets K4 row bytes; the impl writes with qsa_k4.k4_store (keys rotated by H, then\\n#                                     4-bit), reads through qsa_k4.k4_views, rotates the query by the same H; both kernel calls k4=True\\n#   nvidia/ops/qsa.py, ops/qsa_dcp.py the split-K kernel decodes 4-bit K (constexpr K4); wrapper, warmup, DCP partial\\n# FLASHNEXT_QSA_K4 unset = exactly V4 (every edit branches on it). K4 without V4 stops with an error. Safe to run again (\\\"already\\\").\\n# FLASHNEXT_K4_UNDO=1 reverses it (V4 stays).\\nimport os\\nfrom pathlib import Path\\n\\n_root = Path(globals().get(\\\"VROOT\\\") or \\\"/tmp/site-vllm/vllm\\\")\\n_here = Path(globals().get(\\\"K4_DIR\\\") or (Path(__file__).parent if \\\"__file__\\\" in globals() else \\\".\\\"))\\n_MOD_SRC = globals().get(\\\"K4_MODULE_SRC\\\") or (_here / \\\"qsa_k4.py\\\").read_text()\\n_MARK = \\\"# flashnext K4 (session 17)\\\"\\n_V4MARK = \\\"# flashnext V4 (session 17)\\\"\\n_NEW = _root / \\\"models/qwen4_exp/nvidia/ops/qsa_k4.py\\\"\\n\\n\\ndef _write(path, text):\\n    tmp = path.with_name(path.name + \\\".fn_tmp\\\")\\n    tmp.write_text(text)\\n    os.replace(tmp, path)\\n\\n\\n_K4ROW = \\\"((_fn_d // 2 + _fn_d // 2 + 16 + 15) // 16 * 16)\\\"\\nEDITS = {\\n    \\\"platforms/interface.py\\\": [\\n        (\\\"attention page per token\\\",\\n         \\\"\\\"\\\"                attn_page_size_1_token = model_config.get_num_kv_heads(parallel_config) * ((_fn_d + _fn_d // 2 + 8 + 15) // 16 * 16)\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"                attn_page_size_1_token = model_config.get_num_kv_heads(parallel_config) * ((_fn_d + _fn_d // 2 + 8 + 15) // 16 * 16)\\n                if _fn_os.environ.get(\\\"FLASHNEXT_QSA_K4\\\") == \\\"1\\\":   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\": K 4-bit too\\n                    attn_page_size_1_token = model_config.get_num_kv_heads(parallel_config) * \\\"\\\"\\\" + _K4ROW + \\\"\\\"\\\"\\n\\\"\\\"\\\"),\\n    ],\\n    \\\"models/qwen4_exp/common/qsa_cache.py\\\": [\\n        (\\\"compressed-key page\\\",\\n         \\\"\\\"\\\"            _fn_page = _fn_base * _fn_mc.get_num_kv_heads(vllm_config.parallel_config) * ((_fn_d + _fn_d // 2 + 8 + 15) // 16 * 16)\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"            _fn_page = _fn_base * _fn_mc.get_num_kv_heads(vllm_config.parallel_config) * ((_fn_d + _fn_d // 2 + 8 + 15) // 16 * 16)\\n            if _fn_os.environ.get(\\\"FLASHNEXT_QSA_K4\\\") == \\\"1\\\":   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\"\\n                _fn_page = _fn_base * _fn_mc.get_num_kv_heads(vllm_config.parallel_config) * \\\"\\\"\\\" + _K4ROW + \\\"\\\"\\\"\\n\\\"\\\"\\\"),\\n    ],\\n    \\\"models/qwen4_exp/nvidia/qsa.py\\\": [\\n        (\\\"spec bytes\\\",\\n         \\\"\\\"\\\"    return v4_row_bytes(head_dim)\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"    from .ops.qsa_k4 import k4_check_enabled, k4_row_bytes   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\"\\n\\n    if k4_check_enabled():\\n        return k4_row_bytes(head_dim)\\n    return v4_row_bytes(head_dim)\\n\\\"\\\"\\\"),\\n        (\\\"impl flag\\\",\\n         \\\"\\\"\\\"        self._fn_v4 = _fn_v4_enabled()\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"        self._fn_v4 = _fn_v4_enabled()\\n        from .ops.qsa_k4 import k4_check_enabled as _fn_k4_check   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\"\\n        self._fn_k4 = _fn_k4_check()\\n\\\"\\\"\\\"),\\n        (\\\"impl views + query rotation\\\",\\n         \\\"\\\"\\\"            key_cache, value_cache = v4_views(kv_cache, self.head_size)\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"            if self._fn_k4:   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\": packed 4-bit K and V; the query is rotated like the stored keys\\n                from .ops.qsa_k4 import k4_rotate, k4_views\\n\\n                key_cache, value_cache = k4_views(kv_cache, self.head_size)\\n                query = k4_rotate(query)\\n            else:\\n                key_cache, value_cache = v4_views(kv_cache, self.head_size)\\n\\\"\\\"\\\"),\\n        (\\\"impl fp8 key view\\\",\\n         \\\"\\\"\\\"            key_cache = key_cache.view(torch.float8_e4m3fn)\\n            if not self._fn_v4:\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"            if not self._fn_k4:   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\"\\n                key_cache = key_cache.view(torch.float8_e4m3fn)\\n            if not self._fn_v4:\\n\\\"\\\"\\\"),\\n        (\\\"impl dtype check\\\",\\n         \\\"\\\"\\\"        if query.dtype != torch.bfloat16 or key_cache.dtype not in (\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"        if query.dtype != torch.bfloat16 or not self._fn_k4 and key_cache.dtype not in (   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\"\\n\\\"\\\"\\\"),\\n        (\\\"single-GPU call\\\",\\n         \\\"\\\"\\\"            output_gate=output_gate[:num_tokens],\\n            v4=self._fn_v4,   \\\"\\\"\\\" + _V4MARK + \\\"\\\"\\\"\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"            output_gate=output_gate[:num_tokens],\\n            v4=self._fn_v4,   \\\"\\\"\\\" + _V4MARK + \\\"\\\"\\\"\\n            k4=self._fn_k4,   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\"\\n\\\"\\\"\\\"),\\n        (\\\"DCP call\\\",\\n         \\\"\\\"\\\"            self._QSA_DCP_WARPS,\\n            v4=self._fn_v4,   \\\"\\\"\\\" + _V4MARK + \\\"\\\"\\\"\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"            self._QSA_DCP_WARPS,\\n            v4=self._fn_v4,   \\\"\\\"\\\" + _V4MARK + \\\"\\\"\\\"\\n            k4=self._fn_k4,   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\"\\n\\\"\\\"\\\"),\\n        (\\\"cache write\\\",\\n         \\\"\\\"\\\"        v4_store(key, value, kv_cache, slot_mapping, layer._k_scale_float)\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"        if self._fn_k4:   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\"\\n            from .ops.qsa_k4 import k4_store\\n\\n            k4_store(key, value, kv_cache, slot_mapping)\\n            return\\n        v4_store(key, value, kv_cache, slot_mapping, layer._k_scale_float)\\n\\\"\\\"\\\"),\\n    ],\\n    \\\"models/qwen4_exp/nvidia/ops/qsa.py\\\": [\\n        (\\\"kernel constexpr\\\",\\n         \\\"\\\"\\\"    V4: tl.constexpr = False,   \\\"\\\"\\\" + _V4MARK + \\\"\\\"\\\": v_cache holds 4-bit V codes + per-row float32 scale/zero\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"    V4: tl.constexpr = False,   \\\"\\\"\\\" + _V4MARK + \\\"\\\"\\\": v_cache holds 4-bit V codes + per-row float32 scale/zero\\n    K4: tl.constexpr = False,   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\": k_cache holds rotated 4-bit K codes; K scale/zero (float32) at row + HEAD_DIM + 8\\n\\\"\\\"\\\"),\\n        (\\\"kernel K decode\\\",\\n         \\\"\\\"\\\"        keys = tl.load(\\n            k_cache_ptr\\n            + safe_page[None, :] * stride_k_block\\n            + page_offset[None, :] * stride_k_token\\n            + kv_head * stride_k_head\\n            + dim_offsets[:, None],\\n            mask=valid[None, :],\\n            other=0.0,\\n        )\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"        if K4:   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\": decode [BLOCK_N, D] then transpose to the [D, BLOCK_N] tile the dot takes\\n            k_row = k_cache_ptr + safe_page * stride_k_block + page_offset * stride_k_token + kv_head * stride_k_head\\n            k_packed = tl.load(k_row[:, None] + tl.arange(0, HEAD_DIM // 2)[None, :], mask=valid[:, None], other=0)\\n            k_meta = (k_row + HEAD_DIM + 8).to(tl.pointer_type(tl.float32))\\n            k_scale_row = tl.load(k_meta, mask=valid, other=0.0)\\n            k_zero_row = tl.load(k_meta + 1, mask=valid, other=0.0)\\n            k_codes = tl.reshape(tl.join(k_packed & 0xF, k_packed >> 4), (BLOCK_N, HEAD_DIM))\\n            keys = tl.trans((k_codes.to(tl.float32) * k_scale_row[:, None] + k_zero_row[:, None]).to(query.dtype))\\n        else:\\n            keys = tl.load(\\n                k_cache_ptr\\n                + safe_page[None, :] * stride_k_block\\n                + page_offset[None, :] * stride_k_token\\n                + kv_head * stride_k_head\\n                + dim_offsets[:, None],\\n                mask=valid[None, :],\\n                other=0.0,\\n            )\\n\\\"\\\"\\\"),\\n        (\\\"wrapper argument\\\",\\n         \\\"\\\"\\\"    v4: bool = False,   \\\"\\\"\\\" + _V4MARK + \\\"\\\"\\\": v_cache = packed 4-bit V rows (uint8, D/2 wide) from qsa_v4.v4_views\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"    v4: bool = False,   \\\"\\\"\\\" + _V4MARK + \\\"\\\"\\\": v_cache = packed 4-bit V rows (uint8, D/2 wide) from qsa_v4.v4_views\\n    k4: bool = False,   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\": k_cache = packed rotated 4-bit K rows (qsa_k4.k4_views); q must be rotated (k4_rotate)\\n\\\"\\\"\\\"),\\n        (\\\"wrapper head check\\\",\\n         \\\"\\\"\\\"    if q.shape[2] != k_cache.shape[3] or q.shape[1] % k_cache.shape[2]:\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"    if q.shape[2] != k_cache.shape[3] * (2 if k4 else 1) or q.shape[1] % k_cache.shape[2]:   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\"\\n\\\"\\\"\\\"),\\n        (\\\"wrapper fp8 path\\\",\\n         \\\"\\\"\\\"    assert not v4 or v_cache.dtype == torch.uint8   \\\"\\\"\\\" + _V4MARK + \\\"\\\"\\\"\\n    is_fp8 = k_cache.dtype == torch.float8_e4m3fn\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"    assert not v4 or v_cache.dtype == torch.uint8   \\\"\\\"\\\" + _V4MARK + \\\"\\\"\\\"\\n    assert not k4 or (v4 and k_cache.dtype == torch.uint8)   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\"\\n    is_fp8 = k_cache.dtype == torch.float8_e4m3fn or k4\\n\\\"\\\"\\\"),\\n        (\\\"wrapper softmax scale\\\",\\n         \\\"\\\"\\\"        output_scale = 1.0   \\\"\\\"\\\" + _V4MARK + \\\"\\\"\\\": each row's own scale/zero dequantize V\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"        output_scale = 1.0   \\\"\\\"\\\" + _V4MARK + \\\"\\\"\\\": each row's own scale/zero dequantize V\\n    if k4:\\n        softmax_scale = head_dim**-0.5   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\": absolute per-row key scales (no FP8 k_scale)\\n\\\"\\\"\\\"),\\n        (\\\"wrapper launch\\\",\\n         \\\"\\\"\\\"        V4=v4,   \\\"\\\"\\\" + _V4MARK + \\\"\\\"\\\"\\n        num_warps=partial_warps,\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"        V4=v4,   \\\"\\\"\\\" + _V4MARK + \\\"\\\"\\\"\\n        K4=k4,   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\"\\n        num_warps=partial_warps,\\n\\\"\\\"\\\"),\\n        (\\\"warmup views\\\",\\n         \\\"\\\"\\\"    _fn_v4 = v4_enabled()\\n    if _fn_v4:\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"    _fn_v4 = v4_enabled()\\n    from .qsa_k4 import k4_check_enabled, k4_row_bytes, k4_views   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\"\\n\\n    _fn_k4 = k4_check_enabled()\\n    if _fn_k4:\\n        head_dim = next(d for d in (64, 128, 256, 512) if k4_row_bytes(d) == kv_cache.shape[-1])\\n        key_cache, value_cache = k4_views(kv_cache, head_dim)\\n    elif _fn_v4:\\n\\\"\\\"\\\"),\\n        (\\\"warmup K pointer\\\",\\n         \\\"\\\"\\\"    k_cache_ptr = TritonWarmupTensor(\\n        cache_dtype,\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"    k_cache_ptr = TritonWarmupTensor(\\n        torch.uint8 if _fn_k4 else cache_dtype,   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\"\\n\\\"\\\"\\\"),\\n        (\\\"warmup launch\\\",\\n         \\\"\\\"\\\"            V4=_fn_v4,   \\\"\\\"\\\" + _V4MARK + \\\"\\\"\\\"\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"            V4=_fn_v4,   \\\"\\\"\\\" + _V4MARK + \\\"\\\"\\\"\\n            K4=_fn_k4,   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\"\\n\\\"\\\"\\\"),\\n    ],\\n    \\\"models/qwen4_exp/nvidia/ops/qsa_dcp.py\\\": [\\n        (\\\"partial argument\\\",\\n         \\\"\\\"\\\"    v4: bool = False,   \\\"\\\"\\\" + _V4MARK + \\\"\\\"\\\": v_cache = packed 4-bit V rows (qsa_v4.v4_views)\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"    v4: bool = False,   \\\"\\\"\\\" + _V4MARK + \\\"\\\"\\\": v_cache = packed 4-bit V rows (qsa_v4.v4_views)\\n    k4: bool = False,   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\": k_cache = packed rotated 4-bit K rows; q rotated\\n\\\"\\\"\\\"),\\n        (\\\"partial scales\\\",\\n         \\\"\\\"\\\"    is_fp8 = k_cache.dtype == torch.float8_e4m3fn\\n    softmax_scale = (head_dim**-0.5) * (float(k_scale) if is_fp8 else 1.0)\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"    is_fp8 = k_cache.dtype == torch.float8_e4m3fn or k4   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\"\\n    softmax_scale = (head_dim**-0.5) * (float(k_scale) if is_fp8 and not k4 else 1.0)\\n\\\"\\\"\\\"),\\n        (\\\"partial launch\\\",\\n         \\\"\\\"\\\"        V4=v4,   \\\"\\\"\\\" + _V4MARK + \\\"\\\"\\\"\\n        num_warps=num_warps,\\n\\\"\\\"\\\",\\n         \\\"\\\"\\\"        V4=v4,   \\\"\\\"\\\" + _V4MARK + \\\"\\\"\\\"\\n        K4=k4,   \\\"\\\"\\\" + _MARK + \\\"\\\"\\\"\\n        num_warps=num_warps,\\n\\\"\\\"\\\"),\\n    ],\\n}\\n\\nif os.environ.get(\\\"FLASHNEXT_K4_UNDO\\\") == \\\"1\\\":\\n    for _rel, _edits in EDITS.items():\\n        _p = _root / _rel\\n        _s = _p.read_text()\\n        _n = 0\\n        for _l, _o, _w in reversed(_edits):\\n            if _s.count(_w) == 1:\\n                _s, _n = _s.replace(_w, _o), _n + 1\\n        assert _MARK not in _s, f\\\"K4 undo: a marked line is left in {_p}\\\"\\n        compile(_s, str(_p), \\\"exec\\\")\\n        _write(_p, _s)\\n        print(f\\\"K4: UNDONE in {_rel} ({_n} edits)\\\")\\n    if _NEW.exists():\\n        _NEW.unlink()\\n    print(\\\"K4: removed\\\", _NEW)\\nelse:\\n    _texts = {rel: (_root / rel).read_text() for rel in EDITS}\\n    assert all(_V4MARK in s for s in _texts.values()), \\\"K4: install V4 first (v4_edits.py)\\\"\\n    _marked = [rel for rel, s in _texts.items() if _MARK in s]\\n    _complete = _marked and all(_texts[rel].count(_w) == 1 for rel, _edits in EDITS.items() for _l, _o, _w in _edits)\\n    if _marked and not (_complete and _NEW.exists() and _NEW.read_text() == _MOD_SRC):\\n        for _rel in list(_texts):                       # an older K4 install: reverse what is there, then install fresh\\n            for _l, _o, _w in reversed(EDITS[_rel]):\\n                if _texts[_rel].count(_w) == 1:\\n                    _texts[_rel] = _texts[_rel].replace(_w, _o)\\n            assert _MARK not in _texts[_rel], f\\\"K4: cannot undo the older install in {_rel} - start a fresh runtime\\\"\\n        print(\\\"K4: updating an older install\\\")\\n        _marked = []\\n    if _marked:\\n        print(\\\"K4: [already] installed\\\")\\n    else:\\n        _new = {}\\n        for _rel, _edits in EDITS.items():\\n            _s = _texts[_rel]\\n            for _l, _o, _w in _edits:\\n                assert _s.count(_o) == 1, f\\\"K4 edit '{_l}' in {_rel}: anchor found {_s.count(_o)} times\\\"\\n                _s = _s.replace(_o, _w)\\n            compile(_s, _rel, \\\"exec\\\")\\n            _new[_rel] = _s\\n        compile(_MOD_SRC, str(_NEW), \\\"exec\\\")\\n        _write(_NEW, _MOD_SRC)\\n        for _rel, _s in _new.items():\\n            _write(_root / _rel, _s)\\n        print(\\\"K4: [installed] ops/qsa_k4.py +\\\", \\\", \\\".join(f\\\"{rel} ({len(EDITS[rel])})\\\" for rel in EDITS),\\n              \\\"| active only with FLASHNEXT_QSA_V4=1 and FLASHNEXT_QSA_K4=1 at the server start\\\")\\n\"")
K4_MODULE = json.loads("\"# flashnext QSA \\\"K4\\\" KV layout (session 17): keys AND values 4-bit, on top of V4 (v4_edits.py). Installed as\\n# vllm/models/qwen4_exp/nvidia/ops/qsa_k4.py by k4_edits.py; active only when FLASHNEXT_QSA_V4=1 AND FLASHNEXT_QSA_K4=1 at the server start.\\n#\\n# Keys are rotated by the orthonormal Hadamard matrix H (D x D, Sylvester, symmetric) before they are quantized, and every query is rotated\\n# by the same H before attention: (qH).(kH) = q.k, so the scores are unchanged in exact arithmetic, while the rotation spreads outlier\\n# channels over all dimensions so a per-row 4-bit grid loses less. Values are NOT rotated (the output and its gate stay in normal space).\\n# One cache row per (token, KV head), K4_ROW_BYTES = round_up(D/2 + D/2 + 16, 16) bytes (D = 256 -> 272; V4 400; FP8 512):\\n#   [0, D/2)          K (rotated), 4-bit codes, byte i = dim 2i (low nibble) | dim 2i+1 (high nibble)\\n#   [D/2, D)          V, 4-bit codes, same packing\\n#   [D, D+8)          V scale, V zero (float32)      <- right after the V codes, as in V4 (the kernel reads V the same way)\\n#   [D+8, D+16)       K scale, K zero (float32)\\nimport torch\\n\\nfrom vllm.triton_utils import tl, triton\\n\\nfrom .qsa_v4 import v4_enabled\\n\\n_H = {}\\n\\n\\ndef k4_enabled() -> bool:\\n    import os\\n\\n    return os.environ.get(\\\"FLASHNEXT_QSA_K4\\\") == \\\"1\\\"\\n\\n\\ndef k4_row_bytes(head_dim: int) -> int:\\n    return (head_dim // 2 + head_dim // 2 + 16 + 15) // 16 * 16\\n\\n\\ndef hadamard(head_dim: int, device) -> torch.Tensor:\\n    \\\"\\\"\\\"Orthonormal Sylvester Hadamard matrix (float32, symmetric: H = H^T = H^-1), cached per (size, device).\\\"\\\"\\\"\\n    key = (head_dim, str(device))\\n    h = _H.get(key)\\n    if h is None:\\n        assert head_dim & (head_dim - 1) == 0, \\\"the Hadamard rotation needs a power-of-two head size\\\"\\n        m = torch.ones(1, 1)\\n        while m.shape[0] < head_dim:\\n            m = torch.cat([torch.cat([m, m], 1), torch.cat([m, -m], 1)], 0)\\n        h = _H[key] = (m / head_dim ** 0.5).to(device=device, dtype=torch.float32)\\n    return h\\n\\n\\ndef k4_rotate(x: torch.Tensor) -> torch.Tensor:\\n    \\\"\\\"\\\"x [..., D] -> x @ H in x's dtype (queries before attention, keys before they are stored).\\\"\\\"\\\"\\n    return (x.float() @ hadamard(x.shape[-1], x.device)).to(x.dtype)\\n\\n\\n@triton.jit\\ndef _quant_row(x, HALF: tl.constexpr):\\n    lo_v = tl.min(x, axis=0)\\n    scale = (tl.max(x, axis=0) - lo_v) / 15.0\\n    inv = tl.where(scale > 0, 1.0 / scale, 0.0)\\n    code = tl.minimum(tl.maximum(tl.floor((x - lo_v) * inv + 0.5), 0.0), 15.0).to(tl.uint8)\\n    lo, hi = tl.split(tl.reshape(code, (HALF, 2)))\\n    return lo | (hi << 4), scale, lo_v\\n\\n\\n@triton.jit\\ndef _k4_store_kernel(\\n    key_ptr,\\n    value_ptr,\\n    cache_ptr,\\n    slot_ptr,\\n    stride_key_tok,\\n    stride_key_head,\\n    stride_val_tok,\\n    stride_val_head,\\n    stride_cache_block,\\n    stride_cache_head,\\n    stride_cache_token,\\n    num_cache_slots,\\n    PAGE_SIZE: tl.constexpr,\\n    HEAD_DIM: tl.constexpr,\\n):\\n    tok = tl.program_id(0)\\n    head = tl.program_id(1)\\n    slot = tl.load(slot_ptr + tok).to(tl.int64)\\n    if (slot < 0) | (slot >= num_cache_slots):\\n        return\\n    HALF: tl.constexpr = HEAD_DIM // 2\\n    row = cache_ptr + (slot // PAGE_SIZE) * stride_cache_block + head * stride_cache_head + (slot % PAGE_SIZE) * stride_cache_token\\n    dims = tl.arange(0, HEAD_DIM)\\n    halves = tl.arange(0, HALF)\\n    k = tl.load(key_ptr + tok * stride_key_tok + head * stride_key_head + dims).to(tl.float32)      # already rotated (k4_store)\\n    kp, ks, kz = _quant_row(k, HALF)\\n    v = tl.load(value_ptr + tok * stride_val_tok + head * stride_val_head + dims).to(tl.float32)\\n    vp, vs, vz = _quant_row(v, HALF)\\n    tl.store(row + halves, kp)\\n    tl.store(row + HALF + halves, vp)\\n    meta = (row + HEAD_DIM).to(tl.pointer_type(tl.float32))\\n    tl.store(meta, vs)\\n    tl.store(meta + 1, vz)\\n    tl.store(meta + 2, ks)\\n    tl.store(meta + 3, kz)\\n\\n\\ndef k4_store(key: torch.Tensor, value: torch.Tensor, kv_cache: torch.Tensor, slot_mapping: torch.Tensor) -> None:\\n    \\\"\\\"\\\"Write this step's K/V rows: K rotated by H then 4-bit, V 4-bit. key/value [tokens, heads, D]; kv_cache [blocks, heads, page,\\n    K4_ROW_BYTES] uint8; slot_mapping [tokens] (negative = not stored on this rank). The layer's FP8 k_scale is not used (absolute\\n    per-row scales).\\\"\\\"\\\"\\n    n = slot_mapping.shape[0]\\n    if n == 0:\\n        return\\n    head_dim = key.shape[-1]\\n    assert kv_cache.dtype == torch.uint8 and kv_cache.shape[-1] == k4_row_bytes(head_dim) and kv_cache.stride(-1) == 1\\n    assert value.stride(-1) == 1 and key.shape[1] == kv_cache.shape[1]\\n    k_rot = (key[:n].float() @ hadamard(head_dim, key.device)).contiguous()\\n    _k4_store_kernel[(n, key.shape[1])](\\n        k_rot, value, kv_cache, slot_mapping,\\n        k_rot.stride(0), k_rot.stride(1), value.stride(0), value.stride(1),\\n        kv_cache.stride(0), kv_cache.stride(1), kv_cache.stride(2),\\n        kv_cache.shape[0] * kv_cache.shape[2],\\n        PAGE_SIZE=kv_cache.shape[2], HEAD_DIM=head_dim, num_warps=2,\\n    )\\n\\n\\ndef k4_views(kv_cache: torch.Tensor, head_dim: int) -> tuple[torch.Tensor, torch.Tensor]:\\n    \\\"\\\"\\\"(packed K [blocks, page, heads, D/2] uint8, packed V [blocks, page, heads, D/2] uint8) over the same bytes.\\\"\\\"\\\"\\n    t = kv_cache.transpose(1, 2)\\n    return t[..., :head_dim // 2], t[..., head_dim // 2:head_dim]\\n\\n\\ndef k4_check_enabled() -> bool:\\n    \\\"\\\"\\\"True when K4 is on; K4 without V4 is refused (no silent fallback).\\\"\\\"\\\"\\n    if not k4_enabled():\\n        return False\\n    if not v4_enabled():\\n        raise RuntimeError(\\\"FLASHNEXT_QSA_K4=1 needs FLASHNEXT_QSA_V4=1 (4-bit keys come on top of 4-bit values)\\\")\\n    return True\\n\\n\\ndef k4_dequant_reference(kv_cache: torch.Tensor, head_dim: int) -> tuple[torch.Tensor, torch.Tensor]:\\n    \\\"\\\"\\\"Test helper: (rotated K, V) as float32 [blocks, page, heads, D], decoded from the rows the way the kernel decodes them.\\\"\\\"\\\"\\n    kp, vp = k4_views(kv_cache, head_dim)\\n    t = kv_cache.transpose(1, 2)\\n    meta = t[..., head_dim:head_dim + 16].contiguous().view(torch.float32)          # [..., 4]: v scale, v zero, k scale, k zero\\n    dec = lambda p: torch.stack([p & 0xF, p >> 4], dim=-1).reshape(*p.shape[:-1], head_dim).float()\\n    return meta[..., 3:4] + dec(kp) * meta[..., 2:3], meta[..., 1:2] + dec(vp) * meta[..., 0:1]\\n\"")
os.environ['FLASHNEXT_K4_UNDO'] = '0'
exec(compile(K4_EDITS, 'k4_edits', 'exec'), {'__name__': '__main__', 'VROOT': VROOT, 'K4_MODULE_SRC': K4_MODULE})
os.environ['FLASHNEXT_QSA_K4'] = '1'
os.environ['FLASHNEXT_PREFIX_MATCH_UNIT'] = '1504'
print('QSA KV:', 'K4 (keys + values 4-bit)' if os.environ.get('FLASHNEXT_QSA_K4') == '1' else 'V4 (values 4-bit)',
      '| prefix match unit', os.environ['FLASHNEXT_PREFIX_MATCH_UNIT'], '| prefill chunk', os.environ['FLASHNEXT_MAX_BATCHED'])

In [ ]:
import os
from pathlib import Path

_root = Path(globals().get("VROOT") or "/tmp/site-vllm/vllm")
_MARK = "# flashnext state RAM v2 (session 18b)"
_OLD_MARK = "# flashnext state RAM (session 18b)"

_SINGLE_EDITS = [
    ("retire: newest state to RAM instead of pinned",
     """        if _FN_HOLD["mode"] == "pinnew":   # flashnext hold-states switch (session 17): only the newest state stays pinned
""",
     """        if _FN_HOLD["mode"] == "pinnew" and getattr(self.block_pool, "_fn_sr", None) is not None:   """ + _MARK + """
            return _fn_sr_retire(self, request_id, idx, block, aligned, held)
        if _FN_HOLD["mode"] == "pinnew":   # flashnext hold-states switch (session 17): only the newest state stays pinned
"""),
    ("lookup: fine-grained DeltaNet hit may come from RAM",
     """                if cached_block := block_pool.get_cached_block(
                    block_hash, kv_cache_group_ids
                ):
                    block_idx = fine_idx // scale_factor
""",
     """                if cached_block := _fn_sr_cached(block_pool, block_hash, kv_cache_group_ids, True):   """ + _MARK + """
                    block_idx = fine_idx // scale_factor
"""),
    ("lookup: DeltaNet hit may come from RAM",
     """            if cached_block := block_pool.get_cached_block(
                block_hashes[i], kv_cache_group_ids
            ):
                # When enable Mamba prefix caching, `block_size` will be aligned
""",
     """            if cached_block := _fn_sr_cached(block_pool, block_hashes[i], kv_cache_group_ids,   """ + _MARK + """
                                             block_size == alignment_tokens or (i + 1) * block_size % alignment_tokens == 0):
                # When enable Mamba prefix caching, `block_size` will be aligned
"""),
]
_SINGLE_TAIL = '''

''' + _MARK + '''
def _fn_sr_retire(mgr, request_id, idx, block, aligned, held):
    """pinnew with the newest state in RAM: queue one copy to RAM, then free it as a normal cached block (taken back at the end of the
    call if the GPU still has it). Older held states of the request were freed already."""
    from vllm.v1.core.kv_cache_coordinator import _FN_SEG

    pool = mgr.block_pool
    for h in list(held):
        held.remove(h)
        if len(h) > 3:
            _FN_HOLD["prot"].pop(h[1].block_id, None)
        else:
            pool.free_blocks([h[1]])
        _FN_SEG["states_dropped"] += 1
    pool._fn_sr._fn_sr_store(request_id, mgr.kv_cache_group_id, block)
    held.append((idx, block, aligned, block.block_hash))
    pool.free_blocks([block])
    _FN_HOLD["freed"] += 1
    return True


def _fn_sr_cached(block_pool, block_hash, kv_cache_group_ids, aligned_ok):
    """get_cached_block, and for the GPU pool a restore from RAM when the GPU no longer holds the state."""
    sr = getattr(block_pool, "_fn_sr", None)
    blocks = block_pool.get_cached_block(block_hash, kv_cache_group_ids)
    if sr is None or not aligned_ok:
        return blocks
    if blocks:
        return sr._fn_sr_force(block_pool, block_hash, kv_cache_group_ids, blocks) if sr._fn_sr_cfg["force"] else blocks
    return sr._fn_sr_restore(block_pool, block_hash, kv_cache_group_ids)
'''

_MGR_EDITS = [
    ("connector: the GPU pool knows the state RAM",
     """        if self._fn_on:
            gpu_block_pool._fn_on_evict = self._fn_gpu_evicted
            self.cpu_block_pool._fn_on_evict = self._fn_ram_evicted
""",
     """        if self._fn_on:
            gpu_block_pool._fn_on_evict = self._fn_gpu_evicted
            self.cpu_block_pool._fn_on_evict = self._fn_ram_evicted
        self._fn_sr_init(gpu_block_pool)   """ + _MARK + """
"""),
    ("connector: state RAM methods",
     """    def _fn_gpu_evicted(self, block, hashes) -> None:
""",
     """    def _fn_sr_init(self, gpu_block_pool) -> None:   """ + _MARK + """
        import os as _o
        import time as _t

        self._fn_sr_cfg = {"on": _o.environ.get("FLASHNEXT_STATE_RAM") == "1", "force": _o.environ.get("FLASHNEXT_STATE_RAM_FORCE") == "1",
                           "check": _o.environ.get("FLASHNEXT_STATE_RAM_CHECK") == "1",
                           "reserve": int(_o.environ.get("FLASHNEXT_STATE_RAM_RESERVE", "64"))}
        self._fn_sr_loads = []        # (gpu blocks, cpu blocks, original gpu blocks or None) restored this step
        self._fn_sr_last = {}         # (request id, group id) -> hash of its newest RAM copy
        self._fn_sr_forced = set()    # test: boundary hashes already forced through RAM once
        self._fn_sr_n = dict.fromkeys(("stored", "already", "ram_full", "no_hash", "ram_dropped", "restored", "no_room",
                                       "forced"), 0)
        self._fn_sr_tot = dict(self._fn_sr_n)
        self._fn_sr_t = _t.time()
        gpu_block_pool._fn_sr = None
        if not self._fn_sr_cfg["on"]:
            return
        if not self._fn_on:
            raise RuntimeError("FLASHNEXT_STATE_RAM=1 needs the RAM offload with FLASHNEXT_OFFLOAD_EVICT=1 (KV_OFFLOAD_GIB set)")
        from vllm.v1.core import single_type_kv_cache_manager as _st

        _h = getattr(_st, "_FN_HOLD", None)
        if _h is None or _h["mode"] != "pinnew":
            raise RuntimeError(f"FLASHNEXT_STATE_RAM=1 needs FLASHNEXT_HOLD_STATES=pinnew (got {None if _h is None else _h['mode']!r})")
        gpu_block_pool._fn_sr = self
        print(f"flashnext state RAM: the newest DeltaNet state of each running task goes to RAM instead of staying pinned on the GPU "
              f"(force {self._fn_sr_cfg['force']}, check {self._fn_sr_cfg['check']}, restore reserve {self._fn_sr_cfg['reserve']} blocks)",
              flush=True)

    def _fn_sr_store(self, request_id, group_id, block) -> None:
        \"\"\"Queue one GPU -> RAM copy of a retired DeltaNet state (the copy holds a GPU reference until it is done).\"\"\"
        n, cpu = self._fn_sr_n, self.cpu_block_pool
        h = block.block_hash
        if h is None:
            n["no_hash"] += 1
            return
        key = (request_id, group_id)
        old = self._fn_sr_last.pop(key, None)
        if old is not None and old != h:
            ob = cpu.cached_block_hash_to_block.get_one_block(old)
            if ob is not None and ob.ref_cnt == 0 and cpu._maybe_evict_cached_block(ob):
                n["ram_dropped"] += 1
        self._fn_sr_last[key] = h
        if len(self._fn_sr_last) > 200000:
            for k in list(self._fn_sr_last)[:100000]:
                del self._fn_sr_last[k]
        if cpu.cached_block_hash_to_block.get_one_block(h) is not None:
            n["already"] += 1
            return
        if cpu.get_num_free_blocks() < 1:
            n["ram_full"] += 1
            return
        cb = cpu.get_new_blocks(1)[0]
        self._gpu_block_pool.touch([block])
        self._pending_finished_stores.append(TransferMeta([block.block_id], [cb.block_id], None))
        self._fn_ram_held += 1
        n["stored"] += 1

    def _fn_sr_restore(self, pool, block_hash, group_ids, originals=None):
        \"\"\"The state at this boundary is in RAM but not on the GPU: take fresh GPU blocks, register them under the same hashes and
        queue the RAM -> GPU copy for this step (before its forward). Returns the blocks like get_cached_block, or None.\"\"\"
        from vllm.v1.core.kv_cache_utils import make_block_hash_with_group_id as _mk

        cpu, n = self.cpu_block_pool, self._fn_sr_n
        cbs = []
        for g in group_ids:
            cb = cpu.cached_block_hash_to_block.get_one_block(_mk(block_hash, g))
            if cb is None:
                return None
            cbs.append(cb)
        if pool.get_num_free_blocks() < len(group_ids) + self._fn_sr_cfg["reserve"]:
            n["no_room"] += 1
            return None
        gbs = pool.get_new_blocks(len(group_ids))          # ref 1 each until build_connector_meta: nobody else takes them this step
        if originals is not None:
            pool.touch(originals)                          # check mode: the originals must keep their content until the compare
        for g, gb, cb in zip(group_ids, gbs, cbs):
            pool._insert_block_hash(_mk(block_hash, g), gb, num_tokens=cb.block_hash_num_tokens)
        cpu.touch(cbs)
        self._fn_sr_loads.append((gbs, cbs, originals))
        n["restored"] += 1
        return gbs

    def _fn_sr_force(self, pool, block_hash, group_ids, blocks):
        \"\"\"TEST ONLY (FLASHNEXT_STATE_RAM_FORCE=1): use the RAM copy although the GPU still has the state.\"\"\"
        if any(b.ref_cnt != 0 for b in blocks) or block_hash in self._fn_sr_forced:
            return blocks                                  # in use (running request / pending copy) or forced once already
        from vllm.v1.core.kv_cache_utils import make_block_hash_with_group_id as _mk

        cpu = self.cpu_block_pool
        if any(cpu.cached_block_hash_to_block.get_one_block(_mk(block_hash, g)) is None for g in group_ids):
            return blocks
        if self._fn_sr_cfg["check"]:
            got = self._fn_sr_restore(pool, block_hash, group_ids, originals=list(blocks))
        else:
            pool._fn_dropping = True
            try:
                for b in blocks:
                    pool._maybe_evict_cached_block(b)
            finally:
                pool._fn_dropping = False
            got = self._fn_sr_restore(pool, block_hash, group_ids)
        if got is None:
            return blocks if self._fn_sr_cfg["check"] else None
        self._fn_sr_forced.add(block_hash)
        self._fn_sr_n["forced"] += 1
        return got

    def _fn_sr_take(self, scheduler_output):
        \"\"\"This step's restores: block ids for the worker; release the step's references.\"\"\"
        gpu, cpu, chk = [], [], []
        for gbs, cbs, orig in self._fn_sr_loads:
            gpu += [b.block_id for b in gbs]
            cpu += [b.block_id for b in cbs]
            if orig is not None:
                chk += [(o.block_id, b.block_id) for o, b in zip(orig, gbs)]
                self._gpu_block_pool.free_blocks(orig)
            self.cpu_block_pool.free_blocks(cbs)
            self._gpu_block_pool.free_blocks(gbs)
        self._fn_sr_loads.clear()
        if gpu:
            scheduler_output.has_sync_kv_loads = True      # the worker copies before this step's forward
        return gpu, cpu, chk

    def _fn_sr_report(self) -> None:
        import time as _t

        if not self._fn_sr_cfg["on"] or _t.time() - self._fn_sr_t < 60:
            return
        self._fn_sr_t = _t.time()
        n, t = self._fn_sr_n, self._fn_sr_tot
        for k, v in n.items():
            t[k] += v
        from vllm.v1.core import single_type_kv_cache_manager as _st

        h = _st._FN_HOLD
        print("flashnext state RAM (last 60 s | total): " + ", ".join(f"{k} {n[k]} | {t[k]}" for k in n)
              + f"; hold freed {h['freed']} taken back {h['back']} not on the GPU at the call end {h['lost']}", flush=True)
        for k in n:
            n[k] = 0

    def _fn_gpu_evicted(self, block, hashes) -> None:
"""),
    ("connector: this step's restores go to the worker",
     """            need_flush=bool(scheduler_output.preempted_req_ids),
        )
        return result
""",
     """            need_flush=bool(scheduler_output.preempted_req_ids),
        )
        if self._fn_sr_cfg["on"]:   """ + _MARK + """
            result.fn_sr_gpu_blocks, result.fn_sr_cpu_blocks, result.fn_sr_check = self._fn_sr_take(scheduler_output)
            self._fn_sr_report()
        return result
"""),
    ("connector: a state copy whose GPU block lost its hash is released, not cached",
     """            primary_hash = gpu_block.block_hash
            assert primary_hash is not None
""",
     """            primary_hash = gpu_block.block_hash
            if primary_hash is None and getattr(self, "_fn_sr_cfg", {}).get("on"):   """ + _MARK + """
                continue
            assert primary_hash is not None
"""),
]

_META_EDITS = [
    ("metadata: restore block lists",
     """    # Whether any requests were preempted this step and need flush pending transfers.
    need_flush: bool = False
""",
     """    # Whether any requests were preempted this step and need flush pending transfers.
    need_flush: bool = False

    """ + _MARK + """: DeltaNet states copied RAM -> GPU before this step's forward
    fn_sr_gpu_blocks: list[int] = field(default_factory=list)
    fn_sr_cpu_blocks: list[int] = field(default_factory=list)
    fn_sr_check: list[tuple[int, int]] = field(default_factory=list)   # test: (original GPU block, restored GPU block)
"""),
]

_WORKER_EDITS = [
    ("worker: restores at the START of the step (before preprocess_mamba copies a resumed state into the running block)",
     """        \"\"\"Sync all in-flight transfers before preempted blocks are reused.\"\"\"
        if not kv_connector_metadata.need_flush:
            return
""",
     """        \"\"\"Sync all in-flight transfers before preempted blocks are reused.\"\"\"
        if kv_connector_metadata.fn_sr_cpu_blocks and not getattr(kv_connector_metadata, "_fn_sr_done", False):   """ + _MARK + """
            # session 18b fix: vLLM's execute_model runs preprocess_mamba (align mode copies a resumed state block into the request's
            # running block) BEFORE the connector's sync loads; the restore must land before that copy -> here, the first connector
            # call of the step, then a device sync (steps with a restore are rare) so no stream can read the block early.
            import torch as _fn_t

            self._fn_sr_copy(kv_connector_metadata.fn_sr_cpu_blocks, kv_connector_metadata.fn_sr_gpu_blocks, kv_connector_metadata.fn_sr_check)
            _fn_t.cuda.synchronize()
            kv_connector_metadata._fn_sr_done = True
        if not kv_connector_metadata.need_flush:
            return
"""),
    ("worker: a restore that reaches the loads unseen is an error",
     """        metadata = self._connector_metadata
        if metadata is not None and metadata.load_cpu_blocks:
""",
     """        metadata = self._connector_metadata
        if metadata is not None and metadata.fn_sr_cpu_blocks and not getattr(metadata, "_fn_sr_done", False):   """ + _MARK + """
            raise RuntimeError("flashnext state RAM: restore blocks reached start_load_kv without the early copy in handle_preemptions "
                               "(vLLM's step order changed?) - the resumed DeltaNet state would be read before it is restored")
        if metadata is not None and metadata.load_cpu_blocks:
"""),
    ("worker: the copy",
     """    def wait_for_save(self) -> None:
""",
     """    def _fn_sr_copy(self, cpu_ids, gpu_ids, check) -> None:   """ + _MARK + """
        \"\"\"RAM -> GPU copies of whole blocks on the current (compute) stream: ordered before this step's forward. Pinned host rows,
        so the copies are asynchronous for the host.\"\"\"
        import torch as _torch

        if self.disk_mode:
            raise RuntimeError("flashnext state RAM: the disk mode of the offload is not supported")
        for name, g in self.gpu_kv_caches.items():
            c = self.cpu_kv_caches[name]
            for s, d in zip(cpu_ids, gpu_ids):
                g[d].copy_(c[s], non_blocking=True)
        if check:
            bad = [(o, r, name) for o, r in check for name, g in self.gpu_kv_caches.items() if not _torch.equal(g[o], g[r])]
            if bad:
                raise RuntimeError(f"flashnext state RAM CHECK: restored block differs from the GPU original: {bad[:4]}")
            self._fn_sr_checked = getattr(self, "_fn_sr_checked", 0) + len(check)
            if self._fn_sr_checked % 50 < len(check):
                print(f"flashnext state RAM CHECK: {self._fn_sr_checked} restored blocks identical to their GPU originals", flush=True)

    def wait_for_save(self) -> None:
"""),
]

_RUNNER2_EDITS = [
    ("V2 model runner: restores before the Mamba align pre-copy",
     """            block_tables, slot_mappings = self.prepare_attn(input_batch)
            # Mamba "align" pre-copy: migrate recurrent state across block
""",
     """            block_tables, slot_mappings = self.prepare_attn(input_batch)
            _fn_sr_md = getattr(scheduler_output, "kv_connector_metadata", None)   """ + _MARK + """
            if _fn_sr_md is not None and getattr(_fn_sr_md, "fn_sr_cpu_blocks", None):
                # the V2 runner calls the connector (pre_forward -> handle_preemptions) only after preprocess_state below, which
                # copies a resumed DeltaNet state into the running block: restore it from RAM first (handle_preemptions does it once)
                from vllm.distributed.kv_transfer import get_kv_transfer_group as _fn_sr_gkt

                _fn_sr_gkt().handle_preemptions(_fn_sr_md)
            # Mamba "align" pre-copy: migrate recurrent state across block
"""),
]

_FILES = {
    "v1/core/single_type_kv_cache_manager.py": (_SINGLE_EDITS, _SINGLE_TAIL),
    "v1/worker/gpu/model_runner.py": (_RUNNER2_EDITS, ""),
    "v1/simple_kv_offload/manager.py": (_MGR_EDITS, ""),
    "v1/simple_kv_offload/metadata.py": (_META_EDITS, ""),
    "v1/simple_kv_offload/worker.py": (_WORKER_EDITS, ""),
}
_done = []
for _rel, (_edits, _tail) in _FILES.items():
    _f = _root / _rel
    assert _f.exists(), f"{_f} not found: run the earlier steps first"
    _s = _f.read_text()
    if _OLD_MARK in _s:
        raise RuntimeError(f"{_f} has the FIRST state-RAM version (restore too late, session 18b bug): start a fresh runtime / vLLM tree")
    if _MARK in _s:
        _done.append(f"{_rel} [already]")
        continue
    for _label, _old, _new in _edits:
        assert _s.count(_old) == 1, f"state RAM edit '{_label}': anchor found {_s.count(_old)} times in {_f} (vLLM or an earlier cell changed?)"
        _s = _s.replace(_old, _new)
    _s += _tail
    compile(_s, str(_f), "exec")
    _f.write_text(_s)
    _done.append(f"{_rel} [installed]")
assert os.environ.get("FLASHNEXT_HOLD_STATES") == "pinnew", "FLASHNEXT_HOLD_STATES must be pinnew (hold-states cell) for the state RAM"
os.environ["FLASHNEXT_STATE_RAM"] = "1"
print("state RAM:", " | ".join(_done), "| FLASHNEXT_STATE_RAM =", os.environ["FLASHNEXT_STATE_RAM"])

In [ ]:
import os
from pathlib import Path

_root = Path(globals().get("VROOT") or "/tmp/site-vllm/vllm")
_MARK = "# flashnext ckpack (session 18b)"
_f = _root / "models/qwen4_exp/nvidia/indexer_qsa.py"
_EDITS = [
    ("indexer keys FP8",
     """        self.indexer_kv_dtype = vllm_config.attention_config.resolve_indexer_kv_dtype(
            "bf16"
        )
""",
     """        self.indexer_kv_dtype = vllm_config.attention_config.resolve_indexer_kv_dtype(
            "bf16"
        )
        if _FN_CKP["on"]:   """ + _MARK + """
            self.indexer_kv_dtype = "fp8"
"""),
    ("two layers per compressed-key row",
     """        self.compressed_key_cache = QSACompressedKeyCache(
            head_size=self.index_head_dim,
            dtype=indexer_dtype,
            compress_ratio=self.compress_ratio,
            prefix=f"{cache_prefix}compressed_key_cache",
            cache_config=cache_config,
            vllm_config=vllm_config,
        )
""",
     """        if _FN_CKP["on"]:   """ + _MARK + """
            self.compressed_key_cache = _fn_ckp_make(self, QSACompressedKeyCache, indexer_dtype, cache_prefix, cache_config, vllm_config)
        else:
            self.compressed_key_cache = QSACompressedKeyCache(
                head_size=self.index_head_dim,
                dtype=indexer_dtype,
                compress_ratio=self.compress_ratio,
                prefix=f"{cache_prefix}compressed_key_cache",
                cache_config=cache_config,
                vllm_config=vllm_config,
            )
"""),
    ("forward uses this layer's half of the row",
     """        compressed_key_cache = self.compressed_key_cache.kv_cache
""",
     """        compressed_key_cache = _fn_ckp_view(self.compressed_key_cache)   """ + _MARK + """
"""),
]
_TAIL = '''

''' + _MARK + '''
import os as _fn_ckp_os

_FN_CKP = {"on": _fn_ckp_os.environ.get("FLASHNEXT_QSA_CKPACK") == "1", "open": None, "pairs": 0}


class _FnCkPartner:
    """The second layer of a pair: no cache of its own; bytes [w, 2w) of the owner's rows, the owner's metadata (prefix)."""

    def __init__(self, owner, width):
        self._owner, self._w = owner, width
        self.prefix = owner.prefix

    @property
    def kv_cache(self):
        return self._owner.kv_cache[..., self._w:2 * self._w]


def _fn_ckp_make(ix, cls, dtype, cache_prefix, cache_config, vllm_config):
    if dtype != torch.float8_e4m3fn:
        raise RuntimeError(f"flashnext ckpack needs FP8 indexer keys, got {dtype}")
    w = int(ix.index_head_dim)
    owner = _FN_CKP["open"]
    if owner is None or owner._fn_ck_cfg is not vllm_config:
        owner = cls(head_size=2 * w, dtype=dtype, compress_ratio=ix.compress_ratio, prefix=f"{cache_prefix}compressed_key_cache",
                    cache_config=cache_config, vllm_config=vllm_config)
        owner._fn_ck_cfg, owner._fn_ck_w = vllm_config, w
        _FN_CKP["open"] = owner
        return owner
    _FN_CKP["open"] = None
    _FN_CKP["pairs"] += 1
    print(f"flashnext ckpack: {cache_prefix}compressed_key_cache shares the FP8 rows of {owner.prefix} (bytes {w}-{2 * w - 1}); "
          f"pairs so far {_FN_CKP['pairs']}", flush=True)
    return _FnCkPartner(owner, w)


def _fn_ckp_view(c):
    if isinstance(c, _FnCkPartner):
        return c.kv_cache
    w = getattr(c, "_fn_ck_w", None)
    return c.kv_cache if w is None else c.kv_cache[..., :w]
'''
assert _f.exists(), f"{_f} not found: run the earlier steps first"
_s = _f.read_text()
if _MARK in _s:
    print("ckpack: [already] in", _f)
else:
    for _label, _old, _new in _EDITS:
        assert _s.count(_old) == 1, f"ckpack edit '{_label}': anchor found {_s.count(_old)} times in {_f} (vLLM or an earlier cell changed?)"
        _s = _s.replace(_old, _new)
    _s += _TAIL
    compile(_s, str(_f), "exec")
    _f.write_text(_s)
    print("ckpack: [installed] in", _f)
os.environ["FLASHNEXT_QSA_CKPACK"] = "1"
print("FLASHNEXT_QSA_CKPACK =", os.environ["FLASHNEXT_QSA_CKPACK"])

# vLLM Patches: Recompute Prevention

In [ ]:
from pathlib import Path

_root = Path(globals().get("VROOT") or "/tmp/site-vllm/vllm")
_km = _root / "v1/core/kv_cache_manager.py"
_sm = _root / "v1/core/single_type_kv_cache_manager.py"
for _p in (_km, _sm):
    assert _p.exists(), f"{_p} not found: run the earlier steps first"
_MARK = "# flashnext exact resume (session 18b)"
_KM_EDITS = [
    ("adopt after vLLM's hit search",
     """            computed_blocks, num_new_computed_tokens, num_uncached = (
                self.coordinator.find_longest_cache_hit(
                    request.block_hashes, max_cache_hit_length
                )
            )
""",
     """            computed_blocks, num_new_computed_tokens, num_uncached = (
                self.coordinator.find_longest_cache_hit(
                    request.block_hashes, max_cache_hit_length
                )
            )
            if _FN_ER["on"]:   """ + _MARK + """
                computed_blocks, num_new_computed_tokens, num_uncached = _fn_er_adopt(
                    self, request, max_cache_hit_length, computed_blocks, num_new_computed_tokens, num_uncached)
"""),
    ("release after the allocation",
     """        new_blocks = self.coordinator.allocate_new_blocks(
            request.request_id,
            num_tokens_need_slot,
            num_tokens_main_model,
            num_encoder_tokens,
        )
""",
     """        new_blocks = self.coordinator.allocate_new_blocks(
            request.request_id,
            num_tokens_need_slot,
            num_tokens_main_model,
            num_encoder_tokens,
        )
        if _FN_ER["pending"]:   """ + _MARK + """
            _fn_er_done(self, request)
"""),
    ("park in free()",
     '''        """Free the blocks allocated for the request.
        We free the blocks in reverse order so that the tail blocks are evicted
        first when caching is enabled.

        Args:
            request: The request to free the blocks.

        """
''',
     '''        """Free the blocks allocated for the request.
        We free the blocks in reverse order so that the tail blocks are evicted
        first when caching is enabled.

        Args:
            request: The request to free the blocks.

        """
        if _FN_ER["on"]:   ''' + _MARK + '''
            _fn_er_park(self, request)
'''),
    ("park in pop_blocks_for_free()",
     '''        Returns:
            The request's blocks in allocation order.

        """
        return self.coordinator.pop_blocks_for_free(request.request_id)
''',
     '''        Returns:
            The request's blocks in allocation order.

        """
        if _FN_ER["on"]:   ''' + _MARK + '''
            _fn_er_park(self, request)
        return self.coordinator.pop_blocks_for_free(request.request_id)
'''),
]
_SM_EDITS = [
    ("ring hand-over",
     '''    def add_local_computed_blocks(
        self,
        request_id: str,
        new_computed_blocks: Sequence[KVCacheBlock],
        num_local_computed_tokens: int,
        num_external_computed_tokens: int,
    ) -> None:
        return

    def remove_skipped_blocks(
        self,
        request_id: str,
        processed_computed_tokens: int,
        num_prompt_tokens: int | None = None,
    ) -> None:
        return

    def get_num_common_prefix_blocks(self, running_request_id: str) -> int:
        return 0

    def get_num_skipped_tokens(self, num_computed_tokens: int) -> int:
        return 0


class KpoolTailManager(CircularBufferManager):''',
     '''    def add_local_computed_blocks(
        self,
        request_id: str,
        new_computed_blocks: Sequence[KVCacheBlock],
        num_local_computed_tokens: int,
        num_external_computed_tokens: int,
    ) -> None:
        from vllm.v1.core.kv_cache_manager import _FN_ER   ''' + _MARK + '''
        _e = _FN_ER["pending"].get(request_id)
        if _e is not None and self.kv_cache_group_id in _e["ring"] and not self.req_to_blocks[request_id]:
            self.req_to_blocks[request_id].append(_e["ring"][self.kv_cache_group_id])
            _e["ring_taken"].add(self.kv_cache_group_id)
        return

    def remove_skipped_blocks(
        self,
        request_id: str,
        processed_computed_tokens: int,
        num_prompt_tokens: int | None = None,
    ) -> None:
        return

    def get_num_common_prefix_blocks(self, running_request_id: str) -> int:
        return 0

    def get_num_skipped_tokens(self, num_computed_tokens: int) -> int:
        return 0


class KpoolTailManager(CircularBufferManager):'''),
]
_FN = '''

''' + _MARK + '''
import collections as _fer_col
import os as _fer_os
import time as _fer_time

_FN_ER = {"on": _fer_os.environ.get("FLASHNEXT_EXACT_RESUME") == "1", "entries": _fer_col.OrderedDict(), "pending": {},
          "ttl": float(_fer_os.environ.get("FLASHNEXT_EXACT_RESUME_TTL", "600")),
          "min_free": int(_fer_os.environ.get("FLASHNEXT_EXACT_RESUME_MIN_FREE", "64")), "cap": 512,
          "any_finish": _fer_os.environ.get("FLASHNEXT_EXACT_RESUME_TEST_ANY_FINISH") == "1",   # TEST ONLY: also park length-capped calls
          "n": _fer_col.Counter(), "tot": _fer_col.Counter(), "t": _fer_time.time(), "errors": 0, "said": False}
_FER_KEYS = ("parked", "parked blocks pinned", "adopted", "tokens gained", "no entry", "entry not longer", "prompt differs",
             "hashed block lost", "released: ttl", "released: pressure", "released: replaced", "skipped: not stopped", "skipped: short")


def _fer_err(where, e):
    _FN_ER["errors"] += 1
    if _FN_ER["errors"] <= 20:
        print(f"FLASHNEXT EXACT RESUME {where} error: {e!r}"[:400], flush=True)


def _fer_report():
    now = _fer_time.time()
    if now - _FN_ER["t"] < 60:
        return
    _FN_ER["t"] = now
    n, tot = _FN_ER["n"], _FN_ER["tot"]
    tot.update(n)
    print("FLASHNEXT EXACT RESUME (last 60 s | total): " + " | ".join(f"{k} {n[k]} / {tot[k]}" for k in _FER_KEYS)
          + f" || entries {len(_FN_ER['entries'])} pending {len(_FN_ER['pending'])} errors {_FN_ER['errors']}", flush=True)
    n.clear()


def _fer_key(ids):
    from vllm.v1.core.single_type_kv_cache_manager import fn_trim_sink_of
    sink = fn_trim_sink_of(ids)
    return hash(tuple(ids[:sink + 200])) if len(ids) >= sink + 200 else None


def _fer_release(mgr, e, why):
    pins = [b for b in e["pins"] if not any(b is e["ring"][g] for g in e["ring_taken"])]
    e["pins"] = []
    if pins:
        mgr.block_pool.free_blocks(pins)
    if why:
        _FN_ER["n"]["released: " + why] += 1


def _fer_drop(mgr, key, why):
    e = _FN_ER["entries"].pop(key, None)
    if e is not None:
        for rid in [r for r, x in _FN_ER["pending"].items() if x is e]:
            _FN_ER["pending"].pop(rid, None)
        _fer_release(mgr, e, why)


def _fer_housekeeping(mgr):
    ents = _FN_ER["entries"]
    now = _fer_time.time()
    while ents:
        k, e = next(iter(ents.items()))
        if now - e["t"] > _FN_ER["ttl"] or len(ents) > _FN_ER["cap"]:
            _fer_drop(mgr, k, "ttl")
        else:
            break
    while ents and mgr.block_pool.get_num_free_blocks() < _FN_ER["min_free"]:
        _fer_drop(mgr, next(iter(ents)), "pressure")


def _fn_er_park(mgr, request):
    try:
        from vllm.v1.request import RequestStatus
        from vllm.v1.core.single_type_kv_cache_manager import CircularBufferManager, MambaManager
        from vllm.utils.math_utils import cdiv
        if not _FN_ER["said"]:
            _FN_ER["said"] = True
            print(f"flashnext exact resume: ON (ttl {_FN_ER['ttl']:.0f} s, min free blocks {_FN_ER['min_free']})", flush=True)
        if request.status != RequestStatus.FINISHED_STOPPED and not (_FN_ER["any_finish"] and request.status == RequestStatus.FINISHED_LENGTH_CAPPED):
            _FN_ER["n"]["skipped: not stopped"] += 1
            return
        L = int(request.num_computed_tokens)
        ids = list(request.all_token_ids)
        if L < 1024 or L > len(ids):
            _FN_ER["n"]["skipped: short"] += 1
            return
        T = ids[:L]
        key = _fer_key(T)
        if key is None:
            _FN_ER["n"]["skipped: short"] += 1
            return
        coord = mgr.coordinator
        groups, pins, hashed, ring = [], [], [], {}
        for m in coord.single_type_managers:
            blocks = m.req_to_blocks.get(request.request_id)
            if not blocks:
                return
            if isinstance(m, CircularBufferManager):
                ring[m.kv_cache_group_id] = blocks[0]
                pins.append(blocks[0])
                groups.append(("ring",))
            elif isinstance(m, MambaManager):
                idx = (L - 1) // m.block_size
                if idx >= len(blocks) or blocks[idx].is_null:
                    return
                pins.append(blocks[idx])
                groups.append(("state", idx, blocks[idx]))
            else:
                n = cdiv(L, m.block_size)
                if n > len(blocks):
                    return
                sel = list(blocks[:n])
                for j, b in enumerate(sel):
                    if b.is_null:
                        continue
                    if b.block_hash is not None and j < n - 1:
                        hashed.append((b, b.block_hash))
                    else:
                        pins.append(b)
                groups.append(("blocks", sel))
        for b in pins:
            b.ref_cnt += 1                       # the request still holds one: the block stays out of the free queue
        if key in _FN_ER["entries"]:
            _fer_drop(mgr, key, "replaced")
        _FN_ER["entries"][key] = {"L": L, "T": T, "groups": groups, "pins": pins, "hashed": hashed, "ring": ring, "ring_taken": set(),
                                  "t": _fer_time.time()}
        _FN_ER["n"]["parked"] += 1
        _FN_ER["n"]["parked blocks pinned"] += len(pins)
        _fer_housekeeping(mgr)
    except Exception as e:
        _fer_err("park", e)
    try:
        _fer_report()
    except Exception:
        pass


def _fn_er_adopt(mgr, request, max_len, computed_blocks, hit, unc):
    try:
        _FN_ER["pending"].pop(request.request_id, None)
        if not _FN_ER["entries"]:
            return computed_blocks, hit, unc
        _fer_housekeeping(mgr)
        ids = list(request.prompt_token_ids or [])
        key = _fer_key(ids)
        e = _FN_ER["entries"].get(key) if key is not None else None
        if e is None:
            _FN_ER["n"]["no entry"] += 1
            return computed_blocks, hit, unc
        L = e["L"]
        if L <= hit or L > max_len:
            _FN_ER["n"]["entry not longer"] += 1
            return computed_blocks, hit, unc
        if ids[:L] != e["T"]:
            _FN_ER["n"]["prompt differs"] += 1
            return computed_blocks, hit, unc
        if any(b.block_hash != h for b, h in e["hashed"]):
            _FN_ER["n"]["hashed block lost"] += 1
            _fer_drop(mgr, key, "")
            return computed_blocks, hit, unc
        null = mgr.block_pool.null_block
        out = []
        for g in e["groups"]:
            if g[0] == "ring":
                out.append([])
            elif g[0] == "state":
                out.append([null] * g[1] + [g[2]])
            else:
                out.append(list(g[1]))
        _FN_ER["pending"][request.request_id] = e
        _FN_ER["n"]["adopted"] += 1
        _FN_ER["n"]["tokens gained"] += L - hit
        return tuple(out), L, 0
    except Exception as ex:
        _fer_err("adopt", ex)
        _FN_ER["pending"].pop(request.request_id, None)
        return computed_blocks, hit, unc


def _fn_er_done(mgr, request):
    try:
        e = _FN_ER["pending"].pop(request.request_id, None)
        if e is None:
            return
        for k in [k for k, x in _FN_ER["entries"].items() if x is e]:
            _FN_ER["entries"].pop(k, None)
        _fer_release(mgr, e, "")
    except Exception as ex:
        _fer_err("done", ex)
'''

def _fn_er_apply(path, edits, tail):
    s = path.read_text()
    if _MARK in s:
        return "already"
    for label, old, new in edits:
        assert s.count(old) == 1, f"exact resume edit '{label}': anchor found {s.count(old)} times in {path} (vLLM or an earlier cell changed?)"
        s = s.replace(old, new, 1)
    s += tail
    compile(s, str(path), "exec")
    path.write_text(s)
    return "installed"

print("exact resume:", _fn_er_apply(_km, _KM_EDITS, _FN), "in", _km, "|", _fn_er_apply(_sm, _SM_EDITS, ""), "in", _sm,
      "(FLASHNEXT_EXACT_RESUME=1; log lines 'FLASHNEXT EXACT RESUME')")
import os as _os_er
_os_er.environ["FLASHNEXT_EXACT_RESUME"] = "1"

In [ ]:
import os
from pathlib import Path

_root = Path(globals().get("VROOT") or "/tmp/site-vllm/vllm")
_OP = _root / "v1/engine/output_processor.py"
_AL = _root / "v1/engine/async_llm.py"
for _p in (_OP, _AL):
    assert _p.exists(), f"{_p} not found: run the earlier steps first"
_MARK = "# flashnext token splice (session 18b)"

_OP_EDITS = [
    ("record a finished request's tokens",
     """                else:
                    self._finish_request(req_state)
                    if not engine_core_output.finished:
""",
     """                else:
                    if finish_reason == FinishReason.STOP:   """ + _MARK + """
                        _fn_tsp_record(req_state)
                    self._finish_request(req_state)
                    if not engine_core_output.finished:
"""),
]
_AL_EDITS = [
    ("splice the new prompt",
     """        self.input_processor.assign_request_id(request)

        # We start the output_handler on the first call to add_request() so
""",
     """        self.input_processor.assign_request_id(request)
        from vllm.v1.engine.output_processor import _fn_tsp_apply   """ + _MARK + """

        _fn_tsp_apply(self, request)

        # We start the output_handler on the first call to add_request() so
"""),
]
_OP_TAIL = r'''


''' + _MARK + r'''
import array as _tsp_array
import bisect as _tsp_bisect
import collections as _tsp_col
import itertools as _tsp_it
import os as _tsp_os
import time as _tsp_time

_FN_TSP = {"on": _tsp_os.environ.get("FLASHNEXT_TOKEN_SPLICE") == "1", "seqs": _tsp_col.OrderedDict(), "tb": None, "tok": None,
           "n": _tsp_col.Counter(), "tot": _tsp_col.Counter(), "t": _tsp_time.time(), "errors": 0, "said": False}
_TSP_KEYS = ("recorded", "spliced", "tokens re-used", "already exact", "no key", "no candidate", "bytes differ", "straddle re-encoded",
             "straddle bad utf8", "image moved", "image check failed", "max_tokens lowered")


def _tsp_err(where, e):
    _FN_TSP["errors"] += 1
    if _FN_TSP["errors"] <= 20:
        print(f"FLASHNEXT TOKEN SPLICE {where} error: {e!r}"[:400], flush=True)


def _tsp_report():
    now = _tsp_time.time()
    if now - _FN_TSP["t"] < 60:
        return
    _FN_TSP["t"] = now
    n, tot = _FN_TSP["n"], _FN_TSP["tot"]
    tot.update(n)
    print("FLASHNEXT TOKEN SPLICE (last 60 s | total): " + " | ".join(f"{k} {n[k]} / {tot[k]}" for k in _TSP_KEYS)
          + f" || keys {len(_FN_TSP['seqs'])} errors {_FN_TSP['errors']}", flush=True)
    n.clear()


def _tsp_tables(tokenizer_name):
    """Byte string of every token id (byte-level BPE) + the tokenizer for re-encoding a remainder."""
    if _FN_TSP["tb"] is not None:
        return _FN_TSP["tb"]
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(tokenizer_name)
    bs = list(range(ord("!"), ord("~") + 1)) + list(range(ord("¡"), ord("¬") + 1)) + list(range(ord("®"), ord("ÿ") + 1))
    cs, n_ = bs[:], 0
    for b in range(256):                           # GPT-2 byte-level alphabet (bytes_to_unicode)
        if b not in bs:
            bs.append(b)
            cs.append(256 + n_)
            n_ += 1
    dec = {chr(c): b for b, c in zip(bs, cs)}
    added = {i: t.content for i, t in tok.added_tokens_decoder.items()}
    n = max(len(tok), max(added) + 1 if added else 0)
    tb = []
    for i in range(n):
        if i in added:
            tb.append(added[i].encode("utf-8"))
            continue
        s = tok.convert_ids_to_tokens(i)
        tb.append(bytes(dec[c] for c in s) if s is not None and all(c in dec for c in s) else None)
    _FN_TSP["tb"], _FN_TSP["tok"] = tb, tok
    return tb


def _tsp_key(ids):
    """hash of the first prompt = everything before the first "<|im_start|>assistant" (the trim cell's sink rule; its token ids are only
    set up in the engine process, so they are looked up here from the tokenizer); None without such a turn."""
    tok = _FN_TSP["tok"]
    if "im_start" not in _FN_TSP:
        _FN_TSP["im_start"] = tok.convert_tokens_to_ids("<|im_start|>")
        _FN_TSP["assistant"] = tok.encode("assistant", add_special_tokens=False)
    st, mk = _FN_TSP["im_start"], _FN_TSP["assistant"]
    ids = list(ids)
    i = -1
    while True:
        try:
            i = ids.index(st, i + 1)
        except ValueError:
            return None
        if ids[i + 1:i + 1 + len(mk)] == mk:
            return hash(tuple(ids[:i])) if i > 0 else None


def _fn_tsp_record(req_state):
    if not _FN_TSP["on"]:
        return
    try:
        p = req_state.prompt_token_ids
        o = req_state.detokenizer.output_token_ids if req_state.detokenizer is not None else None
        if not p or not o:
            return
        if _FN_TSP["tok"] is None:
            return                                 # tables not loaded yet: the first request has not been seen (cannot happen after it)
        S = _tsp_array.array("i", p)
        S.extend(o)
        key = _tsp_key(p)
        if key is None:
            return
        lst = _FN_TSP["seqs"].pop(key, [])
        lst = [x for x in lst if not (len(x) <= len(S) and x[:len(x)] == S[:len(x)])][-3:] + [S]
        _FN_TSP["seqs"][key] = lst
        while len(_FN_TSP["seqs"]) > 600:
            _FN_TSP["seqs"].popitem(last=False)
        _FN_TSP["n"]["recorded"] += 1
    except Exception as e:
        _tsp_err("record", e)


def _tsp_try(S, P, tb):
    """New prompt ids (list) for P built on S[:L], with the moved image offsets, or None + reason."""
    L = len(S) - 1
    m = min(L, len(P))
    d = 0
    while d < m and S[d] == P[d]:          # first difference (C-level slices would be faster; prompts are <= ~100K)
        d += 1
    if d >= L:
        return "already exact", None
    bS = b"".join(tb[t] for t in S[d:L])
    lens = [len(tb[t]) for t in P[d:]]
    cum = list(_tsp_it.accumulate(lens))
    k = _tsp_bisect.bisect_left(cum, len(bS))     # P[d:d+k+1] covers len(bS) bytes
    if k >= len(cum):
        return "bytes differ", None
    bP = b"".join(tb[t] for t in P[d:d + k + 1])
    if bP[:len(bS)] != bS:
        return "bytes differ", None
    rest_ids = []
    if cum[k] > len(bS):                           # P token d+k straddles the end of S[:L]: re-encode its remaining bytes
        rem = bP[len(bS):]
        try:
            rem_s = rem.decode("utf-8")
        except UnicodeDecodeError:
            return "straddle bad utf8", None
        rest_ids = _FN_TSP["tok"].encode(rem_s, add_special_tokens=False)
        if b"".join(tb[t] for t in rest_ids) != rem:
            return "straddle bad utf8", None
    j = d + k + 1                                  # first P index after the spliced span
    new = list(S[:L]) + list(rest_ids) + list(P[j:])
    return ("straddle" if rest_ids else "ok"), (new, d, j, L + len(rest_ids))


def _fn_tsp_apply(engine, request):
    if not _FN_TSP["on"]:
        return
    try:
        if not _FN_TSP["said"]:
            _FN_TSP["said"] = True
            print("flashnext token splice: ON (follow-up prompts take the model's own token ids where the text is the same)", flush=True)
        P = request.prompt_token_ids
        if not P or request.prompt_embeds is not None:
            return
        tb = _tsp_tables(engine.model_config.tokenizer)
        key = _tsp_key(P)
        if key is None:
            _FN_TSP["n"]["no key"] += 1
            return
        cands = _FN_TSP["seqs"].get(key)
        if not cands:
            _FN_TSP["n"]["no candidate"] += 1
            return
        best, why = None, "bytes differ"
        for S in reversed(cands):
            r, res = _tsp_try(S, P, tb)
            if r == "already exact":
                _FN_TSP["n"]["already exact"] += 1
                return
            if res is not None and (best is None or res[3] > best[0][3]):
                best = (res, r)
            elif res is None:
                why = r
        if best is None:
            _FN_TSP["n"][why] += 1
            return
        (new, d, j, newj), how = best
        delta = newj - j
        feats = request.mm_features or []
        if feats:
            import dataclasses as _dc
            moved = []
            for f in feats:
                o = f.mm_position.offset
                if o < d:
                    moved.append(f.mm_position)
                elif o >= j:
                    moved.append(_dc.replace(f.mm_position, offset=o + delta))
                else:                                          # inside the spliced span: same byte position in S
                    bpos = sum(len(tb[t]) for t in P[d:o])
                    acc, q = 0, d
                    while acc < bpos:
                        acc += len(tb[new[q]])
                        q += 1
                    ln = f.mm_position.length
                    if acc != bpos or new[q:q + ln] != P[o:o + ln]:
                        _FN_TSP["n"]["image check failed"] += 1
                        return
                    moved.append(_dc.replace(f.mm_position, offset=q))
                    _FN_TSP["n"]["image moved"] += 1
            for f, pos in zip(feats, moved):
                f.mm_position = pos
        request.prompt_token_ids = new
        sp = request.sampling_params
        mml = engine.model_config.max_model_len
        if sp is not None and sp.max_tokens is not None and len(new) + sp.max_tokens > mml:
            sp.max_tokens = max(1, mml - len(new))
            _FN_TSP["n"]["max_tokens lowered"] += 1
        _FN_TSP["n"]["spliced"] += 1
        _FN_TSP["n"]["tokens re-used"] += newj - d
        if how == "straddle":
            _FN_TSP["n"]["straddle re-encoded"] += 1
    except Exception as e:
        _tsp_err("apply", e)
    finally:
        try:
            _tsp_report()
        except Exception:
            pass
'''

_done = []
for _f, _edits, _tail in ((_OP, _OP_EDITS, _OP_TAIL), (_AL, _AL_EDITS, "")):
    _s = _f.read_text()
    if _MARK in _s:
        _done.append(f"{_f.name} [already]")
        continue
    for _label, _old, _new in _edits:
        assert _s.count(_old) == 1, f"token splice edit '{_label}': anchor found {_s.count(_old)} times in {_f} (vLLM changed?)"
        _s = _s.replace(_old, _new)
    _s += _tail
    compile(_s, str(_f), "exec")
    _f.write_text(_s)
    _done.append(f"{_f.name} [installed]")
os.environ["FLASHNEXT_TOKEN_SPLICE"] = "1"
print("token splice:", " | ".join(_done), "| FLASHNEXT_TOKEN_SPLICE =", os.environ["FLASHNEXT_TOKEN_SPLICE"])

# vLLM Patches: Decode Kernels

In [ ]:
from pathlib import Path

_root = Path(globals().get("VROOT") or "/tmp/site-vllm/vllm")
_fr = _root / "v1/worker/gpu/model_runner.py"
assert _fr.exists(), f"{_fr} not found: run the earlier steps first"
_MARK = "# flashnext top-k exchange (session 22)"
_HELPERS = '''
_FN_TOPK = __import__("os").environ.get("FLASHNEXT_TOPK_EXCHANGE") == "1"   ''' + _MARK + '''
_FN_TOPK_K = int(__import__("os").environ.get("FLASHNEXT_TOPK_EXCHANGE_K", "64"))
_fn_topk_stats = {"fast": 0, "full": 0, "why": {}, "t": time.time()}


def _fn_topk_reason(runner, batch, grammar_output):
    """None if this step may use the candidate exchange, else the reason (counted)"""
    s = runner.sampler
    idx = batch.idx_mapping_np
    if s is None:
        return "no sampler"
    if grammar_output is not None:
        return "grammar"
    if batch.num_draft_tokens:
        return "speculative tokens"
    if int(batch.cu_num_logits_np[-1]) != idx.shape[0]:
        return "logit rows != requests"
    if s.get_logprobs_dims(idx) is not None:
        return "logprobs"
    if s.return_sampling_mask or s.trace_replay_state is not None or s.compute_nans:
        return "sampling mask / trace / nans"
    st = s.sampling_states
    tk, temp = st.top_k.np[idx], st.temperature.np[idx]
    if not np.all((temp == 0.0) | ((tk >= 1) & (tk <= _FN_TOPK_K))):
        return "top_k outside 1..K"
    for p in s.logits_processors:
        n = type(p).__name__
        if n == "LogitBiasState":
            if np.any(p.use_logit_bias[idx]):
                return "logit bias"
        elif n == "PenaltiesState":
            if np.any(p.use_penalty[idx]):
                return "penalties"
        elif n == "BadWordsState":
            if int(p.num_bad_words.np[idx].max()) != 0:
                return "bad words"
        else:
            return "custom logits processor " + n
    tb = s.thinking_budget_state
    if tb.enabled and np.any(tb.use_thinking_budget[idx]):
        return "thinking budget"
    return None


def _fn_topk_count(runner, why):
    st = _fn_topk_stats
    if why is None:
        st["fast"] += 1
    else:
        st["full"] += 1
        st["why"][why] = st["why"].get(why, 0) + 1
    if time.time() - st["t"] > 60 and runner.batch_sharder is not None and runner.batch_sharder.tp_rank == 0:
        st["t"] = time.time()
        print(f"flashnext top-k exchange: fast {st['fast']} | full {st['full']} {st['why']}", flush=True)


def _fn_topk_all_to_all(logits_shard, metadata, vocab_size, tp_rank):
    """per request, this rank's K best (value, token id) -> the owner; returns compact logits [n, tp*K] and token ids [n, tp*K]"""
    from vllm.distributed.parallel_state import get_tp_group
    k = _FN_TOPK_K
    shard_w = logits_shard.shape[-1]
    vals, ids = torch.topk(logits_shard.float(), k, dim=-1)
    packed = torch.stack([vals, (ids + tp_rank * shard_w).to(torch.float32)], dim=-1).reshape(logits_shard.shape[0], 2 * k)
    recv = packed.new_empty(metadata.tp_size * metadata.num_local_logits, 2 * k)
    torch.distributed.all_to_all_single(
        recv,
        packed.contiguous(),
        output_split_sizes=[metadata.num_local_logits] * metadata.tp_size,
        input_split_sizes=metadata.num_logits_per_rank,
        group=get_tp_group().device_group,
    )
    recv = recv.view(metadata.tp_size, metadata.num_local_logits, k, 2).permute(1, 0, 2, 3).reshape(metadata.num_local_logits, metadata.tp_size * k, 2)
    tok = recv[..., 1].round().to(torch.int64)
    logits = recv[..., 0].masked_fill(tok >= vocab_size, float("-inf")).contiguous()
    return logits, tok


class GPUModelRunner(LoRAModelRunnerMixin):
'''
_EDITS = [
    ("helpers", "\nclass GPUModelRunner(LoRAModelRunnerMixin):\n", _HELPERS),
    ("candidate ids variable",
     """        shard_metadata = None
        global_input_batch = input_batch
""",
     """        shard_metadata = None
        global_input_batch = input_batch
        _fn_ids = None   """ + _MARK + """
"""),
    ("exchange",
     """            sample_hidden_states = hidden_states[sorted_logits_indices]
            local_logits = self.model.compute_logits_local(sample_hidden_states)
            logits = all_to_all_logits(local_logits, shard_metadata)
            logits = logits[:, : self.vocab_size]
""",
     """            sample_hidden_states = hidden_states[sorted_logits_indices]
            local_logits = self.model.compute_logits_local(sample_hidden_states)
            _fn_why = _fn_topk_reason(self, global_input_batch, grammar_output) if _FN_TOPK else "off"   """ + _MARK + """
            if _FN_TOPK:
                _fn_topk_count(self, _fn_why)
            if _fn_why is None:
                logits, _fn_ids = _fn_topk_all_to_all(local_logits, shard_metadata, self.vocab_size, self.batch_sharder.tp_rank)
            else:
                logits = all_to_all_logits(local_logits, shard_metadata)
                logits = logits[:, : self.vocab_size]
"""),
    ("map drawn positions to token ids",
     """            assert self.sampler is not None
            sampler_output = self.sampler(logits, input_batch)
""",
     """            assert self.sampler is not None
            sampler_output = self.sampler(logits, input_batch)
            if _fn_ids is not None:   """ + _MARK + """
                sampler_output.sampled_token_ids = torch.gather(_fn_ids, 1, sampler_output.sampled_token_ids.view(-1, 1).to(torch.int64))
"""),
]
_s = _fr.read_text()
for _label, _old, _new in _EDITS:
    if _new in _s:
        print("top-k exchange: [already]", _label)
        continue
    assert _s.count(_old) == 1, f"top-k exchange edit '{_label}': anchor found {_s.count(_old)} times in {_fr} (patch cell changed?)"
    _s = _s.replace(_old, _new)
    print("top-k exchange: [installed]", _label)
compile(_s, str(_fr), "exec")
_fr.write_text(_s)
print("(FLASHNEXT_TOPK_EXCHANGE=1 before the server starts, with FLASHNEXT_BATCH_SHARDED_SAMPLING=1; unset = unchanged)")

In [ ]:
from pathlib import Path

_MOD_SRC = '# flashnext HC-down split-K (session 22) - installed as vllm/models/qwen4_exp/nvidia/ops/fn_hc_down.py by s05m_hc_down.py.\n# The merged "down + inject" projection of every hyper-connection mixer (FP8 weight 336 x 10240, bf16 activations, M = tokens) runs in\n# Humming as a stream-K GEMM over only 6 output tiles: 27 of 58 SMs, ~33 us at M 60 on an L4 (~120 GB/s; ~30 us even with the weight in\n# L2: the long K loop per tile is the limit). Here the K dimension is split over many thread blocks (each does one output tile over one K\n# slice), the partial sums go to a workspace, a second small kernel adds them in a fixed order (deterministic) and writes bf16.\n# Weights: the checkpoint\'s FP8 bytes [N, K] and per-row or per-(row, K-block) scales, kept by s05m before Humming transforms the layer\n# (Humming\'s copy is then freed). Same math as Humming\'s W8A16 path (FP8 -> bf16 weights, fp32 accumulation), other summation order.\nimport torch\n\nfrom vllm.triton_utils import tl, triton\n\n\n@triton.jit(do_not_specialize=["M"])   # K_PER_SPLIT stays specialized: its alignment lets Triton vectorize the loads\ndef _fn_hcd_partial(x, w, s, part, M, N, K, stride_xm, NG, G_SIZE, K_PER_SPLIT,\n                    BM: tl.constexpr, BN: tl.constexpr, BK: tl.constexpr, ROW_SCALE: tl.constexpr):\n    pid_n, pid_k, pid_m = tl.program_id(0), tl.program_id(1), tl.program_id(2)\n    offs_m = pid_m * BM + tl.arange(0, BM)\n    offs_n = pid_n * BN + tl.arange(0, BN)\n    mm, nm = offs_m < M, offs_n < N\n    acc = tl.zeros([BM, BN], dtype=tl.float32)\n    k0 = pid_k * K_PER_SPLIT\n    for kk in range(0, K_PER_SPLIT, BK):\n        offs_k = k0 + kk + tl.arange(0, BK)\n        a = tl.load(x + offs_m[:, None] * stride_xm + offs_k[None, :], mask=mm[:, None], other=0.0)\n        b = tl.load(w + offs_n[:, None] * K + offs_k[None, :], mask=nm[:, None], other=0.0)\n        if ROW_SCALE:                                    # one scale per output row: applied once after the K loop (bf16 weights: scale 1)\n            acc += tl.dot(a, tl.trans(b.to(tl.bfloat16)))\n        else:                                            # per (row, K-block) scales\n            sc = tl.load(s + offs_n * NG + (k0 + kk) // G_SIZE, mask=nm, other=0.0)\n            acc += tl.dot(a, tl.trans(b.to(tl.bfloat16))) * sc[None, :]\n    if ROW_SCALE:\n        acc = acc * tl.load(s + offs_n, mask=nm, other=0.0)[None, :]\n    tl.store(part + pid_k * M * N + offs_m[:, None] * N + offs_n[None, :], acc, mask=mm[:, None] & nm[None, :])\n\n\n@triton.jit(do_not_specialize=["MN"])\ndef _fn_hcd_reduce(part, out, MN, SPLITS: tl.constexpr, BLOCK: tl.constexpr):\n    offs = tl.program_id(0) * BLOCK + tl.arange(0, BLOCK)\n    m = offs < MN\n    acc = tl.zeros([BLOCK], dtype=tl.float32)\n    for s in range(SPLITS):\n        acc += tl.load(part + s * MN + offs, mask=m, other=0.0)\n    tl.store(out + offs, acc.to(out.dtype.element_ty), mask=m)\n\n\n_ws = {}\n\n\ndef _workspace(numel, device):\n    t = _ws.get(device)\n    if t is None or t.numel() < numel:\n        t = torch.empty(max(numel, 1 << 20), dtype=torch.float32, device=device)\n        _ws[device] = t\n    return t\n\n\ndef plan(M):\n    """(BM, BN, BK, SPLITS, warps), searched on one L4 at N 336, K 10240 with the input in L2 (session 22, tools/l4_hc_down_vs_humming.py,\n    tools/hcd_plan.py). vs Humming: M 60 23.5 vs 28.7 us (weight in L2 16.4 vs 27.6), M 128 25.6 vs 44.0, M 512 ~67 vs 69.6,\n    M 1024 ~125 vs 112.6 (the only case slower: prefill chunks)"""\n    if M <= 64:\n        return (16 if M <= 16 else (32 if M <= 32 else 64)), 32, 128, 5, 8\n    if M <= 128:\n        return 64, 32, 128, 5, 8\n    if M <= 256:\n        return 64, 128, 128, 4, 4\n    return 64, 128, 64, 2, 4\n\n\ndef hc_down(x, w8, scale, out=None, plan_fn=None):\n    """x [M, K] bf16 (contiguous rows), w8 [N, K] float8_e4m3fn (or bf16 with scale ones), scale [N, NG] fp32 -> out [M, N] (x\'s dtype)"""\n    M, K = x.shape\n    N = w8.shape[0]\n    assert w8.shape[1] == K and x.stride(1) == 1\n    NG = scale.shape[1]\n    G = K // NG\n    if out is None:\n        out = torch.empty((M, N), dtype=x.dtype, device=x.device)\n    if M == 0:\n        return out\n    BM, BN, bk, splits, warps = (plan_fn or plan)(M)\n    assert K % (bk * splits) == 0 and (G >= K or G % bk == 0), (K, bk, splits, G)\n    part = _workspace(splits * M * N, x.device)\n    grid = (triton.cdiv(N, BN), splits, triton.cdiv(M, BM))\n    _fn_hcd_partial[grid](x, w8, scale, part, M, N, K, x.stride(0), NG, G, K // splits, BM=BM, BN=BN, BK=bk,\n                          ROW_SCALE=NG == 1, num_warps=warps, num_stages=3)\n    _fn_hcd_reduce[(triton.cdiv(M * N, 1024),)](part, out, M * N, SPLITS=splits, BLOCK=1024, num_warps=4)\n    return out\n'
_root = Path(globals().get("VROOT") or "/tmp/site-vllm/vllm")
_fl = _root / "model_executor/layers/quantization/utils/humming/linear.py"
_fo = _root / "models/qwen4_exp/nvidia/ops/fn_hc_down.py"
assert _fl.exists() and _fo.parent.is_dir(), f"{_fl} not found: run the earlier steps first"
assert _MOD_SRC and "def hc_down(" in _MOD_SRC, "the module source is missing (build the notebook with its builder)"
_MARK = "# flashnext HC-down split-K (session 22)"
_HELPERS = '''def _fn_hcd_target(layer) -> bool:   ''' + _MARK + '''
    import os

    return os.environ.get("FLASHNEXT_HC_DOWN") == "1" and str(getattr(layer, "prefix", "")).endswith("input_mix_weight_down_block_inject")


def _fn_hcd_keep(layer, weight_schema, tensors):
    """keep the FP8 weight [N, K] + fp32 scales [N, NG] of an HC down + inject layer before Humming re-lays them out"""
    if not _fn_hcd_target(layer):
        return
    extra = sorted(set(tensors) - {"weight", "weight_scale"})
    if extra or getattr(layer, "has_bias", False):
        raise RuntimeError(f"FLASHNEXT_HC_DOWN=1: {layer.prefix} has {extra or 'a bias'}; the split-K kernel supports weight + weight_scale only")
    w = tensors["weight"]
    if "float8e4m3" not in str(getattr(weight_schema, "b_dtype", "")).lower() and "e4m3" not in str(getattr(weight_schema, "b_dtype", "")).lower():
        raise RuntimeError(f"FLASHNEXT_HC_DOWN=1: {layer.prefix} weight type {getattr(weight_schema, 'b_dtype', None)} is not FP8 e4m3")
    w8 = w.view(torch.float8_e4m3fn)
    n, k = w8.shape
    sc = tensors["weight_scale"].float()
    if sc.numel() == 1:
        s32 = sc.reshape(1, 1).expand(n, 1)
    elif sc.numel() == n:
        s32 = sc.reshape(n, 1)
    elif sc.dim() == 2 and sc.shape[0] == n and k % sc.shape[1] == 0 and (k // sc.shape[1]) % 64 == 0:
        s32 = sc
    elif (sc.dim() == 2 and k % sc.shape[1] == 0 and (k // sc.shape[1]) % 64 == 0
          and (gn := int(getattr(weight_schema, "weight_scale_group_size_n", 0) or 0)) > 0 and -(-n // gn) == sc.shape[0]):
        s32 = sc.repeat_interleave(gn, 0)[:n]                       # 2-D block scales: one row of blocks per gn output rows
    else:
        raise RuntimeError(f"FLASHNEXT_HC_DOWN=1: {layer.prefix} weight_scale shape {tuple(sc.shape)} for weight {tuple(w8.shape)} not understood")
    w8, s32 = w8.contiguous().clone(), s32.contiguous().clone()
    # the layer's output partitions are [lora rank, hc count, pad]: the pad rows get no checkpoint weight (uninitialized memory, can be
    # NaN) and the model drops those output columns; zero them here so the kernel writes zeros there, and check only the real rows
    parts = list(getattr(layer, "output_partition_sizes", [n]))
    real = sum(parts[:2]) if len(parts) == 3 else n
    if not 0 < real <= n:
        raise RuntimeError(f"FLASHNEXT_HC_DOWN=1: {layer.prefix} output partitions {parts} do not fit the weight rows {n}")
    if real < n:
        w8.view(torch.uint8)[real:] = 0                             # byte 0 = +0.0 in e4m3fn
        s32[real:] = 0
    layer._fn_hcd_pending = (w8, s32, real)


def _fn_hcd_check_and_switch(layer, config, tensors):
    """compare the split-K kernel with Humming's GEMM on one random input, then free Humming's copy and switch the layer over"""
    pending = layer.__dict__.pop("_fn_hcd_pending", None)
    if pending is None:
        return
    from vllm.models.qwen4_exp.nvidia.ops.fn_hc_down import hc_down

    w8, s32, real = pending
    g = torch.Generator(device=w8.device).manual_seed(1234)
    x = (torch.randn(16, w8.shape[1], device=w8.device, generator=g) * 0.5).to(layer.params_dtype)
    ref = apply_humming_linear(layer, x, layer_config=config, compute_config=get_humming_linear_compute_config(),
                               locks=torch.zeros(1024, dtype=torch.int32, device=w8.device)).float()
    got = hc_down(x, w8, s32).float()
    got, ref = got[:, :real], ref[:, :real]                          # the pad columns are dropped by the model
    rel = ((got - ref).abs().max() / ref.abs().max().clamp_min(1e-6)).item()
    if not rel < 2e-2:
        raise RuntimeError(f"FLASHNEXT_HC_DOWN=1: {layer.prefix} split-K vs Humming relative max difference {rel:.3e} (limit 2e-2) on the "
                           f"{real} real rows of {w8.shape[0]} | non-finite: split-K {int((~got.isfinite()).sum())}, Humming "
                           f"{int((~ref.isfinite()).sum())}, kept weight {int((~w8[:real].float().isfinite()).sum())}, scales "
                           f"{int((~s32[:real].isfinite()).sum())}")
    for name in tensors:
        old = getattr(layer, name)
        setattr(layer, name, torch.nn.Parameter(torch.empty(0, dtype=old.dtype, device=old.device), requires_grad=False))
    layer._fn_hcd = (w8, s32)
    _fn_hcd_count[0] += 1
    if _fn_hcd_count[0] == 1:
        print(f"flashnext HC-down split-K: ON | first layer {layer.prefix}: weight {tuple(w8.shape)}, scales {tuple(s32.shape)}, "
              f"real rows {real} (pad rows zeroed), self-check rel diff {rel:.2e}", flush=True)


_fn_hcd_count = [0]


def _flashnext_small_pad(layer) -> bool:'''
_EDITS = [
    ("helpers", "def _flashnext_small_pad(layer) -> bool:", _HELPERS),
    ("keep the FP8 weight before the transform",
     """    tensors.update(input_tensors)
    layer.weight_schema = weight_schema
""",
     """    tensors.update(input_tensors)
    layer.weight_schema = weight_schema
    _fn_hcd_keep(layer, weight_schema, tensors)   """ + _MARK + """
"""),
    ("self-check, free Humming's copy, switch",
     """    for name, tensor in tensors.items():
        param = torch.nn.Parameter(tensor, requires_grad=False)
        setattr(layer, name, param)

    return config
""",
     """    for name, tensor in tensors.items():
        param = torch.nn.Parameter(tensor, requires_grad=False)
        setattr(layer, name, param)
    _fn_hcd_check_and_switch(layer, config, tensors)   """ + _MARK + """

    return config
"""),
    ("apply_humming_linear routes to the split-K kernel",
     """    from vllm.utils.humming import humming_forward

    mode = layer_config.input_quant_mode
""",
     """    _fn = layer.__dict__.get("_fn_hcd")
    if _fn is not None:   """ + _MARK + """
        from vllm.models.qwen4_exp.nvidia.ops.fn_hc_down import hc_down as _fn_hc_down

        _flat = x.reshape(-1, x.size(-1))
        _out = _fn_hc_down(_flat if _flat.stride(-1) == 1 else _flat.contiguous(), _fn[0], _fn[1])
        return _out.view(*x.shape[:-1], _out.size(-1))
    from vllm.utils.humming import humming_forward

    mode = layer_config.input_quant_mode
"""),
]
if not _fo.exists() or _fo.read_text() != _MOD_SRC:
    _fo.write_text(_MOD_SRC)
    compile(_MOD_SRC, str(_fo), "exec")
    print("HC-down split-K: [installed]", _fo)
_s = _fl.read_text()
for _label, _old, _new in _EDITS:
    if _new in _s:
        print("HC-down split-K: [already]", _label)
        continue
    assert _s.count(_old) == 1, f"HC-down split-K edit '{_label}': anchor found {_s.count(_old)} times in {_fl} (patch cell changed?)"
    _s = _s.replace(_old, _new)
    print("HC-down split-K: [installed]", _label)
compile(_s, str(_fl), "exec")
_fl.write_text(_s)
print("(FLASHNEXT_HC_DOWN=1 before the server starts; unset = unchanged)")

In [ ]:
from pathlib import Path

_root = Path(globals().get("VROOT") or "/tmp/site-vllm/vllm")
_f = _root / "models/qwen4_exp/nvidia/qsa.py"
assert _f.exists(), f"{_f} not found: run the earlier steps first"
_MARK = "# flashnext DCP comm (session 22)"
_EDITS = [
    ("query all-gather started before the indexer",
     """        selected = self.indexer(
            projected_qk,
            positions,
            self.topk_indices_buffer[:num_tokens],
        )
""",
     """        _fn_impl = self.impl
        if _FN_DCP_QUERY_EARLY and getattr(_fn_impl, "dcp_world_size", 1) > 1:   """ + _MARK + """
            from vllm.distributed.parallel_state import get_dcp_group

            _q = query
            if getattr(_fn_impl, "_fn_k4", False):
                from .ops.qsa_k4 import k4_rotate

                _q = k4_rotate(query)
            _q = _q[:num_tokens].contiguous()
            _cur = torch.cuda.current_stream()
            _side = _fn_dcp_side_stream()
            _side.wait_stream(_cur)
            with torch.cuda.stream(_side):
                _qa = get_dcp_group().all_gather(_q, dim=1)
            _q.record_stream(_side)
            _fn_impl._fn_query_all = (_qa, _side)
        selected = self.indexer(
            projected_qk,
            positions,
            self.topk_indices_buffer[:num_tokens],
        )
"""),
    ("the attention uses the early all-gather",
     """        # The DCP group is the pair of TP ranks that share this KV head: gather both ranks' query heads.
        query_all = group.all_gather(query[:num_tokens].contiguous(), dim=1)
""",
     """        # The DCP group is the pair of TP ranks that share this KV head: gather both ranks' query heads.
        _fn_early = self.__dict__.pop("_fn_query_all", None)   """ + _MARK + """
        if _fn_early is not None:
            query_all, _side = _fn_early
            torch.cuda.current_stream().wait_stream(_side)
            query_all.record_stream(torch.cuda.current_stream())
        else:
            query_all = group.all_gather(query[:num_tokens].contiguous(), dim=1)
"""),
    ("BF16 partials",
     """        merged = self.dcp_combine(partial, lse, group, is_lse_base_on_e=False)
""",
     """        if _FN_DCP_BF16_PARTIAL:   """ + _MARK + """
            partial = partial.to(torch.bfloat16)
        merged = self.dcp_combine(partial, lse, group, is_lse_base_on_e=False)
"""),
    ("switches + side stream",
     """class Qwen4ExpQSAAttention(Qwen3NextAttention, AttentionLayerBase):
""",
     """_FN_DCP_QUERY_EARLY = __import__("os").environ.get("FLASHNEXT_DCP_QUERY_EARLY") == "1"   """ + _MARK + """
_FN_DCP_BF16_PARTIAL = __import__("os").environ.get("FLASHNEXT_DCP_BF16_PARTIAL") == "1"
_fn_dcp_side = []


def _fn_dcp_side_stream():
    if not _fn_dcp_side:
        _fn_dcp_side.append(torch.cuda.Stream())
    return _fn_dcp_side[0]


if _FN_DCP_QUERY_EARLY:
    print("flashnext DCP comm: query all-gather early ON", flush=True)
if _FN_DCP_BF16_PARTIAL:
    print("flashnext DCP comm: BF16 partials ON", flush=True)


class Qwen4ExpQSAAttention(Qwen3NextAttention, AttentionLayerBase):
"""),
]
_s = _f.read_text()
for _label, _old, _new in _EDITS:
    if _new in _s:
        print("DCP comm: [already]", _label)
        continue
    assert _s.count(_old) == 1, f"DCP comm edit '{_label}': anchor found {_s.count(_old)} times in {_f} (patch cell changed?)"
    _s = _s.replace(_old, _new)
    print("DCP comm: [installed]", _label)
compile(_s, str(_f), "exec")
_f.write_text(_s)
print("(FLASHNEXT_DCP_QUERY_EARLY=1 and/or FLASHNEXT_DCP_BF16_PARTIAL=1 before the server starts; unset = unchanged)")

In [ ]:
from pathlib import Path

_root = Path(globals().get("VROOT") or "/tmp/site-vllm/vllm")
_qsa = _root / "models/qwen4_exp/nvidia/qsa.py"
_idx = _root / "models/qwen4_exp/nvidia/ops/qsa_indexer.py"
_mr = _root / "v1/worker/gpu/model_runner.py"
_k4 = _root / "models/qwen4_exp/nvidia/ops/qsa_k4.py"
_gdn = _root / "model_executor/layers/mamba/gdn/qwen_gdn_linear_attn.py"
_hc = _root / "models/qwen4_exp/nvidia/hyperconnection.py"
_mu = _root / "v1/worker/mamba_utils.py"
_hcd = _root / "models/qwen4_exp/nvidia/ops/fn_hc_down.py"
_hcds = _root / "models/qwen4_exp/nvidia/ops/fn_hcd_silu.py"
_qg = _root / "models/qwen4_exp/nvidia/ops/fn_qsa_gate.py"
for _p in (_qsa, _idx, _mr, _k4, _gdn, _hc, _hcd, _mu):
    assert _p.exists(), f"{_p} not found: run the earlier steps first (fn_hc_down.py comes from the HC-down cell)"
_HCDS_SRC = '# flashnext HC-down + SiLU in one pass (session 22, roadmap C2) - installed as vllm/models/qwen4_exp/nvidia/ops/fn_hcd_silu.py by\n# s05q_decode_trims.py (FLASHNEXT_TRIM_HCD_SILU). On the L4 (sm_89) vLLM runs the hyper-connection "down + inject" projection, then a separate SiLU kernel over its\n# first lora_rank columns (vLLM\'s fused down + SiLU GEMM needs sm_90). With the HC-down split-K kernel (fn_hc_down.py) on, the second\n# (reduce) kernel can apply the SiLU itself: it writes the low-rank part silu(x / hc_count) [M, lora_rank] and the injection logits\n# [M, hc_count] as two contiguous tensors (what the next GEMM and the combine read), the pad columns are dropped. One kernel less per\n# HC mixer (96 per decode step). Bit-identical to fn_hc_down + split + vLLM\'s _hc_silu_kernel (the down output is rounded to bf16 first).\nimport torch\n\nfrom vllm.models.qwen4_exp.nvidia.ops.fn_hc_down import _fn_hcd_partial, _workspace, plan\nfrom vllm.triton_utils import tl, triton\n\n\n@triton.jit(do_not_specialize=["M"])\ndef _fn_hcd_reduce_silu(part, lora_out, inj_out, M, N, LORA: tl.constexpr, HC: tl.constexpr, SPLITS: tl.constexpr,\n                        BLOCK: tl.constexpr):\n    row = tl.program_id(0)\n    cols = tl.arange(0, BLOCK)\n    m = cols < LORA + HC\n    acc = tl.zeros([BLOCK], dtype=tl.float32)\n    for s in range(SPLITS):\n        acc += tl.load(part + s * M * N + row * N + cols, mask=m, other=0.0)\n    v = acc.to(lora_out.dtype.element_ty)               # the bf16 down output, as fn_hc_down\'s reduce writes it\n    x = v.to(tl.float32) / HC                           # then vLLM\'s _hc_silu_kernel on it: bit-identical\n    y = x * tl.sigmoid(x)\n    tl.store(lora_out + row * LORA + cols, y.to(lora_out.dtype.element_ty), mask=cols < LORA)\n    tl.store(inj_out + row * HC + (cols - LORA), v, mask=(cols >= LORA) & m)\n\n\ndef hc_down_silu(x, w8, scale, lora_rank, hc_count, plan_fn=None):\n    """x [M, K] bf16, w8 [N, K] (N = lora_rank + hc_count + pad), scale [N, NG] fp32 -> (silu(down / hc) [M, lora], injection [M, hc])"""\n    M, K = x.shape\n    N = w8.shape[0]\n    assert w8.shape[1] == K and x.stride(1) == 1 and N >= lora_rank + hc_count\n    NG = scale.shape[1]\n    G = K // NG\n    lora = torch.empty((M, lora_rank), dtype=x.dtype, device=x.device)\n    inj = torch.empty((M, hc_count), dtype=x.dtype, device=x.device)\n    if M == 0:\n        return lora, inj\n    BM, BN, bk, splits, warps = (plan_fn or plan)(M)\n    assert K % (bk * splits) == 0 and (G >= K or G % bk == 0), (K, bk, splits, G)\n    part = _workspace(splits * M * N, x.device)\n    grid = (triton.cdiv(N, BN), splits, triton.cdiv(M, BM))\n    _fn_hcd_partial[grid](x, w8, scale, part, M, N, K, x.stride(0), NG, G, K // splits, BM=BM, BN=BN, BK=bk,\n                          ROW_SCALE=NG == 1, num_warps=warps, num_stages=3)\n    _fn_hcd_reduce_silu[(M,)](part, lora, inj, M, N, LORA=lora_rank, HC=hc_count, SPLITS=splits,\n                              BLOCK=triton.next_power_of_2(lora_rank + hc_count), num_warps=4)\n    return lora, inj\n'
assert _HCDS_SRC and "def hc_down_silu(" in _HCDS_SRC, "the fn_hcd_silu.py source is missing (build the notebook with its builder)"
_QG_SRC = '# flashnext QSA output gate in one kernel (session 22) - installed as vllm/models/qwen4_exp/nvidia/ops/fn_qsa_gate.py by\n# s05q_decode_trims.py (FLASHNEXT_TRIM_QSA_GATE). The DCP attention path ends with\n#   output[:T] = (merged.to(bf16).float() * torch.sigmoid(gate.float())).to(output.dtype)\n# = ~6 small kernels per QSA layer (casts, sigmoid, multiply, cast, slice copy). This kernel does the same arithmetic in one pass:\n# torch\'s CUDA sigmoid is 1 / (1 + expf(-x)) in FP32 (IEEE division, accurate expf); here libdevice exp (the same expf) and div_rn,\n# then the FP32 product rounded to bf16 (round to nearest even, as .to()). Bit-identical (Colab L4 check over all 65,536 bf16 gate values).\nimport torch\n\nfrom vllm.triton_utils import tl, triton\n\ntry:\n    from triton.language.extra.cuda import libdevice as _libdevice\nexcept ImportError:  # older Triton layout\n    from triton.language.extra import libdevice as _libdevice\n\n\n@triton.jit\ndef _fn_qsa_gate_kernel(m_ptr, g_ptr, o_ptr, sm0, sm1, sg0, sg1, so0, so1, D: tl.constexpr, BLOCK_D: tl.constexpr):\n    t = tl.program_id(0)\n    h = tl.program_id(1)\n    d = tl.arange(0, BLOCK_D)\n    mask = d < D\n    m = tl.load(m_ptr + t * sm0 + h * sm1 + d, mask=mask, other=0.0).to(tl.bfloat16).to(tl.float32)\n    g = tl.load(g_ptr + t * sg0 + h * sg1 + d, mask=mask, other=0.0).to(tl.float32)\n    s = tl.math.div_rn(1.0, 1.0 + _libdevice.exp(-g))\n    tl.store(o_ptr + t * so0 + h * so1 + d, (m * s).to(o_ptr.dtype.element_ty), mask=mask)\n\n\ndef qsa_gate_store(merged: torch.Tensor, gate: torch.Tensor, out: torch.Tensor) -> None:\n    """out[t, h, :] = bf16(float(bf16(merged)) * sigmoid(float(gate))), all [T, H, D]; same values as the torch expression."""\n    T, H, D = merged.shape\n    assert gate.shape == (T, H, D) and out.shape == (T, H, D), (merged.shape, gate.shape, out.shape)\n    if T == 0:\n        return\n    m = merged if merged.stride(-1) == 1 else merged.contiguous()\n    g = gate if gate.stride(-1) == 1 else gate.contiguous()\n    assert out.stride(-1) == 1, out.stride()\n    _fn_qsa_gate_kernel[(T, H)](m, g, out, m.stride(0), m.stride(1), g.stride(0), g.stride(1), out.stride(0), out.stride(1),\n                                D=D, BLOCK_D=triton.next_power_of_2(D), num_warps=2)\n'
assert _QG_SRC and "def qsa_gate_store(" in _QG_SRC, "the fn_qsa_gate.py source is missing (build the notebook with its builder)"
_MARK = "# flashnext decode trims (session 22)"
_EDITS = {
    _qsa: [
        ("K4 query rotated once (switch)",
         """_FN_DCP_QUERY_EARLY = __import__("os").environ.get("FLASHNEXT_DCP_QUERY_EARLY") == "1"   # flashnext DCP comm (session 22)
""",
         """_FN_DCP_QUERY_EARLY = __import__("os").environ.get("FLASHNEXT_DCP_QUERY_EARLY") == "1"   # flashnext DCP comm (session 22)
_FN_TRIM_K4_ROT = __import__("os").environ.get("FLASHNEXT_TRIM_K4_ROT") == "1"   """ + _MARK + """
_FN_TRIM_QSA_GATE = __import__("os").environ.get("FLASHNEXT_TRIM_QSA_GATE") == "1"   """ + _MARK + """
if _FN_TRIM_K4_ROT:
    print("flashnext decode trims: K4_ROT ON", flush=True)
if _FN_TRIM_QSA_GATE:
    print("flashnext decode trims: QSA_GATE ON", flush=True)
"""),
        ("K4 query rotated once",
         """                key_cache, value_cache = k4_views(kv_cache, self.head_size)
                query = k4_rotate(query)
""",
         """                key_cache, value_cache = k4_views(kv_cache, self.head_size)
                if not (_FN_TRIM_K4_ROT and "_fn_query_all" in self.__dict__):   """ + _MARK + """: the early DCP copy is already rotated
                    query = k4_rotate(query)
"""),
        ("QSA output gate in one kernel",
         """        gate = output_gate[:num_tokens].reshape(num_tokens, num_heads, -1)
        # Same rounding as the unpatched kernel: BF16 attention output, gate applied in FP32.
        output[:num_tokens] = (
""",
         """        gate = output_gate[:num_tokens].reshape(num_tokens, num_heads, -1)
        if _FN_TRIM_QSA_GATE:   """ + _MARK + """: the same arithmetic in one kernel (ops/fn_qsa_gate.py)
            from .ops.fn_qsa_gate import qsa_gate_store

            qsa_gate_store(merged, gate, output[:num_tokens])
            return output
        # Same rounding as the unpatched kernel: BF16 attention output, gate applied in FP32.
        output[:num_tokens] = (
"""),
    ],
    _idx: [
        ("gap math once per forward pass (memo)",
         """def _fn_sw_gap(visible_blocks, token_topk, compress_ratio, row_start=0):
    \"\"\"(sink, first recent group) per row in compressed groups (int64), or None when sink+window is inactive.\"\"\"
""",
         """_FN_TRIM_GAP_ONCE = _fn_os.environ.get("FLASHNEXT_TRIM_GAP_ONCE") == "1"   """ + _MARK + """
_FN_STEP_MEMO = [None, {}]                      # [forward context (strong ref), {key: value}]
if _FN_TRIM_GAP_ONCE:
    print("flashnext decode trims: GAP_ONCE ON", flush=True)


def _fn_step_memo(key, compute):
    \"\"\"compute() once per forward pass for this key (FLASHNEXT_TRIM_GAP_ONCE=1), else every call.\"\"\"
    if not _FN_TRIM_GAP_ONCE:
        return compute()
    from vllm.forward_context import get_forward_context, is_forward_context_available

    if not is_forward_context_available():
        return compute()
    ctx = get_forward_context()
    if _FN_STEP_MEMO[0] is not ctx:
        _FN_STEP_MEMO[0], _FN_STEP_MEMO[1] = ctx, {}
    key = (torch.cuda.is_current_stream_capturing(),) + key
    memo = _FN_STEP_MEMO[1]
    if key not in memo:
        memo[key] = compute()
    return memo[key]


def _fn_sw_gap(visible_blocks, token_topk, compress_ratio, row_start=0):
    \"\"\"(sink, first recent group) per row in compressed groups (int64), or None when sink+window is inactive.\"\"\"
    if _FN_SW_WINDOW <= 0 and _FN_SW_LIMIT <= 0:
        return None
    _sink = _FN_SW_SINK_TOK
    return _fn_step_memo(("gap", visible_blocks.data_ptr(), tuple(visible_blocks.shape), tuple(visible_blocks.stride()), visible_blocks.dtype,
                          token_topk, compress_ratio, row_start, None if _sink is None else _sink.data_ptr()),
                         lambda: _fn_sw_gap_compute(visible_blocks, token_topk, compress_ratio, row_start))


def _fn_sw_gap_compute(visible_blocks, token_topk, compress_ratio, row_start=0):
    \"\"\"(sink, first recent group) per row in compressed groups (int64), or None when sink+window is inactive.\"\"\"
"""),
        ("decode compact arguments once per forward pass",
         """    if _fn_compact:
        _fn_lo = _fn_gap[0].to(torch.int32).contiguous()
        _fn_shift = (_fn_gap[1] - _fn_gap[0]).to(torch.int32).contiguous()
        _fn_vis = (visible_blocks - _fn_shift).to(visible_blocks.dtype)
        _fn_kw = dict(gap_lo_ptr=_fn_lo, gap_shift_ptr=_fn_shift, COMPACT=True)
""",
         """    if _fn_compact:
        def _fn_compact_args():   """ + _MARK + """
            _lo = _fn_gap[0].to(torch.int32).contiguous()
            _shift = (_fn_gap[1] - _fn_gap[0]).to(torch.int32).contiguous()
            return _lo, _shift, (visible_blocks - _shift).to(visible_blocks.dtype)

        _fn_lo, _fn_shift, _fn_vis = _fn_step_memo(("compact", id(_fn_gap), visible_blocks.data_ptr(), tuple(visible_blocks.shape),
                                                    tuple(visible_blocks.stride()), visible_blocks.dtype), _fn_compact_args)
        _fn_kw = dict(gap_lo_ptr=_fn_lo, gap_shift_ptr=_fn_shift, COMPACT=True)
"""),
        ("compact shift-back in one kernel (switch + kernel)",
         """_FN_SW_CHECK = _fn_os.environ.get("FLASHNEXT_QSA_SW_CHECK") == "1"
""",
         """_FN_SW_CHECK = _fn_os.environ.get("FLASHNEXT_QSA_SW_CHECK") == "1"
_FN_TRIM_QSA_SHIFT = _fn_os.environ.get("FLASHNEXT_TRIM_QSA_SHIFT") == "1"   """ + _MARK + """
if _FN_TRIM_QSA_SHIFT:
    print("flashnext decode trims: QSA_SHIFT ON", flush=True)


@triton.jit
def _fn_shift_back_kernel(bi_ptr, lo_ptr, sh_ptr, n_cols, stride_r, BLOCK: tl.constexpr):
    r = tl.program_id(0)
    c = tl.program_id(1) * BLOCK + tl.arange(0, BLOCK)
    m = c < n_cols
    lo = tl.load(lo_ptr + r)
    sh = tl.load(sh_ptr + r)
    b = tl.load(bi_ptr + r * stride_r + c, mask=m)
    tl.store(bi_ptr + r * stride_r + c, tl.where(b >= lo, b + sh, b).to(bi_ptr.dtype.element_ty), mask=m)


def _fn_shift_back(block_indices, lo, shift):
    \"\"\"block_indices[r, c] += shift[r] where block_indices[r, c] >= lo[r] (in place), one kernel.\"\"\"
    rows, cols = block_indices.shape
    if rows == 0 or cols == 0:
        return
    assert block_indices.stride(1) == 1 and lo.is_contiguous() and shift.is_contiguous()
    _fn_shift_back_kernel[(rows, triton.cdiv(cols, 1024))](block_indices, lo, shift, cols, block_indices.stride(0), BLOCK=1024, num_warps=4)
"""),
        ("compact shift-back in one kernel",
         """        block_indices.add_(torch.where(block_indices >= _fn_lo[:, None], _fn_shift[:, None], 0).to(block_indices.dtype))
""",
         """        if _FN_TRIM_QSA_SHIFT:   """ + _MARK + """
            _fn_shift_back(block_indices, _fn_lo, _fn_shift)
        else:
            block_indices.add_(torch.where(block_indices >= _fn_lo[:, None], _fn_shift[:, None], 0).to(block_indices.dtype))
"""),
    ],
    _mr: [
        ("top-k exchange on BF16 logits (switch)",
         """def _fn_topk_all_to_all(logits_shard, metadata, vocab_size, tp_rank):
""",
         """_FN_TRIM_TOPK_BF16 = __import__("os").environ.get("FLASHNEXT_TRIM_TOPK_BF16") == "1"   """ + _MARK + """
if _FN_TRIM_TOPK_BF16:
    print("flashnext decode trims: TOPK_BF16 ON", flush=True)


def _fn_topk_all_to_all(logits_shard, metadata, vocab_size, tp_rank):
"""),
        ("top-k exchange on BF16 logits",
         """    vals, ids = torch.topk(logits_shard.float(), k, dim=-1)
""",
         """    if _FN_TRIM_TOPK_BF16:   """ + _MARK + """: same selected values (BF16 -> FP32 is exact), no full FP32 copy
        vals, ids = torch.topk(logits_shard, k, dim=-1)
        vals = vals.float()
    else:
        vals, ids = torch.topk(logits_shard.float(), k, dim=-1)
"""),
    ],
    _k4: [
        ("K4 rotation as FWHT: helpers + query path",
         """def k4_rotate(x: torch.Tensor) -> torch.Tensor:
    \"\"\"x [..., D] -> x @ H in x's dtype (queries before attention, keys before they are stored).\"\"\"
    return (x.float() @ hadamard(x.shape[-1], x.device)).to(x.dtype)
""",
         """def k4_rotate(x: torch.Tensor) -> torch.Tensor:
    \"\"\"x [..., D] -> x @ H in x's dtype (queries before attention, keys before they are stored).\"\"\"
    if _FN_TRIM_K4_FWHT:   """ + _MARK + """
        return _fn_fwht_rotate(x)
    return (x.float() @ hadamard(x.shape[-1], x.device)).to(x.dtype)


_FN_TRIM_K4_FWHT = __import__("os").environ.get("FLASHNEXT_TRIM_K4_FWHT") == "1"   """ + _MARK + """
if _FN_TRIM_K4_FWHT:
    print("flashnext decode trims: K4_FWHT ON", flush=True)


@triton.jit
def _fn_fwht_stage(x, ROWS: tl.constexpr, D: tl.constexpr, H: tl.constexpr):
    x = tl.reshape(x, (ROWS, D // (2 * H), 2, H))
    x = tl.permute(x, (0, 1, 3, 2))
    a, b = tl.split(x)
    x = tl.join(a + b, a - b)
    x = tl.permute(x, (0, 1, 3, 2))
    return tl.reshape(x, (ROWS, D))


@triton.jit
def _fn_fwht256(x, ROWS: tl.constexpr):
    \"\"\"x [ROWS, 256] float32 -> x @ hadamard(256): 8 butterfly stages (Sylvester order), then the 1/16 scale.\"\"\"
    x = _fn_fwht_stage(x, ROWS, 256, 1)
    x = _fn_fwht_stage(x, ROWS, 256, 2)
    x = _fn_fwht_stage(x, ROWS, 256, 4)
    x = _fn_fwht_stage(x, ROWS, 256, 8)
    x = _fn_fwht_stage(x, ROWS, 256, 16)
    x = _fn_fwht_stage(x, ROWS, 256, 32)
    x = _fn_fwht_stage(x, ROWS, 256, 64)
    x = _fn_fwht_stage(x, ROWS, 256, 128)
    return x * 0.0625


@triton.jit
def _fn_fwht256_kernel(x_ptr, y_ptr, M, stride_x, stride_y, BLOCK_M: tl.constexpr):
    rows = tl.program_id(0) * BLOCK_M + tl.arange(0, BLOCK_M)
    cols = tl.arange(0, 256)
    mask = rows[:, None] < M
    x = tl.load(x_ptr + rows[:, None] * stride_x + cols[None, :], mask=mask, other=0.0).to(tl.float32)
    tl.store(y_ptr + rows[:, None] * stride_y + cols[None, :], _fn_fwht256(x, BLOCK_M).to(y_ptr.dtype.element_ty), mask=mask)


def _fn_fwht_rotate(x: torch.Tensor) -> torch.Tensor:
    if x.shape[-1] != 256:
        raise RuntimeError(f"FLASHNEXT_TRIM_K4_FWHT supports head size 256 only, got {x.shape[-1]}")
    x2 = x.reshape(-1, 256)
    if x2.stride(-1) != 1:
        x2 = x2.contiguous()
    y = torch.empty((x2.shape[0], 256), dtype=x.dtype, device=x.device)
    _fn_fwht256_kernel[(triton.cdiv(x2.shape[0], 2),)](x2, y, x2.shape[0], x2.stride(0), y.stride(0), BLOCK_M=2, num_warps=1)
    return y.reshape(x.shape)
"""),
        ("K4 rotation as FWHT: store kernel flag",
         """    PAGE_SIZE: tl.constexpr,
    HEAD_DIM: tl.constexpr,
):
    tok = tl.program_id(0)
""",
         """    PAGE_SIZE: tl.constexpr,
    HEAD_DIM: tl.constexpr,
    ROTATE: tl.constexpr = False,   """ + _MARK + """: rotate the key here (FWHT) instead of a separate fp32 matmul
):
    tok = tl.program_id(0)
"""),
        ("K4 rotation as FWHT: store kernel rotates",
         """    k = tl.load(key_ptr + tok * stride_key_tok + head * stride_key_head + dims).to(tl.float32)      # already rotated (k4_store)
""",
         """    k = tl.load(key_ptr + tok * stride_key_tok + head * stride_key_head + dims).to(tl.float32)      # already rotated (k4_store)
    if ROTATE:   """ + _MARK + """
        k = tl.reshape(_fn_fwht256(tl.reshape(k, (1, HEAD_DIM)), 1), (HEAD_DIM,))
"""),
        ("K4 rotation as FWHT: store call",
         """    k_rot = (key[:n].float() @ hadamard(head_dim, key.device)).contiguous()
""",
         """    if _FN_TRIM_K4_FWHT:   """ + _MARK + """: the store kernel rotates the BF16 key rows itself
        if head_dim != 256:
            raise RuntimeError(f"FLASHNEXT_TRIM_K4_FWHT supports head size 256 only, got {head_dim}")
        k_in = key if key.stride(-1) == 1 else key.contiguous()
        _k4_store_kernel[(n, key.shape[1])](
            k_in, value, kv_cache, slot_mapping,
            k_in.stride(0), k_in.stride(1), value.stride(0), value.stride(1),
            kv_cache.stride(0), kv_cache.stride(1), kv_cache.stride(2),
            kv_cache.shape[0] * kv_cache.shape[2],
            PAGE_SIZE=kv_cache.shape[2], HEAD_DIM=head_dim, ROTATE=True, num_warps=2,
        )
        return
    k_rot = (key[:n].float() @ hadamard(head_dim, key.device)).contiguous()
"""),
    ],
    _gdn: [
        ("GDN b/a without copies: call site",
         """        torch.ops.vllm.qwen_gdn_attention_core(
            mixed_qkv,
            b.contiguous(),
            a.contiguous(),
            core_attn_out,
""",
         """        torch.ops.vllm.qwen_gdn_attention_core(   """ + _MARK + """: strided b/a views, see _forward_core
            mixed_qkv,
            b if _FN_TRIM_GDN_AB else b.contiguous(),
            a if _FN_TRIM_GDN_AB else a.contiguous(),
            core_attn_out,
"""),
        ("GDN b/a without copies (switch)",
         """logger = init_logger(__name__)
""",
         """logger = init_logger(__name__)
_FN_TRIM_GDN_AB = os.environ.get("FLASHNEXT_TRIM_GDN_AB") == "1"   """ + _MARK + """
if _FN_TRIM_GDN_AB:
    print("flashnext decode trims: GDN_AB ON", flush=True)
"""),
        ("GDN b/a without copies: mixed path copies",
         """        num_accepted_tokens = attn_metadata.num_accepted_tokens

        mixed_qkv = mixed_qkv[:num_actual_tokens]
        b = b[:num_actual_tokens]
        a = a[:num_actual_tokens]
""",
         """        num_accepted_tokens = attn_metadata.num_accepted_tokens

        mixed_qkv = mixed_qkv[:num_actual_tokens]
        b = b[:num_actual_tokens].contiguous()   """ + _MARK + """: no-op when b/a come contiguous (switch off)
        a = a[:num_actual_tokens].contiguous()
"""),
    ],
    _hc: [
        ("HC down + SiLU fused (switch)",
         """from .ops.cute_dsl.hc_down_silu import MAX_FUSED_M, hc_down_silu
""",
         """from .ops.cute_dsl.hc_down_silu import MAX_FUSED_M, hc_down_silu

_FN_TRIM_HCD_SILU = __import__("os").environ.get("FLASHNEXT_TRIM_HCD_SILU") == "1"   """ + _MARK + """
_FN_HCDS_MAX_M = 128                 # decode / small mixed steps; larger M keeps split-K + vLLM's SiLU (M 1024: +6.6 us fused)
_fn_hcds_said = [False]
if _FN_TRIM_HCD_SILU:
    print(f"flashnext decode trims: HCD_SILU ON (M <= {_FN_HCDS_MAX_M})", flush=True)
"""),
        ("HC down + SiLU fused",
         """        down_and_injection = self.input_mix_weight_down_block_inject(xn)
        split_sizes = [self.lora_rank, self.hc_count, self.pad_size]
""",
         """        if _FN_TRIM_HCD_SILU and xn.shape[0] <= _FN_HCDS_MAX_M:   """ + _MARK + """
            _fn = self.input_mix_weight_down_block_inject.__dict__.get("_fn_hcd")
            if _fn is None:
                raise RuntimeError("FLASHNEXT_TRIM_HCD_SILU=1 needs the HC-down split-K kernel (FLASHNEXT_HC_DOWN=1): "
                                   f"{self.input_mix_weight_down_block_inject.prefix} has no kept FP8 weight")
            from .ops.fn_hcd_silu import hc_down_silu as _fn_hcd_silu

            if not _fn_hcds_said[0]:
                _fn_hcds_said[0] = True
                print(f"flashnext HC-down + SiLU fused: ON (M <= {_FN_HCDS_MAX_M}) | first layer "
                      f"{self.input_mix_weight_down_block_inject.prefix}", flush=True)
            return _fn_hcd_silu(xn if xn.stride(-1) == 1 else xn.contiguous(), _fn[0], _fn[1], self.lora_rank, self.hc_count)
        down_and_injection = self.input_mix_weight_down_block_inject(xn)
        split_sizes = [self.lora_rank, self.hc_count, self.pad_size]
"""),
    ],
    _mu: [
        ("DeltaNet align copies: 1 tile",
         """_TEMPORAL_TILES = 16
""",
         """_TEMPORAL_TILES = 1 if __import__("os").environ.get("FLASHNEXT_TRIM_MAMBA_TILES") == "1" else 16   """ + _MARK + """
if _TEMPORAL_TILES == 1:
    print("flashnext decode trims: MAMBA_TILES ON", flush=True)
"""),
    ],
}
for _mod, _src in ((_hcds, _HCDS_SRC), (_qg, _QG_SRC)):
    if not _mod.exists() or _mod.read_text() != _src:
        compile(_src, str(_mod), "exec")
        _mod.write_text(_src)
        print("decode trims: [installed]", _mod)
    else:
        print("decode trims: [already]", _mod)
for _f, _edits in _EDITS.items():
    _s = _f.read_text()
    for _label, _old, _new in _edits:
        if _new in _s:
            print("decode trims: [already]", _label)
            continue
        assert _s.count(_old) == 1, f"decode trims edit '{_label}': anchor found {_s.count(_old)} times in {_f} (an earlier cell changed?)"
        _s = _s.replace(_old, _new)
        print("decode trims: [installed]", _label)
    compile(_s, str(_f), "exec")
    _f.write_text(_s)
print("(FLASHNEXT_TRIM_K4_ROT / _GAP_ONCE / _TOPK_BF16 / _K4_FWHT / _GDN_AB / _HCD_SILU / _MAMBA_TILES / _QSA_GATE / _QSA_SHIFT = 1 before the server starts; unset = unchanged)")

# Kernel Cache Restore

In [ ]:
import glob, os, shutil, tarfile, time
from concurrent.futures import ThreadPoolExecutor
_t = time.time()
_pats = ["startup_caches*.tar.gz", "startup_caches*.tar.gz.bin"]
_tars = sorted({p for pat in _pats for d in ("/kaggle/input/*", "/kaggle/input/*/*", "/kaggle/input/datasets/*/*", "/kaggle/input/datasets/*/*/*")
                for p in glob.glob(f"{d}/{pat}")})
_how = "fixed mount depths"
if not _tars:
    _tars = sorted({p for pat in _pats for p in glob.glob(f"/kaggle/input/**/{pat}", recursive=True)}); _how = "full search"
_done = []
for _tar in _tars:
    with tarfile.open(_tar, "r:gz") as tf:
        tf.extractall("/", filter="tar")
    _done.append(f"unpacked {_tar} ({os.path.getsize(_tar) / 1e6:.1f} MB, found by {_how})")
if not _tars:
    _bases = set()
    for _marker in ("root/.humming", "root/.triton", "root/.cache"):
        for _p in glob.glob(f"/kaggle/input/**/{_marker}", recursive=True):
            _bases.add(_p[: -len(_marker)].rstrip("/"))
    for _b in sorted(_bases):
        _src = os.path.join(_b, "root")
        _files = [os.path.join(d, f) for d, _, fs in os.walk(_src) for f in fs]
        def _cp(f):
            dst = "/root/" + os.path.relpath(f, _src)
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy2(f, dst)
        with ThreadPoolExecutor(32) as ex:
            list(ex.map(_cp, _files))
        _done.append(f"copied {_src} -> /root ({len(_files)} files, 32 threads)")
if not _done:
    raise RuntimeError("no startup caches under /kaggle/input (no startup_caches*.tar.gz(.bin), no unpacked root/ folder). Attach the fast-startup dataset.")
print("\n".join("  " + d for d in _done))
for _f in ("/root/.cache/flashinfer", "/root/.humming", "/root/.triton", "/root/.cache/vllm"):
    print(f"  {_f}: {sum(len(fs) for _, _, fs in os.walk(_f)) if os.path.isdir(_f) else 'not in the dataset'} files")
_aot, _n8 = "/tmp/site-vllm/flashinfer/data/aot", 0
for _so in sorted(glob.glob("/root/.cache/flashinfer/*/*/cached_ops/*/*.so")):
    _name = os.path.basename(os.path.dirname(_so))
    if os.path.basename(_so) != _name + ".so":
        continue
    os.makedirs(f"{_aot}/{_name}", exist_ok=True)
    shutil.copy2(_so, f"{_aot}/{_name}/{_name}.so")
    _n8 += 1
    print(f"  FlashInfer kernel '{_name}' -> {_aot}/{_name}/{_name}.so (loaded as prebuilt, never rebuilt)")
if not _n8:
    print("  no FlashInfer kernels in the restored cache (the sampling kernel will be compiled during the start, ~2 min)")
print(f"caches restored in {time.time() - _t:.1f} s")
CACHE_RESTORED_T = time.time()

# Harness

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

REPO = Path('/tmp/arc-agi-2-submission')
shutil.rmtree(REPO, ignore_errors=True)
shutil.copytree(WHEELHOUSE / 'pivotools', REPO)
print('pivotools copied to', REPO, '| data:', sorted(p.name for p in (REPO / 'data').glob('*.json')))
import json

KERNEL_MEM_GIB = 8
RAM_BRAKE_GIB = 12
MEM_GUARD_LOG = '/kaggle/working/memory_guard.log'

ARC_PY = sys.executable
ARC_ENV = dict(os.environ, PYTHONPATH=os.pathsep.join([str(ASITE), str(REPO / 'src')]), PYTHONUNBUFFERED='1',
               HF_HUB_OFFLINE='1', TOKENIZERS_PARALLELISM='false',
               ARC_KERNEL_MEM_GIB=str(KERNEL_MEM_GIB or 0), ARC_RAM_BRAKE_GIB=str(RAM_BRAKE_GIB or 0), ARC_MEM_GUARD_LOG=MEM_GUARD_LOG)
print(f'memory guard: python tool process limit {KERNEL_MEM_GIB or "off"} GiB | brake below {RAM_BRAKE_GIB or "off"} GiB available | log {MEM_GUARD_LOG}')
r = subprocess.run([sys.executable, '-c', "import arcagi2, jupyter_client, ipykernel, numpy, scipy, shapely, networkx, skimage, ortools, z3, openai; print('arcagi2 from', arcagi2.__file__)"],
                   env=ARC_ENV, capture_output=True, text=True)
print((r.stdout + r.stderr).strip()[-1500:])
assert r.returncode == 0, 'the harness environment is incomplete: paste the message above'

SANDBOX = r'''"""
Docker-free Sandbox for pivotools: one real IPython kernel (a local subprocess) per REPL.

Implements the same Sandbox / REPL interface as ipybox_sandbox.py, with the same IPython semantics and the
same result handling as ipybox, so a solver config only needs sandbox_cls=LocalKernelSandbox, sandbox_kwargs={}.

The kernel is launched in isolation. Hosted notebooks such as Colab register their own kernelspec and ship their
own IPython settings, which crash a kernel started from a separate environment. So this module (1) bypasses the
installed kernelspecs, (2) starts the kernel with every IPython/Jupyter config location emptied, and (3) uses a
throwaway working directory. If the kernel still fails to start, the error includes the kernel's own output.

No isolation beyond a separate process: use it on a machine you trust (Colab, Kaggle).
"""
import asyncio
import logging
import os
import shutil
import signal
import sys
import tempfile
import threading
import time
from pathlib import Path
from typing import Self

from jupyter_client.kernelspec import KernelSpec, KernelSpecManager
from jupyter_client.manager import AsyncKernelManager

from arcagi2.sandbox.base import ExecutionError, ExecutionResult, REPL, Sandbox
from arcagi2.sandbox.exceptions import SandboxInfrastructureError, ExecutionTimeoutError

logger = logging.getLogger(__name__)

_KERNEL_NAME = "pivotools-local"

# Starts the stock ipykernel app, after emptying the system-wide and environment-wide IPython config directories.
_LAUNCHER = (
    "import IPython.core.application as a\n"
    "a.SYSTEM_CONFIG_DIRS[:] = []\n"
    "a.ENV_CONFIG_DIRS[:] = []\n"
    "from ipykernel.kernelapp import launch_new_instance\n"
    "launch_new_instance()\n"
)


class _OwnSpecManager(KernelSpecManager):
    """Always returns our own kernelspec, whatever kernelspecs the machine has installed."""

    def get_kernel_spec(self, kernel_name):
        return KernelSpec(
            argv=[sys.executable, "-c", _LAUNCHER, "-f", "{connection_file}",
                  "--IPKernelApp.kernel_class=ipykernel.ipkernel.IPythonKernel"],
            display_name="pivotools-local",
            language="python",
        )

    def find_kernel_specs(self):
        return {_KERNEL_NAME: "(built in)"}


_SPEC_MANAGER = _OwnSpecManager()


# ---- memory guard: watches every live kernel (and its direct children) once a second ----
_KERNEL_MEM_GIB = float(os.environ.get("ARC_KERNEL_MEM_GIB", "0") or 0)
_RAM_BRAKE_GIB = float(os.environ.get("ARC_RAM_BRAKE_GIB", "0") or 0)
_GUARD_LOG = os.environ.get("ARC_MEM_GUARD_LOG", "")
_live = {}                       # kernel pid -> LocalKernelREPL
_guard_lock = threading.Lock()
_guard_thread = None


def _rss_gib(pid: int) -> float:
    pids = [pid]
    try:
        pids += [int(p) for p in open(f"/proc/{pid}/task/{pid}/children").read().split()]
    except Exception:
        pass
    kib = 0
    for p in pids:
        try:
            for line in open(f"/proc/{p}/status"):
                if line.startswith("VmRSS:"):
                    kib += int(line.split()[1])
                    break
        except Exception:
            pass
    return kib / 1048576


def _avail_gib() -> float:
    for line in open("/proc/meminfo"):
        if line.startswith("MemAvailable:"):
            return int(line.split()[1]) / 1048576
    return float("inf")


def _guard_kill(pid: int, repl, why: str):
    with _guard_lock:
        if _live.pop(pid, None) is None:
            return
    repl._killed = why
    try:
        os.kill(pid, signal.SIGKILL)
    except Exception:
        pass
    msg = f"MEMORY GUARD: python tool process {pid} stopped: it {why}"
    print(msg, file=sys.stderr, flush=True)
    if _GUARD_LOG:
        try:
            with open(_GUARD_LOG, "a") as f:
                f.write(time.strftime("%H:%M:%S ") + msg + "\n")
        except Exception:
            pass


def _guard_loop():
    while True:
        time.sleep(1.0)
        with _guard_lock:
            items = list(_live.items())
        if not items:
            continue
        sizes = [(pid, repl, _rss_gib(pid)) for pid, repl in items]
        for pid, repl, gib in sizes:
            if _KERNEL_MEM_GIB and gib > _KERNEL_MEM_GIB:
                _guard_kill(pid, repl, f"used {gib:.1f} GiB of RAM, more than the {_KERNEL_MEM_GIB:g} GiB limit per python process")
        if _RAM_BRAKE_GIB:
            avail = _avail_gib()
            alive = [x for x in sizes if x[0] in _live]
            if avail < _RAM_BRAKE_GIB and alive:
                pid, repl, gib = max(alive, key=lambda x: x[2])
                if gib > 0.5:
                    _guard_kill(pid, repl, f"was the largest python process ({gib:.1f} GiB) when the machine had only {avail:.1f} GiB "
                                           f"of RAM left (brake below {_RAM_BRAKE_GIB:g} GiB)")
                    time.sleep(5.0)    # give the machine time to free the memory before the next decision


def _guard_register(pid, repl):
    global _guard_thread
    if not (_KERNEL_MEM_GIB or _RAM_BRAKE_GIB) or pid is None:
        return
    with _guard_lock:
        _live[pid] = repl
        if _guard_thread is None:
            _guard_thread = threading.Thread(target=_guard_loop, name="memory-guard", daemon=True)
            _guard_thread.start()


def _guard_unregister(pid):
    with _guard_lock:
        _live.pop(pid, None)


class _MemoryStopped(Exception):
    pass


def _tail(path: Path, n_chars: int = 3000) -> str:
    try:
        return path.read_text(errors="replace")[-n_chars:]
    except Exception:
        return ""


class LocalKernelREPL(REPL):
    def __init__(self, kernel_name: str = "python3"):
        self._km = None
        self._kc = None
        self._workdir = None
        self._log_path = None
        self._pid = None
        self._killed = None

    async def _start_kernel(self):
        env = {k: v for k, v in os.environ.items() if k != "PYTHONSTARTUP"}
        env.update({"IPYTHONDIR": str(self._workdir / "ipython"), "JUPYTER_NO_CONFIG": "1"})
        (self._workdir / "ipython").mkdir(exist_ok=True)
        (self._workdir / "cwd").mkdir(exist_ok=True)
        self._km = AsyncKernelManager(kernel_name=_KERNEL_NAME, kernel_spec_manager=_SPEC_MANAGER)
        with open(self._log_path, "ab") as log:
            await self._km.start_kernel(env=env, cwd=str(self._workdir / "cwd"), stdout=log, stderr=log)
        self._kc = self._km.client()
        self._kc.start_channels()
        await self._kc.wait_for_ready(timeout=60)
        if self._pid is not None:
            _guard_unregister(self._pid)
        prov = getattr(self._km, "provisioner", None)
        self._pid = getattr(getattr(prov, "process", None), "pid", None) or getattr(prov, "pid", None)
        self._killed = None
        _guard_register(self._pid, self)

    async def __aenter__(self) -> Self:
        self._workdir = Path(tempfile.mkdtemp(prefix="pivotools_kernel_"))
        self._log_path = self._workdir / "kernel.log"
        try:
            await self._start_kernel()
        except Exception as e:
            details = _tail(self._log_path)
            await self._shutdown()
            raise SandboxInfrastructureError(f"Failed to start local kernel: {e}\n--- kernel output ---\n{details}") from e
        return self

    async def _shutdown(self):
        try:
            if self._kc is not None:
                self._kc.stop_channels()
            if self._km is not None:
                await self._km.shutdown_kernel(now=True)
        except Exception as e:
            logger.warning(f"Failed to shut down local kernel: {e}")
        finally:
            if self._pid is not None:
                _guard_unregister(self._pid)
                self._pid = None
            self._kc = None
            self._km = None
            if self._workdir is not None:
                shutil.rmtree(self._workdir, ignore_errors=True)
                self._workdir = None

    async def __aexit__(self, exc_type, exc_val, exc_tb):
        await self._shutdown()

    async def _run(self, code: str):
        # Same result handling as ipybox: stream text and the text/plain of results/displays are concatenated;
        # an error carries the kernel's own traceback lines joined by newlines.
        msg_id = self._kc.execute(code, store_history=False)
        chunks, error = [], None
        while True:
            msg = await self._kc.get_iopub_msg()
            if msg["parent_header"].get("msg_id") != msg_id:
                continue
            kind, content = msg["msg_type"], msg["content"]
            if kind == "stream":
                chunks.append(content["text"])
            elif kind in ("execute_result", "display_data"):
                text = content["data"].get("text/plain")
                if text is not None:
                    chunks.append(text)
            elif kind == "error":
                error = "\n".join(content["traceback"])
            elif kind == "status" and content["execution_state"] == "idle":
                return chunks, error

    async def _run_guarded(self, code: str):
        # _run, but gives up as soon as the memory guard has stopped this kernel (then starts a fresh one)
        run = asyncio.ensure_future(self._run(code))
        while not run.done():
            if self._killed is not None:
                run.cancel()
                why = self._killed
                try:
                    self._kc.stop_channels()
                    await self._km.shutdown_kernel(now=True)
                except Exception:
                    pass
                await self._start_kernel()
                raise _MemoryStopped(why)
            await asyncio.wait({run}, timeout=0.5)
        return run.result()

    async def execute(self, code: str, timeout: float) -> ExecutionResult:
        if self._kc is None:
            raise SandboxInfrastructureError("REPL is not started")
        try:
            chunks, error = await asyncio.wait_for(self._run_guarded(code), timeout=timeout)
        except asyncio.TimeoutError:
            # Interrupt the runaway cell (like ipybox). If the kernel does not come back, replace it (state is lost).
            try:
                await self._km.interrupt_kernel()
                await asyncio.wait_for(self._kc.wait_for_ready(timeout=5), timeout=8)
            except Exception:
                try:
                    self._kc.stop_channels()
                    await self._km.shutdown_kernel(now=True)
                    await self._start_kernel()
                except Exception as e:
                    raise SandboxInfrastructureError(f"Kernel could not be recovered: {e}") from e
            raise ExecutionTimeoutError(f"Code execution timed out after {timeout} seconds")
        except _MemoryStopped as e:
            raise ExecutionError(trace=f"MemoryError: this python process {e}, so it was stopped and replaced by a fresh one. "
                                       f"Every variable, function and import from earlier calls is gone: define them again. "
                                       f"Use less memory (smaller arrays, no huge lists or caches).")
        except Exception as e:
            raise SandboxInfrastructureError(f"Code execution failed: {e}") from e
        if error is not None:
            raise ExecutionError(trace=error)
        return ExecutionResult(text="".join(chunks).strip() if chunks else None)


class LocalKernelSandbox(Sandbox):
    """No container to manage: each create_repl() starts its own IPython kernel process."""

    def __init__(self, kernel_name: str = "python3"):
        self.kernel_name = kernel_name

    async def __aenter__(self) -> Self:
        return self

    async def __aexit__(self, exc_type, exc_val, exc_tb):
        return None

    def create_repl(self) -> LocalKernelREPL:
        return LocalKernelREPL(self.kernel_name)
'''

(REPO / 'src/arcagi2/sandbox/local_sandbox.py').write_text(SANDBOX)
print('local_sandbox.py written')

check = '''
import asyncio, os
from arcagi2.sandbox.local_sandbox import LocalKernelSandbox
from arcagi2.sandbox.base import ExecutionError
async def main():
    async with LocalKernelSandbox() as sb:
        async with sb.create_repl() as repl:
            await repl.execute("x = 41", 60)
            print("state ok: x + 1 ->", (await repl.execute("x + 1", 60)).text)
            try:
                await repl.execute("1 / 0", 60)
            except ExecutionError:
                print("errors ok: a ZeroDivisionError came back as an error")
            if os.environ.get("ARC_KERNEL_MEM_GIB", "0") not in ("", "0"):
                lim = float(os.environ["ARC_KERNEL_MEM_GIB"])
                try:
                    await repl.execute(f"big = b'x' * {int((lim + 1) * 2**30)}; import time; time.sleep(30)", 120)
                    print("MEMORY GUARD CHECK FAILED: a process above the limit was not stopped")
                except ExecutionError as e:
                    print("memory guard ok:", str(getattr(e, "trace", e))[:110])
                print("fresh process after the stop:", (await repl.execute("1 + 1", 60)).text)
asyncio.run(main())
'''
r = subprocess.run([sys.executable, '-c', check], env=ARC_ENV, capture_output=True, text=True, timeout=300)
print((r.stdout + r.stderr).strip()[-4000:])
assert r.returncode == 0 and 'CHECK FAILED' not in r.stdout, 'sandbox check failed, paste the WHOLE message above'

In [ ]:
import hashlib, pathlib, subprocess, sys

SRC = REPO / "src/arcagi2"
assert SRC.exists(), "Run the pivotools cell first."
assert (SRC / "sandbox/local_sandbox.py").exists(), "Run the local-sandbox cell first."

FILES = {
    "utils/grid_image.py": r'''"""Dependency-free rendering of ARC grids to PNG (standard library only).

This module is used in two places: by the client, to attach images of the puzzle to the first message, and inside
the Python REPL, where its source is executed so the model can call show(...). Keep it stdlib-only and self-contained.
"""
import base64
import struct
import zlib

PALETTE = [
    (0, 0, 0), (0, 116, 217), (255, 65, 54), (46, 204, 64), (255, 220, 0),
    (170, 170, 170), (240, 18, 190), (255, 133, 27), (127, 219, 255), (135, 12, 37),
]
GRIDLINE = (85, 85, 85)
FILL = (32, 32, 32)


def _grid_pixel_rows(grid, scale, gridlines):
    lines = gridlines and scale >= 4
    rows = []
    for row in grid:
        cells = []
        for v in row:
            color = bytes(PALETTE[v]) if isinstance(v, int) and 0 <= v <= 9 else bytes((255, 255, 255))
            cells.append((bytes(GRIDLINE) + color * (scale - 1)) if lines else color * scale)
        line = b"".join(cells)
        if lines:
            rows.append(bytes(GRIDLINE) * (scale * len(row)))
            rows.extend([line] * (scale - 1))
        else:
            rows.extend([line] * scale)
    width = (len(grid[0]) if grid else 0) * scale
    return rows, width


def render_png(grids, scale=16, gridlines=True, gap=10):
    """Render one grid, or several grids side by side, as PNG bytes."""
    if grids and isinstance(grids[0], (int, float)):
        raise TypeError("render_png expects a list of grids (each a list of lists of ints)")
    blocks = [_grid_pixel_rows(g, scale, gridlines) for g in grids if g]
    if not blocks:
        raise ValueError("nothing to draw: the grid is empty")
    height = max(len(rows) for rows, _ in blocks)
    gap_bytes = bytes(FILL) * gap
    out_rows = []
    for y in range(height):
        parts = []
        for i, (rows, width) in enumerate(blocks):
            if i:
                parts.append(gap_bytes)
            parts.append(rows[y] if y < len(rows) else bytes(FILL) * width)
        out_rows.append(b"".join(parts))
    width_px = len(out_rows[0]) // 3
    raw = b"".join(b"\x00" + r for r in out_rows)

    def chunk(kind, data):
        body = kind + data
        return struct.pack(">I", len(data)) + body + struct.pack(">I", zlib.crc32(body) & 0xFFFFFFFF)

    return (b"\x89PNG\r\n\x1a\n"
            + chunk(b"IHDR", struct.pack(">IIBBBBB", width_px, height, 8, 2, 0, 0, 0))
            + chunk(b"IDAT", zlib.compress(raw, 6))
            + chunk(b"IEND", b""))


def png_size(png):
    return struct.unpack(">II", png[16:24])


def data_url(png):
    return "data:image/png;base64," + base64.b64encode(png).decode()


def image_tokens(width, height):
    """Rough vision-token cost of an image for a Qwen-style model (32x32 pixel merged patches plus 2 sentinels)."""
    return max(64, -(-width // 32) * -(-height // 32) + 2)


def image_tokens_from_data_url(url):
    marker = "base64,"
    i = url.find(marker)
    if i < 0:
        return 256
    try:
        head = base64.b64decode(url[i + len(marker): i + len(marker) + 32], validate=True)
        w, h = struct.unpack(">II", head[16:24])
        return image_tokens(w, h)
    except Exception:
        return 256


def build_first_message_content(prompt_text, puzzle, scale=16):
    """Content parts for the first user message: the prompt text, then one image per training pair
    (input and output side by side) and one per test input."""
    parts = [{"type": "text", "text": prompt_text}]
    if __import__("os").environ.get("ARC_TEXT_GRIDS", "1") == "0":     # session 17: no grid text above, only the sizes
        parts.append({"type": "text", "text": (
            "\n\n# Images of the grids\n\nThe grids of the puzzle above, drawn as images (they are not written out as text). Each square is "
            "one cell and thin lines separate cells. Use the images for overall shape, symmetry and layout, and code (`puzzle`) for exact values.")})
    else:
        parts.append({"type": "text", "text": (
            "\n\n# Images of the grids\n\nThe same grids as above, drawn as images. Each square is one cell and thin lines "
            "separate cells. Use the images for overall shape, symmetry and layout, and the text or code for exact values.")})
    for i, ex in enumerate(puzzle["train"]):
        parts.append({"type": "text", "text": f"\nTraining example {i}: input (left) and output (right)"})
        parts.append({"type": "image_url", "image_url": {"url": data_url(render_png([ex["input"], ex["output"]], scale=scale))}})
    for j, ex in enumerate(puzzle["test"]):
        parts.append({"type": "text", "text": f"\nTest input {j}"})
        parts.append({"type": "image_url", "image_url": {"url": data_url(render_png([ex["input"]], scale=scale))}})
    return parts


# ---------------------------------------------------------------------------------------------------------------------
# Drawing PART of a grid, with the row and column numbers of the original grid on the borders (used by show_part).
# Same palette and grid lines as render_png; render_png itself is unchanged.
# ---------------------------------------------------------------------------------------------------------------------
_FONT = {
    "0": ("111", "101", "101", "101", "111"), "1": ("010", "110", "010", "010", "111"),
    "2": ("111", "001", "111", "100", "111"), "3": ("111", "001", "111", "001", "111"),
    "4": ("101", "101", "111", "001", "001"), "5": ("111", "100", "111", "001", "111"),
    "6": ("111", "100", "111", "101", "111"), "7": ("111", "001", "010", "010", "010"),
    "8": ("111", "101", "111", "101", "111"), "9": ("111", "101", "111", "001", "111"),
}
_LABEL_COLOR = (200, 200, 200)


def _span(spec, n):
    """(start, stop) of a part: 0-based, stop not included. None = everything; a single number = that one row or column."""
    if spec is None:
        return 0, n
    if isinstance(spec, int):
        a, b = spec, spec + 1
    else:
        a, b = spec
    a, b = max(0, int(a)), min(n, int(b))
    if a >= b:
        raise ValueError("empty part: use (start, stop) with start < stop inside the grid (0-based, stop not included), or one number")
    return a, b


def _encode_png(out_rows, width_px):
    raw = b"".join(b"\x00" + bytes(r) for r in out_rows)

    def chunk(kind, data):
        body = kind + data
        return struct.pack(">I", len(data)) + body + struct.pack(">I", zlib.crc32(body) & 0xFFFFFFFF)

    return (b"\x89PNG\r\n\x1a\n"
            + chunk(b"IHDR", struct.pack(">IIBBBBB", width_px, len(out_rows), 8, 2, 0, 0, 0))
            + chunk(b"IDAT", zlib.compress(raw, 6))
            + chunk(b"IEND", b""))


def _draw_text(canvas, x, y, text, fs, color):
    pix = bytes(color)
    cx = x
    for ch in text:
        glyph = _FONT.get(ch)
        if glyph is not None:
            for gy, line in enumerate(glyph):
                for gx, bit in enumerate(line):
                    if bit == "1":
                        for dy in range(fs):
                            row = canvas[y + gy * fs + dy]
                            for dx in range(fs):
                                px = cx + gx * fs + dx
                                row[3 * px:3 * px + 3] = pix
        cx += 4 * fs


def render_part_png(grid, rows=None, cols=None, scale=16, ticks=True):
    """PNG bytes of rows[start:stop] x cols[start:stop] of one grid: every cell a coloured square, thin lines between all cells,
    and (ticks=True) the row numbers on the left and the column numbers on top, counted in the ORIGINAL grid."""
    height = len(grid)
    width = len(grid[0]) if height else 0
    r0, r1 = _span(rows, height)
    c0, c1 = _span(cols, width)
    scale = max(8, int(scale))
    sub = [list(row[c0:c1]) for row in grid[r0:r1]]
    cell_rows, width_px = _grid_pixel_rows(sub, scale, True)
    line = bytes(GRIDLINE)
    body = [bytes(r) + line for r in cell_rows]            # closing line on the right edge
    body.append(line * (width_px + 1))                       # and on the bottom edge
    digits = max(len(str(r1 - 1)), len(str(c1 - 1)))
    fs = 2 if scale >= 14 and (digits < 2 or scale >= 20) else 1     # two-digit numbers need room: smaller digits in small cells
    lm = digits * 4 * fs + 4 if ticks else 0
    tm = 5 * fs + 4 if ticks else 0
    total_w = lm + width_px + 1
    canvas = [bytearray(bytes(FILL) * total_w) for _ in range(tm + len(body))]
    for i, r in enumerate(body):
        canvas[tm + i][3 * lm:3 * lm + len(r)] = r
    if ticks:
        for j in range(c1 - c0):
            label = str(c0 + j)
            w = len(label) * 4 * fs - fs
            _draw_text(canvas, lm + j * scale + (scale - w) // 2 + 1, 2, label, fs, _LABEL_COLOR)
        for i in range(r1 - r0):
            label = str(r0 + i)
            w = len(label) * 4 * fs - fs
            _draw_text(canvas, lm - 3 - w, tm + i * scale + (scale - 5 * fs) // 2 + 1, label, fs, _LABEL_COLOR)
    return _encode_png(canvas, total_w)
''',
    "api/clients_v3.py": r'''"""Chat Completions client (v3, "free" variant): python tool only, no reply cap, nothing injected, trimming at the window.

Built for vLLM with a window of about 32K tokens. What this client does (session 13: unless switched off, ALSO a thinking cap per reply
after the first + a reminder, budget reminders in the tool output, check_attempts.json, and (session 14) one review of a final answer
against the test input(s); see the config fields; the points below
describe the client with those switched off):

* one tool: python (a Jupyter session that keeps everything the model defines). No memory or summary tools, no reminders, no nudges,
  no instruction of any kind is added to the conversation while the model works;
* NO limit on the length of a reply: max_tokens is not sent, so a reply may use whatever is left of the window;
* TRIMMING: when the prompt reaches the window (minus a small margin), or a reply is cut off by the window, the oldest messages are
  dropped in one block until about trim_target_tokens (16K) remain. The first message with the puzzle is always kept. A one-line note
  (trim_note) tells the model that old messages were removed and which functions and variables exist in its Python session;
* a reply that is cut off by the window is kept as a short tail; the model simply continues from there;
* a limit on the tokens the model DECODES per task (max_generated_tokens, default 200,000): the task stops there; no answer is forced;
* a reply without tool calls is the final answer. NO forced final answer and NO token limit: the sample ends when the model gives a
  final answer or at the deadline;
* optional (verify_final, off by default): a final answer is checked on the training examples first;
* at the deadline the client can rebuild a program from the functions defined in the Python session (salvage_at_deadline);
* images are passed on untouched: every image the model asks for with show() is sent, and none is removed from the history except
  together with the messages that are trimmed away.
"""
import asyncio
import ast
import copy
import inspect
import json
import logging
import math
import os
import re
import shutil
import tempfile
import time
from dataclasses import dataclass
from typing import Optional

from jsonschema.exceptions import ValidationError
from openai import BadRequestError as OpenAIBadRequestError
from openai import InternalServerError as OpenAIInternalServerError

from arcagi2.api.clients import AsyncChatCompletionsAPIClient, RawTurnResult, TurnResult
from arcagi2.api.utils import replace_developer_message
from arcagi2.sandbox.base import REPL
from arcagi2.sandbox.exceptions import ExecutionTimeoutError
from arcagi2.tools.base import Tool
from arcagi2.tools.repl_tool import REPLTool
from arcagi2.utils import grid_image
from arcagi2.utils.utils import code_block_defines_function, get_code_matches

logger = logging.getLogger(__name__)

_MSG_OVERHEAD = 8   # rough tokens of chat-template markup per message

# ---- session 13: what the model is told by the thinking cap and the budget reminders
THINK_CAP_NOTE = ("[Note from the harness] Your reply was stopped after {k}K tokens of thinking without running any code. Stop checking by hand: "
                  "write the idea you are working on as code and run it with the python tool right now (check(f) tests a function on every training example).")
BUDGET_MID_NONE = ("[Note from the harness] You have used half of this task's token budget ({used}K of {limit}K) and have not run check() on a solution yet. "
                   "Write your current best idea as `solution` now and run check(), even if it is incomplete. If it is far off, drop that idea and try another hypothesis.")
BUDGET_MID_LOW = ("[Note from the harness] You have used half of this task's token budget ({used}K of {limit}K) and your best check() so far gets only "
                  "{ok}/{n} training examples right. Stop refining this idea in your thinking: test a different hypothesis with the python tool now.")
BUDGET_LATE = ("[Note from the harness] About {left}K tokens are left for this task. If a function passes check() on every training example, "
               "reply with the final answer now (no more visual checks). Otherwise write your best idea as `solution` and run check() right away.")
REVIEW_NOTE = ("[Note from the harness] Your final answer passes every training example. Before it is submitted, check it once more against the "
               "test input(s): the training examples may not show everything a test input contains. Look at each test input for something new: "
               "a colour, size or shape that no training input has, objects touching each other or the border, more or fewer objects, overlaps, "
               "a different layout. Run your solution on the test input(s), look at the outputs with show(), and decide whether each one is "
               "what the rule really asks for.\n"
               "- If they are right, reply with your final answer again (the same code block).\n"
               "- If a test input is an edge case that your code gets wrong, fix the code, run check() on the new version, and reply with "
               "the new code as your final answer.\n"
               "Your current answer is kept as the first attempt; a different final answer is submitted as the second attempt. "
               "Keep the review short: about {k}K tokens.")
_CHECK_LINE_RE = re.compile(r"check: (\d+)/(\d+) training examples right \| ([\d.]+)%")

# ---- recovery: what the model is told when its final answer is not usable (the failed final answer and this message are pinned into the base prompt)
RECOVERY_NO_CODE = (
    "Your reply was not accepted as a final answer: {why}\n"
    "The final answer must be ONE reply that contains a ```python code block that defines `solution(grid)`; the LAST code block of the reply is the one that is used. "
    "The program must be self-contained: all imports and helper functions inside that block (it is run in a fresh Python session where nothing from your notebook exists). "
    "Try again: you may use the python tool again, then reply with the final code block."
)
RECOVERY_CHECK_FAILED = (
    "Your final program was not accepted. It was run in a fresh Python session (only what is inside the code block exists there) on the training examples "
    "and on the test inputs, and these problems came up:\n{details}\n"
    "`solution(grid)` must return a list of lists of ints and must reproduce every training output; the program must be self-contained "
    "(all imports and helper functions inside the block). Fix the problem and reply with the final code block again."
)

def _show_setup_code(show_dir: str, scale: int) -> str:
    source = inspect.getsource(grid_image)
    return source + f"""

import os as _os, itertools as _it
_show_dir = {show_dir!r}
_show_scale = {int(scale)}
_show_counter = _it.count(1)


def show(*grids, labels=None):
    # Draw one or more grids (side by side) as an image. The image is shown to you in the next message.
    if not grids:
        raise ValueError("show() needs at least one grid")
    drawn = []
    for g in grids:
        if hasattr(g, "tolist"):
            g = g.tolist()
        drawn.append([[int(v) for v in row] for row in g])
    png = render_png(drawn, scale=_show_scale)
    n = next(_show_counter)
    path = _os.path.join(_show_dir, "%04d.png" % n)
    with open(path, "wb") as f:
        f.write(png)
    with open(path[:-4] + ".txt", "w") as f:
        f.write(" | ".join(str(x) for x in labels) if labels else ("grid" if len(drawn) == 1 else "grids side by side"))
    print("[image of %d grid(s) saved; it will be shown to you in the next message]" % len(drawn))


def show_part(grid, rows=None, cols=None, labels=None, scale=None):
    # Draw PART of one grid as an image, with the row and column numbers of the ORIGINAL grid written on its borders (0-based).
    # rows=(start, stop) and cols=(start, stop): the stop is not included; None = all of it; a single number = that one row or column.
    # Example: show_part(puzzle["train"][0]["input"], rows=(2, 8), cols=(10, 16)). The image is shown to you in the next message.
    if hasattr(grid, "tolist"):
        grid = grid.tolist()
    grid = [[int(v) for v in row] for row in grid]
    r0, r1 = _span(rows, len(grid))
    c0, c1 = _span(cols, len(grid[0]))
    png = render_part_png(grid, rows=rows, cols=cols, scale=scale or max(_show_scale, 24))
    n = next(_show_counter)
    path = _os.path.join(_show_dir, "%04d.png" % n)
    with open(path, "wb") as f:
        f.write(png)
    with open(path[:-4] + ".txt", "w") as f:
        f.write((" | ".join(str(x) for x in labels) if isinstance(labels, (list, tuple)) else str(labels)) if labels else "part of a %dx%d grid: rows %d-%d, columns %d-%d" % (len(grid), len(grid[0]), r0, r1 - 1, c0, c1 - 1))
    print("[image of rows %d-%d, columns %d-%d saved; it will be shown to you in the next message]" % (r0, r1 - 1, c0, c1 - 1))


_pivot_baseline = set(globals())
"""


# session 12: check(fn) in the Python session (only with check_helper); kept out of the list of the model's own functions
_CHECK_CODE = r"""

def check(fn=None, max_cells=12):
    # Runs fn (default: your current `solution`) on every training example and every test input and prints exactly what is right
    # and what is wrong: per training example OK, the wrong shape, or the wrong cells as (row, col) got/expected; per test input the
    # size and colours of the output. Returns True when every training example is right.
    import copy as _copy, traceback as _tb
    if fn is None:
        fn = globals().get("solution")
        if fn is None:
            print("check(): no function given and no `solution` defined yet")
            return False
    _pz = globals()["puzzle"]

    def _run(grid):
        try:
            out = fn(_copy.deepcopy(grid))
        except Exception as e:
            fr = [f for f in _tb.extract_tb(e.__traceback__)[1:] if "site-packages" not in f.filename and "/lib/python" not in f.filename]
            return None, "%s: %s%s" % (type(e).__name__, e, " (line %d, in %s)" % (fr[-1].lineno, fr[-1].name) if fr else "")
        if hasattr(out, "tolist"):
            out = out.tolist()
        if not (isinstance(out, (list, tuple)) and out and all(isinstance(r, (list, tuple)) and r for r in out)):
            return None, "the output is not a grid (a non-empty list of non-empty lists): %s" % type(out).__name__
        if len({len(r) for r in out}) != 1:
            return None, "the rows of the output have different lengths: %s" % sorted({len(r) for r in out})
        bad = [v for r in out for v in r if not isinstance(v, int) or isinstance(v, bool)]
        if bad:
            return None, "the output contains values that are not ints, e.g. %r (%s)" % (bad[0], type(bad[0]).__name__)
        return [list(r) for r in out], None

    ok = right = total = 0
    n = len(_pz["train"])
    for i, ex in enumerate(_pz["train"]):
        exp = ex["output"]
        got, err = _run(ex["input"])
        size = len(exp) * len(exp[0])
        total += size
        if err:
            print("train %d: ERROR %s" % (i, err))
            continue
        if (len(got), len(got[0])) != (len(exp), len(exp[0])):
            print("train %d: WRONG shape: got %dx%d, expected %dx%d" % (i, len(got), len(got[0]), len(exp), len(exp[0])))
            continue
        wrong = [(r, c) for r in range(len(exp)) for c in range(len(exp[0])) if got[r][c] != exp[r][c]]
        right += size - len(wrong)
        if not wrong:
            ok += 1
            print("train %d: OK" % i)
            continue
        print("train %d: WRONG %d of %d cells, (row,col) got/expected: %s%s" % (i, len(wrong), size,
              ", ".join("(%d,%d) %d/%d" % (r, c, got[r][c], exp[r][c]) for r, c in wrong[:max_cells]),
              " ..." if len(wrong) > max_cells else ""))
        if len(wrong) > max_cells:
            ws = set(wrong)
            print("  map of wrong cells (x = wrong, . = right):")
            for r in range(len(exp)):
                print("  " + "".join("x" if (r, c) in ws else "." for c in range(len(exp[0]))))
    tests, test_grids = [], []
    for j, ex in enumerate(_pz["test"]):
        got, err = _run(ex["input"])
        test_grids.append(got)
        tests.append("test %d: ERROR %s" % (j, err) if err else "test %d: %dx%d, colours %s" % (j, len(got), len(got[0]), sorted({v for r in got for v in r})))
    print("\n".join(tests))
    print("check: %d/%d training examples right | %.1f%% of training cells right" % (ok, n, 100.0 * right / max(1, total)))
    _ck_record(fn, ok, n, 100.0 * right / max(1, total), test_grids)
    return ok == n


# The harness keeps the test outputs of every function run through check(): the best ones fill the two attempts of the submission
# when the task ends (with no final answer: attempt 1 and 2; with a final answer: attempt 2 when it differs). One record per distinct output.
_CHECK_RECORDS = {}
_CK_SEQ = [0]


def _ck_record(fn, ok, n, pct, test_grids):
    import json as _json
    if all(g is None for g in test_grids):
        return
    key = _json.dumps(test_grids)
    _CK_SEQ[0] += 1
    try:                                   # session 15: the code of the function (a second pass is shown the best one)
        import inspect as _inspect
        code = _inspect.getsource(fn)[:6000]
    except Exception:
        code = None
    rec = {"ok": ok, "n": n, "pct": round(pct, 2), "seq": _CK_SEQ[0],
           "name": getattr(fn, "__name__", "?"), "tests": test_grids, "calls": 1, "code": code}
    old = _CHECK_RECORDS.get(key)
    if old is not None:
        rec["calls"] = old["calls"] + 1
        if (old["ok"], old["pct"]) > (ok, rec["pct"]):
            rec.update({k: old.get(k) for k in ("ok", "n", "pct", "name", "code")})
    _CHECK_RECORDS[key] = rec


def _ck_final_run():
    # at the end of a task: check() the current `solution` once more, silently (it may never have been checked)
    import io as _io, contextlib as _cl
    fn = globals().get("solution")
    if callable(fn):
        with _cl.redirect_stdout(_io.StringIO()):
            try:
                check(fn)
            except Exception:
                pass


def _ck_final_dump(top=6):
    # the best records first: most training examples right, then most training cells right, then the most recent
    import json as _json
    recs = sorted(_CHECK_RECORDS.values(), key=lambda r: (r["ok"] / max(1, r["n"]), r["pct"], r["seq"]), reverse=True)[:top]
    print("CHECKREC:" + _json.dumps(recs))


def _ck_check_code(src):
    # session 14: check() the `solution` of a final answer's code silently, in its own namespace (the session's `solution` is not
    # changed), so the final answer is a record too
    import io as _io, contextlib as _cl
    ns = {"__name__": "__final_answer__"}
    with _cl.redirect_stdout(_io.StringIO()):
        try:
            exec(src, ns)
            if callable(ns.get("solution")):
                check(ns["solution"])
        except Exception:
            pass


_pivot_baseline = set(globals().get("_pivot_baseline", set())) | {"check"}
"""


# session 12: grid helpers in the Python session (only with grid_helpers); kept out of the list of the model's own functions
_GRID_HELPERS_CODE = r"""

# Grid helpers, already defined in the notebook. Each takes a grid (a list of lists; a numpy array also works) and returns plain
# lists of ints, never changing the grid it was given.

def _as_grid(grid):
    if hasattr(grid, "tolist"):
        grid = grid.tolist()
    return [[int(v) for v in row] for row in grid]


def rot90(grid, k=1):
    # rotate clockwise by k * 90 degrees (k=1: 90, k=2: 180, k=3: 270 = 90 counter-clockwise)
    g = _as_grid(grid)
    for _ in range(k % 4):
        g = [list(r) for r in zip(*g[::-1])]
    return g


def flip_lr(grid):
    # mirror left <-> right
    return [row[::-1] for row in _as_grid(grid)]


def flip_ud(grid):
    # mirror top <-> bottom
    return _as_grid(grid)[::-1]


def transpose(grid):
    # rows become columns (mirror along the main diagonal)
    return [list(r) for r in zip(*_as_grid(grid))]


def dihedral(grid):
    # all 8 rotations and flips of the grid, by name
    g = _as_grid(grid)
    return {"identity": g, "rot90": rot90(g, 1), "rot180": rot90(g, 2), "rot270": rot90(g, 3),
            "flip_lr": flip_lr(g), "flip_ud": flip_ud(g), "transpose": transpose(g), "anti_transpose": rot90(transpose(g), 2)}


def symmetries_between(a, b):
    # the names of the rotations/flips that turn grid a into grid b ([] = none of them)
    b = _as_grid(b)
    return [name for name, g in dihedral(a).items() if g == b]


def _span_of(spec, n):
    if spec is None:
        return 0, n
    if isinstance(spec, int):
        return spec, spec + 1
    start, stop = spec
    return (0 if start is None else start), (n if stop is None else stop)


def crop(grid, rows=None, cols=None):
    # a part of the grid: rows=(start, stop), cols=(start, stop), 0-based, stop not included (as in show_part);
    # None = all of it; a single number = that one row or column
    g = _as_grid(grid)
    r0, r1 = _span_of(rows, len(g))
    c0, c1 = _span_of(cols, len(g[0]))
    return [row[c0:c1] for row in g[r0:r1]]


def bbox(grid, bg=0):
    # (rows, cols) of the smallest box around every cell that is not bg, as (start, stop) pairs: crop(g, *bbox(g)) cuts it out.
    # None when every cell is bg.
    g = _as_grid(grid)
    rs = [r for r, row in enumerate(g) if any(v != bg for v in row)]
    cs = [c for c in range(len(g[0])) if any(row[c] != bg for row in g)]
    if not rs:
        return None
    return (rs[0], rs[-1] + 1), (cs[0], cs[-1] + 1)


def components(grid, bg=0, diagonal=False, multicolor=False):
    # the objects of the grid: connected groups of cells that are not bg (bg=None: every colour counts, also 0).
    # diagonal=True: cells touching at a corner are connected too. multicolor=True: neighbours of different colours form one object.
    # Returns a list of dicts, sorted top to bottom, then left to right:
    #   color (the most common colour of the object), colors (all its colours), cells (list of (row, col)), size,
    #   rows and cols ((start, stop) of its box, as in crop), shape (the box cut out; cells outside the object = bg, or 0 if bg is None)
    g = _as_grid(grid)
    h, w = len(g), len(g[0])
    steps = [(-1, 0), (1, 0), (0, -1), (0, 1)] + ([(-1, -1), (-1, 1), (1, -1), (1, 1)] if diagonal else [])
    seen = [[False] * w for _ in range(h)]
    objs = []
    for r in range(h):
        for c in range(w):
            if seen[r][c] or (bg is not None and g[r][c] == bg):
                continue
            seen[r][c] = True
            stack, cells = [(r, c)], []
            while stack:
                y, x = stack.pop()
                cells.append((y, x))
                for dy, dx in steps:
                    ny, nx = y + dy, x + dx
                    if 0 <= ny < h and 0 <= nx < w and not seen[ny][nx] and (bg is None or g[ny][nx] != bg) \
                            and (multicolor or g[ny][nx] == g[y][x]):
                        seen[ny][nx] = True
                        stack.append((ny, nx))
            cells.sort()
            counts = {}
            for y, x in cells:
                counts[g[y][x]] = counts.get(g[y][x], 0) + 1
            r0, r1 = min(y for y, _ in cells), max(y for y, _ in cells) + 1
            c0, c1 = min(x for _, x in cells), max(x for _, x in cells) + 1
            fill = 0 if bg is None else bg
            shape = [[fill] * (c1 - c0) for _ in range(r1 - r0)]
            for y, x in cells:
                shape[y - r0][x - c0] = g[y][x]
            objs.append({"color": max(counts, key=lambda k: (counts[k], -k)), "colors": sorted(counts), "cells": cells,
                         "size": len(cells), "rows": (r0, r1), "cols": (c0, c1), "shape": shape})
    objs.sort(key=lambda o: (o["rows"][0], o["cols"][0]))
    return objs


_GRID_HELPERS_SRC = {'_as_grid': 'def _as_grid(grid):\n    if hasattr(grid, "tolist"):\n        grid = grid.tolist()\n    return [[int(v) for v in row] for row in grid]', 'rot90': 'def rot90(grid, k=1):\n    # rotate clockwise by k * 90 degrees (k=1: 90, k=2: 180, k=3: 270 = 90 counter-clockwise)\n    g = _as_grid(grid)\n    for _ in range(k % 4):\n        g = [list(r) for r in zip(*g[::-1])]\n    return g', 'flip_lr': 'def flip_lr(grid):\n    # mirror left <-> right\n    return [row[::-1] for row in _as_grid(grid)]', 'flip_ud': 'def flip_ud(grid):\n    # mirror top <-> bottom\n    return _as_grid(grid)[::-1]', 'transpose': 'def transpose(grid):\n    # rows become columns (mirror along the main diagonal)\n    return [list(r) for r in zip(*_as_grid(grid))]', 'dihedral': 'def dihedral(grid):\n    # all 8 rotations and flips of the grid, by name\n    g = _as_grid(grid)\n    return {"identity": g, "rot90": rot90(g, 1), "rot180": rot90(g, 2), "rot270": rot90(g, 3),\n            "flip_lr": flip_lr(g), "flip_ud": flip_ud(g), "transpose": transpose(g), "anti_transpose": rot90(transpose(g), 2)}', 'symmetries_between': 'def symmetries_between(a, b):\n    # the names of the rotations/flips that turn grid a into grid b ([] = none of them)\n    b = _as_grid(b)\n    return [name for name, g in dihedral(a).items() if g == b]', '_span_of': 'def _span_of(spec, n):\n    if spec is None:\n        return 0, n\n    if isinstance(spec, int):\n        return spec, spec + 1\n    start, stop = spec\n    return (0 if start is None else start), (n if stop is None else stop)', 'crop': 'def crop(grid, rows=None, cols=None):\n    # a part of the grid: rows=(start, stop), cols=(start, stop), 0-based, stop not included (as in show_part);\n    # None = all of it; a single number = that one row or column\n    g = _as_grid(grid)\n    r0, r1 = _span_of(rows, len(g))\n    c0, c1 = _span_of(cols, len(g[0]))\n    return [row[c0:c1] for row in g[r0:r1]]', 'bbox': 'def bbox(grid, bg=0):\n    # (rows, cols) of the smallest box around every cell that is not bg, as (start, stop) pairs: crop(g, *bbox(g)) cuts it out.\n    # None when every cell is bg.\n    g = _as_grid(grid)\n    rs = [r for r, row in enumerate(g) if any(v != bg for v in row)]\n    cs = [c for c in range(len(g[0])) if any(row[c] != bg for row in g)]\n    if not rs:\n        return None\n    return (rs[0], rs[-1] + 1), (cs[0], cs[-1] + 1)', 'components': 'def components(grid, bg=0, diagonal=False, multicolor=False):\n    # the objects of the grid: connected groups of cells that are not bg (bg=None: every colour counts, also 0).\n    # diagonal=True: cells touching at a corner are connected too. multicolor=True: neighbours of different colours form one object.\n    # Returns a list of dicts, sorted top to bottom, then left to right:\n    #   color (the most common colour of the object), colors (all its colours), cells (list of (row, col)), size,\n    #   rows and cols ((start, stop) of its box, as in crop), shape (the box cut out; cells outside the object = bg, or 0 if bg is None)\n    g = _as_grid(grid)\n    h, w = len(g), len(g[0])\n    steps = [(-1, 0), (1, 0), (0, -1), (0, 1)] + ([(-1, -1), (-1, 1), (1, -1), (1, 1)] if diagonal else [])\n    seen = [[False] * w for _ in range(h)]\n    objs = []\n    for r in range(h):\n        for c in range(w):\n            if seen[r][c] or (bg is not None and g[r][c] == bg):\n                continue\n            seen[r][c] = True\n            stack, cells = [(r, c)], []\n            while stack:\n                y, x = stack.pop()\n                cells.append((y, x))\n                for dy, dx in steps:\n                    ny, nx = y + dy, x + dx\n                    if 0 <= ny < h and 0 <= nx < w and not seen[ny][nx] and (bg is None or g[ny][nx] != bg) \\\n                            and (multicolor or g[ny][nx] == g[y][x]):\n                        seen[ny][nx] = True\n                        stack.append((ny, nx))\n            cells.sort()\n            counts = {}\n            for y, x in cells:\n                counts[g[y][x]] = counts.get(g[y][x], 0) + 1\n            r0, r1 = min(y for y, _ in cells), max(y for y, _ in cells) + 1\n            c0, c1 = min(x for _, x in cells), max(x for _, x in cells) + 1\n            fill = 0 if bg is None else bg\n            shape = [[fill] * (c1 - c0) for _ in range(r1 - r0)]\n            for y, x in cells:\n                shape[y - r0][x - c0] = g[y][x]\n            objs.append({"color": max(counts, key=lambda k: (counts[k], -k)), "colors": sorted(counts), "cells": cells,\n                         "size": len(cells), "rows": (r0, r1), "cols": (c0, c1), "shape": shape})\n    objs.sort(key=lambda o: (o["rows"][0], o["cols"][0]))\n    return objs'}


_GRID_DEPS = {"dihedral": ["rot90", "flip_lr", "flip_ud", "transpose"], "symmetries_between": ["dihedral"], "crop": ["_span_of"]}


def helper_source(*names):
    # prints the code of grid helpers (default: all of them) so it can be copied into a final answer, which must be self-contained.
    # Example: helper_source("components", "crop")
    unknown = [n for n in names if n not in _GRID_HELPERS_SRC or n.startswith("_")]
    if unknown:
        print("unknown helper(s):", unknown, "| known:", [n for n in _GRID_HELPERS_SRC if not n.startswith("_")])
        return
    order, todo = [], ["_as_grid"] + list(names or [n for n in _GRID_HELPERS_SRC if not n.startswith("_")])
    while todo:
        n = todo.pop(0)
        if n not in order:
            order.append(n)
            todo += _GRID_DEPS.get(n, [])
    print("\n\n".join(_GRID_HELPERS_SRC[n] for n in sorted(order, key=list(_GRID_HELPERS_SRC).index)))


_pivot_baseline = set(globals().get("_pivot_baseline", set())) | set(['bbox', 'components', 'crop', 'dihedral', 'flip_lr', 'flip_ud', 'helper_source', 'rot90', 'symmetries_between', 'transpose'])
"""


_DESCRIBE_CODE = """
import inspect as _i, types as _t
_f, _v = [], []
for _n, _o in list(globals().items()):
    if _n.startswith("_") or _n in _pivot_baseline or isinstance(_o, _t.ModuleType):
        continue
    if isinstance(_o, _t.FunctionType):
        try:
            _f.append(_n + str(_i.signature(_o)))
        except Exception:
            _f.append(_n + "(...)")
    else:
        _v.append(_n)
print("FUNCS:" + "; ".join(_f)[:700])
print("VARS:" + ", ".join(_v)[:400])
"""


_SALVAGE_CODE = """
import inspect as _i, types as _t, json as _j
_srcs = []
for _n, _o in list(globals().items()):
    if _n.startswith("_") or _n in _pivot_baseline or not isinstance(_o, _t.FunctionType):
        continue
    try:
        _srcs.append(_i.getsource(_o))
    except Exception:
        pass
print("SALVAGE:" + _j.dumps(_srcs))
"""

# Runs a final answer on the training examples (and a crash check on the test inputs) inside the live Python session,
# in a fresh namespace, the way pivotools does it afterwards. Prints one line: VERIFY:<json>.
_VERIFY_CODE = """
import json as _j, copy as _cp, numbers as _nb, traceback as _tb

def _pv_verify(_code):
    # pivotools runs the final code in a fresh session where `puzzle` (train and test inputs, no test outputs) is defined and nothing else of the notebook exists
    _ns = {"__name__": "pv_verify", "puzzle": _cp.deepcopy(puzzle)}
    _lines = _code.splitlines()

    def _where(_e):
        try:
            _fr = [_f for _f in _tb.extract_tb(_e.__traceback__) if _f.filename == "<final answer>"]
            if _fr:
                _n = _fr[-1].lineno
                return " (line %d of your code: %s)" % (_n, _lines[_n - 1].strip()[:110])
        except Exception:
            pass
        return ""

    try:
        exec(compile(_code, "<final answer>", "exec"), _ns)
        _fn = _ns["solution"]
    except SyntaxError as _e:
        return {"fatal": "the code could not be loaded: %s: %s (line %s: %s)" % (type(_e).__name__, _e.msg, _e.lineno, (_e.text or "").strip()[:110])}
    except Exception as _e:
        return {"fatal": "the code could not be loaded: %s: %s%s" % (type(_e).__name__, str(_e)[:200], _where(_e))}

    def _norm(_o):
        # the same rule as pivotools: a list of lists (or an array) of integers; floats and other types are an error
        if hasattr(_o, "tolist"):
            _o = _o.tolist()
        _g = []
        for _ri, _r in enumerate(_o):
            _row = []
            for _ci, _v in enumerate(_r):
                if not isinstance(_v, _nb.Integral):
                    raise TypeError("Non-integer value %r at position (%d, %d)" % (_v, _ri, _ci))
                _row.append(int(_v))
            _g.append(_row)
        return _g

    def _shape(_g):
        return "%dx%d" % (len(_g), len(_g[0]) if _g else 0)

    _res = {"train": [], "test": []}
    for _ex in puzzle["train"]:
        try:
            _out = _norm(_fn(_cp.deepcopy(_ex["input"])))
            _exp = _ex["output"]
            if _out == _exp:
                _res["train"].append("ok")
            elif _shape(_out) != _shape(_exp):
                _res["train"].append("wrong output shape: expected %s, got %s" % (_shape(_exp), _shape(_out)))
            else:
                _bad = sum(1 for _a, _b in zip(_out, _exp) for _x, _y in zip(_a, _b) if _x != _y)
                _res["train"].append("wrong output: %d of %d cells differ (shape %s)" % (_bad, len(_exp) * len(_exp[0]), _shape(_exp)))
        except Exception as _e:
            _res["train"].append("raised %s: %s%s" % (type(_e).__name__, str(_e)[:150], _where(_e)))
    for _t in puzzle["test"]:
        try:
            _norm(_fn(_cp.deepcopy(_t["input"])))
            _res["test"].append("ok")
        except Exception as _e:
            _res["test"].append("raised %s: %s%s" % (type(_e).__name__, str(_e)[:150], _where(_e)))
    return _res

print("VERIFY:" + _j.dumps(_pv_verify(__CODE__)))
"""

_IMPORT_LINE_RE = re.compile(r"^(?:import\s+[\w\.]+(?:\s+as\s+\w+)?(?:\s*,\s*[\w\.]+(?:\s+as\s+\w+)?)*|from\s+[\w\.]+\s+import\s+[\w\*\s,\(\)]+)\s*$")
_DEFAULT_IMPORTS = [
    "import numpy as np", "import copy", "import math", "import itertools", "import collections",
    "from collections import Counter, deque, defaultdict",
]


@dataclass
class _Entry:
    msg: dict
    kind: str                 # assistant | tool | user | images | note
    tokens: int
    exact: bool = False       # tokens come from the server's usage numbers
    chars: int = 0            # text characters, used for calibration
    img_tokens: int = 0       # vision tokens inside this message


class _State:
    def __init__(self) -> None:
        self.generated = 0
        self.requests = 0
        self.cutoffs = 0
        self.window_hit = False         # a reply was cut by the window: trim at the next request
        self.token_limit = False        # the task stopped because it decoded max_generated_tokens
        self.cut_streak = 0
        self.trims = 0
        self.dropped_msgs = 0
        self.fmt_reminders = 0
        self.format_pending = False
        self.show_images = 0
        self.action_requests = 0
        self.salvaged = 0
        self.final_rejects = 0           # = the number of recoveries
        self.recoveries = 0
        self.recovery_kinds: list = []
        self.rebase = False              # the pinned head changed: take its size from the next request
        self.deadline = False
        self.server_errors = 0
        self.replies = 0                 # session 13: model replies received (the thinking cap starts after the first)
        self.think_cuts = 0
        self.best_check: Optional[tuple] = None   # (examples right, examples, % cells) of the best check() output seen
        self.nudged_mid = False
        self.nudged_late = False
        self.check_cands = -1            # records written to check_attempts.json (-1 = not written)
        self.check_seen = False          # session 14: a check() line in the tool output of this reply (write check_attempts.json)
        self.review_from: Optional[int] = None   # session 14: decoded tokens when the final review started
        self.review_response = None      # session 14: the response with the first final answer (what is submitted)
        self.review = "-"                # session 14: - (none) | kept | changed | no_code | budget | deadline
        self.imports: list = []
        self.base_tokens: Optional[int] = None
        self.k_text = 2.5
        self.seen_images: set = set()


def _text_of(msg: dict) -> str:
    content = msg.get("content")
    if isinstance(content, str):
        text = content
    elif isinstance(content, list):
        text = " ".join(p.get("text", "") for p in content if isinstance(p, dict) and p.get("type") == "text")
    else:
        text = ""
    extra = msg.get("reasoning_content") or msg.get("reasoning") or ""
    for c in msg.get("tool_calls") or []:
        try:
            text += c["function"]["arguments"]
        except Exception:
            pass
    return text + (extra if isinstance(extra, str) else "")


def _image_tokens_in(msg: dict) -> int:
    total = 0
    content = msg.get("content")
    if isinstance(content, list):
        for p in content:
            if isinstance(p, dict) and p.get("type") == "image_url":
                total += grid_image.image_tokens_from_data_url(str((p.get("image_url") or {}).get("url", "")))
    return total


def _final_code(content: Optional[str]) -> Optional[str]:
    """The code of a final answer: the LAST code block of the reply (as pivotools reads it), if it defines solution()."""
    if not content:
        return None
    matches = get_code_matches(content)
    if not matches:
        return None
    code = matches[-1].group(1)
    try:
        return code if code_block_defines_function(code, "solution") else None
    except Exception:
        return None


def _reasoning_of(message) -> str:
    return getattr(message, "reasoning_content", None) or getattr(message, "reasoning", None) or ""


def _is_context_error(exc: Exception) -> bool:
    text = str(exc).lower()
    return any(k in text for k in ("context length", "too long", "maximum context", "max_model_len",
                                   "available context size", "exceeds the", "n_ctx"))


_IMAGE_URL_RE = re.compile(r"data:image/[a-z]+;base64,[A-Za-z0-9+/=]+")


class _Transcript:
    """transcript.md + images/ in the task's own folder (the folder of its run.log), appended after every model call."""

    def __init__(self, folder):
        self.folder = folder
        self.path = os.path.join(folder, "transcript.md") if folder else None
        self.n_img = 0
        self.trims = 0
        self.t0 = time.time()

    @staticmethod
    def _task_folder():
        try:
            from arcagi2.utils.logging_utils import task_id_var
            tid = task_id_var.get()
        except Exception:
            return None
        for h in logging.getLogger().handlers:
            if isinstance(h, logging.FileHandler) and os.path.basename(h.baseFilename) == "run.log" \
                    and any(getattr(f, "task_id", object()) == tid for f in h.filters):
                return os.path.dirname(h.baseFilename)
        return None

    @classmethod
    def open(cls, head: list, model: str) -> "_Transcript":
        tr = cls(cls._task_folder())
        if tr.path is None:
            logger.warning("TRANSCRIPT: the task folder was not found (no run.log of this task): no transcript.md for this task")
            return tr
        tr._w(f"# Transcript ({model})\n\nStarted {time.strftime('%Y-%m-%d %H:%M:%S')}. One section per model call; images are in images/.\n\n")
        for m in head:
            tr._w(f"## {m.get('role', '?')} message (pinned)\n\n")
            tr._content(m.get("content"), "prompt")
        return tr

    def _w(self, text: str) -> None:
        if self.path is None:
            return
        try:
            with open(self.path, "a", encoding="utf-8") as f:
                f.write(text)
        except OSError as e:
            logger.warning(f"TRANSCRIPT: could not write {self.path}: {e}")
            self.path = None

    def _save_png(self, png: bytes, stem: str) -> str:
        self.n_img += 1
        name = f"{self.n_img:04d}_{stem}.png"
        try:
            os.makedirs(os.path.join(self.folder, "images"), exist_ok=True)
            with open(os.path.join(self.folder, "images", name), "wb") as f:
                f.write(png)
        except OSError as e:
            logger.warning(f"TRANSCRIPT: could not save image {name}: {e}")
        return f"images/{name}"

    def _content(self, content, stem: str) -> None:
        if isinstance(content, list):
            for p in content:
                if p.get("type") == "text":
                    self._w(p.get("text", "") + "\n\n")
                elif p.get("type") == "image_url":
                    url = (p.get("image_url") or {}).get("url", "")
                    try:
                        import base64 as _b64
                        rel = self._save_png(_b64.b64decode(url.split(",", 1)[1]), stem)
                    except Exception:
                        rel = "(image could not be decoded)"
                    self._w(f"![image]({rel})\n\n")
        elif content:
            self._w(str(content) + "\n\n")

    def reply(self, st, message, prompt_tokens: int, completion_tokens: int, finish) -> None:
        if self.path is None:
            return
        if st.trims != self.trims:
            self._w(f"### Event: the conversation was trimmed (trim #{st.trims}; {st.dropped_msgs} messages dropped so far)\n\n")
            self.trims = st.trims
        self._w(f"---\n\n## Model call {st.requests} (minute {(time.time() - self.t0) / 60:.1f}; prompt {prompt_tokens} tokens; "
                f"decoded {completion_tokens}, {st.generated} in total; finish: {finish})\n\n")
        think = _reasoning_of(message)
        if think:
            self._w(f"### Thinking\n\n{think.strip()}\n\n")
        if message.content:
            self._w(f"### Reply\n\n{message.content.strip()}\n\n")

    def tool(self, name: str, code, raw_args, output: str) -> None:
        if self.path is None:
            return
        body = code if code is not None else raw_args
        self._w(f"### Tool call: {name}\n\n```python\n{str(body).rstrip()}\n```\n\n### Tool output\n\n```\n{str(output).rstrip()}\n```\n\n")

    def images(self, items: list) -> None:
        for label, png in items:
            rel = self._save_png(png, "show")
            self._w(f"![{label}]({rel})\n\n")

    def event(self, text: str) -> None:
        self._w(f"### Event: {text}\n\n")

    def end(self, st, response) -> None:
        msg = response.choices[0].message.content if response is not None and response.choices else None
        how = ("time was up" if st.deadline else "token limit reached" if st.token_limit
               else "final answer given" if msg and _final_code(msg) else "ended")
        self._w(f"---\n\n## End: {how} (minute {(time.time() - self.t0) / 60:.1f}; {st.requests} model calls; {st.generated} tokens decoded; "
                f"{st.recoveries} recoveries; {st.trims} trims; {st.show_images} images from show())\n")


class AsyncChatCompletionsAPIClientV3(AsyncChatCompletionsAPIClient):

    @dataclass
    class ChatCompletionsV3CallConfig(AsyncChatCompletionsAPIClient.ChatCompletionsAPICallConfig):
        context_window: int = 32768          # must equal the server's max_model_len
        trim_target_tokens: int = 16384      # when the window is reached, old messages are dropped until about this many tokens remain
        safety_margin: int = 1024            # the window counts as reached this many tokens before its end
        trim_note: bool = True               # one line after a trim: old messages were removed + what exists in the Python session
        max_generated_tokens: int = 200000   # a task stops when the model has decoded this many tokens in total (0 = no limit). No answer is forced.
        recovery_max: int = 5                # at most this many recoveries per task; after that the next reply is accepted as it is
        recovery_final_chars: int = 12000    # how much of the failed final answer (its text, not its thinking) is kept in the base prompt
        verify_final: bool = False           # no longer used (the recovery above replaces it); kept so that an older config file can still be imported
        verify_timeout: int = 90
        deadline_epoch: float = 0.0          # unix time at which the sample must stop (0 = no deadline)
        sample_seconds: float = 0.0          # longest wall time of ONE sample, counted from its start (0 = no cap): its fair share
        salvage_at_deadline: bool = True     # at the deadline, rebuild the answer from the Python session
        reasoning_field: str = "reasoning_content"   # llama-server's name for the thinking of an assistant message
        tool_output_max_chars: int = 4500
        cutoff_tail_chars: int = 4000        # how much of a reply that was cut off by the window is kept
        enable_show: bool = True
        image_scale: int = 16
        first_message_images: bool = True
        compact_grids: bool = True
        check_helper: bool = False           # session 12: check(fn) in the Python session (ARC_CHECK_HELPER)
        grid_helpers: bool = False           # session 12: rot90, flip_lr, ..., crop, bbox, components in the Python session (ARC_GRID_HELPERS)
        check_attempts: bool = False         # session 13: write the best check() records to check_attempts.json at the end of a task (ARC_CHECK_ATTEMPTS)
        think_cap_tokens: int = 0            # session 13: every reply after the first stops after this many tokens + a reminder (ARC_THINK_CAP; 0 = off)
        budget_reminders: bool = False       # session 13: reminders at half the token budget and near its end (ARC_BUDGET_REMINDERS)
        budget_late_left: int = 15000        # session 13: the late reminder comes when this many decoded tokens are left (ARC_BUDGET_LATE_LEFT)
        final_review: bool = False           # session 14: one review of a final answer that passes training, against the test input(s) (ARC_FINAL_REVIEW)
        review_max_tokens: int = 20000       # session 14: the review stops after this many decoded tokens (ARC_REVIEW_MAX_TOKENS)
        review_min_left: int = 8000          # session 14: no review when fewer decoded tokens than this are left in the task's budget (ARC_REVIEW_MIN_LEFT)
        rerun_notes: str = ""                # session 15: second pass: JSON file {puzzle id: note added to the first user message} (ARC_RERUN_NOTES)
        run_limits: str = ""                 # session 17: JSON file {puzzle id: decoded-token limit of that run, 0 = none} (ARC_RUN_LIMITS)

    # ------------------------------------------------------------------ entry point
    @classmethod
    async def call(cls, config, messages: list[dict], initial_code: Optional[list[str]] = None) -> TurnResult:
        if not isinstance(config, cls.ChatCompletionsV3CallConfig):
            raise ValueError(f"Config must be an instance of {cls.ChatCompletionsV3CallConfig}")
        messages = replace_developer_message(messages=messages, system_prompt=config.system_prompt)
        async with cls(api_provider=config.api_provider, **config.client_kwargs) as client:
            client._v3 = config
            client._t_start = time.time()
            client._show_dir = tempfile.mkdtemp(prefix="pivotools_show_")
            code = list(initial_code or [])
            if config.enable_show:
                code.append(_show_setup_code(client._show_dir, config.image_scale))
            if config.check_helper:
                code.append(_CHECK_CODE)
            if config.grid_helpers:
                code.append(_GRID_HELPERS_CODE)
            try:
                result = await client.ensure_repl_and_run_tool_loop(
                    tools=config.tools,
                    sandbox_cls=config.sandbox_cls,
                    sandbox_kwargs=config.sandbox_kwargs,
                    initial_code=code,
                    initial_code_timeout=config.initial_code_timeout,
                    model=config.model,
                    messages=messages,
                    max_retries=config.max_retries,
                    sleep=config.sleep,
                    **config.request_kwargs,
                )
            finally:
                shutil.rmtree(client._show_dir, ignore_errors=True)
            message = result.response.choices[0].message
            return TurnResult(role=message.role, content=message.content, trace=result.trace)

    # ------------------------------------------------------------------ trace helpers
    async def create_chat_completion(self, model: str, messages: list, **kwargs) -> RawTurnResult:
        result = await super().create_chat_completion(model=model, messages=messages, **kwargs)
        # keep image payloads out of the saved trace: they are repeated in every request and would make it huge
        result.trace[0] = _IMAGE_URL_RE.sub("data:image/png;base64,<omitted>", result.trace[0])
        return result

    @staticmethod
    def get_readable_trace(trace, include_interpreter_output=True, separator="\n\n", include_prompt=False,
                           code_interpreter_tool_name=None):
        trace = list(trace)
        try:
            first = json.loads(trace[0])
            for message in first["messages"]:
                if isinstance(message.get("content"), list):
                    message["content"] = "\n".join(
                        p.get("text", "[image]") if p.get("type") == "text" else "[image]" for p in message["content"]
                    )
            trace[0] = json.dumps(first)
        except Exception:
            pass
        return AsyncChatCompletionsAPIClient.get_readable_trace(
            trace, include_interpreter_output=include_interpreter_output, separator=separator,
            include_prompt=include_prompt, code_interpreter_tool_name=code_interpreter_tool_name,
        )

    # ------------------------------------------------------------------ small helpers
    def _remaining(self) -> Optional[float]:
        """Seconds until this sample must stop: the earlier of the global deadline and its own time share."""
        ends = []
        if self._v3.deadline_epoch:
            ends.append(self._v3.deadline_epoch)
        if self._v3.sample_seconds:
            ends.append(self._t_start + self._v3.sample_seconds)
        return (min(ends) - time.time()) if ends else None

    def _cap_tool_text(self, text: str) -> str:
        limit = self._v3.tool_output_max_chars
        if len(text) <= limit:
            return text
        tail = limit // 3
        head = limit - tail
        return f"{text[:head]}\n... [{len(text) - limit} characters cut from the middle] ...\n{text[-tail:]}"

    def _user_entry(self, st: _State, content, kind: str = "user") -> _Entry:
        msg = {"role": "user", "content": content}
        chars = len(_text_of(msg))
        img = _image_tokens_in(msg)
        return _Entry(msg=msg, kind=kind, tokens=math.ceil(chars / st.k_text) + img + _MSG_OVERHEAD, chars=chars, img_tokens=img)

    def _assistant_entry(self, st: _State, message, completion_tokens: int) -> _Entry:
        msg = json.loads(message.model_dump_json())
        return _Entry(msg=msg, kind="assistant", tokens=completion_tokens + _MSG_OVERHEAD, exact=True)

    def _kept_assistant_entry(self, st: _State, message, marker: str = "") -> _Entry:
        """A reply that was cut off or had no usable action, kept as a short plain-text tail (no tool calls)."""
        cfg = self._v3
        content = message.content or ""
        tail = _reasoning_of(message)[-cfg.cutoff_tail_chars:]
        kept = {"role": "assistant", "content": content[-1500:] if content else ""}
        if tail or marker:
            kept[cfg.reasoning_field] = (marker + tail) if tail else marker.strip()
        chars = len(_text_of(kept))
        return _Entry(msg=kept, kind="assistant", tokens=math.ceil(chars / st.k_text) + _MSG_OVERHEAD, chars=chars)

    def _collect_show_images(self, st: _State) -> list:
        found = []
        try:
            names = sorted(n for n in os.listdir(self._show_dir) if n.endswith(".png"))
        except OSError:
            return found
        for name in names:
            if name in st.seen_images:
                continue
            st.seen_images.add(name)
            path = os.path.join(self._show_dir, name)
            try:
                with open(path, "rb") as f:
                    png = f.read()
                label = ""
                if os.path.exists(path[:-4] + ".txt"):
                    with open(path[:-4] + ".txt") as f:
                        label = f.read().strip()
                found.append((label, png))
            except OSError:
                continue
        return found

    def _images_entry(self, st: _State, items: list) -> _Entry:
        parts = [{"type": "text", "text": "Image(s) from your show() call(s):"}]
        for label, png in items:
            parts.append({"type": "text", "text": f"[{label}]"})
            parts.append({"type": "image_url", "image_url": {"url": grid_image.data_url(png)}})
        st.show_images += len(items)
        return self._user_entry(st, parts, kind="images")

    # ------------------------------------------------------------------ token accounting and trimming
    def _usage_numbers(self, response, req_messages: list) -> tuple:
        """(prompt_tokens, completion_tokens) from the server; estimated (3 characters per token) if the server sends none."""
        u = getattr(response, "usage", None)
        p, c = getattr(u, "prompt_tokens", None), getattr(u, "completion_tokens", None)
        if p is not None and c is not None:
            return int(p), int(c)
        if not getattr(self, "_usage_warned", False):
            self._usage_warned = True
            logger.warning("V3_EVENT the server sent no usage numbers: token counts are estimated")
        try:
            reply = json.loads(response.choices[0].message.model_dump_json())
        except Exception:
            reply = {}
        completion = max(1, math.ceil(len(_text_of(reply)) / 3))
        prompt = 1200 + sum(math.ceil(len(_text_of(m)) / 3) + _MSG_OVERHEAD + _image_tokens_in(m) for m in req_messages)
        return prompt, completion

    def _estimate(self, entries: list, st: _State) -> int:
        return (st.base_tokens or 0) + sum(e.tokens for e in entries)

    def _calibrate(self, entries: list, st: _State, prompt_tokens: int, had_entries: bool) -> None:
        if st.rebase and not had_entries:
            st.base_tokens = prompt_tokens     # the first request after a recovery holds only the pinned head: its exact size
            st.rebase = False
            return
        if st.base_tokens is None:
            if not had_entries:
                st.base_tokens = prompt_tokens     # the first request holds only the system prompt, tools and puzzle
            return
        exact = st.base_tokens + sum(e.tokens for e in entries if e.exact)
        guessed = [e for e in entries if not e.exact]
        chars = sum(e.chars for e in guessed)
        images = sum(e.img_tokens for e in guessed)
        residual = prompt_tokens - exact - images - _MSG_OVERHEAD * len(guessed)
        if chars > 400 and residual > 50:
            st.k_text = min(4.0, max(1.0, 0.5 * st.k_text + 0.5 * (chars / residual)))
            for e in guessed:
                e.tokens = math.ceil(e.chars / st.k_text) + e.img_tokens + _MSG_OVERHEAD

    async def _describe_repl(self, repl: Optional[REPL]) -> str:
        if repl is None or not self._v3.enable_show:
            return ""
        try:
            out = await repl.execute(_DESCRIBE_CODE, timeout=15)
            text = (out.text or "") if out is not None else ""
        except Exception:
            return ""
        funcs = vars_ = ""
        for line in text.splitlines():
            if line.startswith("FUNCS:"):
                funcs = line[6:].strip()
            elif line.startswith("VARS:"):
                vars_ = line[5:].strip()
        bits = []
        if funcs:
            bits.append(f"functions: {funcs}")
        if vars_:
            bits.append(f"variables: {vars_}")
        return " | ".join(bits)

    async def _maybe_trim(self, entries: list, st: _State, repl, level: int = 0, force: bool = False) -> bool:
        """When the prompt reaches the window (minus the margin), or force=True (a reply was cut by the window), drop the oldest
        messages in one block until about trim_target_tokens remain."""
        if st.base_tokens is None or not entries:
            return False
        cfg = self._v3
        trigger = cfg.context_window - cfg.safety_margin
        if level:
            trigger = int(trigger * (1 - 0.1 * level))
        before = self._estimate(entries, st)
        if not force and before <= trigger:
            return False
        target = max(cfg.trim_target_tokens, (st.base_tokens or 0) + 2500)
        if level:
            target = max((st.base_tokens or 0) + 1500, int(target * (1 - 0.25 * level)))
        old_note = entries.pop(0) if entries and entries[0].kind == "note" else None
        dropped = 0
        while self._estimate(entries, st) > target:
            nxt = next((i for i in range(1, len(entries)) if entries[i].kind == "assistant"), None)
            if nxt is None:
                break                              # only the latest exchange is left: always keep it
            del entries[:nxt]
            dropped += nxt
        if dropped:
            if cfg.trim_note:
                desc = await self._describe_repl(repl)
                text = "[Context note] Earlier steps of this conversation were removed to save space. Your Python session is unchanged."
                if desc:
                    text += f" Currently defined in it: {desc}."
                entries.insert(0, self._user_entry(st, text, kind="note"))
            st.trims += 1
            st.dropped_msgs += dropped
            logger.info(f"V3_EVENT trim: dropped {dropped} messages, estimate {before} -> {self._estimate(entries, st)} "
                        f"(trigger {trigger}, target {target}{', forced by the window' if force else ''})")
        elif old_note is not None:
            entries.insert(0, old_note)             # nothing could be dropped: keep the existing note
        return dropped > 0

    # ------------------------------------------------------------------ final answer check and deadline rescue
    async def _verify_final(self, repl: Optional[REPL], code: str) -> Optional[tuple]:
        """(ok, details) from running the final code on the training examples; None if the check itself is unavailable."""
        if repl is None:
            return None
        try:
            out = await repl.execute(_VERIFY_CODE.replace("__CODE__", repr(code)), timeout=self._v3.verify_timeout)
            text = (out.text or "") if out is not None else ""
        except ExecutionTimeoutError:
            return False, f"- the code did not finish within {self._v3.verify_timeout} seconds on the training examples (too slow or an endless loop)"
        except Exception as e:
            logger.info(f"V3_EVENT final check could not run: {type(e).__name__}: {str(e)[:200]}")
            return None
        data = None
        for line in text.splitlines():
            if line.startswith("VERIFY:"):
                try:
                    data = json.loads(line[len("VERIFY:"):])
                except Exception:
                    data = None
        if data is None:
            return None
        if data.get("fatal"):
            return False, f"- {data['fatal']}"
        problems = [f"- training example {i}: {r}" for i, r in enumerate(data.get("train", [])) if r != "ok"]
        problems += [f"- test input {i}: {r}" for i, r in enumerate(data.get("test", [])) if r != "ok"]
        return (not problems), "\n".join(problems[:8])

    async def _final_problem(self, repl: Optional[REPL], message, final_code: Optional[str]) -> Optional[tuple]:
        """Why a reply without tool calls cannot be used as the final answer: (kind, text for the model), or None if it is fine (or cannot be judged)."""
        content = message.content or ""
        if final_code is None:
            if not content.strip():
                return "empty", RECOVERY_NO_CODE.format(why="it was empty: it had no text at all, only thinking.")
            matches = get_code_matches(content)
            if not matches:
                return "no_code", RECOVERY_NO_CODE.format(why="it contains no ```python code block.")
            code = matches[-1].group(1)
            try:
                ast.parse(code)
            except SyntaxError as e:
                line = (e.text or "").strip()[:110]
                return "syntax", RECOVERY_NO_CODE.format(why=f"its last ```python code block has a syntax error: {e.msg} (line {e.lineno}: {line}).")
            except Exception as e:
                return "syntax", RECOVERY_NO_CODE.format(why=f"its last ```python code block could not be parsed: {type(e).__name__}: {str(e)[:150]}.")
            return "no_solution", RECOVERY_NO_CODE.format(why="its last ```python code block does not define a function named `solution`.")
        verdict = await self._verify_final(repl, final_code)
        if verdict is None or verdict[0]:
            return None
        return "check_failed", RECOVERY_CHECK_FAILED.format(details=verdict[1])

    def _recovery_pair(self, message, feedback: str) -> list:
        """The failed final answer (its text only, not the thinking) and the reason it failed: they are pinned into the base prompt."""
        content = (message.content or "").strip()
        limit = self._v3.recovery_final_chars
        if len(content) > limit:
            content = "[...]\n" + content[-limit:]
        return [{"role": "assistant", "content": content or "(empty reply)"}, {"role": "user", "content": feedback}]

    async def _write_check_attempts(self, repl, st: _State, tr, final: bool = True) -> None:
        """session 13: check() the current `solution` once more (silently), then write the best check() records to check_attempts.json.
        session 14: also called with final=False after every reply with a check() line (no extra check(), quiet log)."""
        if repl is None:
            return
        if final:
            try:
                await repl.execute("_ck_final_run()", timeout=60)
            except Exception as e:
                logger.info(f"V3_EVENT check attempts: the last check() of `solution` did not finish: {str(e)[:200]}")
        recs = None
        try:
            out = await repl.execute("_ck_final_dump()", timeout=30)
            for line in ((out.text or "") if out is not None else "").splitlines():
                if line.startswith("CHECKREC:"):
                    recs = json.loads(line[len("CHECKREC:"):])
        except Exception as e:
            logger.info(f"V3_EVENT check attempts: could not read the records: {str(e)[:200]}")
        if recs is None:
            logger.info("V3_EVENT check attempts: no records could be read from the Python session")
            return
        st.check_cands = len(recs)
        best = f"{recs[0]['ok']}/{recs[0]['n']} training examples, {recs[0]['pct']}% cells" if recs else "-"
        if final:
            logger.info(f"V3_EVENT check attempts: {len(recs)} records (best {best})")
        if tr.folder is None:
            logger.warning("V3_EVENT check attempts: no task folder, check_attempts.json not written")
            return
        try:
            with open(os.path.join(tr.folder, "check_attempts.json"), "w") as f:
                json.dump({"token_limit": st.token_limit, "deadline": st.deadline, "records": recs}, f)
        except OSError as e:
            logger.warning(f"V3_EVENT check attempts: could not write check_attempts.json: {e}")

    def _run_limit(self) -> int:
        """session 17: this task's decoded-token limit: from the ARC_RUN_LIMITS file (0 = none), else max_generated_tokens."""
        path = self._v3.run_limits
        if not path:
            return self._v3.max_generated_tokens
        limits = json.load(open(path))
        folder = _Transcript._task_folder()
        pid = None
        for d in ([os.path.dirname(folder), folder] if folder else []):
            try:
                pid = json.load(open(os.path.join(d, "metadata.json")))["puzzle_id"]
                break
            except Exception:
                continue
        if not pid or pid not in limits:
            raise RuntimeError(f"run limits: no limit for puzzle {pid!r} (task folder {folder}) in {path}")
        limit = int(limits[pid])
        logger.info(f"V3_EVENT run limit: {limit or 'none (deadline only)'} decoded tokens for {pid}")
        return limit

    def _with_rerun_note(self, messages: list) -> list:
        """session 15: in a second pass, the note of this puzzle (ARC_RERUN_NOTES file) is added to the end of the first user message."""
        path = self._v3.rerun_notes
        if not path:
            return messages
        notes = json.load(open(path))
        folder = _Transcript._task_folder()
        pid = None
        for d in ([os.path.dirname(folder), folder] if folder else []):
            try:
                pid = json.load(open(os.path.join(d, "metadata.json")))["puzzle_id"]
                break
            except Exception:
                continue
        if pid and pid in notes and notes[pid] is None:     # session 16: round 1 of the stage manager: explicitly no note
            return messages
        if not pid or not notes.get(pid):
            raise RuntimeError(f"second pass: no note for puzzle {pid!r} (task folder {folder}) in {path}")
        note = notes[pid]
        out = copy.deepcopy(messages)
        i = next(k for k, m in enumerate(out) if m.get("role") == "user")
        if isinstance(out[i]["content"], list):
            out[i]["content"].append({"type": "text", "text": "\n\n" + note})
        else:
            out[i]["content"] = f"{out[i]['content']}\n\n{note}"
        logger.info(f"V3_EVENT second pass: note added for {pid} ({len(note)} characters)")
        return out

    async def _record_final(self, repl, code: str) -> None:
        """session 14: check() the `solution` of a final answer's code silently (own namespace), so it becomes a check() record."""
        if repl is None or not code:
            return
        try:
            await repl.execute(f"_ck_check_code({code!r})", timeout=60)
        except Exception as e:
            logger.info(f"V3_EVENT final review: could not check() a final answer's code: {str(e)[:200]}")

    def _budget_note(self, st: _State) -> Optional[str]:
        """session 13: the reminder to add to this tool output, if a budget point was just passed (each fires once)."""
        cfg = self._v3
        lim = st.gen_limit                       # session 17: per-run limit
        if not cfg.budget_reminders or not lim:
            return None
        if not st.nudged_late and st.generated >= lim - cfg.budget_late_left:
            st.nudged_late = st.nudged_mid = True
            return BUDGET_LATE.format(left=max(1, (lim - st.generated) // 1000))
        if not st.nudged_mid and st.generated >= lim // 2:
            st.nudged_mid = True
            if st.best_check is None:
                return BUDGET_MID_NONE.format(used=st.generated // 1000, limit=lim // 1000)
            ok, n, _pct = st.best_check
            if 2 * ok < n:
                return BUDGET_MID_LOW.format(used=st.generated // 1000, limit=lim // 1000, ok=ok, n=n)
        return None

    async def _salvage_solution(self, repl, st: _State) -> Optional[str]:
        """Rebuild a self-contained program from the functions defined in the live Python session."""
        if repl is None:
            return None
        try:
            out = await repl.execute(_SALVAGE_CODE, timeout=20)
            text = (out.text or "") if out is not None else ""
        except Exception as e:
            logger.info(f"V3_EVENT salvage failed to read the session: {e}")
            return None
        sources = []
        for line in text.splitlines():
            if line.startswith("SALVAGE:"):
                try:
                    sources = json.loads(line[len("SALVAGE:"):])
                except Exception:
                    return None
        if not sources:
            return None
        header = list(dict.fromkeys(_DEFAULT_IMPORTS + st.imports))
        code = "\n".join(header) + "\n\n\n" + "\n\n".join(src.rstrip() for src in sources) + "\n"
        try:
            ast.parse(code)
            if not code_block_defines_function(code, "solution"):
                return None
        except Exception:
            return None
        return code

    # ------------------------------------------------------------------ the loop
    async def _run_tool_loop(self, tools: list[Tool], repl: Optional[REPL], model: str, messages: list[dict],
                             max_retries: int, sleep: float, **kwargs) -> RawTurnResult:
        cfg = self._v3
        trace: list = []
        messages = self._with_rerun_note(messages)     # session 15: second pass
        head = copy.deepcopy(messages)                  # system prompt (if any) and the pinned first user message; a recovery adds the last failed final answer and its reason
        base_head = copy.deepcopy(head)                 # the head without any recovery pair
        entries: list = []
        st = _State()
        st.gen_limit = self._run_limit()                # session 17: this run's decoded-token limit (0 = none)
        api_tools = [tool.chat_completions_schema for tool in tools]
        tr = _Transcript.open(head, model)            # live transcript.md + images/ in the task folder
        request_kwargs = dict(kwargs)               # no max_tokens: a reply may use whatever is left of the window
        response = None

        while True:
            remaining = self._remaining()
            if remaining is not None and remaining <= 0:
                st.deadline = True
                logger.info("V3_EVENT deadline reached (time is up for this sample)")
                break
            await self._maybe_trim(entries, st, repl, force=st.window_hit)
            st.window_hit = False

            retry_count = max_retries
            shrink = 0
            result = None
            while True:
                req_messages = head + [e.msg for e in entries]
                call_kwargs = dict(request_kwargs)
                cap_sent = None
                if cfg.think_cap_tokens and st.replies >= 1 \
                        and cfg.context_window - self._estimate(entries, st) - cfg.safety_margin >= cfg.think_cap_tokens:
                    cap_sent = cfg.think_cap_tokens     # session 13: a reply after the first stops here (not when the window is nearly full: the window rules apply)
                    call_kwargs["max_tokens"] = cap_sent
                try:
                    st.requests += 1
                    pending = self.create_chat_completion(model=model, messages=req_messages, tools=api_tools, **call_kwargs)
                    left = self._remaining()
                    result = await pending if left is None else await asyncio.wait_for(pending, timeout=max(1.0, left))
                except asyncio.TimeoutError:
                    st.deadline = True
                    logger.info("V3_EVENT deadline reached while waiting for a reply")
                    break
                except OpenAIBadRequestError as e:
                    if _is_context_error(e) and shrink < 3:
                        shrink += 1
                        logger.warning(f"V3_EVENT server rejected the request as too long; trimming deeper (level {shrink})")
                        if not await self._maybe_trim(entries, st, repl, level=shrink):
                            if entries:       # nothing to drop: shrink the pinned estimate so the next try trims more
                                st.base_tokens = (st.base_tokens or 0) + 2000
                        continue
                    logger.exception("Bad request error during tool loop")
                    if retry_count > 0:
                        retry_count -= 1
                        continue
                    raise
                except OpenAIInternalServerError as e:
                    # llama-server answers a reply it cannot parse (for example a damaged tool call) with a 5xx error
                    st.server_errors += 1
                    logger.warning(f"V3_EVENT server error {st.server_errors}: {str(e)[:200]}")
                    if st.server_errors <= 6:
                        await asyncio.sleep(2)
                        continue
                    raise
                if result.response.choices[0].finish_reason != "length" \
                        and self._response_contains_malformed_tool_calls(result.response, tools):
                    if retry_count > 0:
                        logger.info("Malformed tool calls found in response, retrying")
                        retry_count -= 1
                        trace.extend(result.trace)
                        continue
                break
            if st.deadline:
                break

            response = result.response
            trace.extend(result.trace)
            choice = response.choices[0]
            message = choice.message
            finish = choice.finish_reason
            prompt_tokens, completion_tokens = self._usage_numbers(response, req_messages)
            st.generated += completion_tokens
            logger.info(f"V3_GEN {st.generated}")
            over = bool(st.gen_limit) and st.generated >= st.gen_limit     # session 17: per-run limit (= max_generated_tokens without ARC_RUN_LIMITS)
            if st.review_from is not None and st.generated - st.review_from >= cfg.review_max_tokens:
                over = True     # session 14: the review's own budget is used up: stop after this reply (the first final answer is submitted)
            self._calibrate(entries, st, prompt_tokens, had_entries=bool(entries))
            tool_calls = message.tool_calls or []
            final_code = _final_code(message.content) if not tool_calls else None
            tr.reply(st, message, prompt_tokens, completion_tokens, finish)
            st.replies += 1

            # ---- session 13: the reply was stopped by the thinking cap: keep its end and tell the model to run code
            if finish == "length" and final_code is None and cap_sent and completion_tokens >= cap_sent - 16:
                st.think_cuts += 1
                entries.append(self._kept_assistant_entry(st, message))
                logger.info(f"V3_EVENT thinking cap #{st.think_cuts}: a reply was stopped after {completion_tokens} tokens; generated so far {st.generated}")
                if over:
                    st.token_limit = True
                    logger.info(f"V3_EVENT token limit reached ({st.generated} tokens decoded): the task stops here, no answer is forced")
                    break
                note = THINK_CAP_NOTE.format(k=cap_sent // 1000)
                entries.append(self._user_entry(st, note))
                tr.event(f"thinking cap: the reply was stopped after {completion_tokens} tokens; the model is told:\n\n{note}")
                continue

            # ---- the reply ran into the context window: keep a tail of it and trim at the next request. Nothing is said to the model.
            if finish == "length" and final_code is None:
                st.cutoffs += 1
                st.window_hit = True
                entries.append(self._kept_assistant_entry(st, message))
                logger.info(f"V3_EVENT window reached #{st.cutoffs}: a reply was cut off by the context window; generated so far {st.generated}")
                if over:
                    st.token_limit = True
                    logger.info(f"V3_EVENT token limit reached ({st.generated} tokens decoded): the task stops here, no answer is forced")
                    break
                continue

            # ---- session 14: the reply that ends the final review: the first final answer is submitted; a different new one becomes a record
            if not tool_calls and st.review_from is not None:
                first = _final_code(st.review_response.choices[0].message.content)
                if final_code is None:
                    st.review = "no_code"
                elif final_code.strip() == (first or "").strip():
                    st.review = "kept"
                else:
                    st.review = "changed"
                    if cfg.check_helper:
                        await self._record_final(repl, final_code)
                logger.info(f"V3_EVENT final review ended: {st.review} ({st.generated - st.review_from} tokens)")
                tr.event(f"final review ended: {st.review}; the first final answer is submitted" + (" (the new one is a record for attempt 2)" if st.review == "changed" else ""))
                break

            # ---- no tool call: the reply is the final answer
            if not tool_calls:
                verified = False
                if not over and st.recoveries < cfg.recovery_max:
                    verified = True
                    problem = await self._final_problem(repl, message, final_code)
                    if problem is not None:
                        kind, feedback = problem
                        st.recoveries += 1
                        st.final_rejects = st.recoveries
                        st.recovery_kinds.append(kind)
                        desc = await self._describe_repl(repl)
                        if desc:
                            feedback += f"\n\nYour Python session is unchanged. Currently defined in it: {desc}."
                        # the failed final answer and the reason are saved in the base prompt (a newer failure replaces the older pair); the working steps are dropped
                        head = base_head + self._recovery_pair(message, feedback)
                        entries.clear()
                        st.rebase = True
                        logger.info(f"V3_EVENT final answer rejected (#{st.recoveries}): [{kind}] {feedback[:300]}")
                        tr.event(f"final answer rejected (recovery #{st.recoveries}, {kind}); the model is told:\n\n{feedback}")
                        continue
                # session 14: a final answer that passed the training examples: one review against the test input(s)
                left = (st.gen_limit - st.generated) if st.gen_limit else cfg.review_max_tokens     # session 17: per-run limit
                if cfg.final_review and verified and final_code is not None and repl is not None and left >= cfg.review_min_left:
                    st.review_from = st.generated
                    st.review_response = response
                    if cfg.check_helper:
                        await self._record_final(repl, final_code)
                    entries.append(self._assistant_entry(st, message, completion_tokens))
                    note = REVIEW_NOTE.format(k=max(1, min(cfg.review_max_tokens, left) // 1000))
                    entries.append(self._user_entry(st, note))
                    logger.info(f"V3_EVENT final review started at {st.generated} tokens")
                    tr.event(f"final review: the final answer passes training; the model is told:\n\n{note}")
                    continue
                logger.info("No tool calls in response (the reply is the final answer)")
                break

            # ---- the token limit of this task is reached: stop after this reply (session 13: its tool calls are still run, so a last check()
            #      is recorded for the submission; the model does not see their output). No answer is forced.
            if over:
                st.token_limit = True
                logger.info(f"V3_EVENT token limit reached ({st.generated} tokens decoded): the task stops after the tool calls of this reply, no answer is forced")

            # ---- run the tool calls
            logger.info(f"Found {len(tool_calls)} tool call(s)")
            entries.append(self._assistant_entry(st, message, completion_tokens))
            tool_texts: list = []
            for tool_call in tool_calls:
                tool_name = tool_call.function.name
                logger.info(f"Model called tool named {tool_name}")
                text = None
                args: dict = {}
                tool = None
                try:
                    tool = self._find_tool(tool_name, tools)
                    args = json.loads(tool_call.function.arguments or "{}")
                    tool.validate_arguments(args)
                except ValueError as e:
                    text = str(e)
                except ValidationError as e:
                    text = str(e)
                except Exception as e:
                    text = f"Error parsing tool call arguments {tool_call.function.arguments}. Error message: {e}"
                if text is None:
                    for line in str(args.get("code", "")).splitlines():
                        if _IMPORT_LINE_RE.match(line.strip()) and not line.startswith((" ", "\t")) and line.strip() not in st.imports:
                            st.imports.append(line.strip())
                    out = await tool.run(repl=repl, **args) if isinstance(tool, REPLTool) else await tool.run(**args)
                    text = self._cap_tool_text(out if isinstance(out, str) else str(out))
                    logger.info("Added result to conversation")
                for _m in _CHECK_LINE_RE.finditer(text or ""):     # session 13: the best check() result seen (for the budget reminder)
                    _c = (int(_m.group(1)), int(_m.group(2)), float(_m.group(3)))
                    st.check_seen = True
                    if st.best_check is None or (_c[0] / max(1, _c[1]), _c[2]) > (st.best_check[0] / max(1, st.best_check[1]), st.best_check[2]):
                        st.best_check = _c
                tool_texts.append((tool_call.id, text))
                tr.tool(tool_name, args.get("code") if isinstance(args, dict) else None, tool_call.function.arguments, text)
            if st.check_seen and cfg.check_attempts and cfg.check_helper:     # session 14: keep check_attempts.json up to date
                st.check_seen = False
                await self._write_check_attempts(repl, st, tr, final=False)
            if over:
                break
            note = self._budget_note(st)     # session 13: added to the last tool output of this reply
            if note and tool_texts:
                tool_texts[-1] = (tool_texts[-1][0], f"{tool_texts[-1][1]}\n\n{note}")
                logger.info(f"V3_EVENT budget reminder at {st.generated} tokens: {note[:120]}")
                tr.event(f"budget reminder after this tool output:\n\n{note}")
            for call_id, text in tool_texts:
                chars = len(text)
                entries.append(_Entry(
                    msg={"role": "tool", "tool_call_id": call_id, "content": text},
                    kind="tool", tokens=math.ceil(chars / st.k_text) + _MSG_OVERHEAD, chars=chars))
            if cfg.enable_show:
                items = self._collect_show_images(st)
                if items:
                    entries.append(self._images_entry(st, items))
                    tr.images(items)
            await asyncio.sleep(sleep)

        if st.review_response is not None:      # session 14: whatever ended the review, the first final answer is submitted
            if st.review == "-":
                st.review = "deadline" if st.deadline else "budget"
                logger.info(f"V3_EVENT final review ended: {st.review}")
            response = st.review_response
            st.token_limit = False
        if st.deadline and st.review_response is None:
            if response is None:
                raise RuntimeError("deadline reached before the first model reply")
            if cfg.salvage_at_deadline:
                recovered = await self._salvage_solution(repl, st)
                if recovered:
                    response.choices[0].message.content = (
                        "Final solution (recovered from the Python session when the time was up):\n"
                        f"```python\n{recovered}\n```")
                    st.salvaged = 1
                    logger.info("V3_EVENT salvaged a solution from the Python session")
                else:
                    logger.info("V3_EVENT nothing to salvage from the Python session")
        if cfg.check_attempts and cfg.check_helper:
            await self._write_check_attempts(repl, st, tr)
        tr.end(st, response)
        logger.info(
            f"V3_STATS generated={st.generated} requests={st.requests} cutoffs={st.cutoffs} trims={st.trims} "
            f"dropped_messages={st.dropped_msgs} final_rejects={st.final_rejects} recoveries={st.recoveries} recovery_kinds={'/'.join(st.recovery_kinds) or '-'} fmt_reminders={st.fmt_reminders} "
            f"action_requests={st.action_requests} deadline={int(st.deadline)} token_limit={int(st.token_limit)} salvaged={st.salvaged} "
            f"server_errors={st.server_errors} think_cuts={st.think_cuts} check_cands={st.check_cands} review={st.review} show_images={st.show_images} k_text={st.k_text:.2f} base_tokens={st.base_tokens}")
        return RawTurnResult(response=response, trace=trace)


AsyncChatCompletionsAPIClientV3.ChatCompletionsV3CallConfig.client_class = AsyncChatCompletionsAPIClientV3
''',
    "solver/config/qwen3_8_flash_next_free.py": r'''import hashlib
import os
import tempfile
from pathlib import Path

from arcagi2.api.clients_v3 import AsyncChatCompletionsAPIClientV3
from arcagi2.api.providers import VLLM_API_PROVIDER
from arcagi2.sandbox.local_sandbox import LocalKernelSandbox
from arcagi2.solver.config.base import AgenticCodingConfig, PROMPTS_FOLDER
from arcagi2.tools.repl_tool import REPLToolWithProtection


def _env_int(name: str, default: int) -> int:
    return int(os.environ.get(name, default))


def _env_float(name: str, default: float) -> float:
    return float(os.environ.get(name, default))


def _env_flag(name: str, default: bool) -> bool:
    return os.environ.get(name, "1" if default else "0").strip().lower() in {"1", "true", "yes", "on"}


def _chat_template_kwargs() -> dict:
    """Thinking on; reasoning level from ARC_REASONING_EFFORT (default low). The chat template turns the level into an instruction in the system message, if it has one."""
    effort = os.environ.get("ARC_REASONING_EFFORT", "low")   # session 10: settable (step 8); Flash-Next's template knows xhigh (its default), medium, low
    if effort not in ("xhigh", "medium", "low"):
        raise ValueError(f"ARC_REASONING_EFFORT must be xhigh, medium or low, not {effort!r}")
    return {"enable_thinking": True, "reasoning_effort": effort}


CHECK_AID = ("\n\n- `check(fn=None)`: runs a function (default: your current `solution`) on every training example and every test input and prints "
             "the result: for each training example OK, or the wrong shape, or how many cells are wrong and which ones as (row,col) got/expected; "
             "for each test input the size and colours of the output (and any error, with its line). Example: `check()` or `check(my_idea)`. "
             "It is exact and costs far fewer tokens than checking cells in your thinking.")
CHECK_STYLE = " To test a hypothesis, write it as a function and run `check(f)`; never verify a rule cell by cell in your thinking."

HELPERS_AID = ("\n\n- Grid helpers (already defined; each takes a grid, a list of lists or a numpy array, never changes it and returns plain lists of ints): "
               "`rot90(g, k=1)` (clockwise, k times), `flip_lr(g)`, `flip_ud(g)`, `transpose(g)`, `dihedral(g)` (a dict of all 8 rotations/flips by name), "
               "`symmetries_between(a, b)` (the names of the rotations/flips that turn a into b), `crop(g, rows, cols)` (the same (start, stop) as `show_part`), "
               "`bbox(g, bg=0)` (rows and cols of the part that is not background, so `crop(g, *bbox(g))` cuts it out), "
               "`components(g, bg=0, diagonal=False, multicolor=False)` (the objects: a list of dicts with color, colors, cells, size, rows, cols, shape). "
               "They exist only in this notebook: if your final `solution` uses one, put its code into the final code block too "
               "(`helper_source(\"components\", \"crop\")` prints the code of the ones you name, with what they need).")

TEXT_GRIDS_OFF = [   # session 17: (old, new) prompt sentences when ARC_TEXT_GRIDS=0
    ("Grids below are written one row per line, as digits without spaces (for example `0120` is a row of four cells).",
     "The grids are NOT written out as text in this message: below, each grid is given by its size, and after the puzzle text every "
     "grid is drawn as an image. The exact cell values are in the `puzzle` variable in the Jupyter notebook: get anything exact "
     "(colours, counts, positions, differences, objects) with code."),
    ("Use images for overall shape, symmetry and layout, and use code or the text above for exact cell values.",
     "Use images for overall shape, symmetry and layout, and use code for exact cell values."),
    ("Don't print the `puzzle` dict, because it prints out the grids that are already present in this user message and consumes valuable token budget.",
     "Don't print whole grids to read them cell by cell: compute what you need with code, and test a rule by writing it as a function "
     "and running `check()` on it."),
]


def _rendered_prompt(name: str) -> Path:
    """The prompt file with the numbers of this run filled in (the window and the size it is trimmed to), so that the model is told the real limits."""
    text = (PROMPTS_FOLDER / name).read_text()
    text = text.replace("{{TRIM_TARGET}}", f"{(_env_int('ARC_TRIM_TARGET', 16384) // 1000) * 1000:,}")
    text = text.replace("{{WINDOW}}", f"{(_env_int('ARC_CTX_WINDOW', 32768) // 1000) * 1000:,}")
    _chk = _env_flag("ARC_CHECK_HELPER", True)      # session 12: the check() lines only when the Python session has check()
    text = text.replace("{{CHECK_AID}}", CHECK_AID if _chk else "").replace("{{CHECK_STYLE}}", CHECK_STYLE if _chk else "")
    text = text.replace("{{HELPERS_AID}}", HELPERS_AID if _env_flag("ARC_GRID_HELPERS", True) else "")   # session 12
    if not _env_flag("ARC_TEXT_GRIDS", True):       # session 17: no grid text (images + `puzzle` only)
        if not (_env_flag("ARC_IMAGES", True) and _chk):
            raise RuntimeError("ARC_TEXT_GRIDS=0 needs ARC_IMAGES=1 (the grids are only shown as images) and ARC_CHECK_HELPER=1")
        for _old, _new in TEXT_GRIDS_OFF:
            if name.startswith("agentic_coding"):
                assert text.count(_old) == 1, f"ARC_TEXT_GRIDS=0: the prompt sentence is not there once: {_old[:60]!r}"
                text = text.replace(_old, _new)
    out = Path(tempfile.gettempdir()) / f"arcagi2_free_{hashlib.sha1(text.encode()).hexdigest()[:12]}_{name}"
    if not out.exists():
        out.write_text(text)
    return out


# Qwen3.8-Flash-Next on vLLM, 32K window, "free" variant with RECOVERY: ONE tool (python), no reply cap, nothing injected into the conversation, the oldest
# messages are dropped (down to about 16K) when the window is reached, images untouched, NO forced final answer (the run ends when
# the model answers or at the deadline). A final answer that is unusable is sent back with the reason (recovery, at most ARC_RECOVERY_MAX times).
# Knobs can be changed without editing code through environment variables (read when the config is imported).
QWEN3_8_FLASH_NEXT_FREE_AGENTIC_CODING_CONFIG = AgenticCodingConfig(
    sandbox_cls=LocalKernelSandbox,
    sandbox_kwargs={},
    max_retries=0,
    call_config=AsyncChatCompletionsAPIClientV3.ChatCompletionsV3CallConfig(
        model=os.environ.get("ARC_MODEL_NAME", "Qwen/Qwen3.8-Flash-Next"),   # must match the server's --served-model-name (session 15: ARC_MODEL_NAME, e.g. the 27B)
        api_provider=VLLM_API_PROVIDER,        # only the name is used: llama-server and vLLM speak the same chat API
        system_prompt_path=_rendered_prompt("system_prompt_free.txt"),
        prompt_path=_rendered_prompt("agentic_coding_solver_free.txt"),
        client_kwargs={"timeout": 3600},
        raw_request_kwargs={
            "temperature": 1.0,
            "top_p": 0.95,
            "extra_body": {
                "top_k": 20,
                "min_p": 0.0,
                "presence_penalty": 0.0,
                "chat_template_kwargs": _chat_template_kwargs(),
            },
        },
        tools=[
            REPLToolWithProtection(name="python", timeout=120, protected_variables=["puzzle"], include_strict=False),
        ],
        max_retries=2,
        sleep=0,
        initial_code_timeout=120,
        sandbox_cls=LocalKernelSandbox,
        sandbox_kwargs={},
        context_window=_env_int("ARC_CTX_WINDOW", 32768),
        trim_target_tokens=_env_int("ARC_TRIM_TARGET", 16384),
        safety_margin=_env_int("ARC_TRIM_MARGIN", 1024),
        trim_note=_env_flag("ARC_TRIM_NOTE", True),
        max_generated_tokens=_env_int("ARC_MAX_GEN_TOKENS", 200000),
        recovery_max=_env_int("ARC_RECOVERY_MAX", 5),
        deadline_epoch=_env_float("ARC_DEADLINE_EPOCH", 0.0),
        sample_seconds=_env_float("ARC_SAMPLE_SECONDS", 0.0),
        reasoning_field=os.environ.get("ARC_REASONING_FIELD", "reasoning_content"),   # vLLM calls it "reasoning", llama-server "reasoning_content"
        salvage_at_deadline=_env_flag("ARC_SALVAGE_AT_DEADLINE", True),
        enable_show=_env_flag("ARC_IMAGES", True),
        first_message_images=_env_flag("ARC_IMAGES", True),
        compact_grids=_env_flag("ARC_COMPACT", True),
        check_helper=_env_flag("ARC_CHECK_HELPER", True),
        grid_helpers=_env_flag("ARC_GRID_HELPERS", True),
        check_attempts=_env_flag("ARC_CHECK_ATTEMPTS", True),          # session 13 (owner: on by default)
        think_cap_tokens=_env_int("ARC_THINK_CAP", 12000),              # session 13: 0 = off
        budget_reminders=_env_flag("ARC_BUDGET_REMINDERS", True),      # session 13
        budget_late_left=_env_int("ARC_BUDGET_LATE_LEFT", 15000),
        final_review=_env_flag("ARC_FINAL_REVIEW", False),              # session 14: OFF by default until measured on RTX (owner); the RTX s14 notebook sets it
        review_max_tokens=_env_int("ARC_REVIEW_MAX_TOKENS", 20000),
        review_min_left=_env_int("ARC_REVIEW_MIN_LEFT", 8000),
        rerun_notes=os.environ.get("ARC_RERUN_NOTES", ""),             # session 15: set only by the second pass of the run notebook
        run_limits=os.environ.get("ARC_RUN_LIMITS", ""),               # session 17: per-run limits (Kaggle rounds); empty = max_generated_tokens
    ),
)
''',
    "solver/prompts/system_prompt_free.txt": r'''# Tools

## python

Use this tool to execute Python code in your chain of thought. 

When you send a message containing Python code to python, it will be executed in a stateful Jupyter notebook environment. The environment supports common third party libraries like numpy, scipy, shapely, networkx, scikit-image, more-itertools, pillow, python-constraint, ortools and z3-solver. python will respond with the output of the execution or time out after 120.0 seconds. 

### Usage guideline

1. Use the variables and functions already loaded in the Jupyter notebook instead of defining them yourself. The user message will provide a complete list of such variables and functions.
2. Prefer to write reusable functions. These will persist in the stateful Jupyter notebook and can be referenced in later code snippets without having to define them again.
3. Long tool output is cut off in the middle.
''',
    "solver/prompts/agentic_coding_solver_free.txt": r'''You are an expert ARC AGI puzzle solver. Please solve the puzzle given below.

# Background information about ARC AGI puzzles

Each task consists of a small number of demonstration examples (3.3 on average), and a small number of test examples (1 - 3).

Each example consists of an “input grid” and an “output grid”. Each “grid” is a literal grid of colored squares There are 10 unique colors. A grid can be any height or width between 1x1 and 30x30, inclusive.

The color code is as follows:

0 - black
1 - navy blue
2 - red
3 - green
4 - yellow
5 - gray
6 - pink
7 - orange
8 - sky blue
9 - maroon

When solving an evaluation task, a test-taker has access to the training examples for the task (both the input and output grids), as well as the input grid of the test examples for the task. The test-taker must construct on its own the output grid corresponding to the input grid of each test example. “Constructing the output grid” is done entirely from scratch, meaning that the test-taker must decide what the height and width of the output grid should be, what colors they should place on the grid, and where. The task is successfully solved if the test-taker can produce the exact correct answer on all test examples for the task (binary measure of success).

The mapping from integers (0-9) to color names is arbitrary and should be treated as categorical labels; the transformation rule never relies on the specific integer–color pairing.

ARC AGI puzzle solutions feel "natural" to humans. Natural rules tend to build on visual priors and small-number counting. They are visually immediate or prominent for humans. That is why humans from all educational backgrounds are able to solve these puzzles. Rules that depend on non-visual considerations (e.g. mathematics that's more advanced than small number counting) or inelegant mechanics (e.g. tie-breakers, complex conditional rules, special exceptions) are considered "unnatural."  

This doesn't mean puzzles have to be simple or easy. Common patterns that puzzles use to increase difficulty are:  

- **Rule composition:** multiple simple, natural rules are applied in sequence, with one transformation setting up the next.
- **Contextual application:** a simple if/else based on a visually clear feature.
- **In-grid symbol definition:** certain shapes or patterns act as symbols whose meaning is established using the training examples and which control some aspect of the transformation. The meaning can even vary across training examples.
- **Iterated steps:** a simple action is applied repeatedly (as in assembling pieces of a jigsaw), where each step constrains the next. Repetition is restricted to a small number.
- **Approximate features**: Features that are easily detected by humans, but hard to detect using naive algorithms. Examples are slightly broken symmetry, slightly broken continuity, off-center symmetry axes, objects defined via 8-connectivity etc.
- **Non-locality:** the solution requires applying changes to one region, with the changes depending on features in a different, distant region.

# Puzzle to solve

Grids below are written one row per line, as digits without spaces (for example `0120` is a row of four cells).

{{puzzle}}

# Task aids

You may execute code in the Jupyter notebook environment.

Functions and variables available in the Jupyter notebook:

- `puzzle`: A Python `dict` containing the training and test grids. The key "train" contains the training examples in a list. Similarly, the key "test" contains the test examples as a list. Each training/test example is a dict containing keys "input" and "output". The values are the grids expressed as list of lists of integers (0 - 9). Access: `puzzle["train"][i]["input"]`, `puzzle["train"][i]["output"]`, `puzzle["test"][j]["input"]`.

- `show(*grids, labels=None)`: draws one or more grids as an image and shows it to you in the next message (grids are drawn side by side). You can call it whenever you want to look at grids as images, as often as you like. Example: `show(puzzle["train"][0]["input"], puzzle["train"][0]["output"])`, or `show(candidate_output)` to look at your own result. Use images for overall shape, symmetry and layout, and use code or the text above for exact cell values.

- `show_part(grid, rows=None, cols=None, labels=None)`: like `show`, but draws only a part of one grid, with the row and column numbers of the original grid written on the borders, so you can see exactly which row and which column each cell is in. `rows=(start, stop)` and `cols=(start, stop)` are 0-based and the stop is not included; a single number selects one row or column. Example: `show_part(puzzle["train"][0]["input"], rows=(2, 8), cols=(10, 16))`.{{CHECK_AID}}{{HELPERS_AID}}

Your very first code execution should be `puzzle.keys()` to check data access. 

Don't print the `puzzle` dict, because it prints out the grids that are already present in this user message and consumes valuable token budget.

# Working style

- The conversation window is about {{WINDOW}} tokens. When the conversation reaches it, the oldest messages are removed automatically until about {{TRIM_TARGET}} tokens remain, so do not rely on remembering earlier steps.
- One thing is never lost: everything you define in the Jupyter notebook. Write reusable functions and keep your best `solution` draft defined in the notebook.
- Reason through code, not in your head. Whenever you want to know something about the grids (is this a transposed or flipped copy of that? how many cells have color 3? which cells differ between two grids? is the output symmetric?), write a few lines of code and run them instead of working it out by hand in your thinking. Do not simulate a transformation cell by cell yourself: write the function and compare its output with the expected grid.{{CHECK_STYLE}}
- Do not think for long before acting. As soon as you have a hypothesis, however rough, test it with code right away and read the result. If the first idea is wrong, the output will teach you more than more thinking would. Keep each thinking step short and move on to a tool call.
- Keep working until you have a `solution` that passes every training example, then give it as your final answer.

# Your task

Please solve the puzzle. Solving the puzzle means constructing a function `solution(grid: list[list[int]])` that implements the transformation from input grid to output grid shown in the training examples. The solution must work on all training examples.

```
for idx, example in enumerate(puzzle["train"]):
    assert solution(example["input"]) == example["output"], f"Train example {idx}: mismatch"
```

The solution must generalize to the test input(s). Solutions that cheat by hardcoding the training example outputs are not acceptable.

# Output format (STRICT)

Your response must contain a single Python code block fenced by ```python ... ```. Must include a `solution` function. Must be self-contained i.e. include all necessary imports and helper functions.
''',
}

for rel, text in FILES.items():
    path = SRC / rel
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text)
    assert path.read_text() == text
    print("wrote", rel, f"({len(text.splitlines())} lines, sha256 {hashlib.sha256(text.encode()).hexdigest()[:12]})")

def edit(rel, old, new, marker):
    path = SRC / rel
    s = path.read_text()
    if marker in s:
        print("already edited:", rel, "->", marker)
        return
    assert old in s, f"unexpected content in {rel}: cannot apply edit ({marker})"
    path.write_text(s.replace(old, new, 1))
    print("edited:", rel, "->", marker)

edit("utils/puzzle_utils.py",
     'def grid_to_text(grid: list[list[int]]) -> str:\n    return "\\n".join(" ".join(map(str, row)) for row in grid)\n',
     'def grid_to_text(grid: list[list[int]], compact: bool = False) -> str:\n    # compact: one digit per cell and no spaces, which costs about half the tokens\n    sep = "" if compact else " "\n    return "\\n".join(sep.join(map(str, row)) for row in grid)\n',
     "compact: bool = False) -> str:")
edit("utils/puzzle_utils.py",
     "    include_solutions: bool = False,\n) -> Union[str, list[str]]:",
     "    include_solutions: bool = False,\n    compact: bool = False,\n) -> Union[str, list[str]]:",
     "    compact: bool = False,\n) -> Union")
for part in ("train_example['input']", "train_example['output']", "test_example['input']", "test_example['output']"):
    edit("utils/puzzle_utils.py", "{grid_to_text(" + part + ")}", "{grid_to_text(" + part + ", compact)}", "{grid_to_text(" + part + ", compact)}")
edit("utils/puzzle_utils.py",
     '    sep = "" if compact else " "\n',
     '    if __import__("os").environ.get("ARC_TEXT_GRIDS", "1") == "0":   # session 17: no grid text, only the size (images + `puzzle`)\n'
     '        return f"(a {len(grid)}x{len(grid[0]) if grid else 0} grid: see its image below; the values are in `puzzle`)"\n'
     '    sep = "" if compact else " "\n',
     "ARC_TEXT_GRIDS")

edit("solver/turn.py",
     '''            initial_code: Optional[list[str]] = None,
            ) -> "AbstractTurn.AbstractParsedTurnResult":
        prompt = self._render_prompt(prompt_template_vars)
        logger.info(f"Prompt:\\n{prompt}")
        parsed_turn_result = await self._with_retry(
            self.config.client_class.call, 
            messages=[{"role": "user", "content": prompt}], ''',
     '''            initial_code: Optional[list[str]] = None,
            content_builder: Optional[Callable[[str], object]] = None,
            ) -> "AbstractTurn.AbstractParsedTurnResult":
        prompt = self._render_prompt(prompt_template_vars)
        logger.info(f"Prompt:\\n{prompt}")
        # content_builder turns the rendered prompt text into the message content, e.g. a list of text and image parts
        content = content_builder(prompt) if content_builder is not None else prompt
        parsed_turn_result = await self._with_retry(
            self.config.client_class.call, 
            messages=[{"role": "user", "content": content}], ''',
     "content_builder: Optional")

edit("solver/solver.py",
     "from arcagi2.utils.puzzle_utils import puzzle_to_text, get_copy_without_solutions",
     "from arcagi2.utils import grid_image\nfrom arcagi2.utils.puzzle_utils import puzzle_to_text, get_copy_without_solutions",
     "from arcagi2.utils import grid_image")
edit("solver/solver.py",
     '''            logger.info("Running interleaved thinking solver")
            code_solution_result = await code_solution_turn.run(
                prompt_template_vars={
                    "puzzle": puzzle_to_text(
                        puzzle_json,
                        markdown_level=2,
                        separator="\\n\\n",
                        include_test=True,
                        include_all_tests=True,
                        include_solutions=False
                    ),
                },
                save_to_dir=output_folder,
                initial_code=[f"puzzle={repr(get_copy_without_solutions(puzzle_json))}"],
            )''',
     '''            logger.info("Running interleaved thinking solver")
            call_config = config.call_config
            use_images = bool(getattr(call_config, "first_message_images", False))
            image_scale = int(getattr(call_config, "image_scale", 16))
            code_solution_result = await code_solution_turn.run(
                prompt_template_vars={
                    "puzzle": puzzle_to_text(
                        puzzle_json,
                        markdown_level=2,
                        separator="\\n\\n",
                        include_test=True,
                        include_all_tests=True,
                        include_solutions=False,
                        compact=bool(getattr(call_config, "compact_grids", False)),
                    ),
                },
                save_to_dir=output_folder,
                initial_code=[f"puzzle={repr(get_copy_without_solutions(puzzle_json))}"],
                content_builder=(
                    (lambda prompt: grid_image.build_first_message_content(prompt, puzzle_json, scale=image_scale))
                    if use_images else None
                ),
            )''',
     "content_builder=(")

edit("solver/config/__init__.py",
     "from arcagi2.solver.config.minimax_m2_1 import",
     "from arcagi2.solver.config.qwen3_8_flash_next_free import QWEN3_8_FLASH_NEXT_FREE_AGENTIC_CODING_CONFIG\nfrom arcagi2.solver.config.minimax_m2_1 import",
     "qwen3_8_flash_next_free import")
edit("solver/config/__init__.py",
     '    "minimax_m2_1": MINIMAX',
     '    "qwen3_8_flash_next_free": QWEN3_8_FLASH_NEXT_FREE_AGENTIC_CODING_CONFIG,\n    "minimax_m2_1": MINIMAX',
     '"qwen3_8_flash_next_free":')

check = """
from arcagi2.solver.config import SOLVER_CONFIGS
from arcagi2.api.clients_v3 import AsyncChatCompletionsAPIClientV3
from arcagi2.utils import grid_image
from arcagi2.utils.puzzle_utils import puzzle_to_text
c = SOLVER_CONFIGS["qwen3_8_flash_next_free"].call_config
puz = {"train": [{"input": [[0, 1], [2, 3]], "output": [[3, 2], [1, 0]]}], "test": [{"input": [[1, 1], [0, 0]]}]}
text = puzzle_to_text(puz, compact=c.compact_grids)
parts = grid_image.build_first_message_content(text, puz, scale=c.image_scale)
print("config ok | window", c.context_window, "| trim at", c.context_window - c.safety_margin, "down to", c.trim_target_tokens, "| tools", [t.name for t in c.tools], "| recovery: always on, at most", c.recovery_max, "times per task | reasoning kwargs", c.raw_request_kwargs["extra_body"]["chat_template_kwargs"], "| trim_note", c.trim_note)
print("compact text ok:", "01\\n23" in text, "| first message parts:", len(parts), "| images:", sum(p["type"] == "image_url" for p in parts))
"""
r = subprocess.run([ARC_PY, "-c", check], capture_output=True, text=True, cwd=str(REPO), env=ARC_ENV)
print((r.stdout + r.stderr).strip()[-2500:])
assert r.returncode == 0, "check failed, paste the whole message above"

In [ ]:
from pathlib import Path

_FXC = Path(REPO) / "src/arcagi2/api/clients_v3.py"
_FXK = Path(REPO) / "src/arcagi2/solver/config/qwen3_8_flash_next_free.py"
for _p in (_FXC, _FXK):
    assert _p.exists(), f"{_p} not found: run the harness cell (step 5) first."
_FX_MARK = "# session 18b: harness fixes (step 5v)"

_FX_CUT_OLD = '''        tail = limit // 3
        head = limit - tail
        return f"{text[:head]}\\n... [{len(text) - limit} characters cut from the middle] ...\\n{text[-tail:]}"
'''
_FX_CUT_NEW = '''        tail = limit // 3
        head = limit - tail
        lines = text.split("\\n")             # session 18b: harness fixes (step 5v): whole lines only, and say which lines are missing
        h = used_h = 0
        while h < len(lines) and used_h + len(lines[h]) + 1 <= head:
            used_h += len(lines[h]) + 1
            h += 1
        t = used_t = 0
        while t < len(lines) - h and used_t + len(lines[-1 - t]) + 1 <= tail:
            used_t += len(lines[-1 - t]) + 1
            t += 1
        if h >= 1 and h + t < len(lines):
            n = len(lines)
            return ("\\n".join(lines[:h]) + f"\\n... [output lines {h + 1}-{n - t} of {n} not shown ({n - h - t} lines, "
                    f"{len(text) - used_h - used_t} characters)] ...\\n" + "\\n".join(lines[n - t:]))
        return f"{text[:head]}\\n... [{len(text) - limit} characters cut from the middle] ...\\n{text[-tail:]}"
'''

_BG_DEF = ('def _bg_of(g, bg):\n'
           '    # bg="auto": the most common colour of the grid (ties: the smallest colour); any other value is used as it is\n'
           '    if bg != "auto":\n'
           '        return bg\n'
           '    counts = {}\n'
           '    for row in g:\n'
           '        for v in row:\n'
           '            counts[v] = counts.get(v, 0) + 1\n'
           '    return max(counts, key=lambda k: (counts[k], -k)) if counts else 0')
_BBOX_OLD = ('def bbox(grid, bg=0):\n'
             '    # (rows, cols) of the smallest box around every cell that is not bg, as (start, stop) pairs: crop(g, *bbox(g)) cuts it out.\n'
             '    # None when every cell is bg.\n'
             '    g = _as_grid(grid)\n')
_BBOX_NEW = ('def bbox(grid, bg="auto"):\n'
             '    # (rows, cols) of the smallest box around every cell that is not bg, as (start, stop) pairs: crop(g, *bbox(g)) cuts it out.\n'
             '    # bg="auto" (default): the most common colour of this grid. None when every cell is bg.\n'
             '    g = _as_grid(grid)\n'
             '    bg = _bg_of(g, bg)\n')
_COMP_OLD = ('def components(grid, bg=0, diagonal=False, multicolor=False):\n'
             '    # the objects of the grid: connected groups of cells that are not bg (bg=None: every colour counts, also 0).\n')
_COMP_NEW = ('def components(grid, bg="auto", diagonal=False, multicolor=False):\n'
             '    # the objects of the grid: connected groups of cells that are not bg (bg="auto", the default: the most common colour of this\n'
             '    # grid; a number: that colour; bg=None: every colour counts).\n')
_COMP2_OLD = '    g = _as_grid(grid)\n    h, w = len(g), len(g[0])\n    steps = '
_COMP2_NEW = '    g = _as_grid(grid)\n    bg = _bg_of(g, bg)\n    h, w = len(g), len(g[0])\n    steps = '
_SPAN_END = '    start, stop = spec\n    return (0 if start is None else start), (n if stop is None else stop)'
_esc = lambda s: s.replace("\n", "\\n")
_FX_HELPER_EDITS = [
    ("bbox (session code)", _BBOX_OLD, _BBOX_NEW), ("bbox (helper_source copy)", _esc(_BBOX_OLD), _esc(_BBOX_NEW)),
    ("components (session code)", _COMP_OLD, _COMP_NEW), ("components (helper_source copy)", _esc(_COMP_OLD), _esc(_COMP_NEW)),
    ("components bg (session code)", _COMP2_OLD, _COMP2_NEW), ("components bg (helper_source copy)", _esc(_COMP2_OLD), _esc(_COMP2_NEW)),
    ("_bg_of (session code)", _SPAN_END + "\n\n\ndef crop(", _SPAN_END + "\n\n\n" + _BG_DEF + "\n\n\ndef crop("),
    ("_bg_of (helper_source copy)", "'crop': 'def crop(", "'_bg_of': '" + _esc(_BG_DEF) + "', 'crop': 'def crop("),
    ("helper dependencies", '_GRID_DEPS = {"dihedral": ["rot90", "flip_lr", "flip_ud", "transpose"], "symmetries_between": ["dihedral"], "crop": ["_span_of"]}',
     '_GRID_DEPS = {"dihedral": ["rot90", "flip_lr", "flip_ud", "transpose"], "symmetries_between": ["dihedral"], "crop": ["_span_of"], '
     '"bbox": ["_bg_of"], "components": ["_bg_of"]}'),
]
_FX_PROMPT_EDITS = [
    ("prompt: bbox", "`bbox(g, bg=0)` (rows and cols of the part that is not background, so `crop(g, *bbox(g))` cuts it out), ",
     "`bbox(g, bg='auto')` (rows and cols of the part that is not background, so `crop(g, *bbox(g))` cuts it out), "),
    ("prompt: components", "`components(g, bg=0, diagonal=False, multicolor=False)` (the objects: a list of dicts with color, colors, cells, size, rows, cols, shape). ",
     "`components(g, bg='auto', diagonal=False, multicolor=False)` (the objects: a list of dicts with color, colors, cells, size, rows, cols, shape); "
     "bg='auto' = the most common colour of that grid, or give a colour (bg=None: every colour counts). "),
]

_FX_REC_OLD = """                        # the failed final answer and the reason are saved in the base prompt (a newer failure replaces the older pair); the working steps are dropped
                        head = base_head + self._recovery_pair(message, feedback)
                        entries.clear()
                        st.rebase = True
"""
_FX_REC_NEW = """                        # session 18b (step 5v): the conversation is KEPT after every rejected final answer: the reply and the reason are appended
                        _ae = self._assistant_entry(st, message, completion_tokens)
                        if _ae.msg.get("content") is None and not _ae.msg.get("tool_calls"):
                            _ae.msg["content"] = ""         # an empty reply (only thinking): vLLM refuses content None without tool calls (HTTP 400)
                        entries.append(_ae)
                        entries.append(self._user_entry(st, feedback))
"""

_c, _k = _FXC.read_text(), _FXK.read_text()
assert "an empty reply keeps the conversation" not in _c, "step 5u was run on this harness: run step 5 (the harness cell) again, then only this cell"
if _FX_MARK in _c:
    print("already patched:", _FXC)
else:
    for _name, _old, _new in ([("tool output cut", _FX_CUT_OLD, _FX_CUT_NEW)] + _FX_HELPER_EDITS + [("recovery", _FX_REC_OLD, _FX_REC_NEW)]):
        assert _c.count(_old) == 1, f"harness fixes: unexpected harness source ({_name}: found {_c.count(_old)} times)"
        _c = _c.replace(_old, _new, 1)
    compile(_c, str(_FXC), "exec")
    for _name, _old, _new in _FX_PROMPT_EDITS:
        assert _k.count(_old) == 1, f"harness fixes: unexpected config source ({_name}: found {_k.count(_old)} times)"
        _k = _k.replace(_old, _new, 1)
    compile(_k, str(_FXK), "exec")
    _FXC.write_text(_c)
    _FXK.write_text(_k)
    print("patched:", _FXC, "and", _FXK, "| 1. tool output cut at whole lines | 2. helpers bg='auto' | 3. recovery keeps the conversation")
_r = subprocess.run([ARC_PY, "-c", """
import arcagi2.api.clients_v3 as m, inspect
src = inspect.getsource(m)
assert "harness fixes (step 5v)" in src and "the conversation is KEPT after every rejected final answer" in src
ns = {}
exec(m._GRID_HELPERS_CODE, ns)
g = [[8, 8, 8, 8], [8, 1, 1, 8], [8, 8, 8, 3]]
assert ns["bbox"](g) == ((1, 3), (1, 4)) and [o["color"] for o in ns["components"](g)] == [1, 3] and len(ns["components"](g, bg=0)) == 3
import io, contextlib
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    ns["helper_source"]("components")
alone = {}
exec(buf.getvalue(), alone)
assert [o["color"] for o in alone["components"](g)] == [1, 3], "helper_source copy differs"
from arcagi2.solver.config import SOLVER_CONFIGS
c = SOLVER_CONFIGS["qwen3_8_flash_next_free"].call_config
assert "bg='auto'" in open(c.prompt_path).read()
print("harness fixes: import ok | helpers bg='auto' ok (session and helper_source copy) | prompt lists bg='auto'")
"""], capture_output=True, text=True, cwd=str(REPO), env=ARC_ENV)
print((_r.stdout + _r.stderr).strip()[-1500:])
assert _r.returncode == 0, "the patched harness does not work: paste the message above"

In [ ]:
import subprocess
from pathlib import Path

_ZS = Path(REPO) / "src/arcagi2/sandbox/local_sandbox.py"
_ZC = Path(REPO) / "src/arcagi2/api/clients.py"
_ZV = Path(REPO) / "src/arcagi2/api/clients_v3.py"
for _p in (_ZS, _ZC, _ZV):
    assert _p.exists(), f"{_p} not found: run steps 4, 5 and 5v first."
_Z_MARK = "step 5z fix"

_ZS_EDITS = [("repl init", '''        self._pid = None
        self._killed = None

    async def _start_kernel(self):''', '''        self._pid = None
        self._killed = None
        self.setup_code = []     # step 5z fix (b): the setup cells (clients.py sets them); run again in a fresh python process

    async def _start_kernel(self):'''),
("reader stops", '''        run = asyncio.ensure_future(self._run(code))
        while not run.done():
            if self._killed is not None:
                run.cancel()
                why = self._killed
                try:
                    self._kc.stop_channels()
                    await self._km.shutdown_kernel(now=True)
                except Exception:
                    pass
                await self._start_kernel()
                raise _MemoryStopped(why)
            await asyncio.wait({run}, timeout=0.5)
        return run.result()
''', '''        run = asyncio.ensure_future(self._run(code))
        try:
            while not run.done():
                if self._killed is not None:
                    run.cancel()
                    why = self._killed
                    try:
                        self._kc.stop_channels()
                        await self._km.shutdown_kernel(now=True)
                    except Exception:
                        pass
                    await self._start_kernel()
                    raise _MemoryStopped(why)
                await asyncio.wait({run}, timeout=0.5)
            return run.result()
        finally:
            # step 5z fix (a): a call that is given up (the timeout in execute) stops its reader of the kernel's output too; before, the
            # reader kept going, took the output of later calls and made some fail with an empty error (-> the whole run restarted)
            if not run.done():
                run.cancel()
                await asyncio.wait({run})
'''),
("execute", '''            except Exception:
                try:
                    self._kc.stop_channels()
                    await self._km.shutdown_kernel(now=True)
                    await self._start_kernel()
                except Exception as e:
                    raise SandboxInfrastructureError(f"Kernel could not be recovered: {e}") from e
            raise ExecutionTimeoutError(f"Code execution timed out after {timeout} seconds")
        except _MemoryStopped as e:
            raise ExecutionError(trace=f"MemoryError: this python process {e}, so it was stopped and replaced by a fresh one. "
                                       f"Every variable, function and import from earlier calls is gone: define them again. "
                                       f"Use less memory (smaller arrays, no huge lists or caches).")
        except Exception as e:
            raise SandboxInfrastructureError(f"Code execution failed: {e}") from e
        if error is not None:
            raise ExecutionError(trace=error)
        return ExecutionResult(text="".join(chunks).strip() if chunks else None)
''', '''            except Exception:
                await self._replace_kernel("a timed-out call did not stop when interrupted")
                raise ExecutionError(trace=f"TimeoutError: the code ran longer than {timeout:g} seconds and did not stop when interrupted, "
                                           f"so this python process was replaced by a fresh one. {self._fresh_note()}")
            raise ExecutionTimeoutError(f"Code execution timed out after {timeout} seconds")
        except _MemoryStopped as e:
            await self._rerun_setup()
            raise ExecutionError(trace=f"MemoryError: this python process {e}, so it was stopped and replaced by a fresh one. "
                                       f"{self._fresh_note()} Use less memory (smaller arrays, no huge lists or caches).")
        except Exception as e:
            # step 5z fix (b): a python process that stops answering is replaced (setup run again) and the model is told; before, this
            # was a SandboxInfrastructureError, and the solver restarted the WHOLE run from its first prompt
            logger.warning(f"step 5z: python process broken ({type(e).__name__}: {e}): replacing it")
            await self._replace_kernel(f"{type(e).__name__}: {e}")
            raise ExecutionError(trace=f"PythonProcessError: this python process stopped answering, so it was replaced by a fresh one. "
                                       f"{self._fresh_note()}")
        if error is not None:
            raise ExecutionError(trace=error)
        return ExecutionResult(text="".join(chunks).strip() if chunks else None)

    def _fresh_note(self) -> str:
        # step 5z fix (b): what the model has in the fresh python process
        if self.setup_code:
            return ("`puzzle`, check() and the helper functions are loaded again, but every variable, function and import of your own "
                    "from earlier calls is gone: define them again.")
        return "Every variable, function and import from earlier calls is gone: define them again."

    async def _rerun_setup(self):
        # step 5z fix (b): the setup cells again, in the fresh python process
        for cell in self.setup_code:
            chunks, error = await asyncio.wait_for(self._run(cell), timeout=120)
            if error is not None:
                raise SandboxInfrastructureError(f"the setup code failed in a fresh python process: {error[-800:]}")
        logger.info(f"step 5z: python process replaced, setup run again ({len(self.setup_code)} cells)")

    async def _replace_kernel(self, why: str):
        # step 5z fix (b): a fresh python process + the setup; only if that fails is it an infrastructure error (whole-run restart)
        try:
            try:
                self._kc.stop_channels()
                await self._km.shutdown_kernel(now=True)
            except Exception:
                pass
            await self._start_kernel()
            await self._rerun_setup()
        except Exception as e:
            raise SandboxInfrastructureError(f"Kernel could not be replaced ({why}): {e!r}") from e
''')]

_ZC_OLD = '''                        logger.info(f"Initial code in REPL executed successfully")
'''
_ZC_NEW = '''                        logger.info(f"Initial code in REPL executed successfully")
                        repl.setup_code = list(initial_code)   # step 5z fix (b): a replaced python process runs these again
'''

_ZV_OLD = '''                        _ae = self._assistant_entry(st, message, completion_tokens)
                        if _ae.msg.get("content") is None and not _ae.msg.get("tool_calls"):
                            _ae.msg["content"] = ""         # an empty reply (only thinking): vLLM refuses content None without tool calls (HTTP 400)
                        entries.append(_ae)
                        entries.append(self._user_entry(st, feedback))
'''
_ZV_NEW = '''                        _ae = self._assistant_entry(st, message, completion_tokens)
                        if not _ae.msg.get("content") and not _ae.msg.get("tool_calls"):
                            # step 5z fix (c): a reply with only thinking is sent back with content "" (content None -> vLLM 400)
                            for _z_key in [k for k, v in _ae.msg.items() if v is None]:
                                del _ae.msg[_z_key]
                            _ae.msg["content"] = ""
                        entries.append(_ae)
                        entries.append(self._user_entry(st, feedback))
'''

for _p, _edits in ((_ZS, _ZS_EDITS), (_ZC, [("setup hook", _ZC_OLD, _ZC_NEW)]), (_ZV, [("empty reply", _ZV_OLD, _ZV_NEW)])):
    _t = _p.read_text()
    if _Z_MARK in _t:
        print("already patched:", _p)
        continue
    for _name, _old, _new in _edits:
        assert _t.count(_old) == 1, f"step 5z: unexpected source in {_p.name} ({_name}: found {_t.count(_old)} times; for clients_v3.py run step 5v first)"
        _t = _t.replace(_old, _new, 1)
    compile(_t, str(_p), "exec")
    _p.write_text(_t)
    print("patched:", _p, "|", ", ".join(n for n, _, _ in _edits))

_Z_CHECK = r'''
import asyncio, os, signal, time
from arcagi2.sandbox.local_sandbox import LocalKernelSandbox
from arcagi2.sandbox.base import ExecutionError
from arcagi2.sandbox.exceptions import ExecutionTimeoutError
import arcagi2.api.clients as C, arcagi2.api.clients_v3 as V, inspect
assert "repl.setup_code = list(initial_code)" in inspect.getsource(C) and "step 5z fix (c)" in inspect.getsource(V)
async def quick(repl, n):
    bad = 0
    for k in range(n):
        r = await repl.execute(f"for _ in range(3): print({k} * 7)\nlen(puzzle['train'])", 30)
        bad += r.text != "\n".join([str(k * 7)] * 3) + "\n3"
    return bad
async def main():
    async with LocalKernelSandbox() as sb:
        async with sb.create_repl() as repl:
            repl.setup_code = ["puzzle = {'train': [1, 2, 3]}"]
            await repl.execute(repl.setup_code[0], 60)
            for _ in range(3):
                try:
                    await repl.execute("import time; time.sleep(60)", 2)
                    raise SystemExit("no timeout")
                except ExecutionTimeoutError:
                    pass
                bad = await quick(repl, 15)
                assert bad == 0, f"{bad} of 15 outputs after a timeout were wrong"
            print("a: 3 timeouts, then 45 quick calls: all outputs right, no error")
            async def _no_interrupt(*a, **k):
                raise RuntimeError("the interrupt did not work (live check)")
            repl._km.interrupt_kernel = _no_interrupt
            try:
                await repl.execute("import time; time.sleep(60)", 2)
                raise SystemExit("no error")
            except ExecutionError as e:
                assert "replaced by a fresh one" in e.trace and "`puzzle`" in e.trace, e.trace
            assert await quick(repl, 5) == 0
            print("b: a call that ignores the interrupt -> fresh python process, `puzzle` loaded again, next calls right")
            repl._killed = "used too much memory (live check)"
            try:
                await repl.execute("import time; time.sleep(5)", 30)
                raise SystemExit("no error")
            except ExecutionError as e:
                assert e.trace.startswith("MemoryError") and "`puzzle`" in e.trace, e.trace
            assert await quick(repl, 5) == 0
            print("b: memory stop -> fresh python process, `puzzle` loaded again, next calls right")
            import queue
            async def _empty(*a, **k):
                raise queue.Empty     # what the 64K run's broken calls raised
            repl._kc.get_iopub_msg = _empty
            try:
                await repl.execute("1 + 1", 30)
                raise SystemExit("no error")
            except ExecutionError as e:
                assert e.trace.startswith("PythonProcessError") and "`puzzle`" in e.trace, e.trace
            assert await quick(repl, 5) == 0
            print("b: a broken output channel (queue.Empty) -> fresh python process instead of a whole-run restart, next calls right")
asyncio.run(main())
'''
_r = subprocess.run([ARC_PY, "-c", _Z_CHECK], capture_output=True, text=True, cwd=str(REPO), env=ARC_ENV, timeout=600)
print((_r.stdout + _r.stderr).strip()[-2500:])
assert _r.returncode == 0, "step 5z: the live check failed: paste the WHOLE message above"
print("STEP 5z done: a. timed-out calls stop their reader | b. a broken python process is replaced with the setup run again | "
      "c. empty replies are sent back with content \"\"")

In [ ]:
import subprocess
from pathlib import Path

assert "SINK_JUMP" in globals() and SINK_JUMP and SINK_JUMP[0] == "auto", "run the cell that sets SINK_JUMP first (RTX step 5c / Kaggle patch cell)"
_PT = Path(REPO) / "src/arcagi2/tools/repl_tool.py"
_PS = Path(REPO) / "src/arcagi2/solver/prompts/system_prompt_free.txt"
_PA = Path(REPO) / "src/arcagi2/solver/prompts/agentic_coding_solver_free.txt"
_PC = Path(REPO) / "src/arcagi2/solver/config/qwen3_8_flash_next_free.py"
for _p in (_PT, _PS, _PA, _PC):
    assert _p.exists(), f"{_p} not found: run steps 4, 5, 5v and 5z first."
_RECENT = f"{SINK_JUMP[1] // 1000:,},000"

_P_EDITS = {
    _PT: [("redefine check",
           '            if re.search(rf"{protected_variable}\\s*=", code):\n',
           '            if re.search(rf"(?<![\\w.]){protected_variable}\\s*=(?!=)", code):   # step 5p: not my_puzzle = ..., not puzzle == ...\n'),
          ("print check",
           '            if re.search(rf"(?m)^{protected_variable}(?:\\s*\\n|$)", code) or re.search(rf"print\\({protected_variable}", code):\n',
           '            if re.search(rf"(?m)^{protected_variable}(?:\\s*\\n|$)", code) or re.search(rf"print\\(\\s*{protected_variable}\\s*\\)", code):   # step 5p: only the WHOLE dict\n'),
          ("print message",
           'return f"Code is trying to print the {protected_variable} variable, which is not allowed because it will lead to a very large and wasteful print. Please remove the offending line and try again."',
           'return f"Code is trying to print the whole {protected_variable} variable, which is not allowed because it will lead to a very large and wasteful print. Nothing in this cell was run. Print only the part you need (for example {protected_variable}.keys() or the size of a grid) and try again."')],
    _PS: [("use loaded",
           "1. Use the variables and functions already loaded in the Jupyter notebook instead of defining them yourself. The user message will provide a complete list of such variables and functions.\n",
           "1. While you work, use the variables and functions already loaded in the Jupyter notebook instead of defining them yourself. The user message will provide a complete list of such variables and functions. Your final answer is different: it is run in a fresh Python session where none of them exist, so it must define everything it uses.\n"),
          ("refused cells",
           "3. Long tool output is cut off in the middle.\n",
           "3. Long tool output is cut off in the middle.\n"
           "4. A cell that prints the whole `puzzle` dict (`print(puzzle)` or a line with only `puzzle`) or assigns to `puzzle` is not run; printing parts of it is fine.\n")],
    _PA: [("memory",
           "- The conversation window is about {{WINDOW}} tokens. When the conversation reaches it, the oldest messages are removed automatically until about {{TRIM_TARGET}} tokens remain, so do not rely on remembering earlier steps.\n",
           "- You always see this first message; older parts of the conversation fade, so do not rely on remembering earlier steps exactly.{{BUDGET_LINE}}\n"),
          ("thinking",
           "- Do not think for long before acting. As soon as you have a hypothesis, however rough, test it with code right away and read the result. If the first idea is wrong, the output will teach you more than more thinking would. Keep each thinking step short and move on to a tool call.\n",
           "- Think carefully, and do the validating with code: as soon as you have a hypothesis, however rough, test it with code right away and read the result. If the first idea is wrong, the output will teach you more than more thinking would.\n"),
          ("output format",
           "Your response must contain a single Python code block fenced by ```python ... ```. Must include a `solution` function. Must be self-contained i.e. include all necessary imports and helper functions.",
           "Your final answer is one reply without a tool call that contains a Python code block fenced by ```python ... ``` (if the reply has several code blocks, the LAST one is used). It must include a `solution` function and be self-contained: it is run in a fresh Python session where nothing from the notebook exists, so include all necessary imports and helper functions (also the code of any notebook helper you used).")],
    _PC: [("budget line",
           "    text = text.replace(\"{{WINDOW}}\", f\"{(_env_int('ARC_CTX_WINDOW', 32768) // 1000) * 1000:,}\")\n",
           "    text = text.replace(\"{{WINDOW}}\", f\"{(_env_int('ARC_CTX_WINDOW', 32768) // 1000) * 1000:,}\")\n"
           "    _budget = _env_int('ARC_MAX_GEN_TOKENS', 0)       # step 5p: the model is told its real budget (the run stops there)\n"
           "    _rec = _env_flag('ARC_CHECK_ATTEMPTS', False) and _env_flag('ARC_CHECK_HELPER', True)\n"
           "    text = text.replace(\"{{BUDGET_LINE}}\", (\" The work may stop before you give a final answer\"\n"
           "                                           + (\"; then the functions you ran through `check()` are used instead (most training examples \"\n"
           "                                              \"right first), so run `check()` on every promising `solution`.\" if _rec else \".\")) if _budget else \"\")\n")],
}
for _p, _edits in _P_EDITS.items():
    _t = _p.read_text()
    if all(_new in _t for _, _, _new in _edits):
        print("already patched:", _p.name)
        continue
    for _name, _old, _new in _edits:
        assert _t.count(_old) == 1, f"step 5p: unexpected text in {_p.name} ({_name}: found {_t.count(_old)} times)"
        _t = _t.replace(_old, _new, 1)
    if _p.suffix == ".py":
        compile(_t, str(_p), "exec")
    _p.write_text(_t)
    print("patched:", _p.name, "|", ", ".join(n for n, _, _ in _edits))

_P_CHECK = r'''
import asyncio, os
os.environ.setdefault("ARC_MAX_GEN_TOKENS", "100000"); os.environ.setdefault("ARC_CHECK_ATTEMPTS", "1")
import arcagi2.solver.config.qwen3_8_flash_next_free as F
a = F._rendered_prompt("agentic_coding_solver_free.txt").read_text()
s = F._rendered_prompt("system_prompt_free.txt").read_text()
for gone in ("Do not think for long", "Keep each thinking step short", "conversation window is about", "removed automatically",
             "Your response must contain a single Python code block", "{{BUDGET_LINE}}", "{{WINDOW}}",
             "Your budget", "at least the last", "tokens of your own output"):   # nonum: no token numbers
    assert gone not in a + s, gone
for new in ("Think carefully, and do the validating with code", "You always see this first message; older parts of the conversation fade",
            "The work may stop before you give a final answer; then the functions you ran through `check()` are used instead", "the LAST one is used", "Your final answer is different",
            "assigns to `puzzle` is not run", "Your very first code execution should be `puzzle.keys()`"):
    assert (a + s).count(new) == 1, new
from arcagi2.tools import repl_tool as T
async def _ran(self, repl, code): return "RAN"
T.REPLTool.run = _ran
tool = T.REPLToolWithProtection(name="python", timeout=120, protected_variables=["puzzle"])
cases = {"print(puzzle.keys())": True, "puzzle.keys()": True, "print(puzzle['train'][0]['input'])": True, "my_puzzle = 1": True,
         "if puzzle == {}:\n    pass": True, "x = puzzle['test']": True, "print(puzzle)": False, "print( puzzle )": False,
         "puzzle": False, "a = 1\npuzzle\n": False, "puzzle = {}": False, "puzzle={}": False}
bad = {c: r for c, ok in cases.items() if ((r := asyncio.run(tool.run(None, c))) == "RAN") != ok}
assert not bad, bad
print("prompt and protection check: OK |", len(cases), "cells checked")
'''
_r = subprocess.run([ARC_PY, "-c", _P_CHECK], capture_output=True, text=True, cwd=str(REPO), env=ARC_ENV, timeout=300)
print((_r.stdout + _r.stderr).strip()[-2500:])
assert _r.returncode == 0, "step 5p: the check failed: paste the WHOLE message above"
print(f"STEP 5p done: print(puzzle.keys()) etc. allowed (only the whole dict refused) | thinking: careful, validated with code | memory: first "
      f"message always seen, older parts fade, no token numbers (nonum) | final answer: fresh session, last code block")

# Fast Server Startup

In [ ]:
import json, os
from pathlib import Path
EP_WEIGHT_FILTER = True
DROP_TAR_CACHE = True
GPU_UTIL_FAST = 0.945
_V = Path("/tmp/site-vllm/vllm")
_EDITS = [('model_executor/model_loader/weight_utils.py', 'fast n-gram load: lazy row-shard view', 'def safetensors_weights_iterator(\n', 'class _FNLazyRows:\n    """flashnext fast n-gram load (FLASHNEXT_FAST_NGRAM=1): a checkpoint row shard of the n-gram table, read lazily.\n    Only narrow(0, start, n) reads data (those rows); the PLE loader narrows to the overlap with this rank\'s vocabulary range."""\n\n    _DT = {"F8_E4M3": "float8_e4m3fn", "F8_E5M2": "float8_e5m2", "BF16": "bfloat16", "F16": "float16", "F32": "float32",\n           "U8": "uint8", "I8": "int8"}\n\n    def __init__(self, sl):\n        self._sl = sl\n        self.shape = torch.Size(sl.get_shape())\n        self.ndim = len(self.shape)\n        self.dtype = getattr(torch, self._DT[sl.get_dtype()])\n\n    def numel(self):\n        n = 1\n        for d in self.shape:\n            n *= d\n        return n\n\n    def element_size(self):\n        return torch.empty(0, dtype=self.dtype).element_size()\n\n    def narrow(self, dim, start, length):\n        if dim != 0:\n            raise ValueError("_FNLazyRows only narrows rows (dim 0)")\n        return self._sl[start:start + length]\n\n\ndef safetensors_weights_iterator(\n'), ('model_executor/model_loader/weight_utils.py', 'fast n-gram load: iterator yields lazy n-gram shards', '                    param = f.get_tensor(name)\n                    yield name, param\n', '                    if os.environ.get("FLASHNEXT_FAST_NGRAM") == "1" and "ngram_embedding.shard_" in name:\n                        param = _FNLazyRows(f.get_slice(name))   # flashnext: only this rank\'s rows are read later\n                    else:\n                        param = f.get_tensor(name)\n                    yield name, param\n'), ('models/qwen4_exp/common/ngram_embedding.py', 'n-gram shm: import os', 'import torch\nimport torch.nn.functional as F\n', 'import os  # flashnext n-gram shm\nimport torch\nimport torch.nn.functional as F\n'), ('models/qwen4_exp/common/ngram_embedding.py', 'n-gram shm: the table is a registered view of the /dev/shm file', '        """Allocate the complete PLE weight directly in pinned CPU memory."""\n        return torch.empty(\n            num_embeddings,\n            embedding_dim,\n            dtype=dtype,\n            device="cpu",\n            pin_memory=True,\n        )\n', '        """Allocate the complete PLE weight directly in pinned CPU memory."""\n        if os.environ.get("FLASHNEXT_NGRAM_SHM"):\n            return self._flashnext_shm_weight(num_embeddings, embedding_dim, dtype)\n        return torch.empty(\n            num_embeddings,\n            embedding_dim,\n            dtype=dtype,\n            device="cpu",\n            pin_memory=True,\n        )\n\n    def _flashnext_shm_weight(\n        self,\n        num_embeddings: int,\n        embedding_dim: int,\n        dtype: torch.dtype,\n    ) -> torch.Tensor:\n        """flashnext copy-free n-gram table: this process\'s rows are a registered view of the table file in /dev/shm (global row\n        order, written by step 0). Nothing is allocated or copied. Any mismatch stops the start (no fallback)."""\n        import json\n        import mmap\n        import time as _tm\n\n        path = os.environ["FLASHNEXT_NGRAM_SHM"]\n        meta = json.load(open(path + ".json"))\n        if not meta.get("complete"):\n            raise RuntimeError(f"flashnext n-gram shm: {path} is not complete (step 0 did not finish writing it)")\n        element = torch.empty(0, dtype=dtype).element_size()\n        row = embedding_dim * element\n        if row != meta["row_bytes"] or str(dtype).replace("torch.", "") != meta["torch_dtype"]:\n            raise RuntimeError(f"flashnext n-gram shm: the table file has rows of {meta[\'row_bytes\']} bytes ({meta[\'torch_dtype\']}), vLLM "\n                               f"expects {row} bytes ({dtype})")\n        start = self.shard_indices.org_vocab_start_index\n        begin, end = start * row, (start + num_embeddings) * row\n        if end > os.path.getsize(path):\n            raise RuntimeError(f"flashnext n-gram shm: rows {start}-{start + num_embeddings} need {end} bytes, {path} has "\n                               f"{os.path.getsize(path)}")\n        t0 = _tm.time()\n        base = begin - begin % mmap.PAGESIZE\n        fd = os.open(path, os.O_RDWR)\n        mm = mmap.mmap(fd, end - base, mmap.MAP_SHARED, offset=base)\n        buf = torch.frombuffer(mm, dtype=torch.uint8)\n        err = torch.cuda.cudart().cudaHostRegister(buf.data_ptr(), buf.numel(), 2)\n        if err != torch.cuda.cudart().cudaError.success:\n            raise RuntimeError(f"flashnext n-gram shm: cudaHostRegister of {buf.numel()} bytes failed: {err}")\n        weight = buf[begin - base:].view(dtype).view(num_embeddings, embedding_dim)\n        if not weight.is_pinned():\n            raise RuntimeError("flashnext n-gram shm: the registered table does not count as pinned memory")\n        self._flashnext_shm = (fd, mm, buf)\n        logger.info("flashnext n-gram shm: rows %d-%d (%.2f GB) of %s registered in %.2f s (no allocation, no copy)",\n                    start, start + num_embeddings, (end - begin) / 1e9, path, _tm.time() - t0)\n        return weight\n'), ('models/qwen4_exp/common/ple.py', 'n-gram shm: shard check helper', 'class PLEVocabParallelEmbedding(VocabParallelEmbedding):\n', 'def _flashnext_check_shard(\n    destination: torch.Tensor,\n    loaded_weight: torch.Tensor,\n    *,\n    checkpoint_start: int,\n    tp_start: int,\n    tp_end: int,\n) -> int:\n    """flashnext n-gram shm: the rows are already in the table file; compare 3 rows of the overlap byte for byte instead of copying."""\n    if destination.shape[1:] != loaded_weight.shape[1:]:\n        raise ValueError(\n            "embedding shard dimensions do not match: "\n            f"{tuple(destination.shape[1:])} != {tuple(loaded_weight.shape[1:])}"\n        )\n    overlap = compute_ple_shard_overlap(\n        checkpoint_start=checkpoint_start,\n        checkpoint_rows=loaded_weight.shape[0],\n        tp_start=tp_start,\n        tp_end=tp_end,\n    )\n    if overlap is None:\n        return 0\n    for k in sorted({0, overlap.row_count // 2, overlap.row_count - 1}):\n        source = loaded_weight.narrow(0, overlap.source_start + k, 1).contiguous().view(torch.uint8)\n        target = destination.narrow(0, overlap.destination_start + k, 1).contiguous().view(torch.uint8)\n        if not torch.equal(source, target):\n            raise RuntimeError(\n                f"flashnext n-gram shm: row {checkpoint_start + overlap.source_start + k} differs from the checkpoint "\n                "(the /dev/shm table is wrong; stop and rebuild it in step 0)"\n            )\n    return overlap.row_count\n\n\nclass PLEVocabParallelEmbedding(VocabParallelEmbedding):\n'), ('models/qwen4_exp/common/ple.py', 'n-gram shm: shards are checked, not copied', '        copy_ple_embedding_shard_(\n            param,\n            loaded_weight,\n            checkpoint_start=checkpoint_start,\n            tp_start=self.shard_indices.org_vocab_start_index,\n            tp_end=self.shard_indices.org_vocab_end_index,\n        )\n', '        if getattr(self, "_flashnext_shm", None) is not None:   # flashnext n-gram shm: rows already in place, check them\n            _flashnext_check_shard(\n                param,\n                loaded_weight,\n                checkpoint_start=checkpoint_start,\n                tp_start=self.shard_indices.org_vocab_start_index,\n                tp_end=self.shard_indices.org_vocab_end_index,\n            )\n            return\n        copy_ple_embedding_shard_(\n            param,\n            loaded_weight,\n            checkpoint_start=checkpoint_start,\n            tp_start=self.shard_indices.org_vocab_start_index,\n            tp_end=self.shard_indices.org_vocab_end_index,\n        )\n')]

def meminfo():
    m = {l.split(":")[0]: int(l.split()[1]) / 1048576 for l in open("/proc/meminfo")}
    return f"free {m['MemFree']:.1f} GiB | available {m['MemAvailable']:.1f} GiB | page cache {m['Cached']:.1f} GiB (shmem {m['Shmem']:.1f} GiB)"

assert "NGRAM_SHM" in globals() and NGRAM_SHM and Path(NGRAM_SHM).exists(), f"step 0 did not start the n-gram table in /dev/shm ({globals().get('NGRAM_SHM')!r})"
for _p, _label, _old, _new in _EDITS:
    _f = _V / _p
    _s = _f.read_text()
    if _new in _s:
        print(f"  [already] {_label}"); continue
    assert _s.count(_old) == 1, f"{_label}: the expected text is in {_f} {_s.count(_old)} times (vLLM differs from 0.31.0?)"
    _s = _s.replace(_old, _new); compile(_s, str(_f), "exec"); _f.write_text(_s)
    print(f"  [patch] {_label}")
os.environ["FLASHNEXT_NGRAM_SHM"] = NGRAM_SHM
os.environ["FLASHNEXT_FAST_NGRAM"] = "1"
print("n-gram table: copy-free from", NGRAM_SHM, "(vLLM checks 3 rows of every shard)")
print("memory before:", meminfo())
if DROP_TAR_CACHE:
    for _t in sorted(Path(UTILITY_PATH).glob("*.tar")):
        _fd = os.open(_t, os.O_RDONLY); os.posix_fadvise(_fd, 0, 0, os.POSIX_FADV_DONTNEED); os.close(_fd)
    print("utility tars dropped from the page cache | memory after:", meminfo())
print("EP weight filter:", "ON (--enable-ep-weight-filter)" if EP_WEIGHT_FILTER else "off")
GPU_UTIL = GPU_UTIL_FAST
print(f"GPU_UTIL = {GPU_UTIL}")

In [ ]:
import os
os.environ.pop("VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS", None)
KV_CACHE_BYTES = None
print("KV memory: measured by vLLM at GPU_UTIL", globals().get("GPU_UTIL"), "with the CUDA-graph estimate on (graphs up to 64); "
      "the 104-graph runs had 8.81 GiB per GPU = 4,064,286 tokens")

In [ ]:
import re
from pathlib import Path
_MARK = "flashnext startup test 11: wait for the model prefetch"
_WAIT_CODE = """import os as _fw_os, time as _fw_time
_fw_log = _fw_os.environ.get("FLASHNEXT_WAIT_PREFETCH")
if _fw_log:  # {mark}
    _fw_t0 = _fw_time.time()
    while "PREFETCH DONE" not in (open(_fw_log).read() if _fw_os.path.exists(_fw_log) else ""):
        _fw_err = (open(_fw_log + ".err").read() if _fw_os.path.exists(_fw_log + ".err") else "")
        if "Traceback" in _fw_err or "Error" in _fw_err:
            raise RuntimeError("the model prefetch failed: " + _fw_err[-1500:])
        _fw_pid = int(_fw_os.environ.get("FLASHNEXT_WAIT_PREFETCH_PID") or 0)
        if _fw_pid and not _fw_os.path.exists(f"/proc/{_fw_pid}") and "PREFETCH DONE" not in open(_fw_log).read():
            raise RuntimeError("the model prefetch process ended without PREFETCH DONE: " + _fw_err[-1500:])
        if _fw_time.time() - _fw_t0 > 1800:
            raise RuntimeError("the model prefetch is not done after 30 min (" + _fw_log + ")")
        _fw_time.sleep(1)
    logger.info("flashnext: waited %.0f s for the model prefetch before loading the model", _fw_time.time() - _fw_t0)
""".replace("{mark}", _MARK)
_hits = []
for _f in sorted(Path("/tmp/site-vllm/vllm/v1/worker").rglob("*.py")):
    _s = _f.read_text()
    if "Loading model from scratch" not in _s:
        continue
    if _MARK in _s:
        print("  [already]", _f); _hits.append(_f); continue
    _lines = _s.splitlines(True)
    _idx = [k for k, l in enumerate(_lines) if "Loading model from scratch" in l]
    assert len(_idx) == 1, f"{_f}: 'Loading model from scratch' is on {len(_idx)} lines (vLLM differs from 0.31.0?)"
    _k = _idx[0]
    _start = _k
    while not re.match(r"\s*(logger|self\.logger|log)\b", _lines[_start]) and _start > _k - 3:
        _start -= 1
    assert re.match(r"\s*(logger|self\.logger|log)\b", _lines[_start]), f"{_f}: no logger call at line {_k + 1}: {_lines[_k]!r}"
    _ind = re.match(r"\s*", _lines[_start]).group(0)
    assert "logger" in _s, f"{_f}: no logger in this file"
    _new = "".join(_ind + l if l.strip() else l for l in _WAIT_CODE.splitlines(True))
    _s2 = "".join(_lines[:_start]) + _new + "".join(_lines[_start:])
    compile(_s2, str(_f), "exec")
    _f.write_text(_s2)
    print(f"  [patch] {_f}: wait for the prefetch before line {_start + 1} ({_lines[_k].strip()[:80]})")
    _hits.append(_f)
assert _hits, "no vLLM file logs 'Loading model from scratch...' (vLLM differs from 0.31.0?)"
print(f"loader wait installed in {len(_hits)} file(s); active when step 6 sets FLASHNEXT_WAIT_PREFETCH")

In [ ]:
import os
from pathlib import Path
_f = Path("/tmp/site-vllm/vllm/model_executor/model_loader/weight_utils.py")
_s = _f.read_text()
_MARK = "flashnext startup test 14: skip every tensor of a non-local expert"
_old = "if should_skip_weight(name, local_expert_ids):"
_new = "if should_skip_weight(name, local_expert_ids) or _fn_skip_nonlocal_expert(name, local_expert_ids):"
_helper = """

# ---- {mark}
_FN_EXPERT_RE = __import__("re").compile(r"\\.experts\\.(\\d+)\\.")
_FN_EP_SKIPPED = [0]


def _fn_skip_nonlocal_expert(name, local_expert_ids):
    if local_expert_ids is None or __import__("os").environ.get("FLASHNEXT_EP_FILTER_ALL") != "1":
        return False
    m = _FN_EXPERT_RE.search(name)
    if m is None or int(m.group(1)) in local_expert_ids:
        return False
    _FN_EP_SKIPPED[0] += 1
    if _FN_EP_SKIPPED[0] == 1 or _FN_EP_SKIPPED[0] % 25000 == 0:
        logger.info("flashnext EP filter (all tensors): %d non-local expert tensors skipped so far (e.g. %s)", _FN_EP_SKIPPED[0], name)
    return True
""".replace("{mark}", _MARK)
if _MARK in _s:
    print("  [already]", _f)
else:
    _n = _s.count(_old)
    assert _n >= 1, f"{_f}: '{_old}' not found (vLLM differs from 0.31.0?)"
    assert "logger" in _s, f"{_f}: no logger"
    _s = _s.replace(_old, _new) + _helper
    compile(_s, str(_f), "exec")
    _f.write_text(_s)
    print(f"  [patch] {_f}: {_n} place(s) also skip every tensor of a non-local expert when FLASHNEXT_EP_FILTER_ALL=1")
os.environ["FLASHNEXT_EP_FILTER_ALL"] = "1"
print("FLASHNEXT_EP_FILTER_ALL = 1 (needs --enable-ep-weight-filter, which step 6 passes)")

# Speed Switches and Collapse Fix

In [ ]:
import os
SPEED_SWITCHES = True
_F = {'VLLM_HUMMING_MOE_GEMM_TYPE': 'indexed', 'FLASHNEXT_TOPK_EXCHANGE': '1', 'FLASHNEXT_DCP_QUERY_EARLY': '1', 'FLASHNEXT_DCP_BF16_PARTIAL': '1', 'FLASHNEXT_HC_DOWN': '1'}
_F.update({f'FLASHNEXT_TRIM_{t}': '1' for t in ('K4_ROT', 'GAP_ONCE', 'TOPK_BF16', 'K4_FWHT', 'GDN_AB', 'HCD_SILU', 'MAMBA_TILES', 'QSA_GATE', 'QSA_SHIFT')})
if SPEED_SWITCHES:
    assert os.environ.get('FLASHNEXT_BATCH_SHARDED_SAMPLING') == '1', 'the top-k exchange needs the batch-sharded sampling cell'
    os.environ.update(_F)
else:
    for _k in _F:
        os.environ.pop(_k, None)
print('decode speed switches:', 'ON' if SPEED_SWITCHES else 'OFF', {k: os.environ.get(k) for k in _F})

In [ ]:
import os
from pathlib import Path

_root = Path(globals().get("VROOT") or "/tmp/site-vllm/vllm")
_fs, _fb = _root / "v1/core/sched/scheduler.py", _root / "v1/core/block_pool.py"
_MARK = "# flashnext miss deferral (session 22)"

_HELPERS = '''

''' + _MARK + '''
import os as _fmd_os
import time as _fmd_time

_FN_MD = {"on": _fmd_os.environ.get("FLASHNEXT_MISS_DEFER") == "1",
          "window_s": float(_fmd_os.environ.get("FLASHNEXT_DEFER_WINDOW_S", "60")),
          "max_s": float(_fmd_os.environ.get("FLASHNEXT_DEFER_MAX_S", "1200")),   # replay: 180 s re-started the cascade; harness timeout 3600 s
          "gap_s": float(_fmd_os.environ.get("FLASHNEXT_DEFER_GAP_S", "60")),
          "min_tokens": int(_fmd_os.environ.get("FLASHNEXT_DEFER_MIN_TOKENS", "16384")),
          "frac": float(_fmd_os.environ.get("FLASHNEXT_DEFER_FRAC", "0.5")),
          "clock": _fmd_time.monotonic, "last_att_evict": -1e18, "att_evicted": 0, "deferred": 0, "first": {}, "deferred_reqs": set(),
          "released": 0, "held_back": 0, "rel": None, "rel_done": None, "t": _fmd_time.time()}


def _fn_md_pressure() -> bool:
    return _FN_MD["clock"]() - _FN_MD["last_att_evict"] <= _FN_MD["window_s"]


def _fn_md_defer(request, requests) -> bool:
    """True = skip this large miss now; False once it has waited max_s (aging), one released call at a time."""
    m = _FN_MD
    now = m["clock"]()
    t0 = m["first"].setdefault(request.request_id, now)
    if now - t0 >= m["max_s"]:
        if m["rel"] == request.request_id:
            return False
        if m["rel"] is not None:
            r = requests.get(m["rel"])
            if r is not None and r.num_computed_tokens < r.num_prompt_tokens:
                m["held_back"] += 1
                return True
            if m["rel_done"] is None:
                m["rel_done"] = now
            if now - m["rel_done"] < m["gap_s"]:
                m["held_back"] += 1
                return True
        m["rel"], m["rel_done"] = request.request_id, None
        m["released"] += 1
        return False
    m["deferred"] += 1
    m["deferred_reqs"].add(request.request_id)
    return True


def _fn_md_report(requests) -> None:
    m = _FN_MD
    now = _fmd_time.time()
    if now - m["t"] >= 60:
        m["t"] = now
        for rid in [r for r in m["first"] if r not in requests]:
            del m["first"][rid]
        print(f"flashnext miss defer (total): attention blocks evicted {m['att_evicted']} | scan skips {m['deferred']} "
              f"(distinct calls {len(m['deferred_reqs'])}) | run after waiting {m['max_s']:.0f} s: {m['released']} (one at a time, gap "
              f"{m['gap_s']:.0f} s; held back {m['held_back']}) | waiting now {len(m['first'])} | pressure now {_fn_md_pressure()}", flush=True)
'''

_SCHED_EDITS = [
    ("count scheduler steps",
     "            while token_budget > 0:\n                # Requests holding KV blocks are always drained first.\n",
     "            if _FN_MD[\"on\"]:   " + _MARK + "\n                _fn_md_report(self.requests)\n"
     "            while token_budget > 0:\n                # Requests holding KV blocks are always drained first.\n"),
    ("skip a large miss under pressure",
     "                    # Skip request with pending mm encoding prefetches\n                    if self._ec_transfer_pending(request, num_computed_tokens):\n"
     "                        skip_request(request_queue)\n                        continue\n\n                    # Track first scheduled prefill",
     "                    # Skip request with pending mm encoding prefetches\n                    if self._ec_transfer_pending(request, num_computed_tokens):\n"
     "                        skip_request(request_queue)\n                        continue\n\n"
     "                    if (_FN_MD[\"on\"] and not load_kv_async   " + _MARK + "\n"
     "                            and request.num_prompt_tokens >= _FN_MD[\"min_tokens\"]\n"
     "                            and num_computed_tokens < _FN_MD[\"frac\"] * request.num_prompt_tokens\n"
     "                            and _fn_md_pressure() and (self.running or len(request_queue) > 1)\n"
     "                            and _fn_md_defer(request, self.requests)):\n"
     "                        skip_request(request_queue)\n"
     "                        continue\n\n"
     "                    # Track first scheduled prefill"),
]
_POOL_EDITS = [
    ("count evictions of cached attention blocks",
     "        evicted_hashes = self._remove_cached_block_hashes(block)\n        if not evicted_hashes:\n"
     "            # The block doesn't have hash, eviction is not needed\n            return False\n",
     "        evicted_hashes = self._remove_cached_block_hashes(block)\n        if not evicted_hashes:\n"
     "            # The block doesn't have hash, eviction is not needed\n            return False\n"
     "        if not getattr(self, \"_fn_dropping\", False):   " + _MARK + "\n"
     "            from vllm.v1.core.sched.scheduler import _FN_MD\n"
     "            from vllm.v1.core.kv_cache_coordinator import _FN_SEG as _fmd_seg\n"
     "            from vllm.v1.core.kv_cache_utils import get_group_id as _fmd_gid\n"
     "            if _FN_MD[\"on\"] and _fmd_gid(evicted_hashes[0]) not in _fmd_seg[\"state_groups\"]:\n"
     "                _FN_MD[\"att_evicted\"] += 1\n"
     "                _FN_MD[\"last_att_evict\"] = _FN_MD[\"clock\"]()\n"),
]

def _apply(path, edits, tail):
    s = path.read_text()
    if _MARK in s:
        return "already"
    for label, old, new in edits:
        assert s.count(old) == 1, f"miss deferral edit '{label}': anchor found {s.count(old)} times in {path} (vLLM or a patch cell changed?)"
        s = s.replace(old, new)
    s += tail
    compile(s, str(path), "exec")
    path.write_text(s)
    return "installed"

print("miss deferral:", _fs.relative_to(_root), f"[{_apply(_fs, _SCHED_EDITS, _HELPERS)}]", "|",
      _fb.relative_to(_root), f"[{_apply(_fb, _POOL_EDITS, '')}]")
assert os.environ.get("FLASHNEXT_OFFLOAD_EVICT") == "1", "miss deferral reads the DeltaNet group ids recorded by the evict offload (patch cell)"
os.environ["FLASHNEXT_MISS_DEFER"] = "1"
print("FLASHNEXT_MISS_DEFER =", os.environ["FLASHNEXT_MISS_DEFER"])

# Start the vLLM Server

In [ ]:
# E36 gEAN calibration hook (inserted before the server launch cell).
# gEAN score of expert e in layer L = sum over routed tokens of (router weight) x ||expert output||
# (Yi-Chia Chen, discussion 755801). Measured in the Humming indexed MoE apply, on the per-(token, slot)
# expert outputs right before the weighted sum. Each process (TP/EP rank) adds its local experts and
# writes /kaggle/working/gean/gean_<pid>.pt every 60 s: {layer: float64 tensor [3, E+2]}
# rows = (gEAN sum, routed count, router-weight sum). Needs --enforce-eager (S["eager"] = True).
import os as _os, re as _re, py_compile as _pc
_SITE = "/tmp/site-vllm/vllm"
_F_MOE = f"{_SITE}/model_executor/layers/fused_moe/experts/fused_humming_moe.py"
_F_MODEL = f"{_SITE}/models/qwen4_exp/nvidia/model.py"
_os.makedirs("/kaggle/working/gean", exist_ok=True)

_HOOK_LIB = '''

# ---- E36 gEAN hook ----
GEAN_LAYER = [-1]
_GEAN = {}
_GEAN_T = [None]


def _gean_dump():
    import time as _t
    while True:
        _t.sleep(60)
        try:
            snap = {k: v.detach().to("cpu") for k, v in list(_GEAN.items())}
            p = f"/kaggle/working/gean/gean_{os.getpid()}.pt"
            torch.save(snap, p + ".part")
            os.replace(p + ".part", p)
        except Exception as e:
            print("[GEAN] dump failed:", e, flush=True)


def _gean_acc(down, w, ids, emap, n_experts):
    L = GEAN_LAYER[0]
    if L < 0:
        return
    if _GEAN_T[0] is None:
        import threading as _th
        _GEAN_T[0] = _th.Thread(target=_gean_dump, name="gean-dump", daemon=True)
        _GEAN_T[0].start()
        print(f"[GEAN] hook active in pid {os.getpid()}", flush=True)
    acc = _GEAN.get(L)
    if acc is None:
        acc = _GEAN[L] = torch.zeros(3, n_experts + 2, dtype=torch.float64, device=down.device)
    m = ids.shape[0]
    ids_l = ids.long()
    valid = (ids_l >= 0) & (ids_l <= n_experts)
    safe = ids_l.clamp(min=0, max=n_experts)
    if emap is not None:
        valid &= emap[safe.clamp(max=emap.numel() - 1)] >= 0
    v = valid.double()
    nrm = down[:m].float().norm(dim=-1).double()
    wf = w[:m].double()
    idx = safe.flatten()
    acc[0].index_add_(0, idx, (wf * nrm * v).flatten())
    acc[1].index_add_(0, idx, v.flatten())
    acc[2].index_add_(0, idx, (wf * v).flatten())
'''

_ANCHOR = '''        moe_fused_mul_sum(
            inputs=buffers["down_output"].view(*topk_ids.shape, -1),'''
_CALL = '''        _gean_acc(buffers["down_output"].view(*topk_ids.shape, -1), topk_weights, topk_ids, expert_map, global_num_experts)
'''

src = open(_F_MOE).read()
if "E36 gEAN hook" not in src:
    assert src.count(_ANCHOR) == 1, f"gEAN anchor count {src.count(_ANCHOR)} in {_F_MOE}"
    if "\nimport os\n" not in src:
        src = src.replace("\nimport json\n", "\nimport json\nimport os\n", 1)
    src = src.replace(_ANCHOR, _CALL + _ANCHOR, 1) + _HOOK_LIB
    open(_F_MOE, "w").write(src)
_pc.compile(_F_MOE, doraise=True)

_MODEL_ANCHOR = '''        self.n_shared_experts = int(config.shared_expert_intermediate_size > 0)
'''
_MODEL_ADD = '''        self.n_shared_experts = int(config.shared_expert_intermediate_size > 0)
        import re as _gre
        _gm = _gre.search(r"layers\\.(\\d+)\\.", prefix + ".")
        self._gean_layer = int(_gm.group(1)) if _gm else -1

    def forward(self, hidden_states, already_sequence_parallel=False):
        from vllm.model_executor.layers.fused_moe.experts import fused_humming_moe as _ghm
        _ghm.GEAN_LAYER[0] = self._gean_layer
        return super().forward(hidden_states, already_sequence_parallel)
'''
msrc = open(_F_MODEL).read()
if "_gean_layer" not in msrc:
    assert msrc.count(_MODEL_ANCHOR) == 1, f"model anchor count {msrc.count(_MODEL_ANCHOR)}"
    msrc = msrc.replace(_MODEL_ANCHOR, _MODEL_ADD, 1)
    open(_F_MODEL, "w").write(msrc)
_pc.compile(_F_MODEL, doraise=True)
print("[GEAN] hook installed:", _F_MOE, "|", _F_MODEL, "| dumps -> /kaggle/working/gean/ every 60 s")


In [ ]:
import json, os, re, shlex, shutil, signal, subprocess, sys, time, urllib.request
from pathlib import Path

SERVED_NAME = "Qwen/Qwen3.8-Flash-Next"
BASE_PORT   = 8000
import os; os.environ["VLLM_ALLOW_LONG_MAX_MODEL_LEN"] = "1"
MAX_LEN     = 1048576
MIN_LEN     = 16384
MAX_SEQS    = 128
MTP_TOKENS  = 0
USE_IMAGES  = True
MAX_BATCHED = 4096
DISABLE_CUSTOM_AR = True
STARTUP_TIMEOUT_S = (90 if MODEL_KIND == "flashnext" else 50) * 60

assert "MODEL_DIR" in globals() and "vllm_env" in globals(), "Run steps 1 to 3 first."
assert TP * REPLICAS <= N_GPUS, f"{REPLICAS} copies x {TP} GPUs need {TP * REPLICAS} GPUs, this machine has {N_GPUS}."
MTP_AVAILABLE = bool(MTP_TOKENS and MTP_PRESENT)
if MTP_TOKENS and not MTP_PRESENT:
    print("note: this model folder has no MTP head, so the server starts WITHOUT speculative decoding.")

if globals().get("PREFETCH_READERS"):
    _p = globals().get("PREFETCH_PROC")
    _log = open(PREFETCH_LOG).read() if os.path.exists(PREFETCH_LOG) else ""
    if (_p is None or (_p.poll() is not None and _p.returncode != 0)) and "PREFETCH DONE" not in _log:
        raise RuntimeError(f"the model prefetch of step 0 is not running and not done (exit code {None if _p is None else _p.returncode}):\n{_log[-1500:]}")
    os.environ["FLASHNEXT_WAIT_PREFETCH"] = PREFETCH_LOG
    os.environ["FLASHNEXT_WAIT_PREFETCH_PID"] = str(_p.pid if _p is not None and _p.poll() is None else 0)
    print(f"model prefetch: {(_log.strip().splitlines() or ['starting'])[-1]} | vLLM starts now and waits for it before loading the model")

def gpu_used_mib(ids=None):
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=index,memory.used", "--format=csv,noheader,nounits"],
                             capture_output=True, text=True, timeout=10).stdout.strip().splitlines()
        vals = {int(l.split(",")[0]): float(l.split(",")[1]) for l in out}
        return max(v for i, v in vals.items() if ids is None or i in ids)
    except Exception:
        return -1.0

def gpu_used(ids):
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=index,memory.used,memory.total", "--format=csv,noheader,nounits"],
                             capture_output=True, text=True, timeout=10).stdout.strip().splitlines()
        return " / ".join(f"{int(l.split(',')[1]) / 1024:.1f}G" for l in out if int(l.split(",")[0]) in ids)
    except Exception:
        return "?"

SERVERS = globals().get("SERVERS", [])

def stop_servers():
    for s in SERVERS:
        p = s.get("proc")
        if p is None:
            continue
        for sig, wait in ((signal.SIGTERM, 20), (signal.SIGKILL, 0)):
            try:
                os.killpg(p.pid, sig)
            except (ProcessLookupError, PermissionError):
                pass
            for _ in range(wait):
                if p.poll() is not None:
                    break
                time.sleep(1)
    for pattern in ("vllm.entrypoints", "VLLM::"):
        subprocess.run(["pkill", "-9", "-f", pattern], capture_output=True)
    for _ in range(90):
        used = gpu_used_mib()
        if 0 <= used < 1500:
            break
        time.sleep(1)
    else:
        print(f"WARNING: a GPU still shows memory in use after stopping vLLM ({gpu_used(range(N_GPUS))}). Run  !nvidia-smi  and  !pkill -9 python  if a retry fails.")

DROPPED = set()

def strip_flags(cmd):
    out, i = [], 0
    while i < len(cmd):
        if cmd[i] in DROPPED:
            i += 1
            if i < len(cmd) and not cmd[i].startswith("--"):
                i += 1
            continue
        out.append(cmd[i])
        i += 1
    return out

def build_cmd(port, max_len, util, prefix_cache=True, mamba_align=True, eager=False, mtp=True):
    cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server",
           "--model", str(MODEL_DIR), "--served-model-name", SERVED_NAME,
           "--host", "127.0.0.1", "--port", str(port),
           "--tensor-parallel-size", str(TP),
           "--max-model-len", str(max_len),
           "--gpu-memory-utilization", str(util),
           "--max-num-seqs", str(MAX_SEQS),
           "--max-num-batched-tokens", str(MAX_BATCHED),
           "--reasoning-parser", "qwen3",
           "--enable-auto-tool-choice", "--tool-call-parser", "qwen3_coder",
           "--default-chat-template-kwargs", '{"preserve_thinking": true}']
    if globals().get("KV_CACHE_BYTES"):
        cmd += ["--kv-cache-memory", str(KV_CACHE_BYTES)]
    if globals().get("CUDAGRAPH_CAPTURE_SIZES"):
        cmd += ["--cudagraph-capture-sizes"] + [str(x) for x in CUDAGRAPH_CAPTURE_SIZES]
    if MODEL_KIND != "flashnext":
        cmd += ["--generation-config", "vllm"]
    if DISABLE_CUSTOM_AR:
        cmd += ["--disable-custom-all-reduce"]
    if prefix_cache:
        cmd += ["--enable-prefix-caching"]
    if mamba_align:
        cmd += ["--mamba-cache-mode", "align"]
    if KV_DTYPE != "auto":
        cmd += ["--kv-cache-dtype", KV_DTYPE]
    if eager:
        cmd += ["--enforce-eager"]
    if mtp and MTP_AVAILABLE:
        cmd += ["--speculative-config", json.dumps({"method": "mtp", "num_speculative_tokens": MTP_TOKENS})]
    if MODEL_KIND == "flashnext":
        cmd += ["--engram-config", json.dumps({"cpu_offload": True})]
    if EXPERT_PARALLEL:
        cmd += ["--enable-expert-parallel"] + (["--enable-ep-weight-filter"] if globals().get("EP_WEIGHT_FILTER") else [])
    if not USE_IMAGES:
        cmd += ["--limit-mm-per-prompt", '{"image": 0, "video": 0}' if MODEL_KIND == "flashnext" else '{"image": 0}']
    return strip_flags(cmd)

def log_text(path):
    return Path(path).read_bytes()[-3_000_000:].decode(errors="replace").replace("\r", "\n")

LOG_PREFIX = re.compile(r"^(\(\w+ pid=\d+\)\s*)?((ERROR|WARNING|INFO|CRITICAL)\s+)?\d\d-\d\d \d\d:\d\d:\d\d\s*(\[[^\]]*\])?\s*")
EXC_LINE = re.compile(r"^([\w.]+\.)?\w*(Error|Exception|Exit)\b\s*[:(]"
                      r"|No available memory|Free memory on device|estimated maximum model length|[Oo]ut of memory|not supported|NotImplemented")

def root_cause(text, n=12):
    seen, out = set(), []
    for raw in text.splitlines():
        line = LOG_PREFIX.sub("", raw.strip(), count=1).strip()
        if not line or line.startswith(("File \"", "^", "~", "Traceback")):
            continue
        if EXC_LINE.search(line):
            key = re.sub(r"\d+", "#", line)[:110]
            if key not in seen:
                seen.add(key); out.append(line[:360])
    return out[:n]

LOAD_MARKERS = ("load_draft_model", "speculator", "qwen3_5_mtp", "create_weights", "initialize_model", "load_model")

def classify(causes, tail, full=""):
    text = "\n".join(causes) + "\n" + tail
    m = re.search(r"unrecognized arguments?:\s*(.+)", text)
    if m:
        return "badflags", [t for t in m.group(1).split() if t.startswith("--")]
    m = re.search(r"estimated maximum model length is (\d+)", text)
    if m:
        return "kv", int(m.group(1))
    low = text.lower()
    if "compute capability" in low and any(k in low for k in ("not supported", "requires sm", "requires compute", "only supported on")) \
            and any(k in low for k in ("t4", "capability 7.", "sm89", "sm 8")):
        return "hardware", None
    if "error building extension" in low or "ninja is required" in low or ("ninja" in low and "returned non-zero exit status" in low):
        return "build", None
    if "nccl" in low and any(k in low for k in ("error", "failed", "timeout", "unhandled")):
        return "nccl", None
    if "failed to be inspected" in low or "decorated class should have a forward" in low or "registry" in low and "non-zero exit" in low:
        return "mtp_import", None
    if "outofmemoryerror" in low or "cuda out of memory" in low:
        if any(k in full[-400000:] for k in LOAD_MARKERS):
            return "load_oom", None
        return "memory", None
    if any(k in low for k in ("no available memory for the cache blocks", "free memory on device", "insufficient memory")):
        return "memory", None
    if any(k in low for k in ("prefix caching", "mamba cache", "mamba_cache", "not supported", "not implemented")):
        return "flags", None
    return "other", None

def api_up(port):
    try:
        return urllib.request.urlopen(f"http://127.0.0.1:{port}/v1/models", timeout=3).status == 200
    except Exception:
        return False

_FATAL = re.compile(r"EngineCore failed to start|Engine core initialization failed|WorkerProc hit an exception|WorkerProc failed to start|"
                    r"Worker failed with error|INTERNAL ASSERT FAILED|OutOfMemoryError|CUDA out of memory|"
                    r"Watchdog caught collective operation timeout|died unexpectedly")

def start(rep, cmd, p2p_off):
    s = SERVERS[rep]
    print(shlex.join(cmd), "\n")
    env = vllm_env({"CUDA_VISIBLE_DEVICES": ",".join(map(str, s["gpus"])), "NCCL_DEBUG": "WARN"})
    if p2p_off:
        env["NCCL_P2P_DISABLE"] = "1"
    s["proc"] = subprocess.Popen(cmd, env=env, stdout=open(s["log"], "wb", buffering=0), stderr=subprocess.STDOUT, start_new_session=True)
    print(f"copy {rep}: pid {s['proc'].pid} on GPUs {s['gpus']} -> {s['log']}\n", flush=True)
    t0, last = time.time(), -30
    scan_pos, _carry = 0, ""
    while not api_up(s["port"]):
        if s["proc"].poll() is not None:
            return "exited"
        with open(s["log"], "rb") as _f:
            _f.seek(scan_pos); _new = _carry + _f.read().decode(errors="replace"); scan_pos = _f.tell()
        _carry = _new[-300:]
        if _FATAL.search(_new):
            time.sleep(20)
            return "engine_error"
        if time.time() - t0 > STARTUP_TIMEOUT_S:
            return "timeout"
        if time.time() - t0 - last >= 30:
            last = time.time() - t0
            tail = [l for l in log_text(s["log"]).splitlines() if l.strip()][-1:]
            print(f"[{last:4.0f}s] GPU {gpu_used(s['gpus'])} | {tail[0][:140] if tail else '-'}", flush=True)
        time.sleep(3)
    print(f"\ncopy {rep} READY after {time.time() - t0:.0f}s")
    return "ok"

print("stopping any old vLLM server ...")
stop_servers()
SERVERS = [dict(rep=k, gpus=list(range(k * TP, (k + 1) * TP)), port=BASE_PORT + k, log=f"/kaggle/working/vllm-server-{k}.log", proc=None) for k in range(REPLICAS)]

flag_variants = [dict(prefix_cache=True, mamba_align=True), dict(prefix_cache=True, mamba_align=False), dict(prefix_cache=False, mamba_align=False)]
S = dict(max_len=MAX_LEN, util=GPU_UTIL, vi=0, eager=True,  # E36: eager so the gEAN hook sees every MoE call
          mtp=True, p2p_off=False)

def start_copy(rep):
    for attempt in range(1, 9):
        fv = flag_variants[S["vi"]]
        print(f"\n--- copy {rep}, attempt {attempt}: window {S['max_len']}, {MAX_SEQS} sequences, TP {TP}, KV {KV_DTYPE}, util {S['util']}, images {USE_IMAGES}, "
              f"MTP {MTP_TOKENS if (S['mtp'] and MTP_AVAILABLE) else 'off'}, {'eager (no CUDA graphs)' if S['eager'] else 'CUDA graphs'}, flags {fv}, "
              f"NCCL p2p {'off' if S['p2p_off'] else 'default'} ---")
        state = start(rep, build_cmd(SERVERS[rep]["port"], S["max_len"], S["util"], **fv, eager=S["eager"], mtp=S["mtp"]), S["p2p_off"])
        if state == "ok":
            return
        text = log_text(SERVERS[rep]["log"])
        stop_servers()
        causes = root_cause(text)
        tail = "\n".join([l for l in text.splitlines() if l.strip()][-40:])
        print(f"\nthe server did not start ({state}). The error lines of the log:\n  " + "\n  ".join(causes or ["(none found: see the tail below)"]))
        if state == "engine_error":
            print("\nlast log lines:\n" + "\n".join(l[:300] for l in tail.splitlines()[-30:]))
            raise RuntimeError("vLLM's engine failed during start-up (error lines above). No retry with other settings. Send me this output.")
        kind, value = classify(causes, tail, text)
        print(f"-> classified as: {kind}")
        if kind == "build":
            lines = [l.strip()[:260] for l in text.replace("\r", "\n").splitlines()
                     if re.search(r"cannot find|undefined reference|fatal error|error:|No such file|ld returned", l)][:15]
            print("a kernel that compiles at its first use failed to build. Compiler / linker lines of the log:\n  " + "\n  ".join(lines or ["(none found: see /kaggle/working/vllm-server-0.log)"]))
            raise RuntimeError("A kernel build failed (see the lines above). Retrying with other flags cannot help. Send me those lines.")
        if kind == "hardware":
            raise RuntimeError("The GPUs cannot run this setup (see the error lines above: compute capability / SM89+ / FlashAttention-2). "
                               "Choose the accelerator 'GPU L4 x4' in the Session options, then run again from step 1.")
        if kind == "badflags" and value:
            DROPPED.update(value); print(f"this vLLM version does not know {', '.join(value)}: leaving them out and retrying"); continue
        if kind == "nccl" and not S["p2p_off"]:
            S["p2p_off"] = True; print("a communication (NCCL) problem between the GPUs: retrying with NCCL_P2P_DISABLE=1"); continue
        if kind == "kv" and value:
            new_len = int(value * 0.97) // 1024 * 1024
            if MIN_LEN <= new_len < S["max_len"]:
                print(f"the KV cache cannot hold a window of {S['max_len']}: retrying with {new_len}"); S["max_len"] = new_len; continue
            kind = "memory"
        if kind == "mtp_import" and S["mtp"] and MTP_AVAILABLE:
            S["mtp"] = False; print("the MTP model could not be loaded by vLLM: retrying without MTP"); continue
        if kind == "load_oom":
            if S["mtp"] and MTP_AVAILABLE:
                S["mtp"] = False; print("the GPU ran out of memory while LOADING: retrying without MTP"); continue
            print("the model itself does not fit in the GPU memory with these settings")
        elif kind == "memory":
            if S["util"] > 0.92:
                S["util"] = 0.92; print("not enough GPU memory at that share: retrying with 0.92"); continue
            if not S["eager"]:
                S["eager"] = True; print("not enough GPU memory: retrying without CUDA graphs"); continue
            if S["mtp"] and MTP_AVAILABLE:
                S["mtp"] = False; print("not enough GPU memory: retrying without MTP"); continue
        elif kind == "flags" and S["vi"] + 1 < len(flag_variants):
            S["vi"] += 1; print(f"a flag combination was refused: retrying with {flag_variants[S['vi']]}"); continue
        elif kind == "other":
            if S["mtp"] and MTP_AVAILABLE:
                S["mtp"] = False; print("unknown error: retrying without MTP (the MTP head is the most common cause)"); continue
            if not S["eager"]:
                S["eager"] = True; print("unknown error: retrying without CUDA graphs"); continue
        print("\nlast log lines:\n" + "\n".join(l[:300] for l in tail.splitlines()[-25:]))
        raise RuntimeError("vLLM failed to start. The error lines above are what I need: send me the whole output of this cell. "
                           "Things you can try: USE_IMAGES = False, KV_DTYPE = \"auto\", MAX_SEQS = 16, MAX_LEN = 24576, MTP_TOKENS = 0.")
    raise RuntimeError("vLLM failed to start after 8 attempts (see the output above).")

IMPORT_CHECK_PROC = None
if globals().get("IMPORT_CHECK"):
    _chk = "\n".join([STEP2_CHECK[0]] if globals().get("STEP2_CHECK") else []) + "\n" + IMPORT_CHECK[0]
    _low = (["nice", "-n", "19"] if shutil.which("nice") else []) + (["ionice", "-c", "3"] if shutil.which("ionice") else [])
    print("import checks in the background at", " ".join(_low) or "NORMAL priority (nice/ionice not found)")
    IMPORT_CHECK_PROC = subprocess.Popen(_low + [sys.executable, "-c", _chk], env=IMPORT_CHECK[1],
                                         stdout=open("/kaggle/working/import_check.log", "w"), stderr=subprocess.STDOUT)
for rep in range(REPLICAS):
    start_copy(rep)
if IMPORT_CHECK_PROC is not None:
    _rc = IMPORT_CHECK_PROC.wait()
    print("import check (background):\n" + open("/kaggle/working/import_check.log").read().strip()[-3000:])
    if _rc != 0:
        stop_servers()
        raise RuntimeError("the import check failed (lines above): the server was stopped")

WINDOW, N_SEQS = S["max_len"], MAX_SEQS
PORTS = [s["port"] for s in SERVERS]
USE_IMAGES_EFFECTIVE = USE_IMAGES
MTP_TOKENS_EFFECTIVE = MTP_TOKENS if (S["mtp"] and MTP_AVAILABLE) else 0
for s in SERVERS:
    info = json.load(urllib.request.urlopen(f"http://127.0.0.1:{s['port']}/v1/models", timeout=10))
    print(f"\ncopy {s['rep']}: port {s['port']} | max_model_len {info['data'][0].get('max_model_len')} | GPUs {s['gpus']} memory {gpu_used(s['gpus'])}")
print(f"\nWINDOW {WINDOW} | sequences per copy {N_SEQS} | copies {REPLICAS} x TP {TP} | images {USE_IMAGES_EFFECTIVE} | KV {KV_DTYPE} | MTP tokens {MTP_TOKENS_EFFECTIVE} | "
      f"{'eager' if S['eager'] else 'CUDA graphs'} | util {S['util']}")
if WINDOW != MAX_LEN:
    print(f"NOTE: the window is {WINDOW}, not {MAX_LEN} (the KV cache is the limit).")

KEYS = ("Model loading took", "KV cache", "Maximum concurrency", "kv_cache_dtype", "Marlin", "marlin", "mtp", "MTP", "speculative", "Speculative",
        "block size", "mamba", "prefix caching", "FlashInfer", "FlashAttention", "backend", "Humming", "humming")
seen = []
for l in log_text(SERVERS[0]["log"]).splitlines():
    if any(k in l for k in KEYS) and l not in seen:
        seen.append(l)
print("\nkey log lines of copy 0 (paste these back to me):")
print("\n".join(s_[:230] for s_ in seen[:34]))

In [ ]:
import os, re
_log = open(SERVERS[0]['log'], errors='replace').read()
_need = {'VLLM_HUMMING_MOE_GEMM_TYPE': 'Using indexed gemm for humming moe', 'FLASHNEXT_HC_DOWN': 'flashnext HC-down split-K: ON',
         'FLASHNEXT_DCP_QUERY_EARLY': 'flashnext DCP comm: query all-gather early ON',
         'FLASHNEXT_DCP_BF16_PARTIAL': 'flashnext DCP comm: BF16 partials ON'}
_need.update({f'FLASHNEXT_TRIM_{t}': f'flashnext decode trims: {t} ON' for t in ('K4_ROT', 'GAP_ONCE', 'TOPK_BF16', 'K4_FWHT', 'GDN_AB', 'HCD_SILU', 'MAMBA_TILES', 'QSA_GATE', 'QSA_SHIFT')})
_missing = [k for k, line in _need.items() if os.environ.get(k) and line not in _log]
if _missing:
    raise RuntimeError(f'speed switches set but not active in the server log: {_missing}')
_hcd = re.findall(r'flashnext HC-down split-K: ON \|[^\n]*', _log)
print('speed switches active:', [k for k in _need if os.environ.get(k)], '| top-k exchange:', os.environ.get('FLASHNEXT_TOPK_EXCHANGE'))
print(_hcd[0] if _hcd else 'HC-down off')
_kv = re.findall(r'GPU KV cache size: [^\n]*', _log)
print(_kv[0] if _kv else 'GPU KV cache size: line not found in the server log')

# Run the Tasks and Write the Submission

In [ ]:
import glob, json, os, random, re, shutil, subprocess, sys, time, urllib.request
from datetime import datetime

MAX_GEN_TOKENS = 200_000
TEXT_GRIDS = False
CONCURRENCY = 70
REASONING_EFFORT = "xhigh"
SEED = 0
TRIM_TARGET = 16384
TRIM_MARGIN = 1024
TRIM_NOTE = True
RECOVERY_MAX = 5
REPORT_EVERY_S = 60
STOP_MARGIN_MIN = 1
MAX_REVIVALS = 3
THIRD_TRY = False
TRY3_MIN_LEFT_MIN = 60
MIN_LEFT_FOR_REVIVAL_MIN = 30

assert REASONING_EFFORT in ("low", "medium", "xhigh"), f"REASONING_EFFORT must be low, medium or xhigh, not {REASONING_EFFORT!r}"
assert "DEADLINE" in globals() and "write_submission" in globals(), "Run the START cell first."
assert "PORTS" in globals() and "N_SEQS" in globals() and "ARC_PY" in globals(), "Run the earlier steps first."
assert len(PORTS) == 1, "the stage manager is written for one server copy (step 1: REPLICAS = 1)"
assert TEXT_GRIDS or USE_IMAGES_EFFECTIVE, "TEXT_GRIDS = False needs the images (the server of step 6 has images off)"
STOP_AT = DEADLINE - STOP_MARGIN_MIN * 60
fmt = lambda t: datetime.fromtimestamp(t).strftime("%H:%M")
served_len = json.load(urllib.request.urlopen(f"http://127.0.0.1:{PORTS[0]}/v1/models", timeout=10))["data"][0].get("max_model_len")
assert served_len == WINDOW, f"The server has max_model_len={served_len} but this run assumes {WINDOW}. Run step 6 again (the server cell)."
print("server:", PORTS[0], "| window", WINDOW, "| sequences", N_SEQS, "| runs at a time (both tries)", CONCURRENCY)

OUTS = [f"{WORK}/out_0"]
SUBS = [f"{WORK}/sub_0"]
shutil.rmtree(WORK, ignore_errors=True)
os.makedirs(WORK, exist_ok=True)
_LAST_GOOD.clear(); _LAST_REC.clear()

ids = sorted(TEST_TASKS)
random.Random(SEED).shuffle(ids)
open(f"{WORK}/.env", "w").write("VLLM_API_KEY=dummy\n")
json.dump({k: TEST_TASKS[k] for k in ids}, open(f"{WORK}/subset_0.json", "w"))
_sol_file = os.path.join(os.path.dirname(TEST_FILE), "arc-agi_evaluation_solutions.json")
SOLUTIONS = {}
if os.path.exists(_sol_file):
    SOLUTIONS = {k: v for k, v in json.load(open(_sol_file)).items() if k in TEST_TASKS}
print("tasks:", len(ids), "| solutions known for", len(SOLUTIONS), "(live CORRECT counts)" if SOLUTIONS else "(the hidden test: no CORRECT counts)")

MANAGER_SRC = r'''# STAGE MANAGER (session 16, owner's request): rounds 1, 2 and 3 of the run in ONE harness process, at the same time.
# Started by step 8 with the harness's python: `python stage_manager.py settings.json`. Writes status.json every few seconds (step 8 prints it)
# and runs.json at the end (step 8 merges the attempts from it).
#   round 1: every task, as it is.
#   round 2: a task whose round-1 run ended WITHOUT a final answer runs again in a new conversation, transposed ("T"; "P" = as it is can be
#            added). Session 17 (owner): also when round 1 had a check() passing every training example but gave no answer (s16: 6 of 10 such
#            functions were wrong or half right on the test). Note: round 1's best function and code (with the orientation it was written
#            for); if that function passes every training example, the note asks to check it on the test inputs and confirm or fix it,
#            otherwise to work breadth-first.
#   round 3: a task whose round-1 and round-2 runs all ended without a final answer runs once more, mirrored left-right ("F"), with the best
#            function and code of each earlier run. Each round builds on the ones before. (S["only_final_ends"] = False: s16's rule, where a
#            100% check() also ended a task.)
# SCHEDULING: a free slot takes a queued round-1 run first, else a queued round-2 run, else a queued round-3 run (owner, session 16).
# SECOND TRY (S["second_try"], session 17, owner, Kaggle; default off): no rounds; every task also gets an independent try 2 ("<id>_2": the
#   same grids, a new conversation, no note), queued from the start as a round-2 run. Round 1 first as above, so a try 2 starts only when
#   no try 1 is queued. Among the queued tries 2: tasks whose try 1 ended without a final answer first, then tasks whose try 1 still runs,
#   then tasks whose try 1 gave a final answer (in the order they were queued within each group).
# BUDGET: every run stops at the same token limit (ARC_MAX_GEN_TOKENS). All runs together get TOTAL_TOKEN_BUDGET decoded tokens. Every round-1
# task keeps its full limit reserved until it ends (round 1 always goes first); a round-2/3 run starts only when a free slot exists AND
# the budget minus (tokens of ended runs + max(used, limit) of running runs + the limit for every round-1 task not started yet) is at
# least one limit. A round-1 task that ends early gives its unused tokens back to rounds 2 and 3.
import asyncio, glob, json, logging, os, re, signal, sys, time
from pathlib import Path

S = json.load(open(sys.argv[1]))
LIMIT = int(S["limit"])
TOTAL = int(S["total_budget"])
CONC = int(S["concurrency"])
ROUND2 = list(S["round2_variants"])        # subset of ["P", "T"]
ROUND3 = list(S["round3_variants"])        # subset of ["F"]
assert set(ROUND2) <= {"P", "T"} and set(ROUND3) <= {"F"}, (ROUND2, ROUND3)
OUT = {1: S["out1"], 2: S["out2"], 3: S["out3"]}
SUB = {1: S["sub1"], 2: S["sub2"], 3: S["sub3"]}
NOTES = S["notes_file"]
STATUS = S["status_file"]
ROUND_OF = {"": 1, "P": 2, "T": 3, "F": 3, "2": 2, "3": 3}   # session 18b: "T" = the transposed try 3 | session 22: "3" = the plain try 3
ONLY_FINAL = bool(S.get("only_final_ends", False))   # session 17: only a final answer ends a task
SECOND_TRY = bool(S.get("second_try", False))       # session 17 (owner, Kaggle): an independent try 2 per task instead of rounds
assert not (SECOND_TRY and (ROUND2 or ROUND3)), "second_try replaces the rounds: round2_variants and round3_variants must be empty"
THIRD_TRY = bool(S.get("third_try", False))        # session 18b (owner): transposed try 3 for tasks whose try 1 gave no final answer
TRY3_MIN_LEFT_S = 60 * float(S.get("try3_min_left_min", 60))
assert not THIRD_TRY or SECOND_TRY, "third_try needs second_try"
# session 17 (Kaggle): per-round token limits written for the harness (ARC_RUN_LIMITS file; 0 = no limit, only the deadline), no token
# budget (total_budget 0: time decides), a pause file (while it exists no run starts: a vLLM server is being restarted), an absolute stop time.
LIMITS_FILE = S.get("limits_file")
ROUND_LIMIT = {int(r): int(v) for r, v in (S.get("round_limits") or {}).items()}
PAUSE_FILE = S.get("pause_file")
STOP_EPOCH = S.get("stop_epoch")
BUDGET_ON = TOTAL > 0
_STOP = []                                   # SIGTERM from the notebook: stop like at the time limit (runs cancelled, runs.json written)
signal.signal(signal.SIGTERM, lambda *_: _STOP.append(1))
assert not LIMITS_FILE or set(ROUND_LIMIT) == {1, 2, 3}, "limits_file needs round_limits for rounds 1, 2 and 3"

from dotenv import load_dotenv
from arcagi2.evaluation.evaluate import puzzle_worker
from arcagi2.solver.config import SOLVER_CONFIGS
from arcagi2.utils.logging_utils import setup_infra_logger_console_handler

tp = lambda g: [list(r) for r in zip(*g)]
fl = lambda g: [list(r)[::-1] for r in g]
TRANSFORM = {"": None, "P": None, "T": tp, "F": fl, "2": None, "3": None}     # each is its own inverse: the same function maps answers back

STEPS = ("This time work breadth-first:\n"
         "1. Look at the examples and write down 4-5 DIFFERENT candidate rules, a few lines each. Do not check any of them by hand.\n"
         "2. Write each rule as a function and run them all through check() in one python call.\n"
         "3. Continue from the rule with the best check() result, using its list of wrong cells. If none comes close, write new "
         "candidate rules that differ from the ones you tried.\n"
         "Never verify a rule cell by cell in your thinking: check() does that exactly and costs far fewer tokens.")
ORIENT = {"": "the original grids", "P": "the original grids", "T": "the original grids transposed (rows and columns swapped)",
          "F": "the original grids mirrored left-right (each row reversed)"}
ORDINAL = {2: "a second", 3: "a third"}
# HYBRID INFORMED TRY 2 (owner, session 18b): try 2 waits until its try 1 has ended. Try 1 gave a final answer -> NOTE_ALT (find a
# different rule); no final -> a fresh start without a note (INFORMED_NEW = True adds NOTE_NEW instead; RTX: no measurable effect).
INFORMED_TRY2 = False              # session 22 (owner): no more 'evolve style': try 2 is plain and queued from the start
INFORMED_NEW = False
NOTE_ALT = ("[Note from the harness] This is a second attempt at this puzzle. An earlier attempt found a rule whose function reproduces EVERY "
            "training example; its code is below. A rule can fit every training example and still be wrong on the test input(s): it may miss "
            "a case that only a test input shows. Find a DIFFERENT rule that also reproduces every training example and would give a different "
            "output on at least one test input, and give that as your final answer. Do not submit the earlier rule again.\n{earlier}")
NOTE_NEW = ("[Note from the harness] This is a second attempt at this puzzle. An earlier attempt used its whole budget without finding a rule "
            "that reproduces every training example. Do not follow the same route: find a different rule.\n{earlier}")
NOTE = ("[Note from the harness] This is {nth} attempt at this puzzle. {prev} without finding a rule that reproduces every training example. "
        "You see {orient}; {saw}\n{earlier}\n{ideas}\n" + STEPS)
NOTE_CONFIRM = ("[Note from the harness] This is {nth} attempt at this puzzle. {prev} You see {orient}; {saw}\n{earlier}\n"
                "A function can reproduce every training example and still be wrong on the test input(s). First check that its rule really "
                "explains every training example, then look at what it does on each test input: write it for the grids you see, run check() and "
                "look at the test outputs with show(). If it is right, give it as your final answer. If it misses something a test input needs, "
                "fix it, run check() again and give the fixed version as your final answer.")
PREV_CONFIRM = {2: ("An earlier attempt wrote a function that reproduces EVERY training example, but its budget of {k}K tokens ran out before it "
                    "gave a final answer.", "the earlier attempt may have seen the grids differently, so its code works on the grids IT saw."),
                3: ("The earlier attempts each used their whole budget of {k}K tokens without giving a final answer; at least one of them wrote a "
                    "function that reproduces EVERY training example.", "the earlier attempts saw the grids differently, so their code works on "
                    "the grids THEY saw.")}
PREV = {2: ("An earlier attempt used its whole budget of {k}K tokens", "the earlier attempt may have seen the grids differently, so its code works "
            "on the grids IT saw.", "Its idea is incomplete or wrong, so do not just repeat it."),
        3: ("The earlier attempts each used their whole budget of {k}K tokens", "the earlier attempts saw the grids differently, so their code "
            "works on the grids THEY saw.", "Their ideas are incomplete or wrong, so do not just repeat them.")}


def log(msg):
    logging.getLogger("arcagi2.evaluation.evaluate").info(f"STAGE {msg}")


def view(task, f):
    if f is None:
        return task
    return {"train": [{"input": f(e["input"]), "output": f(e["output"])} for e in task["train"]],
            "test": [{"input": f(e["input"])} for e in task["test"]]}


def _grid(g):
    return isinstance(g, list) and bool(g) and all(isinstance(r, list) and r and all(isinstance(v, int) and not isinstance(v, bool) for v in r) for r in g) \
        and len({len(r) for r in g}) == 1


class Run:
    def __init__(self, pid, var):
        self.pid, self.var, self.rid = pid, var, pid + ("_" + var if var else "")
        self.round = ROUND_OF[var]
        self.state = "queued"           # queued | running | ended | skipped
        self.task = None
        self.folder = None              # the sample folder (out/<uid>/<sample>/)
        self.used = 0
        self.final = False              # a final answer (attempt_1 grid in the round's submission)
        self.full = False               # a check() record passing every training example
        self.t0 = self.t1 = None

    def find_folder(self):
        if self.folder:
            return self.folder
        for m in glob.glob(f"{OUT[self.round]}/*/metadata.json"):
            try:
                if json.load(open(m))["puzzle_id"] == self.rid:
                    subs = [d for d in glob.glob(os.path.join(os.path.dirname(m), "*/")) if os.path.exists(d + "metadata.json")]
                    if subs:
                        self.folder = subs[0]
                        return self.folder
            except Exception:
                continue
        return None

    def read_used(self):
        f = self.find_folder()
        if f and os.path.exists(f + "run.log"):
            g = re.findall(r"V3_GEN (\d+)", open(f + "run.log", errors="replace").read())
            self.used = int(g[-1]) if g else 0
        return self.used

    def records(self):
        f = self.find_folder()
        try:
            return json.load(open(f + "check_attempts.json")).get("records") or [] if f else []
        except Exception:
            return []

    def finish(self):
        self.read_used()
        try:
            sub = json.load(open(f"{SUB[self.round]}/submission.json"))
        except Exception:
            sub = {}
        self.final = any(isinstance(e, dict) and _grid(e.get("attempt_1")) for e in (sub.get(self.rid) or []))
        self.full = any(r.get("n") and r.get("ok") == r.get("n") for r in self.records())

    @property
    def solved_train(self):
        """the task needs no next round: a final answer (session 17), or (s16 rule) also a check() passing every training example"""
        return self.final or (self.full and not ONLY_FINAL)


def best(run):
    rs = sorted(run.records(), key=lambda r: (r["ok"] / max(1, r["n"]), r["pct"]), reverse=True) if run else []
    return rs[0] if rs else None


def earlier_text(n, prev, var):
    """one line (+ code) per earlier run of the task: what it saw, its best check() result, its code"""
    r = best(prev)
    saw = ORIENT[prev.var] + (" (the same grids as you)" if ORIENT[prev.var] == ORIENT[var] else "")
    if r is None:
        return f"- Attempt {n} saw {saw}. It never wrote a function that check() could test: it spent its budget checking ideas by hand."
    s = f"- Attempt {n} saw {saw}. Its best function got {r['ok']}/{r['n']} training examples right ({r['pct']}% of the training cells)"
    if r.get("code"):
        return s + ". Its code (for the grids it saw):\n```python\n" + r["code"].rstrip() + "\n```"
    return s + "."


async def main():
    logging.getLogger().setLevel(logging.INFO)
    logging.getLogger("httpx").setLevel(logging.WARNING)
    setup_infra_logger_console_handler(logging.INFO)
    load_dotenv(S["env_file"])
    config = SOLVER_CONFIGS[S["config_name"]]
    config.set_vllm_base_url(S["base_url"])
    for r in (1, 2, 3):
        Path(OUT[r]).mkdir(parents=True, exist_ok=False)
        Path(SUB[r]).mkdir(parents=True, exist_ok=True)
        json.dump({}, open(f"{SUB[r]}/submission.json", "w"))
    fh = logging.FileHandler(f"{OUT[1]}/run.log")           # the main log (step 10 reads the sample errors from it)
    fh.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))
    lg = logging.getLogger("arcagi2.evaluation.evaluate")
    lg.addHandler(fh); lg.setLevel(logging.INFO)

    tasks = json.load(open(S["challenge_file"]))
    ids = list(tasks)
    notes = {pid: None for pid in ids}                      # round 1: explicitly no note
    def write_notes():
        tmp = NOTES + ".tmp"
        json.dump(notes, open(tmp, "w")); os.replace(tmp, NOTES)
    write_notes()
    limits = {}                                             # session 17: run id -> its decoded-token limit (written before the run starts)
    def write_limits():
        if LIMITS_FILE:
            tmp = LIMITS_FILE + ".tmp"
            json.dump(limits, open(tmp, "w")); os.replace(tmp, LIMITS_FILE)
    write_limits()

    runs = [Run(pid, "") for pid in ids] + ([Run(pid, "2") for pid in ids] if SECOND_TRY else [])
    by = {(r.pid, r.var): r for r in runs}
    if INFORMED_TRY2:
        for r in runs:                                      # informed try 2: waits until its try 1 has ended
            if r.var == "2":
                r.state = "waiting"
    lock = asyncio.Lock()
    sem = asyncio.Semaphore(10 ** 6)                        # concurrency is decided here, not by the semaphore
    k = ROUND_LIMIT.get(1, LIMIT) // 1000                   # the earlier rounds' budget, for the notes
    t_start = time.time()
    deadline = float(STOP_EPOCH) if STOP_EPOCH else t_start + float(S["time_limit_hours"]) * 3600
    events = []

    def event(msg):
        events.append(f"{(time.time() - t_start) / 60:5.1f} min  {msg}")
        log(msg)

    def start(run):
        if run.var == "2":                             # second try: explicitly no note (a missing note stops the harness)
            t1 = by[(run.pid, "")]
            notes[run.rid] = None
            if INFORMED_TRY2 and t1.state == "ended" and (t1.final or INFORMED_NEW):
                notes[run.rid] = (NOTE_ALT if t1.final else NOTE_NEW).format(earlier=earlier_text(1, t1, ""))
            write_notes()
        elif run.var == "3" and THIRD_TRY:           # try 3 (session 22): an independent PLAIN run, explicitly no note
            notes[run.rid] = None
            write_notes()
        elif run.round > 1:
            prev = [r for r in runs if r.pid == run.pid and r.round < run.round]
            prev.sort(key=lambda r: (r.round, r.var))
            earlier = "\n".join(earlier_text(n, r, run.var) for n, r in enumerate(prev, 1))
            if any(r.full for r in prev):              # session 17: an earlier function fits every training example -> confirm or fix it
                pv, saw = PREV_CONFIRM[min(3, len(prev) + 1)]
                notes[run.rid] = NOTE_CONFIRM.format(nth=ORDINAL[run.round], prev=pv.format(k=k), orient=ORIENT[run.var], saw=saw, earlier=earlier)
            else:
                pv, saw, ideas = PREV[min(3, len(prev) + 1)]
                notes[run.rid] = NOTE.format(nth=ORDINAL[run.round], prev=pv.format(k=k), orient=ORIENT[run.var], saw=saw, ideas=ideas,
                                             earlier=earlier)
            write_notes()
        if LIMITS_FILE:
            limits[run.rid] = ROUND_LIMIT[run.round]
            write_limits()
        run.state, run.t0 = "running", time.time()
        run.task = asyncio.create_task(puzzle_worker(semaphore=sem, lock=lock, config=config, puzzle_id=run.rid,
                                                     puzzle_json=view(tasks[run.pid], TRANSFORM[run.var]),
                                                     evaluation_output_folder=Path(OUT[run.round]),
                                                     submission_file=Path(f"{SUB[run.round]}/submission.json"), num_samples=1))
        event(f"start round {run.round} {run.rid}")

    def committed():
        c = 0
        for r in runs:
            if r.state == "ended":
                c += r.used
            elif r.state == "running":
                c += max(r.read_used(), LIMIT)
            elif r.state == "queued" and r.round == 1:
                c += LIMIT
        return c

    def status(done=False, why=""):
        st = {"time_min": round((time.time() - t_start) / 60, 1), "done": done, "why": why, "limit": LIMIT, "total_budget": TOTAL,
              "paused": bool(PAUSE_FILE) and os.path.exists(PAUSE_FILE), "stop_epoch": deadline,
              "concurrency": CONC, "rounds": {}, "events": events[-12:]}
        used = 0
        for rd in (1, 2, 3):
            rr = [r for r in runs if r.round == rd]
            for r in rr:
                if r.state == "running":
                    r.read_used()
            used += sum(r.used for r in rr)
            st["rounds"][rd] = {s: sum(r.state == s for r in rr) for s in ("waiting", "queued", "running", "ended", "skipped")}
            st["rounds"][rd].update(final=sum(r.final for r in rr if r.state == "ended"), full_check=sum(r.full and not r.final for r in rr if r.state == "ended"),
                                    tokens=sum(r.used for r in rr))
        st["tokens_used"] = used
        st["tokens_committed"] = committed()
        tmp = STATUS + ".tmp"
        json.dump(st, open(tmp, "w")); os.replace(tmp, STATUS)

    def follow_up(run):
        """after a run ends: queue the next round of its task when every run so far failed"""
        pid = run.pid
        if THIRD_TRY and run.round == 1 and not run.final and (pid, "3") not in by:
            nr = Run(pid, "3"); runs.append(nr); by[(pid, "3")] = nr
            event(f"try 1 {pid}: no final answer -> try 3 (plain) queued")
        if run.round == 1 and not run.solved_train and ROUND2:
            for v in ROUND2:
                nr = Run(pid, v); runs.append(nr); by[(pid, v)] = nr
            event(f"round 1 {pid}: no final answer" + (", a full check() to confirm" if run.full else ", no full check()") + f" -> round 2 queued ({'+'.join(ROUND2)})")
        elif run.round == 1 and not run.solved_train and not ROUND2 and ROUND3:
            for v in ROUND3:
                nr = Run(pid, v); runs.append(nr); by[(pid, v)] = nr
            event(f"round 1 {pid}: failed -> round 3 queued (no round 2)")
        elif run.round == 2 and ROUND3:
            r2 = [by[(pid, v)] for v in ROUND2]
            if all(r.state == "ended" for r in r2) and not any(r.solved_train for r in r2):
                for v in ROUND3:
                    nr = Run(pid, v); runs.append(nr); by[(pid, v)] = nr
                event(f"round 2 {pid}: no final answer" + (" (a full check() to confirm)" if any(r.full for r in r2) else "") + " -> round 3 queued")

    why = "all runs ended"
    try:
        while True:
            for r in runs:
                if r.state == "running" and r.task.done():
                    r.state, r.t1 = "ended", time.time()
                    r.finish()
                    event(f"end round {r.round} {r.rid}: {r.used // 1000}K tokens | final answer {'yes' if r.final else 'no'} | full check() {'yes' if r.full else 'no'}")
                    follow_up(r)
                    t2 = by.get((r.pid, "2"))
                    if r.var == "" and t2 is not None and t2.state == "waiting":      # informed try 2 may start now
                        t2.state = "queued"
                        event(f"try 2 of {r.pid} queued " + ("with the different-rule note" if r.final else
                                                             ("with the new-route note" if INFORMED_NEW else "fresh (no note)")))
            if time.time() > deadline or _STOP:
                why = "stopped by the notebook" if _STOP else "time limit"
                break
            paused = bool(PAUSE_FILE) and os.path.exists(PAUSE_FILE)
            free_slots = 0 if paused else CONC - sum(r.state == "running" for r in runs)
            for r in runs:                                   # round 1 first: its tokens are reserved
                if free_slots <= 0:
                    break
                if r.state == "queued" and r.round == 1:
                    start(r); free_slots -= 1
            def extra_order(r):                              # session 22 (owner): every try 2 before any try 3 (try 3 last)
                return r.round                               # 2 = try 2, 3 = try 3; queue order within each
            for r in sorted((r for r in runs if r.state == "queued" and r.round in (2, 3)), key=extra_order):
                if free_slots <= 0:
                    break
                if r.round == 3 and THIRD_TRY and deadline - time.time() < TRY3_MIN_LEFT_S:
                    continue                                 # try 3: not in the last minutes
                if r.state == "queued" and (not BUDGET_ON or TOTAL - committed() >= LIMIT):
                    start(r); free_slots -= 1
            running = [r for r in runs if r.state == "running"]
            queued = [r for r in runs if r.state == "queued"]
            if not running and not queued:
                break
            if not running and queued and not paused:        # nothing runs, nothing can start: the budget is used up
                for r in queued:
                    r.state = "skipped"
                event(f"budget used up: {len(queued)} queued runs skipped")
                why = "token budget used up"
                break
            status()
            await asyncio.sleep(float(S.get("poll_s", 5)))
    finally:
        live = [r for r in runs if r.state == "running"]
        for r in live:
            r.task.cancel()
        if live:
            await asyncio.gather(*[r.task for r in live], return_exceptions=True)
            for r in live:
                r.state, r.t1 = "ended", time.time()
                r.finish()
            event(f"stopped ({why}): {len(live)} running runs cancelled")
        for r in runs:
            if r.state in ("queued", "waiting"):
                r.state = "skipped"
        json.dump([{"pid": r.pid, "var": r.var, "rid": r.rid, "round": r.round, "state": r.state, "used": r.used, "final": r.final,
                    "full": r.full, "folder": r.folder, "minutes": round(((r.t1 or time.time()) - r.t0) / 60, 1) if r.t0 else None} for r in runs],
                  open(S["runs_file"], "w"), indent=1)
        status(done=True, why=why)
        fh.close()


asyncio.run(main())
'''
open(f"{WORK}/stage_manager.py", "w").write(MANAGER_SRC)
PAUSE_FILE = f"{WORK}/PAUSE"
SETTINGS = {"challenge_file": f"{WORK}/subset_0.json", "config_name": "qwen3_8_flash_next_free", "base_url": f"http://127.0.0.1:{PORTS[0]}/v1",
            "env_file": f"{WORK}/.env", "limit": MAX_GEN_TOKENS, "total_budget": 0, "concurrency": CONCURRENCY,
            "round2_variants": [], "round3_variants": [], "second_try": True, "third_try": THIRD_TRY, "try3_min_left_min": TRY3_MIN_LEFT_MIN, "time_limit_hours": 12, "stop_epoch": STOP_AT,
            "out1": OUTS[0], "sub1": SUBS[0], "out2": f"{WORK}/out_r2", "sub2": f"{WORK}/sub_r2", "out3": f"{WORK}/out_r3", "sub3": f"{WORK}/sub_r3",
            "notes_file": f"{WORK}/notes.json", "limits_file": f"{WORK}/limits.json", "pause_file": PAUSE_FILE,
            "round_limits": {"1": MAX_GEN_TOKENS, "2": MAX_GEN_TOKENS, "3": MAX_GEN_TOKENS},
            "status_file": f"{WORK}/status.json", "runs_file": f"{WORK}/runs.json", "poll_s": 5, "only_final_ends": True}
json.dump(SETTINGS, open(f"{WORK}/settings.json", "w"), indent=1)

def metric(port, name):
    try:
        text = urllib.request.urlopen(f"http://127.0.0.1:{port}/metrics", timeout=5).read().decode()
        return sum(float(m.group(1)) for m in re.finditer(rf"^vllm:{name}(?:\{{[^}}]*\}})?\s+([\d.eE+-]+)", text, re.M))
    except Exception:
        return None

t_start = time.time()
env = {**ARC_ENV, "ARC_CTX_WINDOW": str(WINDOW), "ARC_TRIM_TARGET": str(TRIM_TARGET), "ARC_TRIM_MARGIN": str(TRIM_MARGIN), "ARC_TRIM_NOTE": "1" if TRIM_NOTE else "0",
       "ARC_MAX_GEN_TOKENS": str(MAX_GEN_TOKENS), "ARC_IMAGES": "1" if USE_IMAGES_EFFECTIVE else "0",
       "ARC_COMPACT": "1", "ARC_RECOVERY_MAX": str(RECOVERY_MAX), "ARC_DEADLINE_EPOCH": str(STOP_AT), "ARC_SAMPLE_SECONDS": "0",
       "ARC_SALVAGE_AT_DEADLINE": "0", "ARC_REASONING_FIELD": "reasoning", "ARC_REASONING_EFFORT": REASONING_EFFORT,
       "ARC_CHECK_HELPER": "1", "ARC_GRID_HELPERS": "1", "ARC_CHECK_ATTEMPTS": "1", "ARC_THINK_CAP": "0", "ARC_BUDGET_REMINDERS": "0",
       "ARC_FINAL_REVIEW": "0", "ARC_RERUN_NOTES": SETTINGS["notes_file"], "ARC_RUN_LIMITS": SETTINGS["limits_file"],
       "ARC_TEXT_GRIDS": "1" if TEXT_GRIDS else "0"}
print(f"started {fmt(t_start)} | THREE PLAIN TRIES: try 1, try 2 (independent, no note), try 3 (plain) for tasks without a final in try 1 (stops starting {TRY3_MIN_LEFT_MIN} min before the end; queue: tries 1, then tries 2, then tries 3) | {MAX_GEN_TOKENS:,} decoded tokens per run | "
      f"text grids: {'ON' if TEXT_GRIDS else 'OFF (sizes + images + `puzzle`)'} | reasoning {REASONING_EFFORT} | stops at {fmt(STOP_AT)} (deadline {fmt(DEADLINE)})")

LOG7 = f"{WORK}/evaluate_0.log"
manager = None

def launch_manager():
    global manager
    manager = subprocess.Popen([ARC_PY, f"{WORK}/stage_manager.py", f"{WORK}/settings.json"], stdout=open(LOG7, "ab"), stderr=subprocess.STDOUT,
                               env=env, cwd=WORK)
    print(f"stage manager started (pid {manager.pid})", flush=True)

launch_manager()

def tail(path, n):
    try:
        return open(path, errors="replace").read().replace("\r", "\n").splitlines()[-n:]
    except Exception:
        return []

_TPb = lambda g: [list(r) for r in zip(*g)]
_FLb = lambda g: [list(r)[::-1] for r in g]

def correct_now(rd):
    if not SOLUTIONS:
        return None, set()
    path = f"{SUBS[0]}/submission.json" if rd == 1 else f"{SETTINGS['sub%d' % rd]}/submission.json"
    try:
        sub = json.load(open(path))
    except Exception:
        return 0, set()
    right, tasks = 0, set()
    for rid, e in sub.items():
        tid, var = (rid.rpartition("_")[0], rid.rpartition("_")[2]) if rd > 1 else (rid, "")
        f = {"T": _TPb, "F": _FLb}.get(var)
        sol = SOLUTIONS.get(tid)
        got = [x.get("attempt_1") if isinstance(x, dict) else None for x in e]
        if sol and len(got) >= len(sol) and all(g is not None and (f(g) if f else g) == s for g, s in zip(got, sol)):
            right += 1; tasks.add(tid)
    return right, tasks

_seen = set()

_LOAD = {"prev": None, "rows": []}
_LOAD_KEYS = ("generation_tokens_total", "prompt_tokens_total", "num_preemptions_total", "prefix_cache_hits_total", "prefix_cache_queries_total",
              "external_prefix_cache_hits_total", "kv_cache_usage_perc", "num_requests_running", "num_requests_waiting")
_NB = None

def _gpu_blocks():
    global _NB
    if _NB is None:
        m = re.findall(r"KVDIAG pool \| num_blocks (\d+)", open(SERVERS[0]["log"], errors="replace").read())
        _NB = int(m[-1]) if m else 0
    return _NB

def _census():
    try:
        for l in reversed(open(SERVERS[0]["log"], errors="replace").readlines()[-400:]):
            if "FLASHNEXT CENSUS free" in l:
                held = sum(int(h) for h in re.findall(r"g\d+:(\d+)/", l))
                if held > 0:
                    return int(re.search(r"CENSUS free (\d+)", l).group(1)), held
    except Exception:
        pass
    return None

def _load_line():
    now = time.time()
    v = {k: metric(PORTS[0], k) for k in _LOAD_KEYS}
    if any(x is None for k, x in v.items() if k != "external_prefix_cache_hits_total"):
        print("    LOAD: /metrics not readable"); return
    prev, _LOAD["prev"] = _LOAD["prev"], (now, v)
    if prev is None:
        return
    dt = now - prev[0]
    d = {k: (v[k] or 0) - (prev[1][k] or 0) for k in _LOAD_KEYS}
    hit = 100 * d["prefix_cache_hits_total"] / d["prefix_cache_queries_total"] if d["prefix_cache_queries_total"] else float("nan")
    ram = 100 * d["external_prefix_cache_hits_total"] / d["prefix_cache_queries_total"] if d["prefix_cache_queries_total"] and v["external_prefix_cache_hits_total"] is not None else float("nan")
    c, nb = _census(), _gpu_blocks()
    row = dict(t=round((now - t_start) / 60, 1), decode=d["generation_tokens_total"] / dt,
               computed=(d["prompt_tokens_total"] - d["prefix_cache_hits_total"]) / dt, hit=hit, ram=ram, preempt=d["num_preemptions_total"],
               usage=100 * v["kv_cache_usage_perc"], running=v["num_requests_running"], waiting=v["num_requests_waiting"],
               held=c[1] if c else None, not_held=(nb - c[0] - c[1]) if c and nb else None)
    _LOAD["rows"].append(row)
    print(f"    LOAD (last {dt:.0f} s): decode {row['decode']:.0f} tok/s | prompt computed {row['computed']:.0f} tok/s | prefix hit {hit:.0f}% "
          f"(RAM {ram:.0f}%) | preemptions {row['preempt']:.0f} | KV usage {row['usage']:.0f}% | running {row['running']:.0f} waiting {row['waiting']:.0f}"
          + (f" | GPU blocks: tasks {row['held']} / held by nothing running {row['not_held']} / pool {nb}" if c else ""), flush=True)
    try:
        _hd = [l for l in open(SERVERS[0]["log"], errors="replace").readlines()[-3000:] if "HITDIAG summary" in l]
        if _hd:
            print("    HITDIAG" + _hd[-1].split("HITDIAG summary")[-1].rstrip()[:420], flush=True)
    except Exception as _e:
        print("    HITDIAG line failed:", repr(_e)[:120])

def _load_summary():
    rows = [r for r in _LOAD["rows"] if r["t"] >= 10]
    if not rows:
        print("LOAD TEST: no rows after 10 min"); return
    mean = lambda k: sum(r[k] for r in rows if r[k] == r[k]) / max(1, sum(1 for r in rows if r[k] == r[k]))
    print(f"\nSPEED SUMMARY (CONCURRENCY {CONCURRENCY}, minutes 10-{rows[-1]['t']:.0f}): decode {mean('decode'):.0f} tok/s | prompt computed "
          f"{mean('computed'):.0f} tok/s | prefix hit {mean('hit'):.0f}% (RAM {mean('ram'):.0f}%) | preemptions {sum(r['preempt'] for r in rows):.0f} | "
          f"KV usage {mean('usage'):.0f}% | running {mean('running'):.0f}"
          + (f" | GPU blocks held by nothing running: mean {mean('not_held'):.0f}, max {max(r['not_held'] for r in rows if r['not_held'] is not None)}"
             if any(r["not_held"] is not None for r in rows) else ""))
    print("Kaggle load test with improvements 1-3 at CONCURRENCY 60 for comparison: decode 651 tok/s (minutes 10-81), prefix hit 94%, preemptions 0")
    json.dump(_LOAD["rows"], open("/kaggle/working/loadtest_rows.json", "w"), indent=1)

def report():
    try:
        st = json.load(open(SETTINGS["status_file"]))
    except Exception:
        print(f"[{(time.time() - t_start) / 60:5.1f} min] no status yet", flush=True); return None
    R = st["rounds"]
    w = lambda rd: R.get(str(rd), {})
    answered, _ = write_submission(quiet=True)
    print(f"[{st['time_min']:5.1f} min] WORKING: try 1 {w(1).get('running', 0)} | try 2 {w(2).get('running', 0)} | try 3 {w(3).get('running', 0)}"
          f"   (at most {CONCURRENCY}) | ANSWERED {answered} of {len(ids)} in submission.json | deadline in {(DEADLINE - time.time()) / 3600:.2f} h"
          + (" | PAUSED (server restart)" if st.get("paused") else "") + f" | restarts {revivals}", flush=True)
    for rd in (1, 2, 3):
        x = w(rd)
        ok, _ = correct_now(rd)
        print(f"    try {rd}: queued {x.get('queued', 0):3d} | running {x.get('running', 0):3d} | ended {x.get('ended', 0):3d} "
              f"(final answer {x.get('final', 0)}" + (f", CORRECT {ok}" if ok is not None else "") + f", full check() only {x.get('full_check', 0)}) | "
              f"skipped {x.get('skipped', 0)} | tokens {x.get('tokens', 0):,}")
    if SOLUTIONS:
        _t1, _t2, _t3 = (correct_now(r)[1] for r in (1, 2, 3))
        print(f"    tasks with a CORRECT final answer so far: {len(_t1 | _t2 | _t3)} (try 1 {len(_t1)}, try 2 {len(_t2)}, try 3 {len(_t3)}, "
              f"only try 2 {len(_t2 - _t1 - _t3)}, only try 3 {len(_t3 - _t1 - _t2)}); "
              f"attempts from check() are in submission.json too")
    g1 = metric(PORTS[0], "generation_tokens_total")
    if g1 is not None:
        print(f"    tokens decoded: {st.get('tokens_used', 0):,} (finished replies) | server: {g1:,.0f} since its start")
    try:
        _load_line()
    except Exception as _e:
        print("    LOAD line failed:", repr(_e)[:200])
    try:
        _mem = {l.split(":")[0]: int(l.split()[1]) / 1048576 for l in open("/proc/meminfo")}
        _gk = sum(1 for _ in open(MEM_GUARD_LOG)) if os.path.exists(MEM_GUARD_LOG) else 0
        print(f"    RAM: available {_mem['MemAvailable']:.0f} of {_mem['MemTotal']:.0f} GiB | guard stops {_gk}")
    except Exception as e:
        print("    RAM: could not read", e)
    for e in st.get("events", []):
        if e not in _seen and ("queued" in e or "skipped" in e or "stopped" in e):
            _seen.add(e); print("    >", e)
    errs = [l for l in tail(LOG7, 400) if "Traceback" in l or " ERROR " in l]
    if errs:
        print("    last error line:", errs[-1][:200])
    return st

SERVER_DOWN_LIMIT = 3
NO_PROGRESS_MIN = 10
_down = [0] * len(SERVERS)
_last_gen = [(None, time.time()) for _ in SERVERS]
def dead_servers():
    out = []
    for r, s in enumerate(SERVERS):
        exited = s.get("proc") is not None and s["proc"].poll() is not None
        g = metric(s["port"], "generation_tokens_total")
        _down[r] = 0 if g is not None else _down[r] + 1
        if g is not None and g != _last_gen[r][0]:
            _last_gen[r] = (g, time.time())
        stuck_min = (time.time() - _last_gen[r][1]) / 60
        if exited:
            out.append((r, f"exited (code {s['proc'].returncode})"))
        elif _down[r] >= SERVER_DOWN_LIMIT:
            out.append((r, f"has not answered /metrics for {_down[r]} reports"))
        elif g is not None and stuck_min >= NO_PROGRESS_MIN:
            out.append((r, f"answers but has generated no token for {stuck_min:.0f} min (hung)"))
    return out

def revive(dead):
    global revivals
    revivals += 1
    tag = f"restart{revivals}"
    open(PAUSE_FILE, "w").write(tag)
    for r, why in dead:
        lg = SERVERS[r]["log"]
        print(f"\nvLLM server {r} {why}. Error lines of {lg}:")
        bad = [l for l in tail(lg, 200000) if re.search(r"Traceback|Error|OutOfMemory|out of memory|Killed", l)]
        print("\n".join(l[:300] for l in bad[-15:]) or "\n".join(l[:300] for l in tail(lg, 15)))
    print(subprocess.run("free -g; dmesg 2>/dev/null | grep -iE 'killed process|out of memory' | tail -3", shell=True, capture_output=True, text=True).stdout)
    write_submission()
    for s in SERVERS:
        if os.path.exists(s["log"]):
            shutil.copy(s["log"], s["log"].replace(".log", f".before-{tag}.log"))
    stop_servers()
    print(f"RESTART {revivals} of {MAX_REVIVALS} (new runs paused): starting vLLM again with window {S['max_len']}, util {S['util']}, "
          f"{'eager' if S['eager'] else 'CUDA graphs'}, flags {flag_variants[S['vi']]}", flush=True)
    for rep in range(len(SERVERS)):
        state = start(rep, build_cmd(SERVERS[rep]["port"], S["max_len"], S["util"], **flag_variants[S["vi"]], eager=S["eager"], mtp=S["mtp"]), S["p2p_off"])
        if state != "ok":
            print(f"copy {rep} did not come back ({state}). Error lines:\n  " + "\n  ".join(root_cause(log_text(SERVERS[rep]["log"])) or ["(none)"]))
            return False
    for r in range(len(SERVERS)):
        _down[r], _last_gen[r] = 0, (None, time.time())
    os.remove(PAUSE_FILE)
    print("server back: new runs start again", flush=True)
    return True

revivals = 0
stop_reason = "all runs ended"
try:
    while manager.poll() is None:
        time.sleep(REPORT_EVERY_S)
        if DEADLINE_HIT.is_set() or time.time() >= STOP_AT:
            stop_reason = f"deadline ({fmt(STOP_AT)})"
            break
        report()
        dead = dead_servers()
        if dead:
            left_min = (STOP_AT - time.time()) / 60
            if revivals < MAX_REVIVALS and left_min >= MIN_LEFT_FOR_REVIVAL_MIN:
                try:
                    if revive(dead):
                        continue
                    stop_reason = "the vLLM server did not come back after a restart"
                except Exception as e:
                    stop_reason = f"the restart failed: {type(e).__name__}: {e}"
            else:
                stop_reason = (f"the vLLM server died ({dead[0][1]}) and " +
                               (f"{MAX_REVIVALS} restarts are used up" if revivals >= MAX_REVIVALS else f"only {left_min:.0f} min are left"))
            break
except KeyboardInterrupt:
    stop_reason = "interrupted by hand"

if manager.poll() is None and stop_reason.startswith("deadline"):
    try:
        manager.wait(120)
    except subprocess.TimeoutExpired:
        pass
if manager.poll() is None:
    manager.terminate()
    try:
        manager.wait(120)
    except subprocess.TimeoutExpired:
        manager.kill()
if DEADLINE_HIT.is_set():
    stop_reason = "deadline (the START cell's timer stopped everything)"
print(f"\nSTOPPED: {stop_reason} | {(time.time() - t_start) / 60:.0f} min | stage manager exit code {manager.returncode} | restarts {revivals}")
report()
if manager.returncode not in (0, None, -15):
    print("\n".join(l[:300] for l in tail(LOG7, 30)))
answered, bad = write_submission()
print(f"FINAL submission.json: {len(TEST_TASKS)} tasks, {answered} with an answer -> {SUBMISSION}")
try:
    _load_summary()
except Exception as _e:
    print("LOAD TEST SUMMARY failed:", repr(_e)[:200])
if SOLUTIONS:
    _score = 0.0
    for tid, sol in SOLUTIONS.items():
        e = json.load(open(SUBMISSION)).get(tid) or []
        _score += sum(i < len(e) and g in (e[i]["attempt_1"], e[i]["attempt_2"]) for i, g in enumerate(sol)) / len(sol)
    print(f"SCORE (trial, solutions known): {_score:.2f} / {len(SOLUTIONS)}")
for f in (LOG7, SETTINGS["runs_file"], SETTINGS["status_file"]):
    if os.path.exists(f):
        shutil.copy(f, "/kaggle/working")
stop_servers()
print("vLLM stopped; done at", fmt(time.time()))